# HaluRISC — Full Training Pipeline (Colab) — corrected grouped-split protocol

Runs the complete experiment protocol on a Colab GPU (T4 or better):

1. HaluEval download + prepare with a **GROUP-AWARE 70/15/15 split** (both answers of one question stay in the same partition; a leakage report is generated and asserted)
2. Full feature extraction (length, lexical, entity/NER, **NLI**, numeric, hedging, semantic) + NLI checkpoint provenance
3. XGBoost tuning (30 iters, 5-fold CV) + baselines + 3-seed protocol + early stopping
4. Platt vs isotonic calibration, ECE/Brier, McNemar, bootstrap CIs, Wilcoxon, 7-group ablations
5. SHAP global + local explanations
6. RAGTruth zero-shot external validation
7. Error analysis (10 FP + 10 FN, auto-tagged for manual review)
8. Latency/efficiency analysis
9. Optional LLM-as-judge comparison (needs OPENAI_API_KEY)
10. Artifact manifest generation (hashes, versions, hardware, split report)
11. Checkpoint all phases to **Google Drive** for crash-safe resume and download

**Before starting:** upload only this notebook. Cell 3 contains the complete runtime source tree, writes it to `/content/HaluRISC/`, and verifies SHA-256 hashes. No source zip is required.

**Runtime:** enable GPU (Runtime > Change runtime type). Use L4 for the full fresh run; cell 2 automatically selects batch 512 on L4/A100 and 256 on T4. Drive checkpoints make restarts resume without repeating completed phases. Keep only one Colab tab open.

**After the run:** download the Drive artifact package, unzip it at the repository root, run cell 7i again locally with `& .venv\Scripts\python.exe src\models\verify_artifacts.py`, then start the API if needed.


In [ ]:
# 1) Mount Google Drive (artifacts persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/HaluRISC'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted at', DRIVE_DIR)


In [ ]:
# 2) Environment check: GPU must be enabled + adaptive batch size
# L4/A100 -> batch 512 (22.5+ GB VRAM); T4 -> batch 256 (16 GB, stable).
# T4 is the recommended runtime: far fewer quota disconnects than L4.
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('!! No GPU detected - enable GPU in Runtime > Change runtime type')
    raise SystemExit(1)
GPU_NAME = torch.cuda.get_device_name(0)
BATCH_SIZE = 512 if any(k in GPU_NAME for k in ('L4', 'A100', 'V100', 'L40')) else 256
print(f'GPU: {GPU_NAME} -> feature batch size {BATCH_SIZE}')


In [ ]:
# 3) SELF-CONTAINED: write the HaluRISC source from this cell (NO zip upload)
# Regenerate with: python colab/build_self_contained.py
# To patch a single file later: edit its EMBEDDED entry below and rerun
# this cell, or paste a small cell that rewrites just that file.
import base64, hashlib, os

ROOT = '/content/HaluRISC'
os.makedirs(ROOT, exist_ok=True)

EMBEDDED = {
 "src/__init__.py": "IiIiCkhhbHVSSVNDOiBDYWxpYnJhdGVkIGFuZCBFeHBsYWluYWJsZSBIYWxsdWNpbmF0aW9uIFJpc2sgUHJlZGljdGlvbiBGcmFtZXdvcmsKIiIiCg==",
 "src/api/main.py": "...",
 "src/claims/decompose.py": "IiIiVDI6IGNsYWltLWxldmVsIGRlY29tcG9zaXRpb24gb2YgYW4gYW5zd2VyIGludG8gYXRvbWljIGNsYWltcy4KCkRldGVybWluaXN0aWMgc2VudGVuY2UvY2xhdXNlIHNwbGl0dGVyIChubyBMTE0gZGVwZW5kZW5jeSBpbiB0aGUgY29yZSBwYXRoKToKCiAgLSBzZW50ZW5jZXMgc3BsaXQgb24gWy4hP10gYm91bmRhcmllcwogIC0gZXZlcnkgc2VudGVuY2UgaXMgcmVmaW5lZCBpbnRvIGF0b21pYyBjbGF1c2VzIG9uIGNvbmp1bmN0aW9uIGJvdW5kYXJpZXMKICAgICgiLCBhbmQiLCAiLCBidXQiLCAiLCB3aGlsZSIsICIsIHdoZXJlYXMiLCAiLCB3aGljaCIpIGFuZCBvbiAsIDsg4oCUIDoKICAgIGJvdW5kYXJpZXMsIHRoZW4gbGVhZGluZyBjb25qdW5jdGlvbnMgYXJlIHN0cmlwcGVkIGZyb20gZWFjaCBjbGF1c2UKICAtIGNsYXVzZXMgc2hvcnRlciB0aGFuIDQwIGNoYXJhY3RlcnMgYXJlIG1lcmdlZCBiYWNrIGludG8gdGhlIHByZXZpb3VzCiAgICBjbGF1c2UsIHNvIGFwcG9zaXRpb25zIGFuZCBzaG9ydCBsaXN0cyBzdGF5IGludGFjdAogIC0gZW1wdHkvd2hpdGVzcGFjZSBjbGFpbXMgZHJvcHBlZCwgZHVwbGljYXRlcyByZW1vdmVkLCBjYXBwZWQgcGVyIGFuc3dlcgoKQXRvbWljIGNsYWltcyBtYXR0ZXIgZm9yIHZlcmlmaWNhdGlvbjogYSBjb21wb3VuZCBjbGFpbSAoIlgsIGFuZCBZIikgY2Fubm90CmJlIGVudGFpbGVkIGJ5IGEgc2luZ2xlIGV2aWRlbmNlIHNlbnRlbmNlLCBhbmQgdGhlIGNyb3NzLWVuY29kZXIgcmVhY3RzIHdpdGggYQpmYWxzZSBjb250cmFkaWN0aW9uLiBTcGxpdHRpbmcgY29uanVuY3Rpb25zIHJlbW92ZXMgdGhhdCBmYWlsdXJlIG1vZGUuCgpSdW4gKHJlcG8gcm9vdCk6CiAgcHl0aG9uIC1jICJmcm9tIHNyYy5jbGFpbXMuZGVjb21wb3NlIGltcG9ydCBzcGxpdF9jbGFpbXM7IHByaW50KHNwbGl0X2NsYWltcygnLi4uJykpIgoiIiIKCmltcG9ydCByZQoKU0VOVEVOQ0VfU1BMSVQgPSByZS5jb21waWxlKHIiKD88PVsuIT9dKVxzKyIpCkNMQVVTRV9TUExJVCA9IHJlLmNvbXBpbGUociIoPzosfDt84oCUfDopXHMrIikKQ09OSlVOQ1RJT05fU1BMSVQgPSByZS5jb21waWxlKHIiLFxzKyg/OmFuZHxidXR8d2hpbGV8d2hlcmVhc3x3aGljaClccysiLCByZS5JR05PUkVDQVNFKQpMRUFESU5HX0NPTkpVTkNUSU9OID0gcmUuY29tcGlsZShyIl4oPzphbmR8YnV0fHdoaWxlfHdoZXJlYXN8d2hpY2h8c28pXHMrIiwgcmUuSUdOT1JFQ0FTRSkKCk1JTl9DTEFVU0VfQ0hBUlMgPSA0MApNQVhfQ0xBSU1fQ0hBUlMgPSA1MDAKTUFYX0NMQUlNUyA9IDEyCgoKZGVmIHNwbGl0X3NlbnRlbmNlcyh0ZXh0OiBzdHIpIC0+IGxpc3Q6CiAgICAiIiJTcGxpdCBwcm9zZSBpbnRvIHNlbnRlbmNlcyAoa2VlcHMgdHJhaWxpbmcgcHVuY3R1YXRpb24gb24gdGhlIHNlbnRlbmNlKS4iIiIKICAgIHJldHVybiBbcy5zdHJpcCgpIGZvciBzIGluIFNFTlRFTkNFX1NQTElULnNwbGl0KHRleHQuc3RyaXAoKSkgaWYgcy5zdHJpcCgpXQoKCmRlZiBfY2xhdXNlc19vZihzZW50ZW5jZTogc3RyKSAtPiBsaXN0OgogICAgIiIiQXRvbWljIGNsYXVzZSByZWZpbmVtZW50OiBjb25qdW5jdGlvbiBzcGxpdCwgcHVuY3R1YXRpb24gY2xhdXNlcywgbWluLWxlbmd0aCBtZXJnZS4iIiIKICAgIHBhcnRzID0gQ09OSlVOQ1RJT05fU1BMSVQuc3BsaXQoc2VudGVuY2UpCiAgICByZWZpbmVkOiBsaXN0ID0gW10KICAgIGZvciBwYXJ0IGluIHBhcnRzOgogICAgICAgIHJlZmluZWQuZXh0ZW5kKENMQVVTRV9TUExJVC5zcGxpdChwYXJ0KSkKICAgIHBhcnRzID0gW0xFQURJTkdfQ09OSlVOQ1RJT04uc3ViKCIiLCBwLnN0cmlwKCkpLnN0cmlwKCkgZm9yIHAgaW4gcmVmaW5lZF0KICAgIHBhcnRzID0gW3AgZm9yIHAgaW4gcGFydHMgaWYgcF0KICAgICMgbWVyZ2UgZnJhZ21lbnRzIHRoYXQgYXJlIHRvbyBzaG9ydCB0byBzdGFuZCBhbG9uZSBhcyBjbGFpbXMKICAgIG1lcmdlZDogbGlzdCA9IFtdCiAgICBmb3IgcCBpbiBwYXJ0czoKICAgICAgICBpZiBtZXJnZWQgYW5kIGxlbihwKSA8IE1JTl9DTEFVU0VfQ0hBUlM6CiAgICAgICAgICAgIG1lcmdlZFstMV0gPSBmInttZXJnZWRbLTFdfSwge3B9IgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG1lcmdlZC5hcHBlbmQocCkKICAgIHJldHVybiBtZXJnZWQKCgpkZWYgc3BsaXRfY2xhaW1zKHRleHQ6IHN0ciwgbWF4X2NsYWltczogaW50ID0gTUFYX0NMQUlNUykgLT4gbGlzdDoKICAgICIiIlNwbGl0IGFuIGFuc3dlciBpbnRvIGF0b21pYyBjbGFpbXMgKGRldGVybWluaXN0aWMpLiIiIgogICAgb3V0OiBsaXN0ID0gW10KICAgIGZvciBzZW50ZW5jZSBpbiBzcGxpdF9zZW50ZW5jZXModGV4dCk6CiAgICAgICAgZm9yIGMgaW4gX2NsYXVzZXNfb2Yoc2VudGVuY2UpOgogICAgICAgICAgICBjbGFpbSA9IGNbOk1BWF9DTEFJTV9DSEFSU10uc3RyaXAoKQogICAgICAgICAgICBpZiBjbGFpbSBhbmQgY2xhaW0gbm90IGluIG91dDoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoY2xhaW0pCiAgICByZXR1cm4gb3V0WzptYXhfY2xhaW1zXQo=",
 "src/claims/judge.py": "IiIiVDQ6IExMTS1qdWRnZSByb3V0aW5nIGZvciB1bmNlcnRhaW4gY2xhaW1zIChoeWJyaWQgdmVyaWZpY2F0aW9uKS4KCkNvc3QtY29udHJvbGxlZDogb25seSBjbGFpbXMgd2hvc2UgTkxJIHNpZ25hbCBpcyBVTkNFUlRBSU4gYXJlIHNlbnQgdG8gdGhlCkxMTSBqdWRnZSB0b2dldGhlciB3aXRoIHRoZWlyIHJldHJpZXZlZCBldmlkZW5jZSBwYXNzYWdlczoKICAtIE5MSSBjb25maWRlbmNlIGJhbmQ6IG1heChlbnRhaWwsIGNvbnRyYSkgaW4gW0NPTkZfTE9XLCBDT05GX0hJR0gpCiAgLSBhYnN0YWluZWQgY2xhaW1zIHRoYXQgaGFkIGV2aWRlbmNlIHJldHJpZXZlZCAoZXZpZGVuY2UgZXhpc3RzIGJ1dCBOTEkKICAgIGdhdmUgbm8gdmVyZGljdCkKQ2FwOiBhdCBtb3N0IE1BWF9DTEFJTVNfUEVSX0FOU1dFUiBjbGFpbXMgcGVyIGFuc3dlci4KCkdyYWNlZnVsOiB3aXRoIG5vIE9QRU5BSV9BUElfS0VZIChvciBhbnkgZmFpbHVyZSkgY2xhaW1zIGtlZXAgdGhlaXIgTkxJCnZlcmRpY3QgYW5kIGFyZSB0YWdnZWQganVkZ2VkX2J5PSJubGkiLgoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBvcwoKQ09ORl9MT1cgPSBmbG9hdChvcy5lbnZpcm9uLmdldCgiSEFMVV9KVURHRV9DT05GX0xPVyIsICIwLjUwIikpCkNPTkZfSElHSCA9IGZsb2F0KG9zLmVudmlyb24uZ2V0KCJIQUxVX0pVREdFX0NPTkZfSElHSCIsICIwLjc1IikpCk1BWF9DTEFJTVNfUEVSX0FOU1dFUiA9IGludChvcy5lbnZpcm9uLmdldCgiSEFMVV9KVURHRV9NQVhfQ0xBSU1TIiwgIjMiKSkKCkpVREdFX1NZU1RFTSA9ICgKICAgICJZb3UgYXJlIGEgc3RyaWN0IGZhY3R1YWwtdmVyaWZpY2F0aW9uIGp1ZGdlLiBHaXZlbiBhIENMQUlNIGFuZCBFVklERU5DRSAiCiAgICAicGFzc2FnZXMsIGRlY2lkZSB3aGV0aGVyIHRoZSBjbGFpbSBpcyBzdXBwb3J0ZWQgYnkgdGhlIGV2aWRlbmNlLCAiCiAgICAiY29udHJhZGljdGVkIGJ5IGl0LCBvciB1bnN1cHBvcnRlZCAodGhlIGV2aWRlbmNlIG5laXRoZXIgc3VwcG9ydHMgbm9yICIKICAgICJjb250cmFkaWN0cyBpdCkuIFJlc3BvbmQgd2l0aCBKU09OIG9ubHk6ICIKICAgICd7InZlcmRpY3QiOiAic3VwcG9ydGVkInwiY29udHJhZGljdGVkInwidW5zdXBwb3J0ZWQiLCAiY29uZmlkZW5jZSI6IDAuMC0xLjAsICcKICAgICcicmVhc29uaW5nIjogIjxvbmUgc2hvcnQgc2VudGVuY2U+In0uJwopCgoKZGVmIF9jb25maWRlbmNlKGNsYWltOiBkaWN0KSAtPiBmbG9hdDoKICAgIHJldHVybiBmbG9hdChjbGFpbS5nZXQoImNvbmZpZGVuY2UiKSBvciAwLjApCgoKZGVmIHNob3VsZF9yb3V0ZShjbGFpbTogZGljdCwgZXZpZGVuY2VfcHJlc2VudDogYm9vbCkgLT4gYm9vbDoKICAgICIiIlJvdXRpbmcgcnVsZTogdW5jZXJ0YWluIE5MSSBiYW5kLCBvciBhYnN0YWluZWQtd2l0aC1ldmlkZW5jZS4iIiIKICAgIGlmIGNsYWltLmdldCgianVkZ2VkX2J5IikgPT0gImxsbSI6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBpZiBjbGFpbS5nZXQoImFic3RhaW5lZCIpOgogICAgICAgIHJldHVybiBldmlkZW5jZV9wcmVzZW50CiAgICBjb25mID0gX2NvbmZpZGVuY2UoY2xhaW0pCiAgICByZXR1cm4gQ09ORl9MT1cgPD0gY29uZiA8IENPTkZfSElHSAoKCmRlZiBqdWRnZV9jbGFpbXMoY2xhaW1zOiBsaXN0LCBwYXNzYWdlc19ieV9jbGFpbTogbGlzdCwganVkZ2VfY2FsbCwgbWF4X2NsYWltczogaW50ID0gTUFYX0NMQUlNU19QRVJfQU5TV0VSKSAtPiBsaXN0OgogICAgIiIiUm91dGUgdW5jZXJ0YWluIGNsYWltcyB0byB0aGUgTExNIGp1ZGdlOyByZXR1cm5zIHRoZSB1cGRhdGVkIGxpc3QuCgogICAganVkZ2VfY2FsbChjbGFpbV90ZXh0LCBldmlkZW5jZV90ZXh0cykgLT4gZGljdCB7dmVyZGljdCwgY29uZmlkZW5jZSwgcmVhc29uaW5nfQogICAgb3IgTm9uZSBvbiBmYWlsdXJlIChjbGFpbSBrZWVwcyBpdHMgTkxJIHZlcmRpY3QpLgogICAgIiIiCiAgICByb3V0ZWQgPSAwCiAgICBmb3IgaSwgY2xhaW0gaW4gZW51bWVyYXRlKGNsYWltcyk6CiAgICAgICAgY2xhaW0uc2V0ZGVmYXVsdCgianVkZ2VkX2J5IiwgIm5saSIpCiAgICAgICAgcHMgPSBwYXNzYWdlc19ieV9jbGFpbVtpXSBpZiBpIDwgbGVuKHBhc3NhZ2VzX2J5X2NsYWltKSBlbHNlIFtdCiAgICAgICAgZXZpZGVuY2VfcHJlc2VudCA9IGFueSgocC5nZXQoInRleHQiKSBvciAiIikuc3RyaXAoKSBmb3IgcCBpbiBwcykKICAgICAgICBpZiBub3Qgc2hvdWxkX3JvdXRlKGNsYWltLCBldmlkZW5jZV9wcmVzZW50KSBvciByb3V0ZWQgPj0gbWF4X2NsYWltczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJlc3VsdCA9IGp1ZGdlX2NhbGwoY2xhaW1bInRleHQiXSwgW3AuZ2V0KCJ0ZXh0IiwgIiIpIGZvciBwIGluIHBzIGlmIHAuZ2V0KCJ0ZXh0IildKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHJlc3VsdCA9IE5vbmUKICAgICAgICBpZiByZXN1bHQgYW5kIHJlc3VsdC5nZXQoInZlcmRpY3QiKSBpbiAoInN1cHBvcnRlZCIsICJjb250cmFkaWN0ZWQiLCAidW5zdXBwb3J0ZWQiKToKICAgICAgICAgICAgY2xhaW1bInZlcmRpY3QiXSA9IHJlc3VsdFsidmVyZGljdCJdCiAgICAgICAgICAgIGNsYWltWyJjb25maWRlbmNlIl0gPSByb3VuZChmbG9hdChyZXN1bHQuZ2V0KCJjb25maWRlbmNlIikgb3IgX2NvbmZpZGVuY2UoY2xhaW0pKSwgNCkKICAgICAgICAgICAgY2xhaW1bImp1ZGdlZF9ieSJdID0gImxsbSIKICAgICAgICAgICAgY2xhaW1bImp1ZGdlX3JlYXNvbmluZyJdID0gc3RyKHJlc3VsdC5nZXQoInJlYXNvbmluZyIpIG9yICIiKVs6MzAwXQogICAgICAgICAgICByb3V0ZWQgKz0gMQogICAgcmV0dXJuIGNsYWltcwoKCmRlZiBvcGVuYWlfanVkZ2VfY2FsbChhcGlfa2V5OiBzdHIsIG1vZGVsOiBzdHIgfCBOb25lID0gTm9uZSkgLT4gY2FsbGFibGU6CiAgICAiIiJSZXR1cm5zIGEganVkZ2VfY2FsbCBib3VuZCB0byB0aGUgT3BlbkFJIGNsaWVudCAobGF6eSBpbXBvcnQpLiIiIgogICAgZnJvbSBvcGVuYWkgaW1wb3J0IE9wZW5BSQoKICAgIGNsaWVudCA9IE9wZW5BSShhcGlfa2V5PWFwaV9rZXkpCiAgICBtb2RlbF9uYW1lID0gbW9kZWwgb3Igb3MuZW52aXJvbi5nZXQoIk9QRU5BSV9NT0RFTCIsICJncHQtNS42LWx1bmEiKQoKICAgIGRlZiBfY2FsbChjbGFpbV90ZXh0OiBzdHIsIGV2aWRlbmNlX3RleHRzOiBsaXN0KSAtPiBkaWN0IHwgTm9uZToKICAgICAgICBldmlkZW5jZSA9ICJcblxuIi5qb2luKGYiW3tpICsgMX1dIHt0fSIgZm9yIGksIHQgaW4gZW51bWVyYXRlKGV2aWRlbmNlX3RleHRzWzo1XSkpIG9yICIobm8gZXZpZGVuY2UpIgogICAgICAgIHVzZXIgPSBmIkNMQUlNOiB7Y2xhaW1fdGV4dH1cblxuRVZJREVOQ0U6XG57ZXZpZGVuY2V9IgogICAgICAgIHJlc3AgPSBjbGllbnQuY2hhdC5jb21wbGV0aW9ucy5jcmVhdGUoCiAgICAgICAgICAgIG1vZGVsPW1vZGVsX25hbWUsCiAgICAgICAgICAgIG1lc3NhZ2VzPVt7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBKVURHRV9TWVNURU19LCB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogdXNlcn1dLAogICAgICAgICAgICB0ZW1wZXJhdHVyZT0wLAogICAgICAgICAgICBtYXhfdG9rZW5zPTE2MCwKICAgICAgICApCiAgICAgICAgY29udGVudCA9IHJlc3AuY2hvaWNlc1swXS5tZXNzYWdlLmNvbnRlbnQuc3RyaXAoKQogICAgICAgIGRhdGEgPSBqc29uLmxvYWRzKGNvbnRlbnRbY29udGVudC5maW5kKCJ7IikgOiBjb250ZW50LnJmaW5kKCJ9IikgKyAxXSkKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAidmVyZGljdCI6IHN0cihkYXRhLmdldCgidmVyZGljdCIsICIiKSkubG93ZXIoKSwKICAgICAgICAgICAgImNvbmZpZGVuY2UiOiBmbG9hdChkYXRhLmdldCgiY29uZmlkZW5jZSIsIDAuMCkpLAogICAgICAgICAgICAicmVhc29uaW5nIjogc3RyKGRhdGEuZ2V0KCJyZWFzb25pbmciLCAiIikpLAogICAgICAgIH0KCiAgICByZXR1cm4gX2NhbGwK",
 "src/claims/verify.py": "IiIiVDI6IHBlci1jbGFpbSBOTEkgdmVyaWZpY2F0aW9uIGFnYWluc3QgZXZpZGVuY2UuCgpGb3IgZXZlcnkgY2xhaW0gdGhlIE5MSSBjcm9zcy1lbmNvZGVyIHNjb3JlcyAoY2xhaW0sIGV2aWRlbmNlX3NlbnRlbmNlKSBwYWlycwpmb3IgZXZlcnkgZXZpZGVuY2Ugc2VudGVuY2UuIFZlcmRpY3RzIGFyZSBkZWNpZGVkIHBlciBwYWlyIGFuZCB0aGVuIGFnZ3JlZ2F0ZWQKd2l0aCBzdXBwb3J0IHByaW9yaXR5OgoKICBwZXIgcGFpciAgICAgZW50YWlsbWVudCA+PSBFTlRBSUxfVEhSRVNIT0xEIGFuZCBlbnRhaWxtZW50ID49IGNvbnRyYWRpY3Rpb24KICAgICAgICAgICAgICAgLT4gc3VwcG9ydCBjYW5kaWRhdGUKICAgICAgICAgICAgICAgY29udHJhZGljdGlvbiA+PSBDT05UUkFfVEhSRVNIT0xEIGFuZCBjb250cmFkaWN0aW9uID4gZW50YWlsbWVudAogICAgICAgICAgICAgICAtPiBjb250cmFkaWN0aW9uIGNhbmRpZGF0ZQogIHBlciBjbGFpbSAgICBhbnkgc3VwcG9ydCBjYW5kaWRhdGUgICAgICAgLT4gc3VwcG9ydGVkCiAgICAgICAgICAgICAgIGVsc2UgYW55IGNvbnRyYWRpY3Rpb24gICAgICAtPiBjb250cmFkaWN0ZWQKICAgICAgICAgICAgICAgZWxzZSAgICAgICAgICAgICAgICAgICAgICAgIC0+IHVuc3VwcG9ydGVkCgpUaGUgc3VwcG9ydC1maXJzdCBhZ2dyZWdhdGlvbiBtYXR0ZXJzIGJlY2F1c2UgYSBjb21wb3VuZCBvciBtaXNtYXRjaGVkIGNsYWltCmNhbiBkcmF3IGEgaGlnaCBjb250cmFkaWN0aW9uIHNjb3JlIGZyb20gYW4gdW5yZWxhdGVkIHNlbnRlbmNlIHdoaWxlIGFub3RoZXIKc2VudGVuY2UgZW50YWlscyBpdCB3aXRoIGhpZ2ggY29uZmlkZW5jZS4gQ29tcGFyaW5nIHRoZSBtYXhpbWEgYWNyb3NzCmRpZmZlcmVudCBzZW50ZW5jZXMgdXNlZCB0byB0dXJuIHN1Y2ggY2xhaW1zIGludG8gZmFsc2UgY29udHJhZGljdGlvbnMuCgpBIHJlbGV2YW5jZSBnYXRlIGFkZGl0aW9uYWxseSByZXF1aXJlcyB0aGUgc2VudGVuY2UgdG8gY292ZXIgcGFydCBvZiB0aGUgY2xhaW0Kdm9jYWJ1bGFyeSwgb3IgdG8gYmUgYSB2ZXJ5IGhpZ2gtY29uZmlkZW5jZSBtYXRjaCB0aGF0IHNoYXJlcyBhIGNvbnRlbnQgd29yZC4KT2ZmLXRvcGljIHJldHJpZXZlZCBzbmlwcGV0cyAoZm9yIGV4YW1wbGUgYW4gdW5yZWxhdGVkIHNwb3J0cyBhcnRpY2xlKSBjYW4KdGhlcmVmb3JlIG5vIGxvbmdlciBkZWNpZGUgYSB2ZXJkaWN0OyBzdWNoIGNsYWltcyBmYWxsIGJhY2sgdG8gdW5zdXBwb3J0ZWQuCgpFdmlkZW5jZSBzZW50ZW5jZSA9IHRoZSBvbmUgdGhhdCBkcm92ZSB0aGUgdmVyZGljdCAoc3VyZmFjZWQgaW4gdGhlIFVJKS4KVmVyZGljdCB0aHJlc2hvbGRzIGFyZSBkb2N1bWVudGVkIGNvbnN0YW50cyAocm9hZG1hcCBCNy41IFQyKSBhbmQgYXJlCmV2YWx1YXRlZCBhZ2FpbnN0IGh1bWFuIGxhYmVscyBieSBzcmMvY2xhaW1zL2V2YWxfY2xhaW1zLnB5IOKAlCB0aGV5IGFyZQpkZWNpc2lvbiBydWxlcywgbm90IHBhc3MvZmFpbCBxdWFsaXR5IGdhdGVzLgoKVGhlIE5MSSBtb2RlbCBpcyB0aGUgc2VudGVuY2UtdHJhbnNmb3JtZXJzIENyb3NzRW5jb2RlciBhbHJlYWR5IGxvYWRlZCBieSB0aGUKQVBJIChwcmVkaWN0KHBhaXJzLCBiYXRjaF9zaXplPS4uLiwgYXBwbHlfc29mdG1heD1UcnVlKSAtPiBwcm9icyBpbiBvcmRlcgpbY29udHJhZGljdGlvbiwgZW50YWlsbWVudCwgbmV1dHJhbF0pLgoiIiIKCmltcG9ydCBqc29uCmltcG9ydCByZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBzcmMuY2xhaW1zLmRlY29tcG9zZSBpbXBvcnQgc3BsaXRfc2VudGVuY2VzCgpMQUJFTFMgPSBbImNvbnRyYWRpY3Rpb24iLCAiZW50YWlsbWVudCIsICJuZXV0cmFsIl0KCkVOVEFJTF9USFJFU0hPTEQgPSAwLjUKQ09OVFJBX1RIUkVTSE9MRCA9IDAuNQpERUZBVUxUX0JBVENIX1NJWkUgPSA2NAoKIyBFdmlkZW5jZSByZWxldmFuY2UgZ2F0ZTogYSByZXRyaWV2ZWQgc2VudGVuY2UgY291bnRzIGFzIGV2aWRlbmNlIG9ubHkgd2hlbiBpdAojIGNvdmVycyBlbm91Z2ggb2YgdGhlIGNsYWltIHZvY2FidWxhcnksIG9yIHdoZW4gdGhlIE5MSSBjb25maWRlbmNlIGlzIHZlcnkKIyBoaWdoIGFuZCBhdCBsZWFzdCBvbmUgY29udGVudCB3b3JkIGlzIHNoYXJlZC4gVGhpcyBrZWVwcyBvZmYtdG9waWMgc25pcHBldHMKIyAoZS5nLiBhbiBORkwgYXJ0aWNsZSBmb3IgYSBmb290YmFsbCBjbGFpbSkgZnJvbSBkZWNpZGluZyBhIHZlcmRpY3QuCk1JTl9FVklERU5DRV9DT1ZFUkFHRSA9IDAuMTUKSElHSF9FTlRBSUxNRU5UID0gMC45MApISUdIX0NPTlRSQURJQ1RJT04gPSAwLjk3CgpfVE9LRU5fUkUgPSByZS5jb21waWxlKHIiW2EtejAtOV0rIikKU1RPUFdPUkRTID0gewogICAgImEiLCAiYWJvdXQiLCAiYWZ0ZXIiLCAiYWdhaW4iLCAiYWxsIiwgImFsc28iLCAiYW4iLCAiYW5kIiwgImFueSIsICJhcmUiLCAiYXMiLCAiYXQiLAogICAgImJlIiwgImJlY2F1c2UiLCAiYmVlbiIsICJiZWZvcmUiLCAiYmVpbmciLCAiYmV0d2VlbiIsICJib3RoIiwgImJ1dCIsICJieSIsICJjYW4iLAogICAgImNvdWxkIiwgImRpZCIsICJkbyIsICJkb2VzIiwgImRvaW5nIiwgImRvd24iLCAiZHVyaW5nIiwgImVhY2giLCAiZmV3IiwgImZvciIsICJmcm9tIiwKICAgICJmdXJ0aGVyIiwgImhhZCIsICJoYXMiLCAiaGF2ZSIsICJoYXZpbmciLCAiaGUiLCAiaGVyIiwgImhlcmUiLCAiaGVycyIsICJoaW0iLCAiaGlzIiwKICAgICJob3ciLCAiaSIsICJpZiIsICJpbiIsICJpbnRvIiwgImlzIiwgIml0IiwgIml0cyIsICJqdXN0IiwgIm1lIiwgIm1vcmUiLCAibW9zdCIsICJteSIsCiAgICAibm8iLCAibm9yIiwgIm5vdCIsICJub3ciLCAib2YiLCAib2ZmIiwgIm9uIiwgIm9uY2UiLCAib25seSIsICJvciIsICJvdGhlciIsICJvdXIiLAogICAgIm91dCIsICJvdmVyIiwgIm93biIsICJzYW1lIiwgInNoZSIsICJzaG91bGQiLCAic28iLCAic29tZSIsICJzdWNoIiwgInRoYW4iLCAidGhhdCIsCiAgICAidGhlIiwgInRoZWlyIiwgInRoZW0iLCAidGhlbiIsICJ0aGVyZSIsICJ0aGVzZSIsICJ0aGV5IiwgInRoaXMiLCAidGhvc2UiLCAidGhyb3VnaCIsCiAgICAidG8iLCAidG9vIiwgInVuZGVyIiwgInVudGlsIiwgInVwIiwgInZlcnkiLCAid2FzIiwgIndlIiwgIndlcmUiLCAid2hhdCIsICJ3aGVuIiwKICAgICJ3aGVyZSIsICJ3aGljaCIsICJ3aGlsZSIsICJ3aG8iLCAid2hvbSIsICJ3aHkiLCAid2lsbCIsICJ3aXRoIiwgIndvdWxkIiwgInlvdSIsICJ5b3VyIiwKfQoKCmRlZiBfdG9rZW5zKHRleHQ6IHN0cikgLT4gc2V0OgogICAgcmV0dXJuIHNldChfVE9LRU5fUkUuZmluZGFsbCh0ZXh0Lmxvd2VyKCkpKQoKCmRlZiBldmlkZW5jZV9nYXRlKGNsYWltOiBzdHIsIHNlbnRlbmNlOiBzdHIpIC0+IHR1cGxlOgogICAgIiIiKGNvdmVyYWdlLCBzaGFyZWRfY29udGVudCkgb2Ygb25lIChjbGFpbSwgc2VudGVuY2UpIHBhaXIuCgogICAgY292ZXJhZ2UgPSBzaGFyZSBvZiB0aGUgY2xhaW0ncyB0b2tlbnMgdGhhdCBhcHBlYXIgaW4gdGhlIHNlbnRlbmNlLgogICAgc2hhcmVkX2NvbnRlbnQgPSB3aGV0aGVyIHRoZSB0d28gc2hhcmUgYXQgbGVhc3Qgb25lIG5vbi1zdG9wd29yZCB0b2tlbi4KICAgICIiIgogICAgY2xhaW1fdG9rZW5zID0gX3Rva2VucyhjbGFpbSkKICAgIHNoYXJlZCA9IGNsYWltX3Rva2VucyAmIF90b2tlbnMoc2VudGVuY2UpCiAgICBjb3ZlcmFnZSA9IGxlbihzaGFyZWQpIC8gbWF4KGxlbihjbGFpbV90b2tlbnMpLCAxKQogICAgcmV0dXJuIGNvdmVyYWdlLCBib29sKHNoYXJlZCAtIFNUT1BXT1JEUykKClRIUkVTSE9MRFNfRklMRSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdIC8gImRhdGEiIC8gInByb2Nlc3NlZCIgLyAidmVyZGljdF90aHJlc2hvbGRzLmpzb24iCgoKZGVmIF9lZmZlY3RpdmVfdGhyZXNob2xkcygpIC0+IHR1cGxlOgogICAgIiIiVGhyZXNob2xkcyBtYXkgYmUgdHVuZWQgZnJvbSBmZWVkYmFjayAoVDQuMiB0dW5lX3RocmVzaG9sZHMucHkgLS1hcHBseSkuIiIiCiAgICB0cnk6CiAgICAgICAgZCA9IGpzb24ubG9hZHMoVEhSRVNIT0xEU19GSUxFLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICByZXR1cm4gZmxvYXQoZFsiZW50YWlsIl0pLCBmbG9hdChkWyJjb250cmEiXSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEVOVEFJTF9USFJFU0hPTEQsIENPTlRSQV9USFJFU0hPTEQKCgpkZWYgc2NvcmVfYmxvY2soZW50YWlsOiBucC5uZGFycmF5LCBjb250cmE6IG5wLm5kYXJyYXksCiAgICAgICAgICAgICAgICBjb3ZlcmFnZTogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICAgc2hhcmVkX2NvbnRlbnQ6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZSkgLT4gdHVwbGU6CiAgICAiIiJPbmUgY2xhaW0ncyB2ZXJkaWN0IGZyb20gaXRzIHBlci1zZW50ZW5jZSBlbnRhaWxtZW50L2NvbnRyYWRpY3Rpb24gYXJyYXlzLgoKICAgIFN1cHBvcnQgcHJpb3JpdHk6IGEgY2xhaW0gaXMgZ3JvdW5kZWQgd2hlbiBhbnkgZXZpZGVuY2Ugc2VudGVuY2UgZW50YWlscwogICAgaXQgKGVudGFpbG1lbnQgPj0gdGhyZXNob2xkIGFuZCBlbnRhaWxtZW50ID49IHRoYXQgcGFpcidzIGNvbnRyYWRpY3Rpb24pLgogICAgQSBjb250cmFkaWN0aW9uIG9ubHkgZGVjaWRlcyB3aGVuIG5vIHNlbnRlbmNlIHN1cHBvcnRzIHRoZSBjbGFpbS4KICAgIFRoZSByZWxldmFuY2UgZ2F0ZSAoY292ZXJhZ2Uvc2hhcmVkX2NvbnRlbnQpIGZpbHRlcnMgb2ZmLXRvcGljIHNlbnRlbmNlczsKICAgIHdoZW4gdGhlIGFycmF5cyBhcmUgTm9uZSB0aGUgZ2F0ZSBpcyBkaXNhYmxlZCAoY2FsbGVycyB3aXRob3V0IHRleHRzKS4KICAgIFJldHVybnMgKHZlcmRpY3QsIGNvbmZpZGVuY2UsIGJlc3Rfc2VudGVuY2VfaW5kZXgpLgogICAgIiIiCiAgICBlbnRfdGhyLCBjb25fdGhyID0gX2VmZmVjdGl2ZV90aHJlc2hvbGRzKCkKICAgIGlmIGNvdmVyYWdlIGlzIE5vbmUgb3Igc2hhcmVkX2NvbnRlbnQgaXMgTm9uZToKICAgICAgICByZWxldmFudCA9IG5wLm9uZXMobGVuKGVudGFpbCksIGR0eXBlPWJvb2wpCiAgICAgICAgc3Ryb25nX2VudCwgc3Ryb25nX2NvbiA9IHJlbGV2YW50LCByZWxldmFudAogICAgZWxzZToKICAgICAgICBjb3ZlcmFnZSA9IG5wLmFzYXJyYXkoY292ZXJhZ2UsIGR0eXBlPWZsb2F0KQogICAgICAgIHNoYXJlZF9jb250ZW50ID0gbnAuYXNhcnJheShzaGFyZWRfY29udGVudCwgZHR5cGU9Ym9vbCkKICAgICAgICByZWxldmFudCA9IGNvdmVyYWdlID49IE1JTl9FVklERU5DRV9DT1ZFUkFHRQogICAgICAgIHN0cm9uZ19lbnQgPSAoZW50YWlsID49IEhJR0hfRU5UQUlMTUVOVCkgJiBzaGFyZWRfY29udGVudAogICAgICAgIHN0cm9uZ19jb24gPSAoY29udHJhID49IEhJR0hfQ09OVFJBRElDVElPTikgJiBzaGFyZWRfY29udGVudAogICAgc3VwcG9ydCA9IChlbnRhaWwgPj0gZW50X3RocikgJiAoZW50YWlsID49IGNvbnRyYSkgJiAocmVsZXZhbnQgfCBzdHJvbmdfZW50KQogICAgY29udHJhZGljdCA9IChjb250cmEgPj0gY29uX3RocikgJiAoY29udHJhID4gZW50YWlsKSAmIChyZWxldmFudCB8IHN0cm9uZ19jb24pCiAgICBpZiBzdXBwb3J0LmFueSgpOgogICAgICAgIGJlc3QgPSBpbnQobnAuYXJnbWF4KG5wLndoZXJlKHN1cHBvcnQsIGVudGFpbCwgLTEuMCkpKQogICAgICAgIHJldHVybiAic3VwcG9ydGVkIiwgZmxvYXQoZW50YWlsW2Jlc3RdKSwgYmVzdAogICAgaWYgY29udHJhZGljdC5hbnkoKToKICAgICAgICBiZXN0ID0gaW50KG5wLmFyZ21heChucC53aGVyZShjb250cmFkaWN0LCBjb250cmEsIC0xLjApKSkKICAgICAgICByZXR1cm4gImNvbnRyYWRpY3RlZCIsIGZsb2F0KGNvbnRyYVtiZXN0XSksIGJlc3QKICAgIGJlc3QgPSBpbnQobnAuYXJnbWF4KG5wLm1heGltdW0oZW50YWlsLCBjb250cmEpKSkKICAgIHJldHVybiAidW5zdXBwb3J0ZWQiLCBmbG9hdChtYXgoZW50YWlsW2Jlc3RdLCBjb250cmFbYmVzdF0pKSwgYmVzdAoKCmRlZiBfc2NvcmVfY2xhaW1fYmxvY2tzKHByb2JzOiBucC5uZGFycmF5LCBuX3NlbnRzOiBpbnQsCiAgICAgICAgICAgICAgICAgICAgICAgIGNvdmVyYWdlOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgIHNoYXJlZF9jb250ZW50OiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUpOgogICAgIiIiUGVyLWNsYWltIHZlcmRpY3QgZGVjaXNpb24gZnJvbSBhbiAobl9jbGFpbXMgKiBuX3NlbnRzLCAzKSBwcm9iIGJsb2NrLiIiIgogICAgb3V0ID0gW10KICAgIGZvciBjaSBpbiByYW5nZShsZW4ocHJvYnMpIC8vIG5fc2VudHMgaWYgbl9zZW50cyBlbHNlIDApOgogICAgICAgIGJsb2NrID0gc2xpY2UoY2kgKiBuX3NlbnRzLCAoY2kgKyAxKSAqIG5fc2VudHMpCiAgICAgICAgb3V0LmFwcGVuZChzY29yZV9ibG9jaygKICAgICAgICAgICAgcHJvYnNbYmxvY2ssIDFdLCBwcm9ic1tibG9jaywgMF0sCiAgICAgICAgICAgIE5vbmUgaWYgY292ZXJhZ2UgaXMgTm9uZSBlbHNlIGNvdmVyYWdlW2Jsb2NrXSwKICAgICAgICAgICAgTm9uZSBpZiBzaGFyZWRfY29udGVudCBpcyBOb25lIGVsc2Ugc2hhcmVkX2NvbnRlbnRbYmxvY2tdLAogICAgICAgICkpCiAgICByZXR1cm4gb3V0CgoKZGVmIF9hZ2dyZWdhdGUob3V0X2NsYWltczogbGlzdCkgLT4gZGljdDoKICAgIGNvdW50cyA9IHsic3VwcG9ydGVkIjogMCwgImNvbnRyYWRpY3RlZCI6IDAsICJ1bnN1cHBvcnRlZCI6IDB9CiAgICBmb3IgYyBpbiBvdXRfY2xhaW1zOgogICAgICAgIGNvdW50c1tjWyJ2ZXJkaWN0Il1dICs9IDEKICAgIG92ZXJhbGwgPSAic3VwcG9ydGVkIgogICAgaWYgY291bnRzWyJjb250cmFkaWN0ZWQiXSA+IDA6CiAgICAgICAgb3ZlcmFsbCA9ICJjb250cmFkaWN0ZWQiCiAgICBlbGlmIGNvdW50c1sidW5zdXBwb3J0ZWQiXSA+IDA6CiAgICAgICAgb3ZlcmFsbCA9ICJ1bnN1cHBvcnRlZCIKICAgIGVudF90aHIsIGNvbl90aHIgPSBfZWZmZWN0aXZlX3RocmVzaG9sZHMoKQogICAgcmV0dXJuIHsKICAgICAgICAibiI6IGxlbihvdXRfY2xhaW1zKSwKICAgICAgICAic3VwcG9ydGVkIjogY291bnRzWyJzdXBwb3J0ZWQiXSwKICAgICAgICAiY29udHJhZGljdGVkIjogY291bnRzWyJjb250cmFkaWN0ZWQiXSwKICAgICAgICAidW5zdXBwb3J0ZWQiOiBjb3VudHNbInVuc3VwcG9ydGVkIl0sCiAgICAgICAgIm92ZXJhbGwiOiBvdmVyYWxsLAogICAgICAgICJydWxlIjogKGYic3VwcG9ydGVkIGlmIGFueSBzZW50ZW5jZSBlbnRhaWxzICg+PXtlbnRfdGhyfSBhbmQgPj0gaXRzIGNvbnRyYWRpY3Rpb24pIHwgIgogICAgICAgICAgICAgICAgIGYiY29udHJhZGljdGVkIGlmIG5vbmUgc3VwcG9ydHMgYW5kIGEgc2VudGVuY2UgY29udHJhZGljdHMgKD57Y29uX3Rocn0gYW5kID4gaXRzIGVudGFpbG1lbnQpIHwgZWxzZSB1bnN1cHBvcnRlZCIpLAogICAgfQoKCmRlZiB2ZXJpZnlfY2xhaW1zKGNsYWltczogbGlzdCwgZXZpZGVuY2U6IHN0ciwgbmxpX21vZGVsLCBiYXRjaF9zaXplOiBpbnQgPSBERUZBVUxUX0JBVENIX1NJWkUpIC0+IGRpY3Q6CiAgICAiIiJSZXR1cm5zIHtjbGFpbXM6IFt7aWQsIHRleHQsIHZlcmRpY3QsIGNvbmZpZGVuY2UsIGV2aWRlbmNlX3NlbnRlbmNlfV0sCiAgICBhZ2dyZWdhdGU6IHtuLCBzdXBwb3J0ZWQsIGNvbnRyYWRpY3RlZCwgdW5zdXBwb3J0ZWQsIG92ZXJhbGx9fS4iIiIKICAgIGV2X3NlbnRzID0gc3BsaXRfc2VudGVuY2VzKGV2aWRlbmNlKSBvciAoW2V2aWRlbmNlXSBpZiBldmlkZW5jZS5zdHJpcCgpIGVsc2UgW10pCiAgICBjbGFpbXMgPSBbYyBmb3IgYyBpbiBjbGFpbXMgaWYgYy5zdHJpcCgpXQoKICAgIG91dF9jbGFpbXMgPSBbXQogICAgaWYgbm90IGV2X3NlbnRzOgogICAgICAgICMgTm8gZXZpZGVuY2UgdG8gY2hlY2sgYWdhaW5zdDogZXZlcnkgY2xhaW0gaXMgdW5zdXBwb3J0ZWQgKG5vIGJhc2lzKS4KICAgICAgICBmb3IgY2ksIGNsYWltIGluIGVudW1lcmF0ZShjbGFpbXMpOgogICAgICAgICAgICBvdXRfY2xhaW1zLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAiaWQiOiBjaSwgInRleHQiOiBjbGFpbSwgInZlcmRpY3QiOiAidW5zdXBwb3J0ZWQiLCAiY29uZmlkZW5jZSI6IDAuMCwKICAgICAgICAgICAgICAgICJldmlkZW5jZV9zZW50ZW5jZSI6ICIiLAogICAgICAgICAgICB9KQogICAgZWxzZToKICAgICAgICAjIENyb3NzRW5jb2RlciBOTEkgZXhwZWN0cyAocHJlbWlzZSwgaHlwb3RoZXNpcyk6IHByZW1pc2UgPSBldmlkZW5jZSwKICAgICAgICAjIGh5cG90aGVzaXMgPSBjbGFpbS4KICAgICAgICBwYWlycyA9IFsocywgYykgZm9yIGMgaW4gY2xhaW1zIGZvciBzIGluIGV2X3NlbnRzXQogICAgICAgIHByb2JzID0gbnAuYXNhcnJheShubGlfbW9kZWwucHJlZGljdChwYWlycywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBhcHBseV9zb2Z0bWF4PVRydWUpKQogICAgICAgICMgcHJvYnNbaSwgOl0gPSBbY29udHJhZGljdGlvbiwgZW50YWlsbWVudCwgbmV1dHJhbF0gZm9yIHBhaXJzW2ldCiAgICAgICAgbl9zZW50cyA9IGxlbihldl9zZW50cykKICAgICAgICBjb3ZlcmFnZSA9IG5wLmVtcHR5KGxlbihwYWlycykpCiAgICAgICAgc2hhcmVkID0gbnAuZW1wdHkobGVuKHBhaXJzKSwgZHR5cGU9Ym9vbCkKICAgICAgICBmb3IgaywgKHMsIGMpIGluIGVudW1lcmF0ZShwYWlycyk6CiAgICAgICAgICAgIGNvdmVyYWdlW2tdLCBzaGFyZWRba10gPSBldmlkZW5jZV9nYXRlKGMsIHMpCiAgICAgICAgZm9yIGNpLCAodmVyZGljdCwgY29uZmlkZW5jZSwgc2VudF9pZHgpIGluIGVudW1lcmF0ZSgKICAgICAgICAgICAgX3Njb3JlX2NsYWltX2Jsb2Nrcyhwcm9icywgbl9zZW50cywgY292ZXJhZ2UsIHNoYXJlZCkKICAgICAgICApOgogICAgICAgICAgICBvdXRfY2xhaW1zLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAiaWQiOiBjaSwKICAgICAgICAgICAgICAgICJ0ZXh0IjogY2xhaW1zW2NpXSwKICAgICAgICAgICAgICAgICJ2ZXJkaWN0IjogdmVyZGljdCwKICAgICAgICAgICAgICAgICJjb25maWRlbmNlIjogcm91bmQoY29uZmlkZW5jZSwgNCksCiAgICAgICAgICAgICAgICAiZXZpZGVuY2Vfc2VudGVuY2UiOiBldl9zZW50c1tzZW50X2lkeF0sCiAgICAgICAgICAgIH0pCgogICAgcmV0dXJuIHsiY2xhaW1zIjogb3V0X2NsYWltcywgImFnZ3JlZ2F0ZSI6IF9hZ2dyZWdhdGUob3V0X2NsYWltcyl9CgoKZGVmIHZlcmlmeV9jbGFpbXNfYWdhaW5zdF9wYXNzYWdlcyhjbGFpbXM6IGxpc3QsIHBhc3NhZ2VzX2J5X2NsYWltOiBsaXN0LCBubGlfbW9kZWwsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYmF0Y2hfc2l6ZTogaW50ID0gREVGQVVMVF9CQVRDSF9TSVpFKSAtPiBkaWN0OgogICAgIiIiUGVyLWNsYWltIHZlcmlmaWNhdGlvbiBhZ2FpbnN0IHJldHJpZXZlZCBwYXNzYWdlcyAoVGllciAzKS4KCiAgICBTRU5URU5DRS1MRVZFTDogZWFjaCBwYXNzYWdlIGlzIHNwbGl0IGludG8gc2VudGVuY2VzIGFuZCBldmVyeQogICAgKGV2aWRlbmNlX3NlbnRlbmNlLCBjbGFpbSkgcGFpciBpcyBzY29yZWQgd2l0aCB0aGUgc2FtZSBzdXBwb3J0LWZpcnN0IHJ1bGUKICAgIGFzIHZlcmlmeV9jbGFpbXMgKGFueSBlbnRhaWxpbmcgc2VudGVuY2Ugd2lucywgY29udHJhZGljdGlvbiBvbmx5IGRlY2lkZXMKICAgIHdoZW4gbm8gc2VudGVuY2Ugc3VwcG9ydHMpLiBUaGUgZHJpdmluZyBzZW50ZW5jZSBiZWNvbWVzIHRoZSBldmlkZW5jZSBvcgogICAgcXVvdGUsIHNvIGxvbmcgcGFzc2FnZXMgY2Fubm90IGRyb3duIGEgc3VwcG9ydGluZyBzZW50ZW5jZS4KICAgIENsYWltcyB3aXRob3V0IHBhc3NhZ2VzIGFic3RhaW4gKHVuc3VwcG9ydGVkLCBhYnN0YWluZWQ9VHJ1ZSkuCiAgICAiIiIKICAgIGNsYWltcyA9IFtjIGZvciBjIGluIGNsYWltcyBpZiBjLnN0cmlwKCldCiAgICBvdXRfY2xhaW1zID0gW10KICAgIGlmIGNsYWltczoKICAgICAgICBwYWlycyA9IFtdICAgICAgICAgICAjIChzZW50ZW5jZSwgY2xhaW0pCiAgICAgICAgYmxvY2tfc2l6ZXMgPSBbXSAgICAgIyBzZW50ZW5jZXMgcGVyIGNsYWltCiAgICAgICAgc2VudF9wYXNzYWdlID0gW10gICAgIyBwZXIgKGNsYWltLCBzZW50ZW5jZSk6IHBhc3NhZ2UgaW5kZXgKICAgICAgICBnYXRlID0gW10gICAgICAgICAgICAjIHBlciBwYWlyOiAoY292ZXJhZ2UsIHNoYXJlZF9jb250ZW50KQogICAgICAgIGZvciBjaSwgY2xhaW0gaW4gZW51bWVyYXRlKGNsYWltcyk6CiAgICAgICAgICAgIHBzID0gcGFzc2FnZXNfYnlfY2xhaW1bY2ldIGlmIGNpIDwgbGVuKHBhc3NhZ2VzX2J5X2NsYWltKSBlbHNlIFtdCiAgICAgICAgICAgIG4gPSAwCiAgICAgICAgICAgIGZvciBwaSwgcCBpbiBlbnVtZXJhdGUocHMpOgogICAgICAgICAgICAgICAgdGV4dCA9IHAuZ2V0KCJ0ZXh0IiwgIiIpCiAgICAgICAgICAgICAgICBzZW50cyA9IHNwbGl0X3NlbnRlbmNlcyh0ZXh0KSBvciAoW3RleHRdIGlmIHRleHQuc3RyaXAoKSBlbHNlIFtdKQogICAgICAgICAgICAgICAgZm9yIHMgaW4gc2VudHM6CiAgICAgICAgICAgICAgICAgICAgcGFpcnMuYXBwZW5kKChzLCBjbGFpbSkpCiAgICAgICAgICAgICAgICAgICAgc2VudF9wYXNzYWdlLmFwcGVuZCgoY2ksIHBpKSkKICAgICAgICAgICAgICAgICAgICBnYXRlLmFwcGVuZChldmlkZW5jZV9nYXRlKGNsYWltLCBzKSkKICAgICAgICAgICAgICAgICAgICBuICs9IDEKICAgICAgICAgICAgYmxvY2tfc2l6ZXMuYXBwZW5kKG4pCgogICAgICAgIGlmIHBhaXJzOgogICAgICAgICAgICBwcm9icyA9IG5wLmFzYXJyYXkobmxpX21vZGVsLnByZWRpY3QocGFpcnMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgYXBwbHlfc29mdG1heD1UcnVlKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcm9icyA9IG5wLmVtcHR5KCgwLCAzKSkKCiAgICAgICAgY3Vyc29yID0gMAogICAgICAgIGZvciBjaSwgY2xhaW0gaW4gZW51bWVyYXRlKGNsYWltcyk6CiAgICAgICAgICAgIG4gPSBibG9ja19zaXplc1tjaV0KICAgICAgICAgICAgcHMgPSBwYXNzYWdlc19ieV9jbGFpbVtjaV0gaWYgY2kgPCBsZW4ocGFzc2FnZXNfYnlfY2xhaW0pIGVsc2UgW10KICAgICAgICAgICAgaWYgbiA9PSAwOgogICAgICAgICAgICAgICAgb3V0X2NsYWltcy5hcHBlbmQoewogICAgICAgICAgICAgICAgICAgICJpZCI6IGNpLCAidGV4dCI6IGNsYWltLCAidmVyZGljdCI6ICJ1bnN1cHBvcnRlZCIsICJjb25maWRlbmNlIjogMC4wLAogICAgICAgICAgICAgICAgICAgICJldmlkZW5jZV9zZW50ZW5jZSI6ICIiLCAiZXZpZGVuY2Vfc291cmNlIjogIiIsICJldmlkZW5jZV91cmwiOiAiIiwKICAgICAgICAgICAgICAgICAgICAiYWJzdGFpbmVkIjogVHJ1ZSwKICAgICAgICAgICAgICAgIH0pCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBibG9jayA9IHByb2JzW2N1cnNvciA6IGN1cnNvciArIG5dCiAgICAgICAgICAgIGJsb2NrX2dhdGUgPSBnYXRlW2N1cnNvciA6IGN1cnNvciArIG5dCiAgICAgICAgICAgIGN1cnNvciArPSBuCiAgICAgICAgICAgIGNvdmVyYWdlID0gbnAuYXJyYXkoW2dbMF0gZm9yIGcgaW4gYmxvY2tfZ2F0ZV0pCiAgICAgICAgICAgIHNoYXJlZCA9IG5wLmFycmF5KFtnWzFdIGZvciBnIGluIGJsb2NrX2dhdGVdLCBkdHlwZT1ib29sKQogICAgICAgICAgICB2ZXJkaWN0LCBjb25maWRlbmNlLCBzZW50X2lkeCA9IHNjb3JlX2Jsb2NrKGJsb2NrWzosIDFdLCBibG9ja1s6LCAwXSwgY292ZXJhZ2UsIHNoYXJlZCkKCiAgICAgICAgICAgIF8sIHBpID0gc2VudF9wYXNzYWdlW2N1cnNvciAtIG4gKyBzZW50X2lkeF0KICAgICAgICAgICAgcGFzc2FnZSA9IHBzW3BpXQogICAgICAgICAgICBlbnRyeSA9IHsKICAgICAgICAgICAgICAgICJpZCI6IGNpLCAidGV4dCI6IGNsYWltLCAidmVyZGljdCI6IHZlcmRpY3QsCiAgICAgICAgICAgICAgICAiY29uZmlkZW5jZSI6IHJvdW5kKGNvbmZpZGVuY2UsIDQpLAogICAgICAgICAgICAgICAgImV2aWRlbmNlX3NlbnRlbmNlIjogcGFpcnNbY3Vyc29yIC0gbiArIHNlbnRfaWR4XVswXSwKICAgICAgICAgICAgICAgICJldmlkZW5jZV9zb3VyY2UiOiBwYXNzYWdlLmdldCgic291cmNlIiwgIiIpLAogICAgICAgICAgICAgICAgImV2aWRlbmNlX3VybCI6IHBhc3NhZ2UuZ2V0KCJ1cmwiLCAiIiksCiAgICAgICAgICAgICAgICAiYWJzdGFpbmVkIjogRmFsc2UsCiAgICAgICAgICAgIH0KICAgICAgICAgICAgaWYgdmVyZGljdCA9PSAiY29udHJhZGljdGVkIjoKICAgICAgICAgICAgICAgIGVudHJ5WyJldmlkZW5jZV9xdW90ZSJdID0gZW50cnlbImV2aWRlbmNlX3NlbnRlbmNlIl1bOjMwMF0KICAgICAgICAgICAgb3V0X2NsYWltcy5hcHBlbmQoZW50cnkpCgogICAgYWdnID0gX2FnZ3JlZ2F0ZShvdXRfY2xhaW1zKQogICAgYWdnWyJhYnN0YWluZWQiXSA9IHN1bSgxIGZvciBjIGluIG91dF9jbGFpbXMgaWYgYy5nZXQoImFic3RhaW5lZCIpKQogICAgcmV0dXJuIHsiY2xhaW1zIjogb3V0X2NsYWltcywgImFnZ3JlZ2F0ZSI6IGFnZ30K",
 "src/data/download.py": "IiIiCkRhdGEgYWNxdWlzaXRpb24gc2NyaXB0IGZvciBIYWx1UklTQy4KCkRvd25sb2FkcyB0aGUgSGFsdUV2YWwgUUEgc3Vic2V0IChxYV9kYXRhLmpzb24sIEpTT05MKSBmcm9tIHRoZSBvZmZpY2lhbApSVUNBSUJveC9IYWx1RXZhbCByZXBvc2l0b3J5LiBQZXIgZG93bmxvYWQgaXQgcmVjb3JkcyB0aGUgcmVwbyBjb21taXQgcmV2aXNpb24KYW5kIHRoZSBmaWxlIFNIQS0yNTYgaGFzaCBpbiBkYXRhL3Jhdy9oYWx1ZXZhbC9yZXZpc2lvbi5qc29uLCBtaXJyb3JpbmcKc3JjL2RhdGEvZG93bmxvYWRfcmFndHJ1dGgucHkgYW5kIHNyYy9kYXRhL2Rvd25sb2FkX2ZhaXRoYmVuY2gucHkuCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvZGF0YS9kb3dubG9hZC5weQoiIiIKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBvcwppbXBvcnQgdXJsbGliLnJlcXVlc3QKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiZG93bmxvYWRfaGFsdWV2YWwiKQoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdCk9VVF9ESVIgPSBST09UIC8gImRhdGEiIC8gInJhdyIgLyAiaGFsdWV2YWwiCgpSRVBPID0gIlJVQ0FJQm94L0hhbHVFdmFsIgpCUkFOQ0ggPSAibWFpbiIKUkFXX0JBU0UgPSBmImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS97UkVQT30ve0JSQU5DSH0iCkFQSV9CQVNFID0gZiJodHRwczovL2FwaS5naXRodWIuY29tL3JlcG9zL3tSRVBPfSIKCiMgbmFtZSAtPiBwYXRoIGluc2lkZSB0aGUgcmVwbwpGSUxFUyA9IHsKICAgICJxYV9kYXRhLmpzb24iOiAiZGF0YS9xYV9kYXRhLmpzb24iLAp9CgoKZGVmIF9odHRwX2dldF9qc29uKHVybDogc3RyKSAtPiBkaWN0OgogICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCh1cmwsIGhlYWRlcnM9eyJVc2VyLUFnZW50IjogImhhbHVyaXNjLWIxIn0pCiAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxLCB0aW1lb3V0PTYwKSBhcyByZXNwOgogICAgICAgIHJldHVybiBqc29uLmxvYWRzKHJlc3AucmVhZCgpLmRlY29kZSgidXRmLTgiKSkKCgpkZWYgX3NoYTI1NihwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBoID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBvcGVuKHBhdGgsICJyYiIpIGFzIGY6CiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBmLnJlYWQoMSA8PCAyMCksIGIiIik6CiAgICAgICAgICAgIGgudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgX2ZldGNoX2NvbW1pdF9zaGEoKSAtPiBzdHI6CiAgICB0cnk6CiAgICAgICAgaW5mbyA9IF9odHRwX2dldF9qc29uKGYie0FQSV9CQVNFfS9jb21taXRzL3tCUkFOQ0h9IikKICAgICAgICByZXR1cm4gc3RyKGluZm8uZ2V0KCJzaGEiLCAidW5rbm93biIpKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgIyBwcm92ZW5hbmNlIHNob3VsZCBub3QgYmxvY2sgdGhlIGRvd25sb2FkCiAgICAgICAgbG9nZ2VyLndhcm5pbmcoZiJDb3VsZCBub3QgZmV0Y2ggY29tbWl0IHJldmlzaW9uOiB7ZX0iKQogICAgICAgIHJldHVybiAidW5rbm93biIKCgpkZWYgX3ZhbGlkYXRlX3FhX2pzb25sKCkgLT4gaW50OgogICAgIiIiVmVyaWZ5IHRoYXQgZXZlcnkgbm9uLWVtcHR5IGxpbmUgb2YgcWFfZGF0YS5qc29uIHBhcnNlcyBhcyBKU09OLiIiIgogICAgY291bnQgPSAwCiAgICB3aXRoIG9wZW4oT1VUX0RJUiAvICJxYV9kYXRhLmpzb24iLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgY291bnQgKz0gMQogICAgbG9nZ2VyLmluZm8oZiJWYWxpZGF0ZWQgSGFsdUV2YWwgUUE6IHtjb3VudH0gSlNPTkwgaXRlbXMgbG9hZGVkIHN1Y2Nlc3NmdWxseS4iKQogICAgcmV0dXJuIGNvdW50CgoKZGVmIGRvd25sb2FkX2hhbHVldmFsX3FhKGZvcmNlOiBib29sID0gRmFsc2UpIC0+IHN0cjoKICAgICIiIkRvd25sb2FkIEhhbHVFdmFsIHFhX2RhdGEuanNvbiBpZiBtaXNzaW5nOyByZXR1cm5zIHRoZSBmaWxlIHBhdGggKHN0cikuIiIiCiAgICBvcy5tYWtlZGlycyhPVVRfRElSLCBleGlzdF9vaz1UcnVlKQogICAgZGVzdCA9IE9VVF9ESVIgLyAicWFfZGF0YS5qc29uIgogICAgcmV2aXNpb25fcGF0aCA9IE9VVF9ESVIgLyAicmV2aXNpb24uanNvbiIKICAgIG1pc3NpbmcgPSBub3QgZGVzdC5leGlzdHMoKQoKICAgIGlmIG1pc3Npbmcgb3IgZm9yY2U6CiAgICAgICAgaWYgZm9yY2UgYW5kIG5vdCBtaXNzaW5nOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiZm9yY2U9VHJ1ZTogcmUtZG93bmxvYWRpbmcgSGFsdUV2YWwgcWFfZGF0YS5qc29uIikKICAgICAgICBmb3IgbmFtZSwgcmVwb19wYXRoIGluIEZJTEVTLml0ZW1zKCk6CiAgICAgICAgICAgIHVybCA9IGYie1JBV19CQVNFfS97cmVwb19wYXRofSIKICAgICAgICAgICAgbG9nZ2VyLmluZm8oZiIgIHt1cmx9IikKICAgICAgICAgICAgdXJsbGliLnJlcXVlc3QudXJscmV0cmlldmUodXJsLCBPVVRfRElSIC8gbmFtZSkKICAgICAgICAgICAgc2l6ZV9tYiA9IChPVVRfRElSIC8gbmFtZSkuc3RhdCgpLnN0X3NpemUgLyAoMTAyNCAqIDEwMjQpCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiICBzYXZlZCB7bmFtZX0gKHtzaXplX21iOi4yZn0gTUIpIikKICAgIGVsc2U6CiAgICAgICAgbG9nZ2VyLmluZm8oIkhhbHVFdmFsIHFhX2RhdGEuanNvbiBhbHJlYWR5IHByZXNlbnQsIHNraXBwaW5nIGRvd25sb2FkLiIpCgogICAgbl9pdGVtcyA9IF92YWxpZGF0ZV9xYV9qc29ubCgpCgogICAgaWYgbWlzc2luZyBvciBmb3JjZSBvciBub3QgcmV2aXNpb25fcGF0aC5leGlzdHMoKToKICAgICAgICByZXZpc2lvbl9wYXRoLndyaXRlX3RleHQoCiAgICAgICAgICAgIGpzb24uZHVtcHMoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgInJlcG8iOiBSRVBPLAogICAgICAgICAgICAgICAgICAgICJicmFuY2giOiBCUkFOQ0gsCiAgICAgICAgICAgICAgICAgICAgImNvbW1pdF9zaGEiOiBfZmV0Y2hfY29tbWl0X3NoYSgpLAogICAgICAgICAgICAgICAgICAgICJmZXRjaGVkX2F0X3V0YyI6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCh0aW1lc3BlYz0ic2Vjb25kcyIpLAogICAgICAgICAgICAgICAgICAgICJmaWxlcyI6IHsKICAgICAgICAgICAgICAgICAgICAgICAgbmFtZTogewogICAgICAgICAgICAgICAgICAgICAgICAgICAgInNoYTI1NiI6IF9zaGEyNTYoT1VUX0RJUiAvIG5hbWUpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgImJ5dGVzIjogKE9VVF9ESVIgLyBuYW1lKS5zdGF0KCkuc3Rfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgfQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgbmFtZSBpbiBGSUxFUwogICAgICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgICAgICB9LAogICAgICAgICAgICAgICAgaW5kZW50PTIsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgKQogICAgICAgIGxvZ2dlci5pbmZvKGYiUmV2aXNpb24gKyBoYXNoZXMgd3JpdHRlbiB0byB7cmV2aXNpb25fcGF0aH0iKQoKICAgIHJldHVybiBzdHIoZGVzdCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgZG93bmxvYWRfaGFsdWV2YWxfcWEoKQo=",
 "src/data/download_faithbench.py": "IiIiCkZhaXRoQmVuY2ggKE5BQUNMIDIwMjUpIGFjcXVpc2l0aW9uIGZvciBIYWx1UklTQyBWZXJzaW9uIEIgKEIxKS4KCkRvd25sb2FkcyB0aGUgb2ZmaWNpYWwgaHVtYW4tYW5ub3RhdGVkIHJlbGVhc2UgYmF0Y2hlcyBmcm9tIHZlY3RhcmEvRmFpdGhCZW5jaAooZGF0YV9mb3JfcmVsZWFzZS9iYXRjaF97MS4uMTZ9Lmpzb247IGJhdGNoIDEzIGRvZXMgbm90IGV4aXN0IHVwc3RyZWFtKS4KCkZhaXRoQmVuY2ggaXMgQ0MgQlktTkMtU0EgNC4wOiB0aGUgcmF3IGZpbGVzIHN0YXkgdW5kZXIgdGhlIGdpdGlnbm9yZWQKYGRhdGEvcmF3L2ZhaXRoYmVuY2gvYCBhbmQgYXJlIE5FVkVSIGJ1bmRsZWQgaW4gdGhlIHJlcG9zaXRvcnkuIE9ubHkKZG93bmxvYWQgaW5zdHJ1Y3Rpb25zLCBjaXRhdGlvbnMsIGhhc2hlcywgYW5kIGxpY2Vuc2Ugbm90ZXMgYXJlIHNoaXBwZWQuCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvZGF0YS9kb3dubG9hZF9mYWl0aGJlbmNoLnB5CiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCB1cmxsaWIucmVxdWVzdApmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJkb3dubG9hZF9mYWl0aGJlbmNoIikKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXQpPVVRfRElSID0gUk9PVCAvICJkYXRhIiAvICJyYXciIC8gImZhaXRoYmVuY2giCgpSRVBPID0gInZlY3RhcmEvRmFpdGhCZW5jaCIKQlJBTkNIID0gIm1haW4iClJBV19CQVNFID0gZiJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20ve1JFUE99L3tCUkFOQ0h9IgpBUElfQkFTRSA9IGYiaHR0cHM6Ly9hcGkuZ2l0aHViLmNvbS9yZXBvcy97UkVQT30iCgojIGJhdGNoXzEzIGRvZXMgbm90IGV4aXN0IGluIHRoZSBvZmZpY2lhbCByZWxlYXNlCkJBVENIX0lEUyA9IFtpIGZvciBpIGluIHJhbmdlKDEsIDE3KSBpZiBpICE9IDEzXQpCQVRDSF9GSUxFID0gImJhdGNoX3tpZH0uanNvbiIKCgpkZWYgX2h0dHBfZ2V0X2pzb24odXJsOiBzdHIpIC0+IGRpY3Q6CiAgICByZXEgPSB1cmxsaWIucmVxdWVzdC5SZXF1ZXN0KHVybCwgaGVhZGVycz17IlVzZXItQWdlbnQiOiAiaGFsdXJpc2MtYjEifSkKICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihyZXEsIHRpbWVvdXQ9NjApIGFzIHJlc3A6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocmVzcC5yZWFkKCkuZGVjb2RlKCJ1dGYtOCIpKQoKCmRlZiBfc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIGltcG9ydCBoYXNobGliCgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggb3BlbihwYXRoLCAicmIiKSBhcyBmOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogZi5yZWFkKDEgPDwgMjApLCBiIiIpOgogICAgICAgICAgICBoLnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgoKZGVmIF9mZXRjaF9jb21taXRfc2hhKCkgLT4gc3RyOgogICAgdHJ5OgogICAgICAgIGluZm8gPSBfaHR0cF9nZXRfanNvbihmIntBUElfQkFTRX0vY29tbWl0cy97QlJBTkNIfSIpCiAgICAgICAgcmV0dXJuIHN0cihpbmZvLmdldCgic2hhIiwgInVua25vd24iKSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBsb2dnZXIud2FybmluZyhmIkNvdWxkIG5vdCBmZXRjaCBjb21taXQgcmV2aXNpb246IHtlfSIpCiAgICAgICAgcmV0dXJuICJ1bmtub3duIgoKCmRlZiBfdmFsaWRhdGVfYmF0Y2hlcygpIC0+IGludDoKICAgICIiIkVhY2ggYmF0Y2ggcGFyc2VzIGFuZCBldmVyeSBzYW1wbGUgaGFzIHNvdXJjZS9zdW1tYXJ5L21ldGFkYXRhLiIiIgogICAgdG90YWwgPSAwCiAgICBmb3IgYmF0Y2hfaWQgaW4gQkFUQ0hfSURTOgogICAgICAgIHBhdGggPSBPVVRfRElSIC8gQkFUQ0hfRklMRS5mb3JtYXQoaWQ9YmF0Y2hfaWQpCiAgICAgICAgYmF0Y2ggPSBqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIHNhbXBsZXMgPSBiYXRjaFsic2FtcGxlcyJdCiAgICAgICAgZm9yIHMgaW4gc2FtcGxlczoKICAgICAgICAgICAgYXNzZXJ0IGlzaW5zdGFuY2Uocy5nZXQoInNvdXJjZSIpLCBzdHIpIGFuZCBzWyJzb3VyY2UiXS5zdHJpcCgpCiAgICAgICAgICAgIGFzc2VydCBpc2luc3RhbmNlKHMuZ2V0KCJzdW1tYXJ5IiksIHN0cikgYW5kIHNbInN1bW1hcnkiXS5zdHJpcCgpCiAgICAgICAgICAgIGFzc2VydCAibWV0YWRhdGEiIGluIHMgYW5kICJzdW1tYXJpemVyIiBpbiBzWyJtZXRhZGF0YSJdCiAgICAgICAgdG90YWwgKz0gbGVuKHNhbXBsZXMpCiAgICBsb2dnZXIuaW5mbyhmIlZhbGlkYXRlZCB7bGVuKEJBVENIX0lEUyl9IEZhaXRoQmVuY2ggYmF0Y2hlcywge3RvdGFsfSBzYW1wbGVzIHRvdGFsLiIpCiAgICByZXR1cm4gdG90YWwKCgpkZWYgZG93bmxvYWRfZmFpdGhiZW5jaChmb3JjZTogYm9vbCA9IEZhbHNlKSAtPiBkaWN0OgogICAgIiIiRG93bmxvYWQgdGhlIG9mZmljaWFsIGJhdGNoZXMgaWYgbWlzc2luZzsgcmV0dXJucyB7YmF0Y2hfaWQ6IHBhdGh9LiIiIgogICAgb3MubWFrZWRpcnMoT1VUX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGhzID0ge30KICAgIG1pc3NpbmcgPSBbYiBmb3IgYiBpbiBCQVRDSF9JRFMgaWYgbm90IChPVVRfRElSIC8gQkFUQ0hfRklMRS5mb3JtYXQoaWQ9YikpLmV4aXN0cygpXQoKICAgIGlmIG1pc3Npbmcgb3IgZm9yY2U6CiAgICAgICAgaWYgZm9yY2UgYW5kIG5vdCBtaXNzaW5nOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiZm9yY2U9VHJ1ZTogcmUtZG93bmxvYWRpbmcgRmFpdGhCZW5jaCBiYXRjaGVzIikKICAgICAgICBjb21taXRfc2hhID0gX2ZldGNoX2NvbW1pdF9zaGEoKQogICAgICAgIGZvciBiYXRjaF9pZCBpbiBCQVRDSF9JRFM6CiAgICAgICAgICAgIG5hbWUgPSBCQVRDSF9GSUxFLmZvcm1hdChpZD1iYXRjaF9pZCkKICAgICAgICAgICAgZGVzdCA9IE9VVF9ESVIgLyBuYW1lCiAgICAgICAgICAgIGlmIGRlc3QuZXhpc3RzKCkgYW5kIG5vdCBmb3JjZToKICAgICAgICAgICAgICAgIHBhdGhzW2JhdGNoX2lkXSA9IHN0cihkZXN0KQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgdXJsID0gZiJ7UkFXX0JBU0V9L2RhdGFfZm9yX3JlbGVhc2Uve25hbWV9IgogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIiAge3VybH0iKQogICAgICAgICAgICB1cmxsaWIucmVxdWVzdC51cmxyZXRyaWV2ZSh1cmwsIGRlc3QpCiAgICAgICAgICAgIGxvZ2dlci5pbmZvKGYiICBzYXZlZCB7bmFtZX0gKHtkZXN0LnN0YXQoKS5zdF9zaXplIC8gMTAyNDouMGZ9IEtCKSIpCiAgICAgICAgX3ZhbGlkYXRlX2JhdGNoZXMoKQogICAgICAgIGhhc2hlcyA9IHsKICAgICAgICAgICAgbmFtZTogeyJzaGEyNTYiOiBfc2hhMjU2KE9VVF9ESVIgLyBuYW1lKSwgImJ5dGVzIjogKE9VVF9ESVIgLyBuYW1lKS5zdGF0KCkuc3Rfc2l6ZX0KICAgICAgICAgICAgZm9yIG5hbWUgaW4gc29ydGVkKHAubmFtZSBmb3IgcCBpbiBPVVRfRElSLmdsb2IoImJhdGNoXyouanNvbiIpKQogICAgICAgIH0KICAgICAgICAoT1VUX0RJUiAvICJyZXZpc2lvbi5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAicmVwbyI6IFJFUE8sCiAgICAgICAgICAgICAgICAgICAgImJyYW5jaCI6IEJSQU5DSCwKICAgICAgICAgICAgICAgICAgICAiY29tbWl0X3NoYSI6IGNvbW1pdF9zaGEsCiAgICAgICAgICAgICAgICAgICAgImZldGNoZWRfYXRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSJzZWNvbmRzIiksCiAgICAgICAgICAgICAgICAgICAgImZpbGVzIjogaGFzaGVzLAogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgIGluZGVudD0yLAogICAgICAgICAgICApLAogICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICkKICAgICAgICBsb2dnZXIuaW5mbyhmIlJldmlzaW9uICsgaGFzaGVzIHdyaXR0ZW4gdG8ge09VVF9ESVIgLyAncmV2aXNpb24uanNvbid9IikKICAgIGVsc2U6CiAgICAgICAgbG9nZ2VyLmluZm8oIkZhaXRoQmVuY2ggYmF0Y2hlcyBhbHJlYWR5IHByZXNlbnQsIHNraXBwaW5nLiIpCgogICAgZm9yIGJhdGNoX2lkIGluIEJBVENIX0lEUzoKICAgICAgICBwYXRoc1tiYXRjaF9pZF0gPSBzdHIoT1VUX0RJUiAvIEJBVENIX0ZJTEUuZm9ybWF0KGlkPWJhdGNoX2lkKSkKICAgIHJldHVybiBwYXRocwoKCmRlZiBsb2FkX2ZhaXRoYmVuY2hfc2FtcGxlcygpOgogICAgIiIiUmV0dXJucyBhIGxpc3Qgb2YgKGJhdGNoX2lkLCBzYW1wbGVfZGljdCkgYWNyb3NzIGFsbCBvZmZpY2lhbCBiYXRjaGVzLiIiIgogICAgcGF0aHMgPSBkb3dubG9hZF9mYWl0aGJlbmNoKCkKICAgIHNhbXBsZXMgPSBbXQogICAgZm9yIGJhdGNoX2lkLCBwYXRoIGluIHBhdGhzLml0ZW1zKCk6CiAgICAgICAgYmF0Y2ggPSBqc29uLmxvYWRzKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGZvciBzIGluIGJhdGNoWyJzYW1wbGVzIl06CiAgICAgICAgICAgIHNhbXBsZXMuYXBwZW5kKChpbnQoYmF0Y2hfaWQpLCBzKSkKICAgIHJldHVybiBzYW1wbGVzCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGRvd25sb2FkX2ZhaXRoYmVuY2goKQo=",
 "src/data/download_ragtruth.py": "IiIiClJBR1RydXRoIChBQ0wgMjAyNCkgT0ZGSUNJQUwgYWNxdWlzaXRpb24gZm9yIEhhbHVSSVNDIFZlcnNpb24gQiAoQjEpLgoKRG93bmxvYWRzIHRoZSB0d28gb2ZmaWNpYWwgZmlsZXMgZnJvbSBQYXJ0aWNsZU1lZGlhL1JBR1RydXRoOgogIGRhdGFzZXQvcmVzcG9uc2UuanNvbmwgICAgKHJlc3BvbnNlcyB3aXRoIHdvcmQtbGV2ZWwgaGFsbHVjaW5hdGlvbiBzcGFucykKICBkYXRhc2V0L3NvdXJjZV9pbmZvLmpzb25sIChzb3VyY2VzLCB0YXNrIHR5cGVzLCBwcm9tcHRzKQoKVGhpcyByZXBsYWNlcyB0aGUgbG9zc3kgSHVnZ2luZ0ZhY2UgbWlycm9yIHVzZWQgaW4gVmVyc2lvbiBBICh3aGljaCBkcm9wcGVkCnNvdXJjZV9pZCwgc3BhbnMsIHRhc2tfdHlwZSwgc3BsaXQsIGFuZCBxdWFsaXR5KS4gQjEga2VlcHMgZXZlcnkgZmllbGQuCgpQZXIgZG93bmxvYWQgaXQgcmVjb3JkcyB0aGUgcmVwbyBjb21taXQgcmV2aXNpb24gYW5kIHBlci1maWxlIFNIQS0yNTYgaGFzaGVzCmluIGRhdGEvcmF3L3JhZ3RydXRoX29mZmljaWFsL3JldmlzaW9uLmpzb24gZm9yIHByb3ZlbmFuY2UuCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvZGF0YS9kb3dubG9hZF9yYWd0cnV0aC5weQoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBvcwppbXBvcnQgdXJsbGliLnJlcXVlc3QKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8sIGZvcm1hdD0iJShhc2N0aW1lKXMgLSAlKGxldmVsbmFtZSlzIC0gJShtZXNzYWdlKXMiKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiZG93bmxvYWRfcmFndHJ1dGgiKQoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdCk9VVF9ESVIgPSBST09UIC8gImRhdGEiIC8gInJhdyIgLyAicmFndHJ1dGhfb2ZmaWNpYWwiCgpSRVBPID0gIlBhcnRpY2xlTWVkaWEvUkFHVHJ1dGgiCkJSQU5DSCA9ICJtYWluIgpSQVdfQkFTRSA9IGYiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL3tSRVBPfS97QlJBTkNIfSIKQVBJX0JBU0UgPSBmImh0dHBzOi8vYXBpLmdpdGh1Yi5jb20vcmVwb3Mve1JFUE99IgoKIyBuYW1lIC0+IHBhdGggaW5zaWRlIHRoZSByZXBvCkZJTEVTID0gewogICAgInJlc3BvbnNlLmpzb25sIjogImRhdGFzZXQvcmVzcG9uc2UuanNvbmwiLAogICAgInNvdXJjZV9pbmZvLmpzb25sIjogImRhdGFzZXQvc291cmNlX2luZm8uanNvbmwiLAp9CgoKZGVmIF9odHRwX2dldF9qc29uKHVybDogc3RyKSAtPiBkaWN0OgogICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCh1cmwsIGhlYWRlcnM9eyJVc2VyLUFnZW50IjogImhhbHVyaXNjLWIxIn0pCiAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxLCB0aW1lb3V0PTYwKSBhcyByZXNwOgogICAgICAgIHJldHVybiBqc29uLmxvYWRzKHJlc3AucmVhZCgpLmRlY29kZSgidXRmLTgiKSkKCgpkZWYgX3NoYTI1NihwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBpbXBvcnQgaGFzaGxpYgoKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZjoKICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IGYucmVhZCgxIDw8IDIwKSwgYiIiKToKICAgICAgICAgICAgaC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiBfZmV0Y2hfY29tbWl0X3NoYSgpIC0+IHN0cjoKICAgIHRyeToKICAgICAgICBpbmZvID0gX2h0dHBfZ2V0X2pzb24oZiJ7QVBJX0JBU0V9L2NvbW1pdHMve0JSQU5DSH0iKQogICAgICAgIHJldHVybiBzdHIoaW5mby5nZXQoInNoYSIsICJ1bmtub3duIikpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAjIHByb3ZlbmFuY2Ugc2hvdWxkIG5vdCBibG9jayB0aGUgZG93bmxvYWQKICAgICAgICBsb2dnZXIud2FybmluZyhmIkNvdWxkIG5vdCBmZXRjaCBjb21taXQgcmV2aXNpb246IHtlfSIpCiAgICAgICAgcmV0dXJuICJ1bmtub3duIgoKCmRlZiBfdmFsaWRhdGVfb2ZmaWNpYWwoKToKICAgICIiIlN0cnVjdHVyYWwgdmFsaWRhdGlvbiBvZiB0aGUgb2ZmaWNpYWwgZmlsZXMgKEIxOiByZXByb2R1Y2libGUgZG93bmxvYWQpLiIiIgogICAgcmVzcF9wYXRoID0gT1VUX0RJUiAvICJyZXNwb25zZS5qc29ubCIKICAgIHNyY19wYXRoID0gT1VUX0RJUiAvICJzb3VyY2VfaW5mby5qc29ubCIKCiAgICByZXNwb25zZXMgPSBbXQogICAgc291cmNlX2lkcyA9IHNldCgpCiAgICB3aXRoIG9wZW4ocmVzcF9wYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAgcmVzcG9uc2VzLmFwcGVuZChqc29uLmxvYWRzKGxpbmUpKQogICAgd2l0aCBvcGVuKHNyY19wYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAgc3JjID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgc291cmNlX2lkcy5hZGQoc3JjWyJzb3VyY2VfaWQiXSkKCiAgICBpZHMgPSBbclsiaWQiXSBmb3IgciBpbiByZXNwb25zZXNdCiAgICBhc3NlcnQgbGVuKGlkcykgPT0gbGVuKHNldChpZHMpKSwgInJlc3BvbnNlIGlkcyBhcmUgbm90IHVuaXF1ZSIKICAgIGFzc2VydCBsZW4oc291cmNlX2lkcykgPT0gbGVuKHtzdHIocykgZm9yIHMgaW4gc291cmNlX2lkc30pLCAic291cmNlIGlkcyBhcmUgbm90IHVuaXF1ZSIKICAgIG1pc3NpbmcgPSBzb3J0ZWQoe3JbInNvdXJjZV9pZCJdIGZvciByIGluIHJlc3BvbnNlc30gLSB7c3RyKHMpIGZvciBzIGluIHNvdXJjZV9pZHN9KQogICAgYXNzZXJ0IG5vdCBtaXNzaW5nLCBmInJlc3BvbnNlcyByZWZlcmVuY2UgdW5rbm93biBzb3VyY2VzOiB7bWlzc2luZ1s6NV19IgogICAgbG9nZ2VyLmluZm8oCiAgICAgICAgZiJWYWxpZGF0ZWQgb2ZmaWNpYWwgUkFHVHJ1dGg6IHtsZW4ocmVzcG9uc2VzKX0gcmVzcG9uc2VzLCAiCiAgICAgICAgZiJ7bGVuKHNvdXJjZV9pZHMpfSBzb3VyY2VzLCBubyBvcnBoYW4gcmVzcG9uc2VzLiIKICAgICkKICAgIHJldHVybiBsZW4ocmVzcG9uc2VzKSwgbGVuKHNvdXJjZV9pZHMpCgoKZGVmIGRvd25sb2FkX3JhZ3RydXRoX29mZmljaWFsKGZvcmNlOiBib29sID0gRmFsc2UpIC0+IGRpY3Q6CiAgICAiIiJEb3dubG9hZCB0aGUgb2ZmaWNpYWwgZmlsZXMgaWYgbWlzc2luZzsgcmV0dXJucyB7bmFtZTogcGF0aH0uIiIiCiAgICBvcy5tYWtlZGlycyhPVVRfRElSLCBleGlzdF9vaz1UcnVlKQogICAgcGF0aHMgPSB7fQogICAgbWlzc2luZyA9IFtuYW1lIGZvciBuYW1lIGluIEZJTEVTIGlmIG5vdCAoT1VUX0RJUiAvIG5hbWUpLmV4aXN0cygpXQoKICAgIGlmIG1pc3Npbmcgb3IgZm9yY2U6CiAgICAgICAgaWYgZm9yY2UgYW5kIG5vdCBtaXNzaW5nOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiZm9yY2U9VHJ1ZTogcmUtZG93bmxvYWRpbmcgb2ZmaWNpYWwgUkFHVHJ1dGggZmlsZXMiKQogICAgICAgIGNvbW1pdF9zaGEgPSBfZmV0Y2hfY29tbWl0X3NoYSgpCiAgICAgICAgZm9yIG5hbWUsIHJlcG9fcGF0aCBpbiBGSUxFUy5pdGVtcygpOgogICAgICAgICAgICB1cmwgPSBmIntSQVdfQkFTRX0ve3JlcG9fcGF0aH0iCiAgICAgICAgICAgIGRlc3QgPSBPVVRfRElSIC8gbmFtZQogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIiAge3VybH0iKQogICAgICAgICAgICB1cmxsaWIucmVxdWVzdC51cmxyZXRyaWV2ZSh1cmwsIGRlc3QpCiAgICAgICAgICAgIHNpemVfbWIgPSBkZXN0LnN0YXQoKS5zdF9zaXplIC8gKDEwMjQgKiAxMDI0KQogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIiAgc2F2ZWQge2Rlc3QubmFtZX0gKHtzaXplX21iOi4yZn0gTUIpIikKICAgICAgICBfdmFsaWRhdGVfb2ZmaWNpYWwoKQogICAgICAgIGhhc2hlcyA9IHtuYW1lOiB7InNoYTI1NiI6IF9zaGEyNTYoT1VUX0RJUiAvIG5hbWUpLCAiYnl0ZXMiOiAoT1VUX0RJUiAvIG5hbWUpLnN0YXQoKS5zdF9zaXplfSBmb3IgbmFtZSBpbiBGSUxFU30KICAgICAgICAoT1VUX0RJUiAvICJyZXZpc2lvbi5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICAgICAganNvbi5kdW1wcygKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAicmVwbyI6IFJFUE8sCiAgICAgICAgICAgICAgICAgICAgImJyYW5jaCI6IEJSQU5DSCwKICAgICAgICAgICAgICAgICAgICAiY29tbWl0X3NoYSI6IGNvbW1pdF9zaGEsCiAgICAgICAgICAgICAgICAgICAgImZldGNoZWRfYXRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSJzZWNvbmRzIiksCiAgICAgICAgICAgICAgICAgICAgImZpbGVzIjogaGFzaGVzLAogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgIGluZGVudD0yLAogICAgICAgICAgICApLAogICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICkKICAgICAgICBsb2dnZXIuaW5mbyhmIlJldmlzaW9uICsgaGFzaGVzIHdyaXR0ZW4gdG8ge09VVF9ESVIgLyAncmV2aXNpb24uanNvbid9IikKICAgIGVsc2U6CiAgICAgICAgbG9nZ2VyLmluZm8oIk9mZmljaWFsIFJBR1RydXRoIGZpbGVzIGFscmVhZHkgcHJlc2VudCwgc2tpcHBpbmcuIikKCiAgICBmb3IgbmFtZSBpbiBGSUxFUzoKICAgICAgICBwYXRoc1tuYW1lXSA9IHN0cihPVVRfRElSIC8gbmFtZSkKICAgIHJldHVybiBwYXRocwoKCmRlZiBsb2FkX3JhZ3RydXRoX29mZmljaWFsKCk6CiAgICAiIiJSZXR1cm5zIChyZXNwb25zZXM6IGxpc3RbZGljdF0sIHNvdXJjZXM6IGRpY3Rbc291cmNlX2lkIC0+IGRpY3RdKS4iIiIKICAgIHBhdGhzID0gZG93bmxvYWRfcmFndHJ1dGhfb2ZmaWNpYWwoKQogICAgcmVzcG9uc2VzID0gW10KICAgIHdpdGggb3BlbihwYXRoc1sicmVzcG9uc2UuanNvbmwiXSwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGZvciBsaW5lIGluIGY6CiAgICAgICAgICAgIGlmIGxpbmUuc3RyaXAoKToKICAgICAgICAgICAgICAgIHJlc3BvbnNlcy5hcHBlbmQoanNvbi5sb2FkcyhsaW5lKSkKICAgIHNvdXJjZXMgPSB7fQogICAgd2l0aCBvcGVuKHBhdGhzWyJzb3VyY2VfaW5mby5qc29ubCJdLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAgc3JjID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgc291cmNlc1tzdHIoc3JjWyJzb3VyY2VfaWQiXSldID0gc3JjCiAgICByZXR1cm4gcmVzcG9uc2VzLCBzb3VyY2VzCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGRvd25sb2FkX3JhZ3RydXRoX29mZmljaWFsKCkK",
 "src/data/mappings.py": "IiIiClB1cmUgbGFiZWwtbWFwcGluZyBmdW5jdGlvbnMgZm9yIHRoZSBCMSB1bmlmaWVkIHNjaGVtYSAocm9hZG1hcCDCpzE0IEIxLjEvQjEuNCkuCgpFdmVyeSBtYXBwaW5nIGlzIGEgZGV0ZXJtaW5pc3RpYywgdW5pdC10ZXN0ZWQgZnVuY3Rpb24uIFRoZSBiaW5hcnkgYGxhYmVsYAppcyBhIGRvY3VtZW50ZWQgaW50ZXJmYWNlOyBvcmlnaW5hbCB0YXhvbm9teSAoUkFHVHJ1dGggc3BhbnMsIEZhaXRoQmVuY2gKYW5ub3RhdGlvbiBsYWJlbHMpIGlzIHByZXNlcnZlZCBsb3NzbGVzc2x5IGluIGBzcGFuX2Fubm90YXRpb25zYC4KCk1hcHBpbmdzOgogIC0gSGFsdUV2YWw6ICAgIGNvcnJlY3QgYW5zd2VyIC0+IDAsIGhhbGx1Y2luYXRlZCBhbnN3ZXIgLT4gMSAoZml4ZWQgYXQgYnVpbGQpLgogIC0gUkFHVHJ1dGg6ICAgIDEgaWYgdGhlIHJlc3BvbnNlIGNhcnJpZXMgYW55IGhhbGx1Y2luYXRpb24gc3BhbiwgZWxzZSAwCiAgICAgICAgICAgICAgICAgKG9mZmljaWFsIHNwYW5zIGFyZSB0aGUgaHVtYW4gYW5ub3RhdGlvbiBvZiBoYWxsdWNpbmF0aW9uKS4KICAtIEZhaXRoQmVuY2g6ICBzZXZlcml0eSBhZ2dyZWdhdGlvbiBvdmVyIGFubm90YXRpb24gbGFiZWxzOgogICAgICAgICAgICAgICAgICAgbm8gYW5ub3RhdGlvbnMgLT4gMAogICAgICAgICAgICAgICAgICAgQmVuaWduIC0+IDAKICAgICAgICAgICAgICAgICAgIFF1ZXN0aW9uYWJsZSAtPiAxIChwcmltYXJ5IG1hcHBpbmcpCiAgICAgICAgICAgICAgICAgICBVbndhbnRlZCAvIFVud2FudGVkLkludHJpbnNpYyAvIFVud2FudGVkLkV4dHJpbnNpYyAtPiAxCiAgICAgICAgICAgICAgICAgUHJpbWFyeSBhZ2dyZWdhdGlvbiA9IHdvcnN0IHNldmVyaXR5IChvZmZpY2lhbCBzY3JpcHQgZGVmYXVsdCkuCiIiIgoKUkFHVFJVVEhfTEFCRUxfTUFQUElORyA9ICJyYWd0cnV0aC1zcGFuLXYxIgpGQUlUSEJFTkNIX0xBQkVMX01BUFBJTkdfUFJJTUFSWSA9ICJmYWl0aGJlbmNoLXdvcnN0LXErdW53YW50ZWQtdjEiCgojIFNldmVyaXR5IHVzZWQgYnkgdGhlIG9mZmljaWFsIEZhaXRoQmVuY2ggYmluYXJpemUucHkgKDEgPSBsZWFzdCwgMyA9IG1vc3Qgc2V2ZXJlKQpGQUlUSEJFTkNIX1NFVkVSSVRZID0gewogICAgIkJlbmlnbiI6IDEsCiAgICAiUXVlc3Rpb25hYmxlIjogMiwKICAgICJVbndhbnRlZCI6IDMsCiAgICAiVW53YW50ZWQuSW50cmluc2ljIjogMywKICAgICJVbndhbnRlZC5FeHRyaW5zaWMiOiAzLAp9CgojIFRoZSBvZmZpY2lhbCBSRUFETUUgZXhhbXBsZSBjb250YWlucyBhIHR5cG8gKCJJbnN0cmluc2ljIik7IHRoZSBzY2hlbWEgYW5kCiMgcmVsZWFzZWQgZGF0YSB1c2UgIkludHJpbnNpYyIuIE5vcm1hbGl6ZSBzbyBib3RoIHNwZWxsaW5ncyBtYXAgaWRlbnRpY2FsbHkuCl9GQUlUSEJFTkNIX1RZUE9fTUFQID0geyJVbndhbnRlZC5JbnN0cmluc2ljIjogIlVud2FudGVkLkludHJpbnNpYyJ9CgpGQUlUSEJFTkNIX1BSSU1BUllfQ0xBU1NFUyA9IGZyb3plbnNldCgKICAgIHsiUXVlc3Rpb25hYmxlIiwgIlVud2FudGVkIiwgIlVud2FudGVkLkludHJpbnNpYyIsICJVbndhbnRlZC5FeHRyaW5zaWMifQopCkZBSVRIQkVOQ0hfU1RSSUNUX0NMQVNTRVMgPSBmcm96ZW5zZXQoeyJVbndhbnRlZCIsICJVbndhbnRlZC5JbnRyaW5zaWMiLCAiVW53YW50ZWQuRXh0cmluc2ljIn0pCgojIFNlbnNpdGl2aXR5IGNvbmZpZ3VyYXRpb25zIHJlcG9ydGVkIGluIHRoZSBtYXBwaW5nIHJlcG9ydCAoQjEuNikKRkFJVEhCRU5DSF9TRU5TSVRJVklUWV9DT05GSUdTID0gewogICAgInByaW1hcnlfd29yc3RfcV9wbHVzX3Vud2FudGVkIjogewogICAgICAgICJhZ2dyZWdhdGlvbiI6ICJ3b3JzdCIsCiAgICAgICAgImhhbGx1Y2luYXRlZF9jbGFzc2VzIjogRkFJVEhCRU5DSF9QUklNQVJZX0NMQVNTRVMsCiAgICB9LAogICAgIm1ham9yaXR5X3FfcGx1c191bndhbnRlZCI6IHsKICAgICAgICAiYWdncmVnYXRpb24iOiAibWFqb3JpdHkiLAogICAgICAgICJoYWxsdWNpbmF0ZWRfY2xhc3NlcyI6IEZBSVRIQkVOQ0hfUFJJTUFSWV9DTEFTU0VTLAogICAgfSwKICAgICJzdHJpY3Rfd29yc3RfdW53YW50ZWRfb25seSI6IHsKICAgICAgICAiYWdncmVnYXRpb24iOiAid29yc3QiLAogICAgICAgICJoYWxsdWNpbmF0ZWRfY2xhc3NlcyI6IEZBSVRIQkVOQ0hfU1RSSUNUX0NMQVNTRVMsCiAgICB9LAp9CgoKZGVmIG5vcm1hbGl6ZV9mYWl0aGJlbmNoX2xhYmVsKGxhYmVsOiBzdHIpIC0+IHN0cjoKICAgIHJldHVybiBfRkFJVEhCRU5DSF9UWVBPX01BUC5nZXQobGFiZWwsIGxhYmVsKQoKCmRlZiByYWd0cnV0aF9sYWJlbF9mcm9tX3NwYW5zKHNwYW5zKSAtPiBpbnQ6CiAgICAiIiJBbnkgaHVtYW4tYW5ub3RhdGVkIGhhbGx1Y2luYXRpb24gc3BhbiA9PiAxOyBlbXB0eSBhbm5vdGF0aW9uIGxpc3QgPT4gMC4iIiIKICAgIHJldHVybiAxIGlmIHNwYW5zIGVsc2UgMAoKCmRlZiBmYWl0aGJlbmNoX2Fubm90YXRpb25fbGFiZWxzKGFubm90YXRpb25zKSAtPiBzZXQ6CiAgICAiIiJTZXQgb2Ygbm9ybWFsaXplZCBsYWJlbCBzdHJpbmdzIGFjcm9zcyBhbGwgYW5ub3RhdGlvbnMgb2YgYSBzYW1wbGUuIiIiCiAgICBsYWJlbHMgPSBzZXQoKQogICAgZm9yIGFubiBpbiBhbm5vdGF0aW9ucyBvciBbXToKICAgICAgICBmb3IgbGFiIGluIGFubi5nZXQoImxhYmVsIikgb3IgW106CiAgICAgICAgICAgIGxhYmVscy5hZGQobm9ybWFsaXplX2ZhaXRoYmVuY2hfbGFiZWwobGFiKSkKICAgIHJldHVybiBsYWJlbHMKCgpkZWYgZmFpdGhiZW5jaF9zZXZlcml0eShsYWJlbDogc3RyKSAtPiBpbnQ6CiAgICBub3JtYWxpemVkID0gbm9ybWFsaXplX2ZhaXRoYmVuY2hfbGFiZWwobGFiZWwpCiAgICBpZiBub3JtYWxpemVkIG5vdCBpbiBGQUlUSEJFTkNIX1NFVkVSSVRZOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmtub3duIEZhaXRoQmVuY2ggbGFiZWw6IHtsYWJlbCFyfSIpCiAgICByZXR1cm4gRkFJVEhCRU5DSF9TRVZFUklUWVtub3JtYWxpemVkXQoKCmRlZiBmYWl0aGJlbmNoX2xhYmVsKAogICAgYW5ub3RhdGlvbnMsCiAgICBhZ2dyZWdhdGlvbjogc3RyID0gIndvcnN0IiwKICAgIGhhbGx1Y2luYXRlZF9jbGFzc2VzPUZBSVRIQkVOQ0hfUFJJTUFSWV9DTEFTU0VTLAopIC0+IGludDoKICAgICIiIkFnZ3JlZ2F0ZSBhbm5vdGF0aW9uIGxhYmVscyB0byBhIGJpbmFyeSBsYWJlbC4KCiAgICBhZ2dyZWdhdGlvbj0id29yc3QiOiBtb3N0IHNldmVyZSBsYWJlbCB3aW5zIChvZmZpY2lhbCBzY3JpcHQgZGVmYXVsdCkuCiAgICBhZ2dyZWdhdGlvbj0ibWFqb3JpdHkiOiBtb3N0IGZyZXF1ZW50IHNldmVyaXR5IHdpbnMgKHRpZXMgLT4gbW9zdCBzZXZlcmUpLgogICAgIiIiCiAgICBsYWJlbHMgPSBmYWl0aGJlbmNoX2Fubm90YXRpb25fbGFiZWxzKGFubm90YXRpb25zKQogICAgaWYgbm90IGxhYmVsczoKICAgICAgICByZXR1cm4gMAogICAgc2V2ZXJpdGllcyA9IFtmYWl0aGJlbmNoX3NldmVyaXR5KGwpIGZvciBsIGluIGxhYmVsc10KICAgIGlmIGFnZ3JlZ2F0aW9uID09ICJ3b3JzdCI6CiAgICAgICAgY2hvc2VuX3NldiA9IG1heChzZXZlcml0aWVzKQogICAgZWxpZiBhZ2dyZWdhdGlvbiA9PSAibWFqb3JpdHkiOgogICAgICAgIGNob3Nlbl9zZXYgPSBtYXgoc2V0KHNldmVyaXRpZXMpLCBrZXk9c2V2ZXJpdGllcy5jb3VudCkKICAgIGVsc2U6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gYWdncmVnYXRpb24gc3RyYXRlZ3k6IHthZ2dyZWdhdGlvbiFyfSIpCiAgICBjaG9zZW4gPSBuZXh0KGwgZm9yIGwgaW4gc2V0KGxhYmVscykgaWYgZmFpdGhiZW5jaF9zZXZlcml0eShsKSA9PSBjaG9zZW5fc2V2KQogICAgcmV0dXJuIDEgaWYgY2hvc2VuIGluIGhhbGx1Y2luYXRlZF9jbGFzc2VzIGVsc2UgMAoKCmRlZiBmYWl0aGJlbmNoX2xhYmVsX3NlbnNpdGl2aXR5KGFubm90YXRpb25zKSAtPiBkaWN0OgogICAgIiIiQWxsIGNvbmZpZ3VyZWQgRmFpdGhCZW5jaCBsYWJlbGluZ3MgZm9yIHRoZSBzZW5zaXRpdml0eSByZXBvcnQuIiIiCiAgICByZXR1cm4gewogICAgICAgIG5hbWU6IGludChmYWl0aGJlbmNoX2xhYmVsKGFubm90YXRpb25zLCAqKmNmZykpCiAgICAgICAgZm9yIG5hbWUsIGNmZyBpbiBGQUlUSEJFTkNIX1NFTlNJVElWSVRZX0NPTkZJR1MuaXRlbXMoKQogICAgfQo=",
 "src/data/prepare.py": "IiIiCkRhdGEgcHJlcGFyYXRpb24gc2NyaXB0IGZvciBIYWx1UklTQy4KUGFyc2VzIHFhX2RhdGEuanNvbiAoSlNPTkwpIGludG8gYSBiaW5hcnkgY2xhc3NpZmljYXRpb24gZGF0YXNldCAodHdvIHJvd3MgcGVyIGVudHJ5OiBjb3JyZWN0ICYgaGFsbHVjaW5hdGVkKSwKcGVyZm9ybXMgYSBHUk9VUC1BV0FSRSB0cmFpbi92YWwvdGVzdCBzcGxpdCAoNzAvMTUvMTUpIHNvIHRoYXQgYm90aCBhbnN3ZXIgdmFyaWFudHMgb2Ygb25lIG9yaWdpbmFsIHF1ZXN0aW9uCnN0YXkgaW4gdGhlIHNhbWUgcGFydGl0aW9uLCBzYXZlcyBzcGxpdCBpbmRpY2VzLCBhIGxlYWthZ2UgcmVwb3J0LCBhbmQgYW4gYXV0by1zYW1wbGVkIGF1ZGl0IGZpbGUuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQKCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikKClJBV19EQVRBX1BBVEggPSBvcy5wYXRoLmpvaW4oImRhdGEiLCAicmF3IiwgImhhbHVldmFsIiwgInFhX2RhdGEuanNvbiIpClBST0NFU1NFRF9ESVIgPSBvcy5wYXRoLmpvaW4oImRhdGEiLCAicHJvY2Vzc2VkIikKQVJUSUZBQ1RTX0RJUiA9IG9zLnBhdGguam9pbigiYXJ0aWZhY3RzIikKUFJPQ0VTU0VEX1BBUlFVRVQgPSBvcy5wYXRoLmpvaW4oUFJPQ0VTU0VEX0RJUiwgInFhX2NsZWFuLnBhcnF1ZXQiKQpBVURJVF9KU09OID0gb3MucGF0aC5qb2luKFBST0NFU1NFRF9ESVIsICJhdWRpdF81MF9zYW1wbGVzLmpzb24iKQpTUExJVF9JTkRJQ0VTX05QWSA9IG9zLnBhdGguam9pbihBUlRJRkFDVFNfRElSLCAic3BsaXRfaW5kaWNlcy5ucHkiKQpTUExJVF9JTkRJQ0VTX0pTT04gPSBvcy5wYXRoLmpvaW4oQVJUSUZBQ1RTX0RJUiwgInNwbGl0X2luZGljZXMuanNvbiIpClNQTElUX1JFUE9SVF9KU09OID0gb3MucGF0aC5qb2luKEFSVElGQUNUU19ESVIsICJzcGxpdF9pbnRlZ3JpdHlfcmVwb3J0Lmpzb24iKQoKU1BMSVRfU0VFRCA9IDQyCgoKZGVmIGxvYWRfYW5kX3BhcnNlX3Jhd19kYXRhKHJhd19wYXRoOiBzdHIgPSBSQVdfREFUQV9QQVRIKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJMb2FkcyBIYWx1RXZhbCBRQSBKU09OTCBhbmQgZXhwYW5kcyBpbnRvIDIgcm93cyBwZXIgcXVlc3Rpb24gKGNvcnJlY3Q9MCwgaGFsbHVjaW5hdGVkPTEpLiIiIgogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKHJhd19wYXRoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIlJhdyBkYXRhIG5vdCBmb3VuZCBhdCB7cmF3X3BhdGh9LiBSdW4gc3JjL2RhdGEvZG93bmxvYWQucHkgZmlyc3QuIikKCiAgICByYXdfaXRlbXMgPSBbXQogICAgd2l0aCBvcGVuKHJhd19wYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICAgICAgcmF3X2l0ZW1zLmFwcGVuZChqc29uLmxvYWRzKGxpbmUpKQoKICAgIGxvZ2dpbmcuaW5mbyhmIkxvYWRlZCB7bGVuKHJhd19pdGVtcyl9IHJhdyBRQSBpdGVtcy4iKQogICAgcm93cyA9IFtdCgogICAgZm9yIGlkeCwgaXRlbSBpbiBlbnVtZXJhdGUocmF3X2l0ZW1zKToKICAgICAgICBxdWVzdGlvbiA9IGl0ZW0uZ2V0KCJxdWVzdGlvbiIsICIiKS5zdHJpcCgpCiAgICAgICAgY29udGV4dCA9IGl0ZW0uZ2V0KCJrbm93bGVkZ2UiLCAiIikuc3RyaXAoKQogICAgICAgIHJpZ2h0X2FucyA9IGl0ZW0uZ2V0KCJyaWdodF9hbnN3ZXIiLCBpdGVtLmdldCgiYW5zd2VyIiwgIiIpKS5zdHJpcCgpCiAgICAgICAgaGFsbHVjaW5hdGVkX2FucyA9IGl0ZW0uZ2V0KCJoYWxsdWNpbmF0ZWRfYW5zd2VyIiwgIiIpLnN0cmlwKCkKCiAgICAgICAgaWYgbm90IHF1ZXN0aW9uIG9yIG5vdCByaWdodF9hbnM6CiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICMgQ29ycmVjdCBzYW1wbGUgKGxhYmVsID0gMCkKICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICJzYW1wbGVfaWQiOiBmInFfe2lkeH1fY29ycmVjdCIsCiAgICAgICAgICAgICJpdGVtX2lkeCI6IGlkeCwKICAgICAgICAgICAgInF1ZXN0aW9uIjogcXVlc3Rpb24sCiAgICAgICAgICAgICJjb250ZXh0IjogY29udGV4dCwKICAgICAgICAgICAgImFuc3dlciI6IHJpZ2h0X2FucywKICAgICAgICAgICAgImxhYmVsIjogMAogICAgICAgIH0pCgogICAgICAgICMgSGFsbHVjaW5hdGVkIHNhbXBsZSAobGFiZWwgPSAxKQogICAgICAgIGlmIGhhbGx1Y2luYXRlZF9hbnM6CiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsKICAgICAgICAgICAgICAgICJzYW1wbGVfaWQiOiBmInFfe2lkeH1faGFsbHVjaW5hdGVkIiwKICAgICAgICAgICAgICAgICJpdGVtX2lkeCI6IGlkeCwKICAgICAgICAgICAgICAgICJxdWVzdGlvbiI6IHF1ZXN0aW9uLAogICAgICAgICAgICAgICAgImNvbnRleHQiOiBjb250ZXh0LAogICAgICAgICAgICAgICAgImFuc3dlciI6IGhhbGx1Y2luYXRlZF9hbnMsCiAgICAgICAgICAgICAgICAibGFiZWwiOiAxCiAgICAgICAgICAgIH0pCgogICAgZGYgPSBwZC5EYXRhRnJhbWUocm93cykKICAgIGxvZ2dpbmcuaW5mbyhmIkNyZWF0ZWQgZGF0YXNldCB3aXRoIHtsZW4oZGYpfSByb3dzICh7ZGZbJ2xhYmVsJ10udmFsdWVfY291bnRzKCkudG9fZGljdCgpfSkuIikKICAgIHJldHVybiBkZgoKCmRlZiBidWlsZF9pbnRlZ3JpdHlfcmVwb3J0KGRmOiBwZC5EYXRhRnJhbWUpIC0+IGRpY3Q6CiAgICAiIiJMZWFrYWdlIHJlcG9ydDogZXZlcnkgaXRlbV9pZHggKHNvdXJjZSBxdWVzdGlvbikgbXVzdCBtYXAgdG8gZXhhY3RseSBvbmUgc3BsaXQuIiIiCiAgICBwZXIgPSBkZi5ncm91cGJ5KCJpdGVtX2lkeCIpWyJzcGxpdCJdLm51bmlxdWUoKQogICAgY3Jvc3MgPSBpbnQoKHBlciA+IDEpLnN1bSgpKQogICAgcmVwb3J0ID0gewogICAgICAgICJzcGxpdCI6ICJncm91cF9ieV9pdGVtX2lkeCIsCiAgICAgICAgInNlZWQiOiBTUExJVF9TRUVELAogICAgICAgICJuX2dyb3Vwc190b3RhbCI6IGludChwZXIuc2l6ZSksCiAgICAgICAgIm5fZ3JvdXBzX3Blcl9zcGxpdCI6IHtzdHIoayk6IGludCh2KSBmb3IgaywgdiBpbiBkZi5ncm91cGJ5KCJzcGxpdCIpWyJpdGVtX2lkeCJdLm51bmlxdWUoKS50b19kaWN0KCkuaXRlbXMoKX0sCiAgICAgICAgIm5fcm93c19wZXJfc3BsaXQiOiB7c3RyKGspOiBpbnQodikgZm9yIGssIHYgaW4gZGYuZ3JvdXBieSgic3BsaXQiKS5zaXplKCkudG9fZGljdCgpLml0ZW1zKCl9LAogICAgICAgICJsYWJlbF9tZWFuX3Blcl9zcGxpdCI6IHtzdHIoayk6IHJvdW5kKGZsb2F0KHYpLCA0KSBmb3IgaywgdiBpbiBkZi5ncm91cGJ5KCJzcGxpdCIpWyJsYWJlbCJdLm1lYW4oKS50b19kaWN0KCkuaXRlbXMoKX0sCiAgICAgICAgImdyb3Vwc19zcGFubmluZ19tdWx0aXBsZV9zcGxpdHMiOiBjcm9zcywKICAgICAgICAibGVha2FnZV9mcmVlIjogY3Jvc3MgPT0gMCwKICAgIH0KICAgIGlmIGNyb3NzID4gMDoKICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcihmIkdyb3VwIGxlYWthZ2UgZGV0ZWN0ZWQ6IHtjcm9zc30gaXRlbV9pZHggdmFsdWVzIHNwYW4gbXVsdGlwbGUgc3BsaXRzIikKICAgIHJldHVybiByZXBvcnQKCgpkZWYgZ3JvdXBfc3BsaXRfYnlfaXRlbShkZjogcGQuRGF0YUZyYW1lLCB0ZXN0X3NpemU6IGZsb2F0ID0gMC4zMCwgdmFsX3NoYXJlOiBmbG9hdCA9IDAuNSwgc2VlZDogaW50ID0gU1BMSVRfU0VFRCk6CiAgICAiIiJHcm91cC1hd2FyZSA3MC8xNS8xNSBzcGxpdDogZWFjaCBvcmlnaW5hbCBxdWVzdGlvbiAoaXRlbV9pZHgpIGdvZXMgdG8gZXhhY3RseSBvbmUgcGFydGl0aW9uLgoKICAgIFJldHVybnMgKGRmX3dpdGhfc3BsaXRfY29sdW1uLCBpbnRlZ3JpdHlfcmVwb3J0KS4KICAgICIiIgogICAgZ3JwID0gKAogICAgICAgIGRmLmdyb3VwYnkoIml0ZW1faWR4Iiwgc29ydD1GYWxzZSkKICAgICAgICAuYWdnKGxhYmVsPSgibGFiZWwiLCAibWF4IiksIG5fcm93cz0oImxhYmVsIiwgInNpemUiKSkKICAgICAgICAucmVzZXRfaW5kZXgoKQogICAgKQogICAgc3RyYXQgPSBncnBbImxhYmVsIl0gaWYgZ3JwWyJsYWJlbCJdLm51bmlxdWUoKSA+IDEgZWxzZSBOb25lCiAgICB0cmFpbl9nLCB0ZW1wX2cgPSB0cmFpbl90ZXN0X3NwbGl0KGdycCwgdGVzdF9zaXplPXRlc3Rfc2l6ZSwgcmFuZG9tX3N0YXRlPXNlZWQsIHN0cmF0aWZ5PXN0cmF0KQogICAgc3RyYXRfdCA9IHRlbXBfZ1sibGFiZWwiXSBpZiB0ZW1wX2dbImxhYmVsIl0ubnVuaXF1ZSgpID4gMSBlbHNlIE5vbmUKICAgIHZhbF9nLCB0ZXN0X2cgPSB0cmFpbl90ZXN0X3NwbGl0KHRlbXBfZywgdGVzdF9zaXplPXZhbF9zaGFyZSwgcmFuZG9tX3N0YXRlPXNlZWQsIHN0cmF0aWZ5PXN0cmF0X3QpCgogICAgc3BsaXRfb2Y6IGRpY3QgPSB7fQogICAgZm9yIGcsIG5hbWUgaW4gKCh0cmFpbl9nLCAidHJhaW4iKSwgKHZhbF9nLCAidmFsIiksICh0ZXN0X2csICJ0ZXN0IikpOgogICAgICAgIGZvciBpIGluIGdbIml0ZW1faWR4Il06CiAgICAgICAgICAgIHNwbGl0X29mW2ludChpKV0gPSBuYW1lCgogICAgb3V0ID0gZGYuY29weSgpCiAgICBvdXRbInNwbGl0Il0gPSBvdXRbIml0ZW1faWR4Il0ubWFwKHNwbGl0X29mKQogICAgcmVwb3J0ID0gYnVpbGRfaW50ZWdyaXR5X3JlcG9ydChvdXQpCiAgICBsb2dnaW5nLmluZm8oCiAgICAgICAgZiJHcm91cCBzcGxpdDogdHJhaW4ge2xlbih0cmFpbl9nKX0gLyB2YWwge2xlbih2YWxfZyl9IC8gdGVzdCB7bGVuKHRlc3RfZyl9IGdyb3VwczsgIgogICAgICAgIGYibGVha2FnZV9mcmVlPXtyZXBvcnRbJ2xlYWthZ2VfZnJlZSddfSIKICAgICkKICAgIHJldHVybiBvdXQsIHJlcG9ydAoKCmRlZiBwcmVwYXJlX3NwbGl0c19hbmRfc2F2ZShkZjogcGQuRGF0YUZyYW1lKToKICAgICIiIlBlcmZvcm1zIHRoZSBncm91cC1hd2FyZSB0cmFpbi92YWwvdGVzdCBzcGxpdCBhbmQgc2F2ZXMgYWxsIGFydGlmYWN0cy4iIiIKICAgIG9zLm1ha2VkaXJzKFBST0NFU1NFRF9ESVIsIGV4aXN0X29rPVRydWUpCiAgICBvcy5tYWtlZGlycyhBUlRJRkFDVFNfRElSLCBleGlzdF9vaz1UcnVlKQoKICAgIGRmLCByZXBvcnQgPSBncm91cF9zcGxpdF9ieV9pdGVtKGRmLCBzZWVkPVNQTElUX1NFRUQpCgogICAgIyBTYXZlIHByb2Nlc3NlZCBkYXRhc2V0IHRvIFBhcnF1ZXQKICAgIGRmLnRvX3BhcnF1ZXQoUFJPQ0VTU0VEX1BBUlFVRVQsIGluZGV4PUZhbHNlKQogICAgbG9nZ2luZy5pbmZvKGYiU2F2ZWQgcHJvY2Vzc2VkIGRhdGFzZXQgdG8ge1BST0NFU1NFRF9QQVJRVUVUfSIpCgogICAgIyBTYXZlIHNwbGl0IGluZGljZXMgKyBncm91cCBpZHMgKGV4YWN0IHJlcHJvZHVjaWJpbGl0eSkKICAgIHNwbGl0X2luZGljZXMgPSB7CiAgICAgICAgInNwbGl0IjogImdyb3VwX2J5X2l0ZW1faWR4IiwKICAgICAgICAidHJhaW4iOiBkZi5pbmRleFtkZlsic3BsaXQiXSA9PSAidHJhaW4iXS50b2xpc3QoKSwKICAgICAgICAidmFsIjogZGYuaW5kZXhbZGZbInNwbGl0Il0gPT0gInZhbCJdLnRvbGlzdCgpLAogICAgICAgICJ0ZXN0IjogZGYuaW5kZXhbZGZbInNwbGl0Il0gPT0gInRlc3QiXS50b2xpc3QoKSwKICAgICAgICAiZ3JvdXBfaWRzIjogewogICAgICAgICAgICAidHJhaW4iOiBzb3J0ZWQoZGYubG9jW2RmWyJzcGxpdCJdID09ICJ0cmFpbiIsICJpdGVtX2lkeCJdLnVuaXF1ZSgpLnRvbGlzdCgpKSwKICAgICAgICAgICAgInZhbCI6IHNvcnRlZChkZi5sb2NbZGZbInNwbGl0Il0gPT0gInZhbCIsICJpdGVtX2lkeCJdLnVuaXF1ZSgpLnRvbGlzdCgpKSwKICAgICAgICAgICAgInRlc3QiOiBzb3J0ZWQoZGYubG9jW2RmWyJzcGxpdCJdID09ICJ0ZXN0IiwgIml0ZW1faWR4Il0udW5pcXVlKCkudG9saXN0KCkpLAogICAgICAgIH0sCiAgICAgICAgInNlZWQiOiBTUExJVF9TRUVELAogICAgICAgICJuX3RyYWluIjogaW50KChkZlsic3BsaXQiXSA9PSAidHJhaW4iKS5zdW0oKSksCiAgICAgICAgIm5fdmFsIjogaW50KChkZlsic3BsaXQiXSA9PSAidmFsIikuc3VtKCkpLAogICAgICAgICJuX3Rlc3QiOiBpbnQoKGRmWyJzcGxpdCJdID09ICJ0ZXN0Iikuc3VtKCkpLAogICAgfQoKICAgIHdpdGggb3BlbihTUExJVF9JTkRJQ0VTX0pTT04sICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoc3BsaXRfaW5kaWNlcywgZiwgaW5kZW50PTIpCgogICAgbnAuc2F2ZShTUExJVF9JTkRJQ0VTX05QWSwgc3BsaXRfaW5kaWNlcywgYWxsb3dfcGlja2xlPVRydWUpCiAgICBsb2dnaW5nLmluZm8oZiJTYXZlZCBzcGxpdCBpbmRpY2VzIHRvIHtTUExJVF9JTkRJQ0VTX0pTT059IGFuZCB7U1BMSVRfSU5ESUNFU19OUFl9IikKCiAgICB3aXRoIG9wZW4oU1BMSVRfUkVQT1JUX0pTT04sICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAocmVwb3J0LCBmLCBpbmRlbnQ9MikKICAgIGxvZ2dpbmcuaW5mbyhmIlNhdmVkIHNwbGl0IGludGVncml0eSByZXBvcnQgdG8ge1NQTElUX1JFUE9SVF9KU09OfSIpCgogICAgIyBTYW1wbGUgNTAgcm93cyBmb3IgdGhlIE1BTlVBTCBhdWRpdCAobGFiZWxzIG11c3QgYmUgcmV2aWV3ZWQgYnkgYSBodW1hbiBiZWZvcmUgdGhlIHBhcGVyKQogICAgYXVkaXRfc2FtcGxlcyA9IGRmLnNhbXBsZShuPW1pbig1MCwgbGVuKGRmKSksIHJhbmRvbV9zdGF0ZT1TUExJVF9TRUVEKS50b19kaWN0KG9yaWVudD0icmVjb3JkcyIpCiAgICB3aXRoIG9wZW4oQVVESVRfSlNPTiwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChhdWRpdF9zYW1wbGVzLCBmLCBpbmRlbnQ9MikKICAgIGxvZ2dpbmcuaW5mbyhmIlNhdmVkIDUwIGF1dG8tc2FtcGxlZCBhdWRpdCByb3dzIHRvIHtBVURJVF9KU09OfSAobWFudWFsIHJldmlldyByZXF1aXJlZCkiKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBkZiA9IGxvYWRfYW5kX3BhcnNlX3Jhd19kYXRhKCkKICAgIHByZXBhcmVfc3BsaXRzX2FuZF9zYXZlKGRmKQo=",
 "src/data/prepare_unified.py": "...",
 "src/data/registry.py": "IiIiCkIxIGRhdGFzZXQgcmVnaXN0cnk6IGxpY2Vuc2UsIHByb3ZlbmFuY2UsIGdyb3VwaW5nIHJ1bGVzLCBhbmQgbGFiZWwgcnVsZXMKKHJvYWRtYXAgwqcxNCBCMS42L0IxLjcpLgoKUmVzdHJpY3RlZCBkYXRhc2V0cyAoRmFpdGhCZW5jaCwgQ0MgQlktTkMtU0EpIGFyZSBORVZFUiBidW5kbGVkIGluIHRoZQpyZXBvc2l0b3J5OiByYXcgZmlsZXMgc3RheSB1bmRlciBnaXRpZ25vcmVkIGBkYXRhL3Jhdy9gLCBhbmQgb25seSBkb3dubG9hZAppbnN0cnVjdGlvbnMsIGNpdGF0aW9ucywgaGFzaGVzLCBhbmQgbGljZW5zZSBub3RlcyBhcmUgc2hpcHBlZC4KIiIiCgppbXBvcnQganNvbgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcwpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBEYXRhc2V0UmVjb3JkOgogICAgbmFtZTogc3RyCiAgICBkaXNwbGF5X25hbWU6IHN0cgogICAgdXJsOiBzdHIKICAgIGxpY2Vuc2U6IHN0cgogICAgcmVkaXN0cmlidXRpb25fYWxsb3dlZDogYm9vbAogICAgZ3JvdXBpbmdfcnVsZTogc3RyCiAgICBsYWJlbF9kZWZpbml0aW9uOiBzdHIKICAgIGxhYmVsX21hcHBpbmdfdmVyc2lvbjogc3RyCiAgICBjaXRhdGlvbjogc3RyCiAgICByYXdfZGlyOiBzdHIgICMgcmVsYXRpdmUgdG8gcmVwbyByb290LCB1bmRlciBkYXRhL3Jhdy8KCgpEQVRBU0VUX1JFR0lTVFJZID0gKAogICAgRGF0YXNldFJlY29yZCgKICAgICAgICBuYW1lPSJoYWx1ZXZhbCIsCiAgICAgICAgZGlzcGxheV9uYW1lPSJIYWx1RXZhbCAoUUEpIiwKICAgICAgICB1cmw9Imh0dHBzOi8vZ2l0aHViLmNvbS9SVUNBSUJveC9IYWx1RXZhbCIsCiAgICAgICAgbGljZW5zZT0iTUlUIiwKICAgICAgICByZWRpc3RyaWJ1dGlvbl9hbGxvd2VkPVRydWUsCiAgICAgICAgZ3JvdXBpbmdfcnVsZT0iZ3JvdXAgYnkgaXRlbV9pZHggKG9yaWdpbmFsIHF1ZXN0aW9uKTsgYm90aCBhbnN3ZXIgdmFyaWFudHMgc3RheSBpbiBvbmUgcGFydGl0aW9uIiwKICAgICAgICBsYWJlbF9kZWZpbml0aW9uPSJjb3JyZWN0IGFuc3dlciAtPiAwOyBoYWxsdWNpbmF0ZWQgYW5zd2VyIC0+IDEiLAogICAgICAgIGxhYmVsX21hcHBpbmdfdmVyc2lvbj0iYjEtbGFiZWxzLXYxIiwKICAgICAgICBjaXRhdGlvbj0iTGkgZXQgYWwuLCBBQ0wgMjAyMywgRE9JIDEwLjE4NjUzL3YxLzIwMjMuZW1ubHAtbWFpbi4zOTciLAogICAgICAgIHJhd19kaXI9ImRhdGEvcmF3L2hhbHVldmFsIiwKICAgICksCiAgICBEYXRhc2V0UmVjb3JkKAogICAgICAgIG5hbWU9InJhZ3RydXRoIiwKICAgICAgICBkaXNwbGF5X25hbWU9IlJBR1RydXRoIChvZmZpY2lhbCkiLAogICAgICAgIHVybD0iaHR0cHM6Ly9naXRodWIuY29tL1BhcnRpY2xlTWVkaWEvUkFHVHJ1dGgiLAogICAgICAgIGxpY2Vuc2U9Ik1JVCIsCiAgICAgICAgcmVkaXN0cmlidXRpb25fYWxsb3dlZD1UcnVlLAogICAgICAgIGdyb3VwaW5nX3J1bGU9Imdyb3VwIGJ5IHNvdXJjZV9pZCAob25lIHNvdXJjZSBlbGljaXRzIHNpeCByZXNwb25zZXMpIiwKICAgICAgICBsYWJlbF9kZWZpbml0aW9uPSJhbnkgaHVtYW4tYW5ub3RhdGVkIGhhbGx1Y2luYXRpb24gc3BhbiAtPiAxOyBubyBzcGFucyAtPiAwIiwKICAgICAgICBsYWJlbF9tYXBwaW5nX3ZlcnNpb249ImIxLWxhYmVscy12MSIsCiAgICAgICAgY2l0YXRpb249Ik5pdSBldCBhbC4sIEFDTCAyMDI0LCBET0kgMTAuMTg2NTMvdjEvMjAyNC5hY2wtbG9uZy41ODUiLAogICAgICAgIHJhd19kaXI9ImRhdGEvcmF3L3JhZ3RydXRoX29mZmljaWFsIiwKICAgICksCiAgICBEYXRhc2V0UmVjb3JkKAogICAgICAgIG5hbWU9ImZhaXRoYmVuY2giLAogICAgICAgIGRpc3BsYXlfbmFtZT0iRmFpdGhCZW5jaCAoc3VtbWFyaXphdGlvbikiLAogICAgICAgIHVybD0iaHR0cHM6Ly9naXRodWIuY29tL3ZlY3RhcmEvRmFpdGhCZW5jaCIsCiAgICAgICAgbGljZW5zZT0iQ0MgQlktTkMtU0EgNC4wIiwKICAgICAgICByZWRpc3RyaWJ1dGlvbl9hbGxvd2VkPUZhbHNlLAogICAgICAgIGdyb3VwaW5nX3J1bGU9Imdyb3VwIGJ5IHJhd19zYW1wbGVfaWQgd2hlbiBhdmFpbGFibGUsIGVsc2Ugc3RhYmxlIGhhc2ggb2Ygc291cmNlIHRleHQiLAogICAgICAgIGxhYmVsX2RlZmluaXRpb249IndvcnN0LXNldmVyaXR5IGFnZ3JlZ2F0aW9uOyBCZW5pZ24vZW1wdHkgLT4gMDsgUXVlc3Rpb25hYmxlL1Vud2FudGVkKiAtPiAxIiwKICAgICAgICBsYWJlbF9tYXBwaW5nX3ZlcnNpb249ImIxLWxhYmVscy12MSIsCiAgICAgICAgY2l0YXRpb249IkJhbyBldCBhbC4sIE5BQUNMIDIwMjUsIERPSSAxMC4xODY1My92MS8yMDI1Lm5hYWNsLXNob3J0LjM4IiwKICAgICAgICByYXdfZGlyPSJkYXRhL3Jhdy9mYWl0aGJlbmNoIiwKICAgICksCikKCgpkZWYgcmVhZF9oYXNoZXNfanNvbihwYXRoOiBQYXRoKSAtPiBkaWN0OgogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIHt9CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIE9TRXJyb3IpOgogICAgICAgIHJldHVybiB7fQoKCmRlZiBsaWNlbnNlX21hbmlmZXN0KCkgLT4gZGljdDoKICAgICIiIkFzc2VtYmxlIHRoZSBCMSBsaWNlbnNlL3Byb3ZlbmFuY2UgbWFuaWZlc3QgZnJvbSB0aGUgcmVnaXN0cnkgKyBkaXNrIGhhc2hlcy4iIiIKICAgIGRhdGFzZXRzID0gW10KICAgIGZvciByZWMgaW4gREFUQVNFVF9SRUdJU1RSWToKICAgICAgICByZXZpc2lvbiA9IHJlYWRfaGFzaGVzX2pzb24oUk9PVCAvIHJlYy5yYXdfZGlyIC8gInJldmlzaW9uLmpzb24iKQogICAgICAgIGZpbGVzID0ge30KICAgICAgICBmb3IgbmFtZSwgbWV0YSBpbiAocmV2aXNpb24uZ2V0KCJmaWxlcyIpIG9yIHt9KS5pdGVtcygpOgogICAgICAgICAgICBmaWxlc1tuYW1lXSA9IHsKICAgICAgICAgICAgICAgICJzaGEyNTYiOiBtZXRhLmdldCgic2hhMjU2IiksCiAgICAgICAgICAgICAgICAiYnl0ZXMiOiBtZXRhLmdldCgiYnl0ZXMiKSwKICAgICAgICAgICAgfQogICAgICAgIGRhdGFzZXRzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgKiphc2RpY3QocmVjKSwKICAgICAgICAgICAgICAgICJkb3dubG9hZF9yZXZpc2lvbiI6IHJldmlzaW9uLmdldCgiY29tbWl0X3NoYSIpLAogICAgICAgICAgICAgICAgImRvd25sb2FkZWRfYXRfdXRjIjogcmV2aXNpb24uZ2V0KCJmZXRjaGVkX2F0X3V0YyIpLAogICAgICAgICAgICAgICAgImZpbGVzIjogZmlsZXMsCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICByZXR1cm4gewogICAgICAgICJzY2hlbWEiOiAiYjEtbGljZW5zZS1tYW5pZmVzdC12MSIsCiAgICAgICAgImdlbmVyYXRlZF9hdF91dGMiOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQodGltZXNwZWM9InNlY29uZHMiKSwKICAgICAgICAibm90ZSI6ICJSZXN0cmljdGVkIGRhdGFzZXRzIGFyZSBub3QgcmVkaXN0cmlidXRlZDsgZG93bmxvYWQgaW5zdHJ1Y3Rpb25zLCBoYXNoZXMsIGFuZCBsaWNlbnNlIG5vdGVzIGFyZSBzaGlwcGVkIGluc3RlYWQuIiwKICAgICAgICAiZGF0YXNldHMiOiBkYXRhc2V0cywKICAgIH0K",
 "src/data/schema.py": "IiIiCkhhbHVSSVNDIFZlcnNpb24gQiB1bmlmaWVkIGRhdGFzZXQgc2NoZW1hIChyb2FkbWFwIMKnMTQgQjEpLgoKQ2Fub25pY2FsLCB2ZXJzaW9uZWQgcm93IGNvbnRyYWN0IHNoYXJlZCBieSBIYWx1RXZhbCwgUkFHVHJ1dGgsIGFuZCBGYWl0aEJlbmNoLgoKRGVzaWduIHJ1bGVzOgogIC0gVmVyc2lvbiBBIHByZXByb2Nlc3NpbmcgKGBzcmMvZGF0YS9wcmVwYXJlLnB5YCkgaXMgZnJvemVuIGFuZCB1bmNoYW5nZWQuCiAgLSBUaGlzIG1vZHVsZSBpcyBhZGRpdGl2ZTogaXQgZGVmaW5lcyB0aGUgQjEgY29udHJhY3QgYW5kIHZhbGlkYXRpb24gb25seS4KICAtIE9yaWdpbmFsIGFubm90YXRpb25zIGFuZCBtZXRhZGF0YSBhcmUgcHJlc2VydmVkIGxvc3NsZXNzbHkgaW4gSlNPTiBjb2x1bW5zOwogICAgdGhlIGJpbmFyeSBgbGFiZWxgIGlzIGEgZGVyaXZlZCwgZG9jdW1lbnRlZCBpbnRlcmZhY2UsIG5ldmVyIGEgcmVwbGFjZW1lbnQuCiAgLSBgc2FtcGxlX2lkYCBpcyBnbG9iYWxseSB1bmlxdWU7IGBzb3VyY2VfZ3JvdXBfaWRgIGlzIHRoZSBsZWFrYWdlLWNvbnRyb2wKICAgIGdyb3VwIGtleSAoSGFsdUV2YWw6IGl0ZW1faWR4LCBSQUdUcnV0aDogc291cmNlX2lkLCBGYWl0aEJlbmNoOiByYXcgaWQpLgoiIiIKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCgpTQ0hFTUFfVkVSU0lPTiA9ICJiMS1zY2hlbWEtdjEiCkxBQkVMX01BUFBJTkdfVkVSU0lPTiA9ICJiMS1sYWJlbHMtdjEiCgpTT1VSQ0VfREFUQVNFVFMgPSAoImhhbHVldmFsIiwgInJhZ3RydXRoIiwgImZhaXRoYmVuY2giKQpUQVNLUyA9ICgicWEiLCAic3VtbWFyaXphdGlvbiIsICJkYXRhX3RvX3RleHQiKQpTUExJVFMgPSAoInRyYWluIiwgInZhbCIsICJ0ZXN0IikKClVOSUZJRURfQ09MVU1OUyA9IFsKICAgICJzYW1wbGVfaWQiLCAgICAgICAgICAjIGdsb2JhbGx5IHVuaXF1ZSByb3cgaWQKICAgICJzb3VyY2VfZGF0YXNldCIsICAgICAjIGhhbHVldmFsIHwgcmFndHJ1dGggfCBmYWl0aGJlbmNoCiAgICAic291cmNlX2dyb3VwX2lkIiwgICAgIyBsZWFrYWdlLWNvbnRyb2wgZ3JvdXAga2V5CiAgICAidGFzayIsICAgICAgICAgICAgICAgIyBxYSB8IHN1bW1hcml6YXRpb24gfCBkYXRhX3RvX3RleHQKICAgICJkb21haW4iLCAgICAgICAgICAgICAjIHN0YWJsZSBkb21haW4vc291cmNlIGNhdGVnb3J5CiAgICAicXVlc3Rpb24iLCAgICAgICAgICAgIyB1c2VyIHF1ZXN0aW9uOyAiIiBmb3Igc3VtbWFyaXphdGlvbi9kYXRhLXRvLXRleHQKICAgICJjb250ZXh0IiwgICAgICAgICAgICAjIGV2aWRlbmNlIC8gc291cmNlIHRleHQKICAgICJhbnN3ZXIiLCAgICAgICAgICAgICAjIG1vZGVsIHJlc3BvbnNlIC8gc3VtbWFyeSAobXVzdCBtYXRjaCBzcGFuIG9mZnNldHMpCiAgICAibGFiZWwiLCAgICAgICAgICAgICAgIyB1bmlmaWVkIGJpbmFyeSBsYWJlbCAoMCA9IGZhaXRoZnVsLCAxID0gaGFsbHVjaW5hdGVkKQogICAgInNwYW5fYW5ub3RhdGlvbnMiLCAgICMgSlNPTiBzdHJpbmcsIG9yaWdpbmFsIGFubm90YXRpb25zIHByZXNlcnZlZCB2ZXJiYXRpbQogICAgImdlbmVyYXRvcl9tb2RlbCIsICAgICMgb3JpZ2luYWwgbW9kZWwgd2hlbiBhdmFpbGFibGUKICAgICJvZmZpY2lhbF9zcGxpdCIsICAgICAjIG5hdGl2ZSBkYXRhc2V0IHNwbGl0IChSQUdUcnV0aCB0cmFpbi90ZXN0KSBvciAiIgogICAgImV4cGVyaW1lbnRfc3BsaXQiLCAgICMgSGFsdUV2YWwgZ3JvdXBlZCBzcGxpdCAodHJhaW4vdmFsL3Rlc3QpIG9yICIiCiAgICAicXVhbGl0eSIsICAgICAgICAgICAgIyBSQUdUcnV0aCBxdWFsaXR5IGZsYWcgKGdvb2QvdHJ1bmNhdGVkLy4uLikgb3IgIiIKICAgICJuYXRpdmVfcmVjb3JkX2lkIiwgICAjIG9yaWdpbmFsIGRhdGFzZXQgcm93IGlkCiAgICAibmF0aXZlX21ldGFkYXRhIiwgICAgIyBKU09OIHN0cmluZywgb3RoZXIgc291cmNlLXNwZWNpZmljIG1ldGFkYXRhCiAgICAibGFiZWxfbWFwcGluZ192ZXJzaW9uIiwKXQoKRU1QVFlfU1BBTlMgPSAiW10iCkVNUFRZX01FVEEgPSAie30iCgoKZGVmIGpzb25fZHVtcHMob2JqKSAtPiBzdHI6CiAgICAiIiJEZXRlcm1pbmlzdGljIEpTT04gc2VyaWFsaXphdGlvbiBmb3IgbWV0YWRhdGEgY29sdW1ucy4iIiIKICAgIHJldHVybiBqc29uLmR1bXBzKG9iaiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBzb3J0X2tleXM9VHJ1ZSkKCgpkZWYgc2hhMjU2X3RleHQodGV4dDogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYodGV4dC5lbmNvZGUoInV0Zi04IikpLmhleGRpZ2VzdCgpCgoKZGVmIGZyYW1lX2ZpbmdlcnByaW50KGRmKSAtPiBzdHI6CiAgICAiIiJTdGFibGUgY29udGVudCBoYXNoIG9mIGEgY2Fub25pY2FsIGZyYW1lIChkZXRlcm1pbmlzbSBjaGVjaykuIiIiCiAgICByZXR1cm4gc2hhMjU2X3RleHQoanNvbl9kdW1wcyhkZi50b19kaWN0KG9yaWVudD0icmVjb3JkcyIpKSkKCgpkZWYgX2FsbF9qc29uX3N0cmluZ3MoZGYsIGNvbDogc3RyKSAtPiBib29sOgogICAgZGVmIG9rKHYpOgogICAgICAgIGlmIHYgaXMgTm9uZSBvciAoaXNpbnN0YW5jZSh2LCBzdHIpIGFuZCBub3Qgdi5zdHJpcCgpKToKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZSh2LCBzdHIpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICB0cnk6CiAgICAgICAgICAgIGpzb24ubG9hZHModikKICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsIFR5cGVFcnJvcik6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHJldHVybiBUcnVlCgogICAgcmV0dXJuIGRmW2NvbF0ubWFwKG9rKS5hbGwoKQoKCmRlZiB2YWxpZGF0ZV91bmlmaWVkX2RmKGRmKToKICAgICIiIlJhaXNlIFZhbHVlRXJyb3Igd2l0aCBhIHByZWNpc2UgbWVzc2FnZSBvbiBhbnkgY29udHJhY3QgdmlvbGF0aW9uLiIiIgogICAgbWlzc2luZyA9IFtjIGZvciBjIGluIFVOSUZJRURfQ09MVU1OUyBpZiBjIG5vdCBpbiBkZi5jb2x1bW5zXQogICAgaWYgbWlzc2luZzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYibWlzc2luZyBjYW5vbmljYWwgY29sdW1uczoge21pc3Npbmd9IikKICAgIGV4dHJhID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiBjIG5vdCBpbiBVTklGSUVEX0NPTFVNTlNdCiAgICBpZiBleHRyYToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5leHBlY3RlZCBjb2x1bW5zOiB7ZXh0cmF9IikKCiAgICBpZiBkZlsic2FtcGxlX2lkIl0uZHVwbGljYXRlZCgpLmFueSgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNhbXBsZV9pZCBtdXN0IGJlIGdsb2JhbGx5IHVuaXF1ZSIpCiAgICBibGFuayA9IGRmWyJzYW1wbGVfaWQiXS5pc25hKCkgfCAoZGZbInNhbXBsZV9pZCJdLmFzdHlwZShzdHIpLnN0ci5zdHJpcCgpID09ICIiKQogICAgaWYgYmxhbmsuYW55KCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2FtcGxlX2lkIG11c3QgYmUgbm9uLWVtcHR5IikKICAgIGJsYW5rID0gZGZbInNvdXJjZV9ncm91cF9pZCJdLmlzbmEoKSB8IChkZlsic291cmNlX2dyb3VwX2lkIl0uYXN0eXBlKHN0cikuc3RyLnN0cmlwKCkgPT0gIiIpCiAgICBpZiBibGFuay5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzb3VyY2VfZ3JvdXBfaWQgbXVzdCBiZSBub24tZW1wdHkiKQoKICAgIGlmIG5vdCBzZXQoZGZbImxhYmVsIl0udW5pcXVlKCkpLmlzc3Vic2V0KHswLCAxfSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibGFiZWwgbXVzdCBiZSAwIG9yIDEiKQogICAgYmFkX2RzID0gc2V0KGRmWyJzb3VyY2VfZGF0YXNldCJdLnVuaXF1ZSgpKSAtIHNldChTT1VSQ0VfREFUQVNFVFMpCiAgICBpZiBiYWRfZHM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gc291cmNlX2RhdGFzZXQgdmFsdWVzOiB7c29ydGVkKGJhZF9kcyl9IikKICAgIGJhZF90YXNrID0gc2V0KGRmWyJ0YXNrIl0udW5pcXVlKCkpIC0gc2V0KFRBU0tTKQogICAgaWYgYmFkX3Rhc2s6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gdGFzayB2YWx1ZXM6IHtzb3J0ZWQoYmFkX3Rhc2spfSIpCgogICAgZW1wdHlfYW5zID0gZGZbImFuc3dlciJdLmlzbmEoKSB8IChkZlsiYW5zd2VyIl0uYXN0eXBlKHN0cikuc3RyLnN0cmlwKCkgPT0gIiIpCiAgICBpZiBlbXB0eV9hbnMuYW55KCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYW5zd2VyIG11c3QgYmUgbm9uLWVtcHR5IikKCiAgICBleHBfdmFsdWVzID0gc2V0KGRmWyJleHBlcmltZW50X3NwbGl0Il1bZGZbImV4cGVyaW1lbnRfc3BsaXQiXSAhPSAiIl0udW5pcXVlKCkpCiAgICBiYWRfZXhwID0gZXhwX3ZhbHVlcyAtIHNldChTUExJVFMpCiAgICBpZiBiYWRfZXhwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHBlcmltZW50X3NwbGl0IG11c3QgYmUgaW4ge1NQTElUU30gb3IgZW1wdHksIGdvdDoge3NvcnRlZChiYWRfZXhwKX0iKQogICAgb2ZmX3ZhbHVlcyA9IHNldChkZlsib2ZmaWNpYWxfc3BsaXQiXVtkZlsib2ZmaWNpYWxfc3BsaXQiXSAhPSAiIl0udW5pcXVlKCkpCiAgICBmb3IgdiBpbiBvZmZfdmFsdWVzOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHYsIHN0cik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJvZmZpY2lhbF9zcGxpdCBtdXN0IGJlIGEgc3RyaW5nLCBnb3Qge3Yhcn0iKQoKICAgIGZvciBjb2wgaW4gKCJzcGFuX2Fubm90YXRpb25zIiwgIm5hdGl2ZV9tZXRhZGF0YSIpOgogICAgICAgIGlmIG5vdCBfYWxsX2pzb25fc3RyaW5ncyhkZiwgY29sKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntjb2x9IG11c3QgYmUgYSBKU09OIHN0cmluZyAob3IgZW1wdHkpIikKICAgIHJldHVybiBUcnVlCg==",
 "src/explain/fig_b5_importance.py": "IiIiUmVnZW5lcmF0ZSB0aGUgU0hBUCBpbXBvcnRhbmNlIGZpZ3VyZSBmb3IgdGhlIHBhcGVyIGZyb20gdmVyaWZpZWQgQjUgZGF0YS4KClJlYWRzIGFydGlmYWN0cy9yZXN1bHRzL2I1L2I1X2ZlYXR1cmVfaW1wb3J0YW5jZS5qc29uIChmcm96ZW4gQi1ydW4pIGFuZApyZW5kZXJzIHRoZSB0b3AtMTIgbWVhbiB8U0hBUHwgYmFyIGNoYXJ0LiBSdW4gZnJvbSB0aGUgcmVwbyByb290OgoKICAudmVudi9TY3JpcHRzL3B5dGhvbi5leGUgc3JjL2V4cGxhaW4vZmlnX2I1X2ltcG9ydGFuY2UucHkKIiIiCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG1hdHBsb3RsaWIKCm1hdHBsb3RsaWIudXNlKCJBZ2ciKQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KU1JDID0gUk9PVCAvICJhcnRpZmFjdHMiIC8gInJlc3VsdHMiIC8gImI1IiAvICJiNV9mZWF0dXJlX2ltcG9ydGFuY2UuanNvbiIKT1VUID0gUk9PVCAvICJhcnRpZmFjdHMiIC8gImZpZ3VyZXMiIC8gImZpZ19zaGFwX2ltcG9ydGFuY2VfYjUucG5nIgoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIGRhdGEgPSBqc29uLmxvYWRzKFNSQy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBtZWFuX2Fic19zaGFwID0gZGF0YVsibWVhbl9hYnNfc2hhcCJdCiAgICBpdGVtcyA9IHNvcnRlZChtZWFuX2Fic19zaGFwLml0ZW1zKCksIGtleT1sYW1iZGEga3Y6IGt2WzFdLCByZXZlcnNlPVRydWUpWzoxMl0KICAgIG5hbWVzID0gW2sgZm9yIGssIF8gaW4gaXRlbXNdCiAgICB2YWx1ZXMgPSBbdiBmb3IgaywgdiBpbiBpdGVtc10KCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDksIDUpKQogICAgYXguYmFyaChyYW5nZShsZW4odmFsdWVzKSksIHZhbHVlcywgY29sb3I9IiM0ZjQ2ZTUiKQogICAgYXguc2V0X3l0aWNrcyhyYW5nZShsZW4obmFtZXMpKSkKICAgIGF4LnNldF95dGlja2xhYmVscyhuYW1lcykKICAgIGF4LmludmVydF95YXhpcygpCiAgICBheC5zZXRfeGxhYmVsKCJtZWFuIHxTSEFQfCIpCiAgICBheC5zZXRfdGl0bGUoIlRvcC0xMiBmZWF0dXJlcyBieSBtZWFuIHxTSEFQfCAoQi1ydW4sIHNlZWQgNDIpIikKICAgIGF4LnNwaW5lc1tbInRvcCIsICJyaWdodCJdXS5zZXRfdmlzaWJsZShGYWxzZSkKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgZmlnLnNhdmVmaWcoT1VULCBkcGk9MTUwKQogICAgcHJpbnQoZiJzYXZlZCB7T1VUfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
 "src/explain/shap_analysis.py": "IiIiClNIQVAgZXhwbGFpbmFiaWxpdHkgYW5hbHlzaXMgZm9yIEhhbHVSSVNDIChibHVlcHJpbnQgwqc3LjMsIHJvYWRtYXAgUGhhc2UgNikuCgpQcm9kdWNlcyAoc2F2ZWQgdG8gYXJ0aWZhY3RzL2ZpZ3VyZXMgKyBhcnRpZmFjdHMvcmVzdWx0cyk6CiAgLSBHbG9iYWw6IFNIQVAgYmVlc3dhcm0gc3VtbWFyeSArIG1lYW58U0hBUHwgYmFyIGNoYXJ0CiAgLSBMb2NhbDogd2F0ZXJmYWxsIHBsb3RzIGZvciAzIGhhbmQtcGlja2VkIHRlc3QgY2FzZXMKICAtIFJPQyAvIFBSIGN1cnZlcyArIHJlbGlhYmlsaXR5IGRpYWdyYW0gKHdpdGggRUNFL0JyaWVyIGFubm90YXRpb24pCiAgLSBzaGFwX3N1bW1hcnkuanNvbiAoZ2xvYmFsIHRvcCBmZWF0dXJlcyArIHBlci1jYXNlIGxvY2FsIFNIQVAgZm9yIHRoZSBkYXNoYm9hcmQpCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvZXhwbGFpbi9zaGFwX2FuYWx5c2lzLnB5CiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkKCmltcG9ydCBqb2JsaWIKaW1wb3J0IG1hdHBsb3RsaWIKCm1hdHBsb3RsaWIudXNlKCJBZ2ciKQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJzaGFwX2FuYWx5c2lzIikKCmZyb20gc3JjLm1vZGVscy5jb25maWcgaW1wb3J0IEZFQVRVUkVTX0ZBTExCQUNLLCBGRUFUVVJFU19GVUxMLCBGSUdVUkVTX0RJUiwgTU9ERUxTX0RJUiwgUUFfQ0xFQU4sIFJFU1VMVFNfRElSLCBST09UCgpTSEFQX1NVQlNBTVBMRSA9IDEwMDAgICMga2VwdCBmb3IgcmVmZXJlbmNlOyBmdWxsIHRlc3Qgc2V0IGlzIHVzZWQgKHRyZWUgZXhwbGFpbmVyIGlzIGZhc3QpCk5fVE9QX0ZFQVRVUkVTID0gMTAKCgpkZWYgbG9hZF90ZXN0X3NldCgpOgogICAgcGF0aCA9IEZFQVRVUkVTX0ZVTEwgaWYgRkVBVFVSRVNfRlVMTC5leGlzdHMoKSBlbHNlIEZFQVRVUkVTX0ZBTExCQUNLCiAgICBkZiA9IHBkLnJlYWRfcGFycXVldChwYXRoKQogICAgY2xlYW4gPSBwZC5yZWFkX3BhcnF1ZXQoUUFfQ0xFQU4pCiAgICB0ZXh0X2NvbHMgPSBbYyBmb3IgYyBpbiBbInF1ZXN0aW9uIiwgImFuc3dlciIsICJjb250ZXh0Il0gaWYgYyBpbiBjbGVhbi5jb2x1bW5zXQogICAgaWYgdGV4dF9jb2xzOgogICAgICAgIGRmID0gcGQuY29uY2F0KFtkZiwgY2xlYW5bdGV4dF9jb2xzXV0sIGF4aXM9MSkKICAgIGZlYXR1cmVfY29scyA9IGpzb24ubG9hZHMoKE1PREVMU19ESVIgLyAiZmVhdHVyZV9uYW1lcy5qc29uIikucmVhZF90ZXh0KCkpCiAgICB0ZXN0X2RmID0gZGZbZGZbInNwbGl0Il0gPT0gInRlc3QiXS5jb3B5KCkKICAgIFhfdGVzdCA9IHRlc3RfZGZbZmVhdHVyZV9jb2xzXS52YWx1ZXMKICAgIHlfdGVzdCA9IHRlc3RfZGZbImxhYmVsIl0udmFsdWVzCiAgICByZXR1cm4gdGVzdF9kZiwgWF90ZXN0LCB5X3Rlc3QsIGZlYXR1cmVfY29scwoKCmRlZiBjYXNlX2luZGV4ZXMoeV9wcm9iOiBucC5uZGFycmF5KSAtPiBkaWN0OgogICAgIiIiMyBoYW5kLXBpY2tlZCBjYXNlczogY2xlYXIgaGFsbHVjaW5hdGlvbiwgY2xlYXJseSBjb3JyZWN0LCBib3JkZXJsaW5lLiIiIgogICAgaWR4X2hpZ2ggPSBpbnQobnAuYXJnbWF4KHlfcHJvYikpCiAgICBpZHhfbG93ID0gaW50KG5wLmFyZ21pbih5X3Byb2IpKQogICAgaWR4X2JvcmRlciA9IGludChucC5hcmdtaW4obnAuYWJzKHlfcHJvYiAtIDAuNSkpKQogICAgcmV0dXJuIHsiaGlnaF9yaXNrIjogaWR4X2hpZ2gsICJsb3dfcmlzayI6IGlkeF9sb3csICJib3JkZXJsaW5lIjogaWR4X2JvcmRlcn0KCgpkZWYgX3NhdmVfZmlnKGZpZywgbmFtZTogc3RyKToKICAgICIiIlNhdmUgYSBmaWd1cmUgYXMgUE5HIChkYXNoYm9hcmQpICsgUERGIChwYXBlciwgYmx1ZXByaW50IMKnMTEgdmVjdG9yIGZvcm1hdCkuIiIiCiAgICBmb3IgZXh0IGluICgicG5nIiwgInBkZiIpOgogICAgICAgIGZpZy5zYXZlZmlnKEZJR1VSRVNfRElSIC8gZiJ7bmFtZX0ue2V4dH0iLCBkcGk9MTUwLCBiYm94X2luY2hlcz0idGlnaHQiKQogICAgcGx0LmNsb3NlKGZpZykKICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQge25hbWV9LnBuZy8ucGRmIikKCgpkZWYgcGxvdF9yb2NfcHIoeV90ZXN0LCB5X3Byb2IsIG5hbWU6IHN0cik6CiAgICBmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUsIHByZWNpc2lvbl9yZWNhbGxfY3VydmUsIHJvY19hdWNfc2NvcmUsIHJvY19jdXJ2ZQoKICAgIGZwciwgdHByLCBfID0gcm9jX2N1cnZlKHlfdGVzdCwgeV9wcm9iKQogICAgcHJlYywgcmVjLCBfID0gcHJlY2lzaW9uX3JlY2FsbF9jdXJ2ZSh5X3Rlc3QsIHlfcHJvYikKICAgIGZpZywgYXhlcyA9IHBsdC5zdWJwbG90cygxLCAyLCBmaWdzaXplPSgxMiwgNC41KSkKICAgIGF4ZXNbMF0ucGxvdChmcHIsIHRwciwgbHc9MiwgY29sb3I9IiM4YjVjZjYiKQogICAgYXhlc1swXS5wbG90KFswLCAxXSwgWzAsIDFdLCBscz0iLS0iLCBjb2xvcj0iZ3JheSIsIGFscGhhPTAuNikKICAgIGF4ZXNbMF0uc2V0X3RpdGxlKGYiUk9DIChBVUM9e3JvY19hdWNfc2NvcmUoeV90ZXN0LCB5X3Byb2IpOi40Zn0pIikKICAgIGF4ZXNbMF0uc2V0X3hsYWJlbCgiRmFsc2UgcG9zaXRpdmUgcmF0ZSIpCiAgICBheGVzWzBdLnNldF95bGFiZWwoIlRydWUgcG9zaXRpdmUgcmF0ZSIpCiAgICBheGVzWzFdLnBsb3QocmVjLCBwcmVjLCBsdz0yLCBjb2xvcj0iIzYzNjZmMSIpCiAgICBheGVzWzFdLnNldF90aXRsZShmIlBSIGN1cnZlIChBVUM9e2F2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlKHlfdGVzdCwgeV9wcm9iKTouNGZ9KSIpCiAgICBheGVzWzFdLnNldF94bGFiZWwoIlJlY2FsbCIpCiAgICBheGVzWzFdLnNldF95bGFiZWwoIlByZWNpc2lvbiIpCiAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgIF9zYXZlX2ZpZyhmaWcsIG5hbWUpCgoKZGVmIHBsb3RfcmVsaWFiaWxpdHkoeV90ZXN0LCB5X3Byb2IsIG5hbWU6IHN0ciwgbl9iaW5zOiBpbnQgPSAxMCk6CiAgICBiaW5zID0gbnAubGluc3BhY2UoMCwgMSwgbl9iaW5zICsgMSkKICAgIGlkeHMgPSBucC5jbGlwKG5wLnNlYXJjaHNvcnRlZChiaW5zLCB5X3Byb2IsIHNpZGU9InJpZ2h0IikgLSAxLCAwLCBuX2JpbnMgLSAxKQogICAgY29uZnMsIGFjY3MsIGNvdW50cyA9IFtdLCBbXSwgW10KICAgIGZvciBiIGluIHJhbmdlKG5fYmlucyk6CiAgICAgICAgbWFzayA9IGlkeHMgPT0gYgogICAgICAgIGlmIG1hc2suc3VtKCkgPT0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBjb25mcy5hcHBlbmQoeV9wcm9iW21hc2tdLm1lYW4oKSkKICAgICAgICBhY2NzLmFwcGVuZCh5X3Rlc3RbbWFza10ubWVhbigpKQogICAgICAgIGNvdW50cy5hcHBlbmQobWFzay5zdW0oKSkKCiAgICBmcm9tIHNyYy5tb2RlbHMudHJhaW5fcGlwZWxpbmUgaW1wb3J0IGVjZQogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGJyaWVyX3Njb3JlX2xvc3MKCiAgICBlY2VfdmFsID0gZWNlKHlfdGVzdCwgeV9wcm9iKQogICAgYnJpZXIgPSBicmllcl9zY29yZV9sb3NzKHlfdGVzdCwgeV9wcm9iKQoKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNiwgNSkpCiAgICBheC5wbG90KFswLCAxXSwgWzAsIDFdLCBscz0iLS0iLCBjb2xvcj0iZ3JheSIsIGFscGhhPTAuNywgbGFiZWw9IlBlcmZlY3QgY2FsaWJyYXRpb24iKQogICAgYXgucGxvdChjb25mcywgYWNjcywgbWFya2VyPSJvIiwgbHc9MiwgY29sb3I9IiM4YjVjZjYiLCBsYWJlbD0iTW9kZWwiKQogICAgZm9yIGMsIGEsIG4gaW4gemlwKGNvbmZzLCBhY2NzLCBjb3VudHMpOgogICAgICAgIGF4LmFubm90YXRlKHN0cihpbnQobikpLCAoYywgYSksIHRleHRjb29yZHM9Im9mZnNldCBwb2ludHMiLCB4eXRleHQ9KDQsIDQpLCBmb250c2l6ZT04LCBhbHBoYT0wLjcpCiAgICBheC5zZXRfeGxpbSgwLCAxKQogICAgYXguc2V0X3lsaW0oMCwgMSkKICAgIGF4LnNldF94bGFiZWwoIkNvbmZpZGVuY2UgKHByZWRpY3RlZCBwcm9iYWJpbGl0eSkiKQogICAgYXguc2V0X3lsYWJlbCgiQWNjdXJhY3kgKGVtcGlyaWNhbCBmcmVxdWVuY3kpIikKICAgIGF4LnNldF90aXRsZShmIlJlbGlhYmlsaXR5IGRpYWdyYW1cbkVDRT17ZWNlX3ZhbDouNGZ9IHwgQnJpZXI9e2JyaWVyOi40Zn0iKQogICAgYXgubGVnZW5kKCkKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgX3NhdmVfZmlnKGZpZywgbmFtZSkKICAgIHJldHVybiBlY2VfdmFsLCBicmllcgoKCmRlZiBtYWluKCk6CiAgICBvcy5tYWtlZGlycyhGSUdVUkVTX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBpbXBvcnQgc2hhcAoKICAgIHRlc3RfZGYsIFhfdGVzdCwgeV90ZXN0LCBmZWF0dXJlX2NvbHMgPSBsb2FkX3Rlc3Rfc2V0KCkKICAgIHJhdyA9IGpvYmxpYi5sb2FkKE1PREVMU19ESVIgLyAibW9kZWxfeGdib29zdF9yYXcuam9ibGliIikKICAgIHlfcHJvYiA9IHJhdy5wcmVkaWN0X3Byb2JhKFhfdGVzdClbOiwgMV0KICAgIGxvZ2dlci5pbmZvKGYiVGVzdCBzZXQ6IHtsZW4oWF90ZXN0KX0gc2FtcGxlcywge2xlbihmZWF0dXJlX2NvbHMpfSBmZWF0dXJlcyIpCgogICAgIyAtLS0tIEdsb2JhbCBTSEFQIChmdWxsIHRlc3Qgc2V0OyB0cmVlIGV4cGxhaW5lciBpcyBjaGVhcCkgLS0tLQogICAgZXhwbGFpbmVyID0gc2hhcC5UcmVlRXhwbGFpbmVyKHJhdykKICAgIGpvYmxpYi5kdW1wKGV4cGxhaW5lciwgTU9ERUxTX0RJUiAvICJzaGFwX2V4cGxhaW5lci5qb2JsaWIiKSAgIyBBMTg6IHNhdmVkIGV4cGxhaW5lciBhcnRpZmFjdAogICAgbG9nZ2VyLmluZm8oIlNhdmVkIHNoYXBfZXhwbGFpbmVyLmpvYmxpYiIpCiAgICBzaGFwX3ZhbHVlcyA9IGV4cGxhaW5lci5zaGFwX3ZhbHVlcyhYX3Rlc3QpCgogICAgIyBCZWVzd2FybSBzdW1tYXJ5CiAgICBzaGFwLnN1bW1hcnlfcGxvdChzaGFwX3ZhbHVlcywgWF90ZXN0LCBmZWF0dXJlX25hbWVzPWZlYXR1cmVfY29scywgc2hvdz1GYWxzZSwgbWF4X2Rpc3BsYXk9MTUpCiAgICBfc2F2ZV9maWcocGx0LmdjZigpLCAiZmlnX3NoYXBfc3VtbWFyeSIpCgogICAgIyBNZWFuIHxTSEFQfCBiYXIKICAgIG1lYW5fYWJzID0gbnAubWVhbihucC5hYnMoc2hhcF92YWx1ZXMpLCBheGlzPTApCiAgICBvcmRlciA9IG5wLmFyZ3NvcnQobWVhbl9hYnMpWzo6LTFdCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDgsIDYpKQogICAgYXguYmFyaCgKICAgICAgICBbZmVhdHVyZV9jb2xzW2ldIGZvciBpIGluIG9yZGVyWzpOX1RPUF9GRUFUVVJFU11dWzo6LTFdLAogICAgICAgIG1lYW5fYWJzW29yZGVyWzpOX1RPUF9GRUFUVVJFU11dWzo6LTFdLAogICAgICAgIGNvbG9yPSIjOGI1Y2Y2IiwKICAgICkKICAgIGF4LnNldF90aXRsZSgiTWVhbiB8U0hBUHwgZmVhdHVyZSBpbXBvcnRhbmNlIikKICAgIGF4LnNldF94bGFiZWwoIm1lYW4gfFNIQVAgdmFsdWV8IikKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgX3NhdmVfZmlnKGZpZywgImZpZ19zaGFwX2ltcG9ydGFuY2UiKQoKICAgICMgLS0tLSBMb2NhbDogMyB3YXRlcmZhbGwgY2FzZXMgLS0tLQogICAgY2FzZXMgPSBjYXNlX2luZGV4ZXMoeV9wcm9iKQogICAgY2FzZV9zaGFwID0ge30KICAgIGZvciBsYWJlbCwgaWR4IGluIGNhc2VzLml0ZW1zKCk6CiAgICAgICAgc2hhcC53YXRlcmZhbGxfcGxvdCgKICAgICAgICAgICAgc2hhcC5FeHBsYW5hdGlvbigKICAgICAgICAgICAgICAgIHNoYXBfdmFsdWVzW2lkeF0sCiAgICAgICAgICAgICAgICBiYXNlX3ZhbHVlcz1leHBsYWluZXIuZXhwZWN0ZWRfdmFsdWUsCiAgICAgICAgICAgICAgICBkYXRhPVhfdGVzdFtpZHhdLAogICAgICAgICAgICAgICAgZmVhdHVyZV9uYW1lcz1mZWF0dXJlX2NvbHMsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIG1heF9kaXNwbGF5PTEwLAogICAgICAgICAgICBzaG93PUZhbHNlLAogICAgICAgICkKICAgICAgICBfc2F2ZV9maWcocGx0LmdjZigpLCBmImZpZ19zaGFwX3dhdGVyZmFsbF97bGFiZWx9IikKICAgICAgICBsb2dnZXIuaW5mbyhmIkNhc2UgJ3tsYWJlbH0nOiBpbmRleD17aWR4fSwgcHJvYj17eV9wcm9iW2lkeF06LjRmfSwgdHJ1ZV9sYWJlbD17eV90ZXN0W2lkeF19IikKICAgICAgICBjYXNlX3NoYXBbbGFiZWxdID0gewogICAgICAgICAgICAic2FtcGxlX2lkIjogc3RyKHRlc3RfZGYuaWxvY1tpZHhdWyJzYW1wbGVfaWQiXSksCiAgICAgICAgICAgICJxdWVzdGlvbiI6IHN0cih0ZXN0X2RmLmlsb2NbaWR4XVsicXVlc3Rpb24iXSlbOjIwMF0sCiAgICAgICAgICAgICJhbnN3ZXIiOiBzdHIodGVzdF9kZi5pbG9jW2lkeF1bImFuc3dlciJdKVs6MjAwXSwKICAgICAgICAgICAgInByb2JhYmlsaXR5IjogZmxvYXQoeV9wcm9iW2lkeF0pLAogICAgICAgICAgICAidHJ1ZV9sYWJlbCI6IGludCh5X3Rlc3RbaWR4XSksCiAgICAgICAgfQoKICAgICMgLS0tLSBDYWxpYnJhdGlvbiAmIHJhbmtpbmcgZmlndXJlcyAtLS0tCiAgICBlY2VfdmFsLCBicmllcl92YWwgPSBwbG90X3JlbGlhYmlsaXR5KHlfdGVzdCwgeV9wcm9iLCAiZmlnX3JlbGlhYmlsaXR5IikKICAgIHBsb3Rfcm9jX3ByKHlfdGVzdCwgeV9wcm9iLCAiZmlnX3JvY19wciIpCgogICAgIyAtLS0tIE1hY2hpbmUtcmVhZGFibGUgc3VtbWFyeSBmb3IgdGhlIGRhc2hib2FyZCAtLS0tCiAgICBzdW1tYXJ5ID0gewogICAgICAgICJtb2RlbF92ZXJzaW9uIjogInhnYm9vc3QtdjEuMCIsCiAgICAgICAgIm5fdGVzdCI6IGludChsZW4oWF90ZXN0KSksCiAgICAgICAgImVjZSI6IGVjZV92YWwsCiAgICAgICAgImJyaWVyIjogYnJpZXJfdmFsLAogICAgICAgICJ0b3BfZmVhdHVyZXMiOiBbCiAgICAgICAgICAgIHsiZmVhdHVyZSI6IGZlYXR1cmVfY29sc1tpXSwgIm1lYW5fYWJzX3NoYXAiOiBmbG9hdChtZWFuX2Fic1tpXSl9CiAgICAgICAgICAgIGZvciBpIGluIG9yZGVyWzpOX1RPUF9GRUFUVVJFU10KICAgICAgICBdLAogICAgICAgICJjYXNlcyI6IGNhc2Vfc2hhcCwKICAgIH0KICAgIHdpdGggb3BlbihSRVNVTFRTX0RJUiAvICJzaGFwX3N1bW1hcnkuanNvbiIsICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoc3VtbWFyeSwgZiwgaW5kZW50PTIpCiAgICBsb2dnZXIuaW5mbyhmIlNhdmVkIHNoYXBfc3VtbWFyeS5qc29uIGFuZCBmaWd1cmVzIHRvIHtGSUdVUkVTX0RJUn0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJPT1QpKQogICAgbWFpbigpCg==",
 "src/features/entity_features.py": "IiIiCkVudGl0eSAoTkVSKSBmZWF0dXJlIGV4dHJhY3Rpb24gZm9yIEhhbHVSSVNDLgoKR3JvdXAgMyBmZWF0dXJlcyAocm9hZG1hcCDCpzYpOgogIG5fZW50aXRpZXNfYW5zd2VyLCBuX2VudGl0aWVzX2NvbnRleHQsIGVudGl0eV9vdmVybGFwX3JhdGlvLCBub3ZlbF9lbnRpdHlfcmF0aW8KClVzZXMgc3BhQ3kgYGVuX2NvcmVfd2ViX3NtYCBmb3IgbmFtZWQgZW50aXR5IHJlY29nbml0aW9uLgpFbXB0eSBjb250ZXh0IC0+IGFuc3dlciBlbnRpdGllcyBhcmUgYWxsIG5vdmVsIChvdmVybGFwIDApLiBObyBlbnRpdGllcyBpbgphbnN3ZXIgLT4gbm8gdW5zdXBwb3J0ZWQtZW50aXR5IHNpZ25hbCAob3ZlcmxhcCAxLjAsIG5vdmVsIDAuMCkuCiIiIgoKaW1wb3J0IGxvZ2dpbmcKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCk1PREVMX05BTUUgPSAiZW5fY29yZV93ZWJfc20iCiMgTkVSIG9ubHk6IHRhZ2dlci9wYXJzZXIvbGVtbWF0aXplci9hdHRyaWJ1dGVfcnVsZXIgZG8gbm90IGFmZmVjdCAuZW50cyBhbmQKIyBjb3N0IG1vc3Qgb2YgdGhlIHJ1bnRpbWUgb24gbG9uZyBkb2N1bWVudHMuCkRJU0FCTEVfQ09NUE9ORU5UUyA9IFsidGFnZ2VyIiwgInBhcnNlciIsICJhdHRyaWJ1dGVfcnVsZXIiLCAibGVtbWF0aXplciJdCgoKZGVmIGxvYWRfbmVyX21vZGVsKCk6CiAgICAiIiJMb2FkIHRoZSBzcGFDeSBORVIgcGlwZWxpbmUgKGxhenksIGNhY2hlZCBhdCBjYWxsIHNpdGUpLiIiIgogICAgaW1wb3J0IHNwYWN5CgogICAgdHJ5OgogICAgICAgIHJldHVybiBzcGFjeS5sb2FkKE1PREVMX05BTUUsIGRpc2FibGU9RElTQUJMRV9DT01QT05FTlRTKQogICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgcmV0dXJuIHNwYWN5LmxvYWQoTU9ERUxfTkFNRSkKCgpkZWYgX2VudGl0eV9mZWF0dXJlc19mcm9tX3NldHMoYW5zX2VudGl0aWVzOiBzZXQsIGN0eF9lbnRpdGllczogc2V0KSAtPiBkaWN0OgogICAgbl9hbnMgPSBsZW4oYW5zX2VudGl0aWVzKQogICAgbl9jdHggPSBsZW4oY3R4X2VudGl0aWVzKQogICAgaWYgbl9hbnMgPT0gMDoKICAgICAgICBlbnRpdHlfb3ZlcmxhcF9yYXRpbyA9IDEuMAogICAgICAgIG5vdmVsX2VudGl0eV9yYXRpbyA9IDAuMAogICAgZWxzZToKICAgICAgICBpbnRlciA9IGFuc19lbnRpdGllcy5pbnRlcnNlY3Rpb24oY3R4X2VudGl0aWVzKQogICAgICAgIGVudGl0eV9vdmVybGFwX3JhdGlvID0gbGVuKGludGVyKSAvIG5fYW5zCiAgICAgICAgbm92ZWxfZW50aXR5X3JhdGlvID0gKG5fYW5zIC0gbGVuKGludGVyKSkgLyBuX2FucwogICAgcmV0dXJuIHsKICAgICAgICAibl9lbnRpdGllc19hbnN3ZXIiOiBuX2FucywKICAgICAgICAibl9lbnRpdGllc19jb250ZXh0Ijogbl9jdHgsCiAgICAgICAgImVudGl0eV9vdmVybGFwX3JhdGlvIjogcm91bmQoZmxvYXQoZW50aXR5X292ZXJsYXBfcmF0aW8pLCA2KSwKICAgICAgICAibm92ZWxfZW50aXR5X3JhdGlvIjogcm91bmQoZmxvYXQobm92ZWxfZW50aXR5X3JhdGlvKSwgNiksCiAgICB9CgoKZGVmIGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzKHF1ZXN0aW9uOiBzdHIsIGNvbnRleHQ6IHN0ciwgYW5zd2VyOiBzdHIsIG5scCkgLT4gZGljdDoKICAgICIiIkdyb3VwIDM6IG5hbWVkLWVudGl0eSBvdmVybGFwIGJldHdlZW4gYW5zd2VyIGFuZCBjb250ZXh0LiIiIgogICAgYW5zX2VudGl0aWVzID0ge2UudGV4dC5sb3dlcigpIGZvciBlIGluIG5scChhbnN3ZXIpLmVudHN9CiAgICBjdHhfZW50aXRpZXMgPSB7ZS50ZXh0Lmxvd2VyKCkgZm9yIGUgaW4gbmxwKGNvbnRleHQpLmVudHN9IGlmIGNvbnRleHQuc3RyaXAoKSBlbHNlIHNldCgpCiAgICByZXR1cm4gX2VudGl0eV9mZWF0dXJlc19mcm9tX3NldHMoYW5zX2VudGl0aWVzLCBjdHhfZW50aXRpZXMpCgoKZGVmIGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzX2RmKAogICAgZGY6IHBkLkRhdGFGcmFtZSwgbmxwLCBiYXRjaF9zaXplOiBpbnQgPSA2NCwgbG9nX2V2ZXJ5OiBpbnQgPSA1MDAKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJCYXRjaCBlbnRpdHkgZmVhdHVyZXMgd2l0aCBzcGFDeSBgYG5scC5waXBlYGAuCgogICAgQW5zd2VycyBhbmQgY29udGV4dHMgYXJlIHN0cmVhbWVkIHRocm91Z2ggb25lIHBpcGUgKDIgZG9jdW1lbnRzIHBlciByb3cpLgogICAgYGBIQUxVX1NQQUNZX05fUFJPQ0VTU2BgIHBhcmFsbGVsaXplcyBhY3Jvc3MgQ1BVIGNvcmVzIChXaW5kb3dzLXNhZmUgdW5kZXIKICAgIHRoZSBzY3JpcHQgX19tYWluX18gZ3VhcmQpLiBQcm9ncmVzcyBpcyBsb2dnZWQgd2l0aCBwZXJjZW50IGFuZCBFVEEgc28gYQogICAgbG9uZyBydW4gaXMgdmlzaWJsZSB3aGlsZSBpdCB3b3Jrcy4KICAgICIiIgogICAgaW1wb3J0IG9zCiAgICBpbXBvcnQgdGltZQoKICAgIHRvdGFsID0gbGVuKGRmKQogICAgbG9nZ2VyLmluZm8oZiJFeHRyYWN0aW5nIGVudGl0eSAoTkVSKSBmZWF0dXJlcyBmb3Ige3RvdGFsfSBzYW1wbGVzLi4uIikKICAgIHRleHRzID0gW10KICAgIGZvciBfLCByb3cgaW4gZGYuaXRlcnJvd3MoKToKICAgICAgICBjb250ZXh0ID0gc3RyKHJvd1siY29udGV4dCJdKSBpZiBzdHIocm93WyJjb250ZXh0Il0pLnN0cmlwKCkgZWxzZSAiIgogICAgICAgIHRleHRzLmFwcGVuZChzdHIocm93WyJhbnN3ZXIiXSkpCiAgICAgICAgdGV4dHMuYXBwZW5kKGNvbnRleHQpCgogICAgbl9wcm9jZXNzID0gbWF4KDEsIGludChvcy5lbnZpcm9uLmdldCgiSEFMVV9TUEFDWV9OX1BST0NFU1MiLCAiMSIpKSkKICAgIGxvZ2dlci5pbmZvKGYiTkVSIHBpcGU6IHtsZW4odGV4dHMpfSBkb2NzLCBiYXRjaF9zaXplPXtiYXRjaF9zaXplfSwgbl9wcm9jZXNzPXtuX3Byb2Nlc3N9IikKCiAgICByb3dzID0gW10KICAgIHBlbmRpbmdfYW5zd2VyOiBzZXQgfCBOb25lID0gTm9uZQogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgZm9yIGksIGRvYyBpbiBlbnVtZXJhdGUobmxwLnBpcGUodGV4dHMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgbl9wcm9jZXNzPW5fcHJvY2VzcykpOgogICAgICAgIGVudGl0aWVzID0ge2UudGV4dC5sb3dlcigpIGZvciBlIGluIGRvYy5lbnRzfQogICAgICAgIGlmIGkgJSAyID09IDA6CiAgICAgICAgICAgIHBlbmRpbmdfYW5zd2VyID0gZW50aXRpZXMKICAgICAgICAgICAgY29udGludWUKICAgICAgICByb3dzLmFwcGVuZChfZW50aXR5X2ZlYXR1cmVzX2Zyb21fc2V0cyhwZW5kaW5nX2Fuc3dlciBvciBzZXQoKSwgZW50aXRpZXMpKQogICAgICAgIGRvbmUgPSBsZW4ocm93cykKICAgICAgICBpZiBsb2dfZXZlcnkgYW5kIGRvbmUgJSBsb2dfZXZlcnkgPT0gMDoKICAgICAgICAgICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gdDAKICAgICAgICAgICAgcmF0ZSA9IGRvbmUgLyBlbGFwc2VkIGlmIGVsYXBzZWQgPiAwIGVsc2UgMC4wCiAgICAgICAgICAgIGV0YSA9ICh0b3RhbCAtIGRvbmUpIC8gcmF0ZSBpZiByYXRlID4gMCBlbHNlIDAuMAogICAgICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgICAgIGYiRW50aXR5IGZlYXR1cmVzOiB7ZG9uZX0ve3RvdGFsfSAoezEwMCAqIGRvbmUgLyB0b3RhbDouMWZ9JSkgfCB7cmF0ZTouMWZ9IHNhbXBsZXMvcyB8IEVUQSB7ZXRhIC8gNjA6LjFmfSBtaW4iCiAgICAgICAgICAgICkKICAgIGxvZ2dlci5pbmZvKGYiRW50aXR5IGZlYXR1cmVzIGRvbmU6IHtsZW4ocm93cyl9IHJvd3MgaW4ge3RpbWUudGltZSgpIC0gdDA6LjFmfXMiKQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzLCBpbmRleD1kZi5pbmRleCkK",
 "src/features/extract_features.py": "IiIiCkZlYXR1cmUgZXh0cmFjdGlvbiBwaXBlbGluZSBmb3IgSGFsdVJJU0MuCkNvbXB1dGVzIGxpZ2h0d2VpZ2h0IGxpbmd1aXN0aWMsIGxleGljYWwgb3ZlcmxhcCwgbnVtZXJpYyBjb25zaXN0ZW5jeSwgYW5kIGhlZGdpbmcgZmVhdHVyZXMuClByZXBhcmVzIG1vZHVsYXIgYXJjaGl0ZWN0dXJlIGZvciBORVIsIFNlbWFudGljLCBhbmQgTkxJIGZlYXR1cmVzLgoiIiIKCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgbnVtcHkgYXMgbnAKCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikKCiMgTGV4aWNvbiBvZiBoZWRnaW5nIC8gdW5jZXJ0YWludHkgaW5kaWNhdG9ycwpIRURHRV9MRVhJQ09OID0gewogICAgIm1heWJlIiwgIm1pZ2h0IiwgImxpa2VseSIsICJwb3NzaWJseSIsICJwcm9iYWJseSIsICJjb3VsZCIsICJzZWVtcyIsIAogICAgInVuY2VydGFpbiIsICJ1bmNsZWFyIiwgImFsbGVnZWRseSIsICJyZXBvcnRlZGx5IiwgInByZXN1bWFibHkiLCAKICAgICJzdXBwb3NlZGx5IiwgImkgdGhpbmsiLCAiaSBiZWxpZXZlIiwgImFwcGVhcnMgdG8iLCAiaXQgc2VlbXMiCn0KCmRlZiBleHRyYWN0X2xlbmd0aF9mZWF0dXJlcyhxdWVzdGlvbjogc3RyLCBjb250ZXh0OiBzdHIsIGFuc3dlcjogc3RyKSAtPiBkaWN0OgogICAgIiIiR3JvdXAgMTogTGVuZ3RoIGFuZCBzdHlsaXN0aWMgZmVhdHVyZXMuIiIiCiAgICB3b3JkcyA9IGFuc3dlci5zcGxpdCgpCiAgICBuX3dvcmRzID0gbGVuKHdvcmRzKQogICAgbl9jaGFycyA9IGxlbihhbnN3ZXIpCiAgICBzZW50ZW5jZXMgPSBbcyBmb3IgcyBpbiByZS5zcGxpdChyJ1suIT9dKycsIGFuc3dlcikgaWYgcy5zdHJpcCgpXQogICAgbl9zZW50ZW5jZXMgPSBtYXgoMSwgbGVuKHNlbnRlbmNlcykpCiAgICBhdmdfd29yZF9sZW4gPSBuX2NoYXJzIC8gbWF4KDEsIG5fd29yZHMpCgogICAgcmV0dXJuIHsKICAgICAgICAibl9jaGFycyI6IG5fY2hhcnMsCiAgICAgICAgIm5fd29yZHMiOiBuX3dvcmRzLAogICAgICAgICJuX3NlbnRlbmNlcyI6IG5fc2VudGVuY2VzLAogICAgICAgICJhdmdfd29yZF9sZW4iOiBhdmdfd29yZF9sZW4sCiAgICB9CgpkZWYgZXh0cmFjdF9sZXhpY2FsX2ZlYXR1cmVzKHF1ZXN0aW9uOiBzdHIsIGNvbnRleHQ6IHN0ciwgYW5zd2VyOiBzdHIpIC0+IGRpY3Q6CiAgICAiIiJHcm91cCAyOiBMZXhpY2FsIG92ZXJsYXAgYW5kIGdyb3VuZGluZyBmZWF0dXJlcy4iIiIKICAgIGFuc190b2tlbnMgPSBzZXQocmUuZmluZGFsbChyJ1x3KycsIGFuc3dlci5sb3dlcigpKSkKICAgIGN0eF90b2tlbnMgPSBzZXQocmUuZmluZGFsbChyJ1x3KycsIGNvbnRleHQubG93ZXIoKSkpCiAgICBxX3Rva2VucyA9IHNldChyZS5maW5kYWxsKHInXHcrJywgcXVlc3Rpb24ubG93ZXIoKSkpCgogICAgaWYgbm90IGFuc190b2tlbnM6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIm92ZXJsYXBfYW5zd2VyX2NvbnRleHQiOiAwLjAsCiAgICAgICAgICAgICJvdmVybGFwX2Fuc3dlcl9xdWVzdGlvbiI6IDAuMCwKICAgICAgICAgICAgImphY2NhcmRfYW5zX2N0eCI6IDAuMCwKICAgICAgICAgICAgImphY2NhcmRfYW5zX3EiOiAwLjAKICAgICAgICB9CgogICAgYW5zX2N0eF9pbnRlcnNlY3QgPSBhbnNfdG9rZW5zLmludGVyc2VjdGlvbihjdHhfdG9rZW5zKQogICAgYW5zX3FfaW50ZXJzZWN0ID0gYW5zX3Rva2Vucy5pbnRlcnNlY3Rpb24ocV90b2tlbnMpCgogICAgb3ZlcmxhcF9hbnNfY3R4ID0gbGVuKGFuc19jdHhfaW50ZXJzZWN0KSAvIGxlbihhbnNfdG9rZW5zKQogICAgb3ZlcmxhcF9hbnNfcSA9IGxlbihhbnNfcV9pbnRlcnNlY3QpIC8gbGVuKGFuc190b2tlbnMpCgogICAgdW5pb25fYW5zX2N0eCA9IGFuc190b2tlbnMudW5pb24oY3R4X3Rva2VucykKICAgIGphY2NhcmRfYW5zX2N0eCA9IGxlbihhbnNfY3R4X2ludGVyc2VjdCkgLyBtYXgoMSwgbGVuKHVuaW9uX2Fuc19jdHgpKQoKICAgIHVuaW9uX2Fuc19xID0gYW5zX3Rva2Vucy51bmlvbihxX3Rva2VucykKICAgIGphY2NhcmRfYW5zX3EgPSBsZW4oYW5zX3FfaW50ZXJzZWN0KSAvIG1heCgxLCBsZW4odW5pb25fYW5zX3EpKQoKICAgIHJldHVybiB7CiAgICAgICAgIm92ZXJsYXBfYW5zd2VyX2NvbnRleHQiOiBvdmVybGFwX2Fuc19jdHgsCiAgICAgICAgIm92ZXJsYXBfYW5zd2VyX3F1ZXN0aW9uIjogb3ZlcmxhcF9hbnNfcSwKICAgICAgICAiamFjY2FyZF9hbnNfY3R4IjogamFjY2FyZF9hbnNfY3R4LAogICAgICAgICJqYWNjYXJkX2Fuc19xIjogamFjY2FyZF9hbnNfcQogICAgfQoKZGVmIGV4dHJhY3RfbnVtZXJpY19mZWF0dXJlcyhxdWVzdGlvbjogc3RyLCBjb250ZXh0OiBzdHIsIGFuc3dlcjogc3RyKSAtPiBkaWN0OgogICAgIiIiR3JvdXAgNTogTnVtZXJpYyBjb25zaXN0ZW5jeSBmZWF0dXJlcy4iIiIKICAgIG51bV9wYXR0ZXJuID0gcidcYlxkKyg/OlwuXGQrKT8lP1xiJwogICAgYW5zX251bXMgPSBzZXQocmUuZmluZGFsbChudW1fcGF0dGVybiwgYW5zd2VyKSkKICAgIGN0eF9udW1zID0gc2V0KHJlLmZpbmRhbGwobnVtX3BhdHRlcm4sIGNvbnRleHQpKQoKICAgIG5fbnVtc19hbnMgPSBsZW4oYW5zX251bXMpCiAgICBuX251bXNfY3R4ID0gbGVuKGN0eF9udW1zKQoKICAgIGlmIG5fbnVtc19hbnMgPT0gMDoKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAibl9udW1iZXJzX2Fuc3dlciI6IDAsCiAgICAgICAgICAgICJuX251bWJlcnNfY29udGV4dCI6IG5fbnVtc19jdHgsCiAgICAgICAgICAgICJudW1iZXJfb3ZlcmxhcF9yYXRpbyI6IDEuMCwgICMgTm8gbnVtYmVycyBpbiBhbnN3ZXIgLT4gbm8gbnVtZXJpYyBoYWxsdWNpbmF0aW9uCiAgICAgICAgICAgICJub3ZlbF9udW1iZXJzIjogMAogICAgICAgIH0KCiAgICBvdmVybGFwX251bXMgPSBhbnNfbnVtcy5pbnRlcnNlY3Rpb24oY3R4X251bXMpCiAgICBvdmVybGFwX3JhdGlvID0gbGVuKG92ZXJsYXBfbnVtcykgLyBuX251bXNfYW5zCiAgICBub3ZlbF9udW1zID0gbGVuKGFuc19udW1zIC0gY3R4X251bXMpCgogICAgcmV0dXJuIHsKICAgICAgICAibl9udW1iZXJzX2Fuc3dlciI6IG5fbnVtc19hbnMsCiAgICAgICAgIm5fbnVtYmVyc19jb250ZXh0Ijogbl9udW1zX2N0eCwKICAgICAgICAibnVtYmVyX292ZXJsYXBfcmF0aW8iOiBvdmVybGFwX3JhdGlvLAogICAgICAgICJub3ZlbF9udW1iZXJzIjogbm92ZWxfbnVtcwogICAgfQoKZGVmIGV4dHJhY3RfaGVkZ2luZ19mZWF0dXJlcyhxdWVzdGlvbjogc3RyLCBjb250ZXh0OiBzdHIsIGFuc3dlcjogc3RyKSAtPiBkaWN0OgogICAgIiIiR3JvdXAgNjogSGVkZ2luZyBhbmQgdW5jZXJ0YWludHkgZmVhdHVyZXMuIiIiCiAgICBhbnNfbG93ZXIgPSBhbnN3ZXIubG93ZXIoKQogICAgaGVkZ2VfY291bnQgPSAwCiAgICBmb3IgaGVkZ2UgaW4gSEVER0VfTEVYSUNPTjoKICAgICAgICBpZiBoZWRnZSBpbiBhbnNfbG93ZXI6CiAgICAgICAgICAgIGhlZGdlX2NvdW50ICs9IGFuc19sb3dlci5jb3VudChoZWRnZSkKCiAgICB3b3JkcyA9IGFuc19sb3dlci5zcGxpdCgpCiAgICBoZWRnZV9kZW5zaXR5ID0gaGVkZ2VfY291bnQgLyBtYXgoMSwgbGVuKHdvcmRzKSkKCiAgICByZXR1cm4gewogICAgICAgICJoZWRnZV9jb3VudCI6IGhlZGdlX2NvdW50LAogICAgICAgICJoZWRnZV9kZW5zaXR5IjogaGVkZ2VfZGVuc2l0eQogICAgfQoKZGVmIGV4dHJhY3RfYWxsX2NvcmVfZmVhdHVyZXMoZGY6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQ29tcHV0ZXMgYWxsIGNvcmUgZmVhdHVyZXMgZm9yIGEgRGF0YUZyYW1lIGNvbnRhaW5pbmcgcXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlci4iIiIKICAgIGxvZ2dpbmcuaW5mbyhmIkV4dHJhY3RpbmcgY29yZSBmZWF0dXJlcyBmb3Ige2xlbihkZil9IHNhbXBsZXMuLi4iKQogICAgZmVhdHVyZV9yb3dzID0gW10KCiAgICBmb3IgaWR4LCByb3cgaW4gZGYuaXRlcnJvd3MoKToKICAgICAgICBxLCBjLCBhID0gcm93WyJxdWVzdGlvbiJdLCByb3dbImNvbnRleHQiXSwgcm93WyJhbnN3ZXIiXQogICAgICAgIGZlYXRzID0ge30KICAgICAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9sZW5ndGhfZmVhdHVyZXMocSwgYywgYSkpCiAgICAgICAgZmVhdHMudXBkYXRlKGV4dHJhY3RfbGV4aWNhbF9mZWF0dXJlcyhxLCBjLCBhKSkKICAgICAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9udW1lcmljX2ZlYXR1cmVzKHEsIGMsIGEpKQogICAgICAgIGZlYXRzLnVwZGF0ZShleHRyYWN0X2hlZGdpbmdfZmVhdHVyZXMocSwgYywgYSkpCiAgICAgICAgZmVhdHVyZV9yb3dzLmFwcGVuZChmZWF0cykKCiAgICBmZWF0dXJlc19kZiA9IHBkLkRhdGFGcmFtZShmZWF0dXJlX3Jvd3MsIGluZGV4PWRmLmluZGV4KQogICAgcmVzdWx0X2RmID0gcGQuY29uY2F0KFtkZltbInNhbXBsZV9pZCIsICJpdGVtX2lkeCIsICJsYWJlbCIsICJzcGxpdCJdXSwgZmVhdHVyZXNfZGZdLCBheGlzPTEpCiAgICBsb2dnaW5nLmluZm8oZiJTdWNjZXNzZnVsbHkgZXh0cmFjdGVkIHtmZWF0dXJlc19kZi5zaGFwZVsxXX0gY29yZSBmZWF0dXJlcy4iKQogICAgcmV0dXJuIHJlc3VsdF9kZgoKCmRlZiBsb2FkX2hlYXZ5X21vZGVscyhubGlfbW9kZWxfbmFtZTogc3RyIHwgTm9uZSA9IE5vbmUsIGRldmljZTogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Q6CiAgICAiIiJMb2FkIE5FUiArIE5MSSArIGVtYmVkZGluZyBtb2RlbHMgb25jZSAodXNlZCBieSBiYXRjaCBleHRyYWN0aW9uIGFuZCBBUEkpLgoKICAgIGRldmljZT1Ob25lIC0+IGxpYnJhcnkgZGVmYXVsdDsgcGFzcyAiY3B1IiBmb3Igc3RhYmlsaXR5IChubyBWUkFNIE9PTSkuCiAgICBPbiBDVURBLCBtb2RlbHMgYXJlIGxvYWRlZCBpbiBmbG9hdDE2IHRvIGhhbHZlIFZSQU0gYW5kIHJlZHVjZSBPT00gcmlzawogICAgb24gc21hbGwgR1BVcyAoUlRYIDMwNjAgNiBHQikuCiAgICAiIiIKICAgIGltcG9ydCB0aW1lCgogICAgaW1wb3J0IHRvcmNoCgogICAgaWYgZGV2aWNlID09ICJjdWRhIiBhbmQgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICBtb2RlbF9rd2FyZ3MgPSB7InRvcmNoX2R0eXBlIjogdG9yY2guZmxvYXQxNn0KICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgIGVsc2U6CiAgICAgICAgbW9kZWxfa3dhcmdzID0gTm9uZQoKICAgIGZyb20gc3JjLmZlYXR1cmVzLmVudGl0eV9mZWF0dXJlcyBpbXBvcnQgbG9hZF9uZXJfbW9kZWwKICAgIGZyb20gc3JjLmZlYXR1cmVzLm5saV9mZWF0dXJlcyBpbXBvcnQgbG9hZF9ubGlfbW9kZWwKICAgIGZyb20gc3JjLmZlYXR1cmVzLnNlbWFudGljX2ZlYXR1cmVzIGltcG9ydCBsb2FkX2VtYmVkZGluZ19tb2RlbAoKICAgIG1vZGVscyA9IHt9CiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBtb2RlbHNbIm5scCJdID0gbG9hZF9uZXJfbW9kZWwoKQogICAgbG9nZ2luZy5pbmZvKGYiTkVSIG1vZGVsIGxvYWRlZCBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyIpCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBtb2RlbHNbIm5saSJdLCBtb2RlbHNbIm5saV9uYW1lIl0gPSBsb2FkX25saV9tb2RlbChubGlfbW9kZWxfbmFtZSwgZGV2aWNlPWRldmljZSwgbW9kZWxfa3dhcmdzPW1vZGVsX2t3YXJncykKICAgIGxvZ2dpbmcuaW5mbyhmIk5MSSBtb2RlbCAoe21vZGVsc1snbmxpX25hbWUnXX0pIGxvYWRlZCBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyIpCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBtb2RlbHNbImVtYmVkZGVyIl0gPSBsb2FkX2VtYmVkZGluZ19tb2RlbChkZXZpY2U9ZGV2aWNlLCBtb2RlbF9rd2FyZ3M9bW9kZWxfa3dhcmdzKQogICAgbG9nZ2luZy5pbmZvKGYiRW1iZWRkaW5nIG1vZGVsIGxvYWRlZCBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyIpCiAgICBpZiBtb2RlbF9rd2FyZ3MgaXMgbm90IE5vbmU6CiAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICByZXR1cm4gbW9kZWxzCgoKZGVmIGV4dHJhY3RfYWxsX2ZlYXR1cmVzX3NpbmdsZShxdWVzdGlvbjogc3RyLCBjb250ZXh0OiBzdHIsIGFuc3dlcjogc3RyLCBtb2RlbHM6IGRpY3QpIC0+IGRpY3Q6CiAgICAiIiJBbGwgNyBmZWF0dXJlIGdyb3VwcyBmb3Igb25lIHNhbXBsZS4gVXNlZCBieSB0aGUgRmFzdEFQSSBpbmZlcmVuY2Ugc2VydmVyLiIiIgogICAgZnJvbSBzcmMuZmVhdHVyZXMuZW50aXR5X2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X2VudGl0eV9mZWF0dXJlcwogICAgZnJvbSBzcmMuZmVhdHVyZXMubmxpX2ZlYXR1cmVzIGltcG9ydCBleHRyYWN0X25saV9mZWF0dXJlcwogICAgZnJvbSBzcmMuZmVhdHVyZXMuc2VtYW50aWNfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3Rfc2VtYW50aWNfZmVhdHVyZXMKCiAgICBmZWF0cyA9IHt9CiAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9sZW5ndGhfZmVhdHVyZXMocXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlcikpCiAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9sZXhpY2FsX2ZlYXR1cmVzKHF1ZXN0aW9uLCBjb250ZXh0LCBhbnN3ZXIpKQogICAgZmVhdHMudXBkYXRlKGV4dHJhY3RfbnVtZXJpY19mZWF0dXJlcyhxdWVzdGlvbiwgY29udGV4dCwgYW5zd2VyKSkKICAgIGZlYXRzLnVwZGF0ZShleHRyYWN0X2hlZGdpbmdfZmVhdHVyZXMocXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlcikpCiAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9lbnRpdHlfZmVhdHVyZXMocXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlciwgbW9kZWxzWyJubHAiXSkpCiAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9ubGlfZmVhdHVyZXMocXVlc3Rpb24sIGNvbnRleHQsIGFuc3dlciwgbW9kZWxzWyJubGkiXSkpCiAgICBmZWF0cy51cGRhdGUoZXh0cmFjdF9zZW1hbnRpY19mZWF0dXJlcyhxdWVzdGlvbiwgY29udGV4dCwgYW5zd2VyLCBtb2RlbHNbImVtYmVkZGVyIl0pKQogICAgcmV0dXJuIGZlYXRzCgoKZGVmIGV4dHJhY3RfZnVsbF9mZWF0dXJlX3NldChkZjogcGQuRGF0YUZyYW1lLCBtb2RlbHM6IGRpY3QgfCBOb25lID0gTm9uZSwgYmF0Y2hfc2l6ZTogaW50ID0gMTI4KSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJFeHRyYWN0cyBhbGwgNyBmZWF0dXJlIGdyb3VwcyAoY29yZSArIGVudGl0eSArIE5MSSArIHNlbWFudGljKSB3aXRoIHBlci1ncm91cCBsYXRlbmN5LgoKICAgIGJhdGNoX3NpemUgY29udHJvbHMgdGhlIE5MSS9lbWJlZGRpbmcgaW5mZXJlbmNlIGJhdGNoIChsYXJnZXIgb24gR1BVID0gZmFzdGVyKS4KICAgICIiIgogICAgaW1wb3J0IHRpbWUKCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5lbnRpdHlfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzX2RmCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5ubGlfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3RfbmxpX2ZlYXR1cmVzX2RmCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5zZW1hbnRpY19mZWF0dXJlcyBpbXBvcnQgZXh0cmFjdF9zZW1hbnRpY19mZWF0dXJlc19kZgoKICAgIGlmIG1vZGVscyBpcyBOb25lOgogICAgICAgIG1vZGVscyA9IGxvYWRfaGVhdnlfbW9kZWxzKCkKCiAgICByZXN1bHRfZGYgPSBleHRyYWN0X2FsbF9jb3JlX2ZlYXR1cmVzKGRmKQoKICAgIHQwID0gdGltZS50aW1lKCkKICAgIGVudGl0eV9kZiA9IGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzX2RmKGRmLCBtb2RlbHNbIm5scCJdKQogICAgbG9nZ2luZy5pbmZvKGYiRW50aXR5IGZlYXR1cmVzIGRvbmUgaW4ge3RpbWUudGltZSgpIC0gdDA6LjFmfXMiKQoKICAgIHQwID0gdGltZS50aW1lKCkKICAgIG5saV9kZiA9IGV4dHJhY3RfbmxpX2ZlYXR1cmVzX2RmKGRmLCBtb2RlbHNbIm5saSJdLCBiYXRjaF9zaXplPWJhdGNoX3NpemUpCiAgICBsb2dnaW5nLmluZm8oZiJOTEkgZmVhdHVyZXMgZG9uZSBpbiB7dGltZS50aW1lKCkgLSB0MDouMWZ9cyAoYmF0Y2g9e2JhdGNoX3NpemV9KSIpCgogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgc2VtYW50aWNfZGYgPSBleHRyYWN0X3NlbWFudGljX2ZlYXR1cmVzX2RmKGRmLCBtb2RlbHNbImVtYmVkZGVyIl0sIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSkKICAgIGxvZ2dpbmcuaW5mbyhmIlNlbWFudGljIGZlYXR1cmVzIGRvbmUgaW4ge3RpbWUudGltZSgpIC0gdDA6LjFmfXMgKGJhdGNoPXtiYXRjaF9zaXplfSkiKQoKICAgIHJlc3VsdF9kZiA9IHBkLmNvbmNhdChbcmVzdWx0X2RmLCBlbnRpdHlfZGYsIG5saV9kZiwgc2VtYW50aWNfZGZdLCBheGlzPTEpCiAgICBsb2dnaW5nLmluZm8oZiJGdWxsIGZlYXR1cmUgbWF0cml4OiB7cmVzdWx0X2RmLnNoYXBlWzFdfSBmZWF0dXJlcyB0b3RhbC4iKQogICAgcmV0dXJuIHJlc3VsdF9kZgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBpbXBvcnQgYXJncGFyc2UKICAgIGltcG9ydCBzeXMKICAgIGZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0pKQoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJIYWx1UklTQyBmdWxsIGZlYXR1cmUgZXh0cmFjdGlvbiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWlucHV0IiwgZGVmYXVsdD1vcy5wYXRoLmpvaW4oImRhdGEiLCAicHJvY2Vzc2VkIiwgInFhX2NsZWFuLnBhcnF1ZXQiKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0IiwgZGVmYXVsdD1vcy5wYXRoLmpvaW4oImRhdGEiLCAicHJvY2Vzc2VkIiwgImZlYXR1cmVzX2Z1bGwucGFycXVldCIpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ubGktbW9kZWwiLCBkZWZhdWx0PU5vbmUsIGhlbHA9Ik92ZXJyaWRlIE5MSSBDcm9zc0VuY29kZXIgY2hlY2twb2ludCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iY3VkYXxjcHUgKGRlZmF1bHQ6IGF1dG87IGN1ZGEgbG9hZHMgbW9kZWxzIGZwMTYpIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTEyOCwgaGVscD0iTkxJL2VtYmVkZGluZyBpbmZlcmVuY2UgYmF0Y2ggKHVzZSAyNTYrIG9uIEdQVSkiKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoYXJncy5pbnB1dCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7YXJncy5pbnB1dH0gbm90IGZvdW5kLiBSdW4gc3JjL2RhdGEvcHJlcGFyZS5weSBmaXJzdC4iKQoKICAgIGRmID0gcGQucmVhZF9wYXJxdWV0KGFyZ3MuaW5wdXQpCiAgICBtb2RlbHMgPSBsb2FkX2hlYXZ5X21vZGVscyhhcmdzLm5saV9tb2RlbCwgZGV2aWNlPWFyZ3MuZGV2aWNlKQogICAgZmVhdHVyZXNfZGYgPSBleHRyYWN0X2Z1bGxfZmVhdHVyZV9zZXQoZGYsIG1vZGVscywgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoX3NpemUpCiAgICBmZWF0dXJlc19kZi50b19wYXJxdWV0KGFyZ3Mub3V0cHV0LCBpbmRleD1GYWxzZSkKICAgIGxvZ2dpbmcuaW5mbyhmIlNhdmVkIGZ1bGwgZmVhdHVyZSBtYXRyaXggdG8ge2FyZ3Mub3V0cHV0fSIpCgogICAgIyBSZWNvcmQgd2hpY2ggTkxJIGNoZWNrcG9pbnQgd2FzIGFjdHVhbGx5IHVzZWQgKHByb3ZlbmFuY2UgZm9yIHBhcmFtcy5qc29uL21hbmlmZXN0KQogICAgbmxpX3VzZWQgPSB7CiAgICAgICAgIm5saV9tb2RlbCI6IG1vZGVscy5nZXQoIm5saV9uYW1lIiksCiAgICAgICAgImRldmljZSI6IHN0cihnZXRhdHRyKG1vZGVscy5nZXQoIm5saSIpLCAiZGV2aWNlIiwgInVua25vd24iKSksCiAgICAgICAgImJhdGNoX3NpemUiOiBhcmdzLmJhdGNoX3NpemUsCiAgICAgICAgImV4dHJhY3RlZF9hdCI6IHBkLlRpbWVzdGFtcC5ub3coIlVUQyIpLmlzb2Zvcm1hdCgpLAogICAgfQogICAgb3MubWFrZWRpcnMob3MucGF0aC5qb2luKCJkYXRhIiwgInByb2Nlc3NlZCIpLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBvcGVuKG9zLnBhdGguam9pbigiZGF0YSIsICJwcm9jZXNzZWQiLCAibmxpX21vZGVsX3VzZWQuanNvbiIpLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKG5saV91c2VkLCBmLCBpbmRlbnQ9MikKICAgIGxvZ2dpbmcuaW5mbyhmIlJlY29yZGVkIE5MSSBwcm92ZW5hbmNlOiB7bmxpX3VzZWRbJ25saV9tb2RlbCddfSIpCg==",
 "src/features/nli_features.py": "IiIiCk5MSSBjb25zaXN0ZW5jeSBmZWF0dXJlIGV4dHJhY3Rpb24gZm9yIEhhbHVSSVNDLgoKR3JvdXAgNCBmZWF0dXJlcyAocm9hZG1hcCDCpzYsIG1hbmRhdG9yeSBwZXIgYmx1ZXByaW50IMKnNik6CiAgbmxpX2N0eF9lbnRhaWxzX2FucywgbmxpX2N0eF9jb250cmFkaWN0c19hbnMsIG5saV9jdHhfbmV1dHJhbF9hbnMKICBubGlfYW5zX2VudGFpbHNfY3R4LCBubGlfYW5zX2NvbnRyYWRpY3RzX2N0eCwgbmxpX2Fuc19uZXV0cmFsX2N0eAoKUHJpbWFyeSBtb2RlbDogY3Jvc3MtZW5jb2Rlci9ubGktZGViZXJ0YS12My1iYXNlIChibHVlcHJpbnQgwqc2KS4KRmFsbGJhY2sgaWYgZG93bmxvYWQgZmFpbHM6IGNyb3NzLWVuY29kZXIvbmxpLU1pbmlMTTItTDYtSDc2OC4KCkNyb3NzRW5jb2RlciBvdXRwdXQgaXMgYSAzLWNsYXNzIHNvZnRtYXggaW4gb3JkZXIKW2NvbnRyYWRpY3Rpb24sIGVudGFpbG1lbnQsIG5ldXRyYWxdIChTTkxJL011bHRpTkxJIHNjaGVtYSkuCgpFbXB0eSBjb250ZXh0IC0+IGFsbCB0aHJlZSBwcm9icyBzZXQgdG8gMS8zIChuZXV0cmFsKSwgcGVyIHByZXBhcmUucHkgcnVsZS4KIiIiCgppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCk5MSV9NT0RFTF9QUklNQVJZID0gImNyb3NzLWVuY29kZXIvbmxpLWRlYmVydGEtdjMtYmFzZSIKTkxJX01PREVMX0ZBTExCQUNLID0gImNyb3NzLWVuY29kZXIvbmxpLU1pbmlMTTItTDYtSDc2OCIKTkxJX01PREVMX0VOViA9ICJIQUxVX05MSV9NT0RFTCIKCk5FVVRSQUwgPSAxLjAgLyAzLjAKCiMgQ3Jvc3NFbmNvZGVyIGxhYmVsIG9yZGVyIGZvciBOTEkgY2hlY2twb2ludHMKTEFCRUxTID0gWyJjb250cmFkaWN0aW9uIiwgImVudGFpbG1lbnQiLCAibmV1dHJhbCJdCgoKZGVmIF9zYWZlX2RldmljZShkZXZpY2U6IE9wdGlvbmFsW3N0cl0pIC0+IE9wdGlvbmFsW3N0cl06CiAgICAiIiJSZXR1cm4gdGhlIGRldmljZSB0byB1c2U7IGN1ZGEgaXMgb25seSBob25vcmVkIHdoZW4gdG9yY2ggc3VwcG9ydHMgaXQuIiIiCiAgICBpZiBkZXZpY2UgPT0gImN1ZGEiOgogICAgICAgIHRyeToKICAgICAgICAgICAgaW1wb3J0IHRvcmNoCgogICAgICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICAgICAgcmV0dXJuICJjdWRhIgogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICAgICAgcGFzcwogICAgICAgIGxvZ2dlci53YXJuaW5nKCJkZXZpY2U9Y3VkYSByZXF1ZXN0ZWQgYnV0IHRvcmNoIGhhcyBubyBDVURBIHN1cHBvcnQ7IHVzaW5nIENQVSIpCiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiBkZXZpY2UKCgpkZWYgbG9hZF9ubGlfbW9kZWwobW9kZWxfbmFtZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUsIGRldmljZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUsIG1vZGVsX2t3YXJnczogT3B0aW9uYWxbZGljdF0gPSBOb25lKToKICAgICIiIkxvYWQgdGhlIE5MSSBDcm9zc0VuY29kZXIgKGZhbGxzIGJhY2sgdG8gTWluaUxNMiBvbiBmYWlsdXJlKS4KCiAgICBkZXZpY2U9Tm9uZSAtPiBsaWJyYXJ5IGRlZmF1bHQgKEdQVSBpZiBhdmFpbGFibGUpOyBzZXQgImNwdSIgZm9yIHN0YWJpbGl0eS4KICAgIG1vZGVsX2t3YXJncyAtPiBleHRyYSBrd2FyZ3MgZm9yIHRoZSBtb2RlbCBsb2FkZXIgKGUuZy4gdG9yY2hfZHR5cGU9ZmxvYXQxNikuCiAgICAiIiIKICAgIGZyb20gc2VudGVuY2VfdHJhbnNmb3JtZXJzIGltcG9ydCBDcm9zc0VuY29kZXIKCiAgICBkZXZpY2UgPSBfc2FmZV9kZXZpY2UoZGV2aWNlKQogICAgY2hvc2VuID0gbW9kZWxfbmFtZSBvciBvcy5lbnZpcm9uLmdldChOTElfTU9ERUxfRU5WLCBOTElfTU9ERUxfUFJJTUFSWSkKICAgIHRyeToKICAgICAgICBsb2dnZXIuaW5mbyhmIkxvYWRpbmcgTkxJIENyb3NzRW5jb2Rlcjoge2Nob3Nlbn0gKGRldmljZT17ZGV2aWNlIG9yICdhdXRvJ30pIC4uLiIpCiAgICAgICAgbW9kZWwgPSBDcm9zc0VuY29kZXIoY2hvc2VuLCBkZXZpY2U9ZGV2aWNlLCBtb2RlbF9rd2FyZ3M9bW9kZWxfa3dhcmdzKQogICAgICAgIGxvZ2dlci5pbmZvKCJOTEkgQ3Jvc3NFbmNvZGVyIGxvYWRlZC4iKQogICAgICAgIHJldHVybiBtb2RlbCwgY2hvc2VuCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgaWYgY2hvc2VuICE9IE5MSV9NT0RFTF9GQUxMQkFDSzoKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoZiJOTEkgbW9kZWwge2Nob3Nlbn0gZmFpbGVkICh7ZX0pOyBmYWxsaW5nIGJhY2sgdG8ge05MSV9NT0RFTF9GQUxMQkFDS30iKQogICAgICAgICAgICByZXR1cm4gbG9hZF9ubGlfbW9kZWwoTkxJX01PREVMX0ZBTExCQUNLLCBkZXZpY2U9ZGV2aWNlLCBtb2RlbF9rd2FyZ3M9bW9kZWxfa3dhcmdzKQogICAgICAgIHJhaXNlCgoKZGVmIF9uZXV0cmFsX3JvdygpIC0+IGRpY3Q6CiAgICByZXR1cm4gewogICAgICAgICJubGlfY3R4X2VudGFpbHNfYW5zIjogTkVVVFJBTCwKICAgICAgICAibmxpX2N0eF9jb250cmFkaWN0c19hbnMiOiBORVVUUkFMLAogICAgICAgICJubGlfY3R4X25ldXRyYWxfYW5zIjogTkVVVFJBTCwKICAgICAgICAibmxpX2Fuc19lbnRhaWxzX2N0eCI6IE5FVVRSQUwsCiAgICAgICAgIm5saV9hbnNfY29udHJhZGljdHNfY3R4IjogTkVVVFJBTCwKICAgICAgICAibmxpX2Fuc19uZXV0cmFsX2N0eCI6IE5FVVRSQUwsCiAgICB9CgoKZGVmIGV4dHJhY3RfbmxpX2ZlYXR1cmVzKHF1ZXN0aW9uOiBzdHIsIGNvbnRleHQ6IHN0ciwgYW5zd2VyOiBzdHIsIG1vZGVsKSAtPiBkaWN0OgogICAgIiIiR3JvdXAgNDogZW50YWlsbWVudC9jb250cmFkaWN0aW9uIHByb2JhYmlsaXRpZXMsIGJvdGggZGlyZWN0aW9ucy4iIiIKICAgIGlmIG5vdCBjb250ZXh0LnN0cmlwKCk6CiAgICAgICAgcmV0dXJuIF9uZXV0cmFsX3JvdygpCgogICAgbG9naXRzID0gbW9kZWwucHJlZGljdChbW2NvbnRleHQsIGFuc3dlcl0sIFthbnN3ZXIsIGNvbnRleHRdXSwgYXBwbHlfc29mdG1heD1UcnVlKQogICAgcHJvYnMgPSBsb2dpdHMgaWYgbG9naXRzLnNoYXBlWzFdID09IDMgZWxzZSBOb25lCiAgICBpZiBwcm9icyBpcyBOb25lOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmV4cGVjdGVkIE5MSSBvdXRwdXQgc2hhcGUge2xvZ2l0cy5zaGFwZX07IGV4cGVjdGVkIChuLCAzKSIpCgogICAgcF9jdHggPSB7TEFCRUxTW2ldOiBmbG9hdChwcm9ic1swLCBpXSkgZm9yIGkgaW4gcmFuZ2UoMyl9CiAgICBwX2FucyA9IHtMQUJFTFNbaV06IGZsb2F0KHByb2JzWzEsIGldKSBmb3IgaSBpbiByYW5nZSgzKX0KCiAgICByZXR1cm4gewogICAgICAgICJubGlfY3R4X2VudGFpbHNfYW5zIjogcm91bmQocF9jdHhbImVudGFpbG1lbnQiXSwgNiksCiAgICAgICAgIm5saV9jdHhfY29udHJhZGljdHNfYW5zIjogcm91bmQocF9jdHhbImNvbnRyYWRpY3Rpb24iXSwgNiksCiAgICAgICAgIm5saV9jdHhfbmV1dHJhbF9hbnMiOiByb3VuZChwX2N0eFsibmV1dHJhbCJdLCA2KSwKICAgICAgICAibmxpX2Fuc19lbnRhaWxzX2N0eCI6IHJvdW5kKHBfYW5zWyJlbnRhaWxtZW50Il0sIDYpLAogICAgICAgICJubGlfYW5zX2NvbnRyYWRpY3RzX2N0eCI6IHJvdW5kKHBfYW5zWyJjb250cmFkaWN0aW9uIl0sIDYpLAogICAgICAgICJubGlfYW5zX25ldXRyYWxfY3R4Ijogcm91bmQocF9hbnNbIm5ldXRyYWwiXSwgNiksCiAgICB9CgoKZGVmIGV4dHJhY3RfbmxpX2ZlYXR1cmVzX2RmKGRmOiBwZC5EYXRhRnJhbWUsIG1vZGVsLCBiYXRjaF9zaXplOiBpbnQgPSA2NCwgbG9nX2V2ZXJ5OiBpbnQgPSAxMDAwKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJCYXRjaCBOTEkgZmVhdHVyZXM7IHByb2Nlc3NlcyBib3RoIChjdHgsIGFucykgYW5kIChhbnMsIGN0eCkgZGlyZWN0aW9ucy4iIiIKICAgIGltcG9ydCB0aW1lCgogICAgdG90YWwgPSBsZW4oZGYpCiAgICBsb2dnZXIuaW5mbyhmIkV4dHJhY3RpbmcgTkxJIGZlYXR1cmVzIGZvciB7dG90YWx9IHNhbXBsZXMgKDIgZGlyZWN0aW9ucyBlYWNoKS4uLiIpCiAgICByb3dzID0gW10KICAgIGJhdGNoX2N0eF9hbnMsIGJhdGNoX2Fuc19jdHgsIGJhdGNoX2lkeCA9IFtdLCBbXSwgW10KICAgIHQwID0gdGltZS50aW1lKCkKICAgIGxvZ2dlZCA9IDAKCiAgICBkZWYgZmx1c2goKToKICAgICAgICBub25sb2NhbCBiYXRjaF9jdHhfYW5zLCBiYXRjaF9hbnNfY3R4LCBiYXRjaF9pZHgsIGxvZ2dlZAogICAgICAgIGlmIG5vdCBiYXRjaF9pZHg6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIGFsbF9wcm9icyA9IG1vZGVsLnByZWRpY3QoYmF0Y2hfY3R4X2FucyArIGJhdGNoX2Fuc19jdHgsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgYXBwbHlfc29mdG1heD1UcnVlKQogICAgICAgIG4gPSBsZW4oYmF0Y2hfaWR4KQogICAgICAgIGZvciBrLCBpZHggaW4gZW51bWVyYXRlKGJhdGNoX2lkeCk6CiAgICAgICAgICAgIHBfY3R4ID0ge0xBQkVMU1tpXTogZmxvYXQoYWxsX3Byb2JzW2ssIGldKSBmb3IgaSBpbiByYW5nZSgzKX0KICAgICAgICAgICAgcF9hbnMgPSB7TEFCRUxTW2ldOiBmbG9hdChhbGxfcHJvYnNbbiArIGssIGldKSBmb3IgaSBpbiByYW5nZSgzKX0KICAgICAgICAgICAgcm93cy5hcHBlbmQoKGlkeCwgewogICAgICAgICAgICAgICAgIm5saV9jdHhfZW50YWlsc19hbnMiOiByb3VuZChwX2N0eFsiZW50YWlsbWVudCJdLCA2KSwKICAgICAgICAgICAgICAgICJubGlfY3R4X2NvbnRyYWRpY3RzX2FucyI6IHJvdW5kKHBfY3R4WyJjb250cmFkaWN0aW9uIl0sIDYpLAogICAgICAgICAgICAgICAgIm5saV9jdHhfbmV1dHJhbF9hbnMiOiByb3VuZChwX2N0eFsibmV1dHJhbCJdLCA2KSwKICAgICAgICAgICAgICAgICJubGlfYW5zX2VudGFpbHNfY3R4Ijogcm91bmQocF9hbnNbImVudGFpbG1lbnQiXSwgNiksCiAgICAgICAgICAgICAgICAibmxpX2Fuc19jb250cmFkaWN0c19jdHgiOiByb3VuZChwX2Fuc1siY29udHJhZGljdGlvbiJdLCA2KSwKICAgICAgICAgICAgICAgICJubGlfYW5zX25ldXRyYWxfY3R4Ijogcm91bmQocF9hbnNbIm5ldXRyYWwiXSwgNiksCiAgICAgICAgICAgIH0pKQogICAgICAgIGJhdGNoX2N0eF9hbnMsIGJhdGNoX2Fuc19jdHgsIGJhdGNoX2lkeCA9IFtdLCBbXSwgW10KICAgICAgICBpZiBsb2dfZXZlcnkgYW5kIGxlbihyb3dzKSAtIGxvZ2dlZCA+PSBsb2dfZXZlcnk6CiAgICAgICAgICAgIGxvZ2dlZCA9IGxlbihyb3dzKQogICAgICAgICAgICBlbGFwc2VkID0gdGltZS50aW1lKCkgLSB0MAogICAgICAgICAgICByYXRlID0gbGVuKHJvd3MpIC8gZWxhcHNlZCBpZiBlbGFwc2VkID4gMCBlbHNlIDAuMAogICAgICAgICAgICBldGEgPSAodG90YWwgLSBsZW4ocm93cykpIC8gcmF0ZSBpZiByYXRlID4gMCBlbHNlIDAuMAogICAgICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgICAgIGYiTkxJIGZlYXR1cmVzOiB7bGVuKHJvd3MpfS97dG90YWx9ICh7MTAwICogbGVuKHJvd3MpIC8gdG90YWw6LjFmfSUpIHwgIgogICAgICAgICAgICAgICAgZiJ7cmF0ZTouMWZ9IHNhbXBsZXMvcyB8IEVUQSB7ZXRhIC8gNjA6LjFmfSBtaW4iCiAgICAgICAgICAgICkKCiAgICBmb3IgaWR4LCByb3cgaW4gZGYuaXRlcnJvd3MoKToKICAgICAgICBjb250ZXh0LCBhbnN3ZXIgPSBzdHIocm93WyJjb250ZXh0Il0pLCBzdHIocm93WyJhbnN3ZXIiXSkKICAgICAgICBpZiBub3QgY29udGV4dC5zdHJpcCgpOgogICAgICAgICAgICByb3dzLmFwcGVuZCgoaWR4LCBfbmV1dHJhbF9yb3coKSkpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgYmF0Y2hfY3R4X2Fucy5hcHBlbmQoKGNvbnRleHQsIGFuc3dlcikpCiAgICAgICAgYmF0Y2hfYW5zX2N0eC5hcHBlbmQoKGFuc3dlciwgY29udGV4dCkpCiAgICAgICAgYmF0Y2hfaWR4LmFwcGVuZChpZHgpCiAgICAgICAgaWYgbGVuKGJhdGNoX2lkeCkgPj0gYmF0Y2hfc2l6ZSAqIDQ6CiAgICAgICAgICAgIGZsdXNoKCkKICAgIGZsdXNoKCkKCiAgICByb3dzLnNvcnQoa2V5PWxhbWJkYSB0OiB0WzBdKQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShbciBmb3IgXywgciBpbiByb3dzXSwgaW5kZXg9W2kgZm9yIGksIF8gaW4gcm93c10pCg==",
 "src/features/semantic_features.py": "IiIiClNlbWFudGljIHNpbWlsYXJpdHkgZmVhdHVyZSBleHRyYWN0aW9uIGZvciBIYWx1UklTQy4KCkdyb3VwIDcgZmVhdHVyZXMgKHJvYWRtYXAgwqc2KToKICBjb3NpbmVfY3R4X2FucywgY29zaW5lX3FfYW5zCgpVc2VzIHNlbnRlbmNlLXRyYW5zZm9ybWVycyBgYWxsLU1pbmlMTS1MNi12MmAgZW1iZWRkaW5ncy4KRW1wdHkgY29udGV4dCAtPiBjb3NpbmVfY3R4X2FucyA9IDAuMCAobm8gZXZpZGVuY2UgYXZhaWxhYmxlKS4KIiIiCgppbXBvcnQgbG9nZ2luZwpmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCkVNQkVERElOR19NT0RFTCA9ICJzZW50ZW5jZS10cmFuc2Zvcm1lcnMvYWxsLU1pbmlMTS1MNi12MiIKCgpkZWYgX3NhZmVfZGV2aWNlKGRldmljZTogT3B0aW9uYWxbc3RyXSkgLT4gT3B0aW9uYWxbc3RyXToKICAgICIiIlJldHVybiB0aGUgZGV2aWNlIHRvIHVzZTsgY3VkYSBpcyBvbmx5IGhvbm9yZWQgd2hlbiB0b3JjaCBzdXBwb3J0cyBpdC4iIiIKICAgIGlmIGRldmljZSA9PSAiY3VkYSI6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgdG9yY2gKCiAgICAgICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgICAgICByZXR1cm4gImN1ZGEiCiAgICAgICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgICAgICBwYXNzCiAgICAgICAgbG9nZ2VyLndhcm5pbmcoImRldmljZT1jdWRhIHJlcXVlc3RlZCBidXQgdG9yY2ggaGFzIG5vIENVREEgc3VwcG9ydDsgdXNpbmcgQ1BVIikKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIGRldmljZQoKCmRlZiBsb2FkX2VtYmVkZGluZ19tb2RlbChkZXZpY2U6IE9wdGlvbmFsW3N0cl0gPSBOb25lLCBtb2RlbF9rd2FyZ3M6IE9wdGlvbmFsW2RpY3RdID0gTm9uZSk6CiAgICAiIiJMb2FkIHRoZSBTQkVSVCBlbWJlZGRpbmcgbW9kZWwgKGxhenksIGNhY2hlZCBhdCBjYWxsIHNpdGUpLgoKICAgIGRldmljZT1Ob25lIC0+IGxpYnJhcnkgZGVmYXVsdCAoR1BVIGlmIGF2YWlsYWJsZSk7IHNldCAiY3B1IiBmb3Igc3RhYmlsaXR5LgogICAgbW9kZWxfa3dhcmdzIC0+IGV4dHJhIGt3YXJncyBmb3IgdGhlIG1vZGVsIGxvYWRlciAoZS5nLiB0b3JjaF9kdHlwZT1mbG9hdDE2KS4KICAgICIiIgogICAgZnJvbSBzZW50ZW5jZV90cmFuc2Zvcm1lcnMgaW1wb3J0IFNlbnRlbmNlVHJhbnNmb3JtZXIKCiAgICByZXR1cm4gU2VudGVuY2VUcmFuc2Zvcm1lcihFTUJFRERJTkdfTU9ERUwsIGRldmljZT1fc2FmZV9kZXZpY2UoZGV2aWNlKSwgbW9kZWxfa3dhcmdzPW1vZGVsX2t3YXJncykKCgpkZWYgX2Nvc2luZShhOiBucC5uZGFycmF5LCBiOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgIG5hLCBuYiA9IG5wLmxpbmFsZy5ub3JtKGEpLCBucC5saW5hbGcubm9ybShiKQogICAgaWYgbmEgPT0gMC4wIG9yIG5iID09IDAuMDoKICAgICAgICByZXR1cm4gMC4wCiAgICByZXR1cm4gZmxvYXQobnAuZG90KGEsIGIpIC8gKG5hICogbmIpKQoKCmRlZiBleHRyYWN0X3NlbWFudGljX2ZlYXR1cmVzKHF1ZXN0aW9uOiBzdHIsIGNvbnRleHQ6IHN0ciwgYW5zd2VyOiBzdHIsIG1vZGVsKSAtPiBkaWN0OgogICAgIiIiR3JvdXAgNzogY29zaW5lIHNpbWlsYXJpdHkgYmV0d2VlbiBhbnN3ZXIgYW5kIGNvbnRleHQvcXVlc3Rpb24uIiIiCiAgICB0ZXh0cyA9IFthbnN3ZXJdCiAgICBpZiBjb250ZXh0LnN0cmlwKCk6CiAgICAgICAgdGV4dHMuYXBwZW5kKGNvbnRleHQpCiAgICB0ZXh0cy5hcHBlbmQocXVlc3Rpb24pCiAgICB2ZWNzID0gbW9kZWwuZW5jb2RlKHRleHRzLCBjb252ZXJ0X3RvX251bXB5PVRydWUpCgogICAgYW5zX3ZlYyA9IHZlY3NbMF0KICAgIG9mZnNldCA9IDEKICAgIGNvc2luZV9jdHhfYW5zID0gX2Nvc2luZShhbnNfdmVjLCB2ZWNzW29mZnNldF0pIGlmIGNvbnRleHQuc3RyaXAoKSBlbHNlIDAuMAogICAgaWYgY29udGV4dC5zdHJpcCgpOgogICAgICAgIG9mZnNldCArPSAxCiAgICBjb3NpbmVfcV9hbnMgPSBfY29zaW5lKGFuc192ZWMsIHZlY3Nbb2Zmc2V0XSkKCiAgICByZXR1cm4gewogICAgICAgICJjb3NpbmVfY3R4X2FucyI6IHJvdW5kKGNvc2luZV9jdHhfYW5zLCA2KSwKICAgICAgICAiY29zaW5lX3FfYW5zIjogcm91bmQoY29zaW5lX3FfYW5zLCA2KSwKICAgIH0KCgpkZWYgZXh0cmFjdF9zZW1hbnRpY19mZWF0dXJlc19kZihkZjogcGQuRGF0YUZyYW1lLCBtb2RlbCwgYmF0Y2hfc2l6ZTogaW50ID0gNjQpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkJhdGNoIHNlbWFudGljIGZlYXR1cmVzOyBlbmNvZGVzIGVhY2ggY29sdW1uIG9uY2UgYW5kIHZlY3Rvcml6ZXMuIiIiCiAgICBsb2dnZXIuaW5mbyhmIkV4dHJhY3Rpbmcgc2VtYW50aWMgKFNCRVJUKSBmZWF0dXJlcyBmb3Ige2xlbihkZil9IHNhbXBsZXMuLi4iKQogICAgYW5zd2VycyA9IGRmWyJhbnN3ZXIiXS5hc3R5cGUoc3RyKS50b2xpc3QoKQogICAgY29udGV4dHMgPSBkZlsiY29udGV4dCJdLmFzdHlwZShzdHIpLnRvbGlzdCgpCiAgICBxdWVzdGlvbnMgPSBkZlsicXVlc3Rpb24iXS5hc3R5cGUoc3RyKS50b2xpc3QoKQoKICAgIGFuc192ZWNzID0gbW9kZWwuZW5jb2RlKGFuc3dlcnMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgY29udmVydF90b19udW1weT1UcnVlLCBzaG93X3Byb2dyZXNzX2Jhcj1UcnVlKQogICAgcV92ZWNzID0gbW9kZWwuZW5jb2RlKHF1ZXN0aW9ucywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBjb252ZXJ0X3RvX251bXB5PVRydWUpCiAgICBjdHhfdmVjcyA9ICgKICAgICAgICBtb2RlbC5lbmNvZGUoW2MgZm9yIGMgaW4gY29udGV4dHMgaWYgYy5zdHJpcCgpXSwgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBjb252ZXJ0X3RvX251bXB5PVRydWUpCiAgICAgICAgaWYgYW55KGMuc3RyaXAoKSBmb3IgYyBpbiBjb250ZXh0cykKICAgICAgICBlbHNlIG5wLnplcm9zKCgwLCBhbnNfdmVjcy5zaGFwZVsxXSkpCiAgICApCgogICAgY3R4X2l0ZXIgPSBpdGVyKGN0eF92ZWNzKQogICAgcm93cyA9IFtdCiAgICBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoY29udGV4dHMpOgogICAgICAgIGFuc192ZWMsIHFfdmVjID0gYW5zX3ZlY3NbaV0sIHFfdmVjc1tpXQogICAgICAgIGN0eF92ZWMgPSBuZXh0KGN0eF9pdGVyKSBpZiBjLnN0cmlwKCkgZWxzZSBOb25lCiAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAiY29zaW5lX2N0eF9hbnMiOiByb3VuZChfY29zaW5lKGFuc192ZWMsIGN0eF92ZWMpLCA2KSBpZiBjdHhfdmVjIGlzIG5vdCBOb25lIGVsc2UgMC4wLAogICAgICAgICAgICAiY29zaW5lX3FfYW5zIjogcm91bmQoX2Nvc2luZShhbnNfdmVjLCBxX3ZlYyksIDYpLAogICAgICAgIH0pCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MsIGluZGV4PWRmLmluZGV4KQo=",
 "src/models/config.py": "IiIiCkhhbHVSSVNDIHNpbmdsZSBzb3VyY2Ugb2YgdHJ1dGggZm9yIGV4cGVyaW1lbnQgY29uZmlndXJhdGlvbgooYmx1ZXByaW50IMKnMTE6ICJBbGwgcmFuZG9tIHNlZWRzIGRvY3VtZW50ZWQgaW4gYSBzaW5nbGUgY29uZmlnIGZpbGUiKS4KCkltcG9ydGVkIGJ5IHRyYWluX3BpcGVsaW5lLCBzaGFwX2FuYWx5c2lzLCBlcnJvcl9hbmFseXNpcywgZXZhbF9sbG1fanVkZ2UsCmV2YWxfZWZmaWNpZW5jeSDigJQgbmV2ZXIgcmVkZWZpbmUgc2VlZHMgZWxzZXdoZXJlLgoiIiIKCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdCgojIEV4cGVyaW1lbnQgc2VlZHMgKGJsdWVwcmludCDCpzcuMjogcmVwZWF0IGV2ZXJ5IGV4cGVyaW1lbnQgd2l0aCA0MiwgMTIzLCA0NTYpClNFRURTID0gWzQyLCAxMjMsIDQ1Nl0KCiMgQm9vdHN0cmFwIC8gc2FtcGxpbmcgc2VlZHMgKGZpeGVkLCBzZXBhcmF0ZSBmcm9tIGV4cGVyaW1lbnQgc2VlZHMpCkJPT1RTVFJBUF9TRUVEID0gNzc3ClNBTVBMRV9TRUVEID0gNDIKCiMgQ291bnRzCk5fQk9PVFNUUkFQID0gMTAwMAoKIyBQYXRocwpBUlRJRkFDVFNfRElSID0gUk9PVCAvICJhcnRpZmFjdHMiCk1PREVMU19ESVIgPSBBUlRJRkFDVFNfRElSIC8gIm1vZGVscyIKUkVTVUxUU19ESVIgPSBBUlRJRkFDVFNfRElSIC8gInJlc3VsdHMiCkZJR1VSRVNfRElSID0gQVJUSUZBQ1RTX0RJUiAvICJmaWd1cmVzIgpEQVRBX1BST0NFU1NFRCA9IFJPT1QgLyAiZGF0YSIgLyAicHJvY2Vzc2VkIgoKRkVBVFVSRVNfRlVMTCA9IERBVEFfUFJPQ0VTU0VEIC8gImZlYXR1cmVzX2Z1bGwucGFycXVldCIKRkVBVFVSRVNfRkFMTEJBQ0sgPSBEQVRBX1BST0NFU1NFRCAvICJmZWF0dXJlc19jb3JlLnBhcnF1ZXQiClFBX0NMRUFOID0gREFUQV9QUk9DRVNTRUQgLyAicWFfY2xlYW4ucGFycXVldCIK",
 "src/models/error_analysis.py": "IiIiCkhhbHVSSVNDIGVycm9yIGFuYWx5c2lzIChibHVlcHJpbnQgwqc4LjU6IGluc3BlY3QgMjAgd3JvbmcgcHJlZGljdGlvbnMsIDEwIEZQICsgMTAgRk4pLgoKU3RlcHM6CiAgMS4gUHJlZGljdCB0aGUgdGVzdCBzcGxpdCB3aXRoIHRoZSBjYWxpYnJhdGVkIG1vZGVsLgogIDIuIFNhbXBsZSAxMCBmYWxzZSBwb3NpdGl2ZXMgKyAxMCBmYWxzZSBuZWdhdGl2ZXMgKHNlZWRlZCkuCiAgMy4gQXV0by10YWcgZWFjaCBjYXNlIHdpdGggdGhlIGJsdWVwcmludCBlcnJvciB0YXhvbm9teSAoaGV1cmlzdGljIHJ1bGVzKS4KICA0LiBTYXZlIGEgcmV2aWV3YWJsZSBjYXNlIGR1bXAgKyBhIGNhdGVnb3J5LWNvdW50IHRhYmxlLgoKVGhlIHRheG9ub215IHRhZ3MgYXJlIGEgc3RhcnRpbmcgcG9pbnQgZm9yIG1hbnVhbCByZXZpZXcg4oCUIHZlcmlmeSBhbmQgYWRqdXN0CnRoZSBjYXRlZ29yaWVzIHdoZW4gd3JpdGluZyB0aGUgcGFwZXIncyBlcnJvciBhbmFseXNpcyBzZWN0aW9uLgoKUnVuIChyZXBvIHJvb3QsIC52ZW52KToKICBweXRob24gc3JjL21vZGVscy9lcnJvcl9hbmFseXNpcy5weQoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBvcwppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXSkpCgppbXBvcnQgam9ibGliCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJlcnJvcl9hbmFseXNpcyIpCgpmcm9tIHNyYy5tb2RlbHMuY29uZmlnIGltcG9ydCBGRUFUVVJFU19GQUxMQkFDSywgRkVBVFVSRVNfRlVMTCwgTU9ERUxTX0RJUiwgUUFfQ0xFQU4sIFJFU1VMVFNfRElSLCBTQU1QTEVfU0VFRAoKTl9GUCA9IDEwCk5fRk4gPSAxMAoKCmRlZiBsb2FkX3Rlc3Rfc2V0KCk6CiAgICBwYXRoID0gRkVBVFVSRVNfRlVMTCBpZiBGRUFUVVJFU19GVUxMLmV4aXN0cygpIGVsc2UgRkVBVFVSRVNfRkFMTEJBQ0sKICAgIGRmID0gcGQucmVhZF9wYXJxdWV0KHBhdGgpCiAgICBjbGVhbiA9IHBkLnJlYWRfcGFycXVldChRQV9DTEVBTikKICAgIHRleHRfY29scyA9IFtjIGZvciBjIGluIFsicXVlc3Rpb24iLCAiY29udGV4dCIsICJhbnN3ZXIiXSBpZiBjIGluIGNsZWFuLmNvbHVtbnNdCiAgICBpZiB0ZXh0X2NvbHM6CiAgICAgICAgZGYgPSBwZC5jb25jYXQoW2RmLCBjbGVhblt0ZXh0X2NvbHNdXSwgYXhpcz0xKQogICAgZmVhdHVyZV9jb2xzID0ganNvbi5sb2FkcygoTU9ERUxTX0RJUiAvICJmZWF0dXJlX25hbWVzLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIHRlc3RfZGYgPSBkZltkZlsic3BsaXQiXSA9PSAidGVzdCJdLmNvcHkoKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICBYX3Rlc3QgPSB0ZXN0X2RmW2ZlYXR1cmVfY29sc10udmFsdWVzCiAgICB5X3Rlc3QgPSB0ZXN0X2RmWyJsYWJlbCJdLnZhbHVlcwogICAgcmV0dXJuIHRlc3RfZGYsIFhfdGVzdCwgeV90ZXN0LCBmZWF0dXJlX2NvbHMKCgpkZWYgbG9hZF9wcmVkaWN0aW9ucyhYX3Rlc3QsIGZlYXR1cmVfY29scyk6CiAgICBidW5kbGUgPSBqb2JsaWIubG9hZChNT0RFTFNfRElSIC8gIm1vZGVsX3hnYm9vc3RfY2FsaWJyYXRlZC5qb2JsaWIiKQogICAgaWYgaXNpbnN0YW5jZShidW5kbGUsIGRpY3QpIGFuZCBidW5kbGUuZ2V0KCJraW5kIikgPT0gInhnYitwbGF0dCI6CiAgICAgICAgcmF3LCBwbGF0dCA9IGJ1bmRsZVsibW9kZWwiXSwgYnVuZGxlWyJjYWxpYnJhdG9yIl0KCiAgICAgICAgZGVmIHByZWRpY3RfcHJvYmEoWCk6CiAgICAgICAgICAgIHAgPSByYXcucHJlZGljdF9wcm9iYShYKVs6LCAxXQogICAgICAgICAgICByZXR1cm4gcGxhdHQucHJlZGljdF9wcm9iYShwLnJlc2hhcGUoLTEsIDEpKVs6LCAxXQoKICAgIGVsc2U6CiAgICAgICAgcHJlZGljdF9wcm9iYSA9IGJ1bmRsZS5wcmVkaWN0X3Byb2JhCiAgICB5X3Byb2IgPSBwcmVkaWN0X3Byb2JhKFhfdGVzdCkKICAgIHJldHVybiB5X3Byb2IKCgpkZWYgdGFnX2Nhc2Uocm93OiBwZC5TZXJpZXMpIC0+IHN0cjoKICAgICIiIkhldXJpc3RpYyB0YXhvbm9teSB0YWdnaW5nIChibHVlcHJpbnQgwqc4LjUgY2F0ZWdvcmllcykgLSByZXZpZXcgbWFudWFsbHkuIiIiCiAgICBuX3dvcmRzID0gZmxvYXQocm93LmdldCgibl93b3JkcyIsIDApKQogICAgb3ZlcmxhcCA9IGZsb2F0KHJvdy5nZXQoIm92ZXJsYXBfYW5zd2VyX2NvbnRleHQiLCAwLjApKQogICAgbmxpX2NvbnRyYSA9IGZsb2F0KHJvdy5nZXQoIm5saV9jdHhfY29udHJhZGljdHNfYW5zIiwgMC4wKSkKICAgIG5saV9lbnRhaWwgPSBmbG9hdChyb3cuZ2V0KCJubGlfY3R4X2VudGFpbHNfYW5zIiwgMC4wKSkKICAgIG5fZW50cyA9IGZsb2F0KHJvdy5nZXQoIm5fZW50aXRpZXNfYW5zd2VyIiwgMCkpCiAgICBjdHhfbGVuID0gbGVuKHN0cihyb3cuZ2V0KCJjb250ZXh0IiwgIiIpKSkKICAgIGFuc19sZW4gPSBsZW4oc3RyKHJvdy5nZXQoImFuc3dlciIsICIiKSkpCgogICAgaWYgbl93b3JkcyA8PSA0OgogICAgICAgIHJldHVybiAic2hvcnRfYW5zd2VyX2FtYmlndWl0eSIKICAgIGlmIG5saV9lbnRhaWwgPiAwLjggYW5kIG92ZXJsYXAgPiAwLjY6CiAgICAgICAgcmV0dXJuICJsYWJlbF9hbWJpZ3VpdHkiCiAgICBpZiBubGlfY29udHJhID4gMC41IGFuZCBvdmVybGFwIDwgMC4zOgogICAgICAgIHJldHVybiAibGFiZWxfYW1iaWd1aXR5IgogICAgaWYgbl9lbnRzID09IDA6CiAgICAgICAgcmV0dXJuICJlbnRpdHlfZXh0cmFjdGlvbl9mYWlsdXJlIgogICAgaWYgY3R4X2xlbiA8IDgwOgogICAgICAgIHJldHVybiAid2Vha19jb250ZXh0IgogICAgaWYgYW5zX2xlbiA8IDQwIGFuZCBuX3dvcmRzIDw9IDg6CiAgICAgICAgcmV0dXJuICJzaG9ydF9hbnN3ZXJfYW1iaWd1aXR5IgogICAgaWYgb3ZlcmxhcCA+IDAuNjoKICAgICAgICByZXR1cm4gInVuc3VwcG9ydGVkX2J1dF9zZW1hbnRpY2FsbHlfc2ltaWxhciIKICAgIHJldHVybiAib3RoZXIiCgoKZGVmIG1haW4oKToKICAgIHRlc3RfZGYsIFhfdGVzdCwgeV90ZXN0LCBmZWF0dXJlX2NvbHMgPSBsb2FkX3Rlc3Rfc2V0KCkKICAgIHlfcHJvYiA9IGxvYWRfcHJlZGljdGlvbnMoWF90ZXN0LCBmZWF0dXJlX2NvbHMpCiAgICB5X3ByZWQgPSAoeV9wcm9iID49IDAuNSkuYXN0eXBlKGludCkKCiAgICBmcF9pZHggPSBucC53aGVyZSgoeV9wcmVkID09IDEpICYgKHlfdGVzdCA9PSAwKSlbMF0KICAgIGZuX2lkeCA9IG5wLndoZXJlKCh5X3ByZWQgPT0gMCkgJiAoeV90ZXN0ID09IDEpKVswXQogICAgbG9nZ2VyLmluZm8oZiJNaXNjbGFzc2lmaWVkOiBGUD17bGVuKGZwX2lkeCl9LCBGTj17bGVuKGZuX2lkeCl9IikKCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoU0FNUExFX1NFRUQpCiAgICBmcF9zYW1wbGUgPSBybmcuY2hvaWNlKGZwX2lkeCwgc2l6ZT1taW4oTl9GUCwgbGVuKGZwX2lkeCkpLCByZXBsYWNlPUZhbHNlKQogICAgZm5fc2FtcGxlID0gcm5nLmNob2ljZShmbl9pZHgsIHNpemU9bWluKE5fRk4sIGxlbihmbl9pZHgpKSwgcmVwbGFjZT1GYWxzZSkKCiAgICByb3dzID0gW10KICAgIGZvciBpZHggaW4gbnAuY29uY2F0ZW5hdGUoW2ZwX3NhbXBsZSwgZm5fc2FtcGxlXSk6CiAgICAgICAgcm93ID0gdGVzdF9kZi5pbG9jW2lkeF0KICAgICAgICBjYXNlID0gewogICAgICAgICAgICAic2FtcGxlX2lkIjogc3RyKHJvd1sic2FtcGxlX2lkIl0pLAogICAgICAgICAgICAiZXJyb3JfdHlwZSI6ICJmYWxzZV9wb3NpdGl2ZSIgaWYgeV90ZXN0W2lkeF0gPT0gMCBlbHNlICJmYWxzZV9uZWdhdGl2ZSIsCiAgICAgICAgICAgICJ0cnVlX2xhYmVsIjogaW50KHlfdGVzdFtpZHhdKSwKICAgICAgICAgICAgInByZWRpY3RlZF9sYWJlbCI6IGludCh5X3ByZWRbaWR4XSksCiAgICAgICAgICAgICJwcm9iYWJpbGl0eSI6IHJvdW5kKGZsb2F0KHlfcHJvYltpZHhdKSwgNCksCiAgICAgICAgICAgICJjYXRlZ29yeSI6IHRhZ19jYXNlKHJvdyksCiAgICAgICAgICAgICJxdWVzdGlvbiI6IHN0cihyb3cuZ2V0KCJxdWVzdGlvbiIsICIiKSlbOjMwMF0sCiAgICAgICAgICAgICJjb250ZXh0Ijogc3RyKHJvdy5nZXQoImNvbnRleHQiLCAiIikpWzo0MDBdLAogICAgICAgICAgICAiYW5zd2VyIjogc3RyKHJvdy5nZXQoImFuc3dlciIsICIiKSlbOjMwMF0sCiAgICAgICAgICAgICJmZWF0dXJlcyI6IHsKICAgICAgICAgICAgICAgICJuX3dvcmRzIjogZmxvYXQocm93LmdldCgibl93b3JkcyIsIDApKSwKICAgICAgICAgICAgICAgICJvdmVybGFwX2Fuc3dlcl9jb250ZXh0Ijogcm91bmQoZmxvYXQocm93LmdldCgib3ZlcmxhcF9hbnN3ZXJfY29udGV4dCIsIDAuMCkpLCA0KSwKICAgICAgICAgICAgICAgICJubGlfY3R4X2NvbnRyYWRpY3RzX2FucyI6IHJvdW5kKGZsb2F0KHJvdy5nZXQoIm5saV9jdHhfY29udHJhZGljdHNfYW5zIiwgMC4wKSksIDQpLAogICAgICAgICAgICAgICAgIm5saV9jdHhfZW50YWlsc19hbnMiOiByb3VuZChmbG9hdChyb3cuZ2V0KCJubGlfY3R4X2VudGFpbHNfYW5zIiwgMC4wKSksIDQpLAogICAgICAgICAgICAgICAgIm5fZW50aXRpZXNfYW5zd2VyIjogZmxvYXQocm93LmdldCgibl9lbnRpdGllc19hbnN3ZXIiLCAwKSksCiAgICAgICAgICAgICAgICAiZW50aXR5X292ZXJsYXBfcmF0aW8iOiByb3VuZChmbG9hdChyb3cuZ2V0KCJlbnRpdHlfb3ZlcmxhcF9yYXRpbyIsIDAuMCkpLCA0KSwKICAgICAgICAgICAgICAgICJjb3NpbmVfY3R4X2FucyI6IHJvdW5kKGZsb2F0KHJvdy5nZXQoImNvc2luZV9jdHhfYW5zIiwgMC4wKSksIDQpLAogICAgICAgICAgICB9LAogICAgICAgIH0KICAgICAgICByb3dzLmFwcGVuZChjYXNlKQoKICAgIGNhc2VzX2RmID0gcGQuRGF0YUZyYW1lKHJvd3MpCiAgICBjb3VudHMgPSBjYXNlc19kZi5ncm91cGJ5KFsiZXJyb3JfdHlwZSIsICJjYXRlZ29yeSJdKS5zaXplKCkudW5zdGFjayhmaWxsX3ZhbHVlPTApCgogICAgc3VtbWFyeSA9IHsKICAgICAgICAibl90ZXN0IjogaW50KGxlbih5X3Rlc3QpKSwKICAgICAgICAibl9mYWxzZV9wb3NpdGl2ZXMiOiBpbnQobGVuKGZwX2lkeCkpLAogICAgICAgICJuX2ZhbHNlX25lZ2F0aXZlcyI6IGludChsZW4oZm5faWR4KSksCiAgICAgICAgInNhbXBsZWQiOiB7ImZwIjogaW50KGxlbihmcF9zYW1wbGUpKSwgImZuIjogaW50KGxlbihmbl9zYW1wbGUpKX0sCiAgICAgICAgImNhdGVnb3J5X2NvdW50cyI6IHsKICAgICAgICAgICAgImZhbHNlX3Bvc2l0aXZlIjogY2FzZXNfZGZbY2FzZXNfZGZbImVycm9yX3R5cGUiXSA9PSAiZmFsc2VfcG9zaXRpdmUiXVsiY2F0ZWdvcnkiXS52YWx1ZV9jb3VudHMoKS50b19kaWN0KCksCiAgICAgICAgICAgICJmYWxzZV9uZWdhdGl2ZSI6IGNhc2VzX2RmW2Nhc2VzX2RmWyJlcnJvcl90eXBlIl0gPT0gImZhbHNlX25lZ2F0aXZlIl1bImNhdGVnb3J5Il0udmFsdWVfY291bnRzKCkudG9fZGljdCgpLAogICAgICAgIH0sCiAgICAgICAgIm5vdGUiOiAiQXV0by10YWdnZWQgd2l0aCBoZXVyaXN0aWMgcnVsZXMgLSB2ZXJpZnkgY2F0ZWdvcmllcyBtYW51YWxseSBiZWZvcmUgcGFwZXIgdXNlLiIsCiAgICB9CgogICAgd2l0aCBvcGVuKFJFU1VMVFNfRElSIC8gImVycm9yX2FuYWx5c2lzX2Nhc2VzLmpzb24iLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHJvd3MsIGYsIGluZGVudD0yKQogICAgd2l0aCBvcGVuKFJFU1VMVFNfRElSIC8gImVycm9yX2FuYWx5c2lzLmpzb24iLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHN1bW1hcnksIGYsIGluZGVudD0yKQoKICAgIHByaW50KCJcbiIgKyAiPSIgKiA4MCkKICAgIHByaW50KCIgSGFsdVJJU0MgRXJyb3IgQW5hbHlzaXMgKDEwIEZQICsgMTAgRk4gb24gdGVzdCBzZXQpIikKICAgIHByaW50KCI9IiAqIDgwKQogICAgcHJpbnQoY291bnRzLmZpbGxuYSgwKS5hc3R5cGUoaW50KS50b19zdHJpbmcoKSkKICAgIHByaW50KCI9IiAqIDgwKQogICAgbG9nZ2VyLmluZm8oZiJTYXZlZCBlcnJvcl9hbmFseXNpcy5qc29uICsgZXJyb3JfYW5hbHlzaXNfY2FzZXMuanNvbiB0byB7UkVTVUxUU19ESVJ9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
 "src/models/eval_claims.py": "IiIiClQyOiBjbGFpbS12ZXJpZmljYXRpb24gZXZhbHVhdGlvbiBzZXQgKyBtZXRyaWNzIChyb2FkbWFwIEI3LjUgVDIpLgoKVHdvIG1vZGVzOgoKICAtLWJ1aWxkICAgICAgc2FtcGxlIE4gYW5zd2VycyAoZnJvbSBiNV9yZXZpZXdfY2FzZXMuanNvbiksIHNwbGl0IGludG8KICAgICAgICAgICAgICAgY2xhaW1zLCBydW4gdGhlIE5MSSB2ZXJpZmllciwgYW5kIHdyaXRlIGEgaHVtYW4tbGFiZWxpbmcgQ1NWOgogICAgICAgICAgICAgICBhcnRpZmFjdHMvcmVzdWx0cy9iNS9jbGFpbV9ldmFsX2Nhc2VzLmNzdgogICAgICAgICAgICAgICAoaHVtYW5fdmVyZGljdCBjb2x1bW4gc3RhcnRzIGVtcHR5IOKAlCBmaWxsIGJ5IGhhbmQsIEI1LjUtc3R5bGUpCgogIC0tZXZhbHVhdGUgICBjb21wYXJlIGEgZmlsbGVkIHNoZWV0IChodW1hbl92ZXJkaWN0KSBhZ2FpbnN0IG1vZGVsX3ZlcmRpY3Q6CiAgICAgICAgICAgICAgIHBlci1jbGFzcyBhZ3JlZW1lbnQsIGJpbmFyeSBwcmVjaXNpb24vcmVjYWxsIG9mICJmbGFnZ2VkIgogICAgICAgICAgICAgICAoY29udHJhZGljdGVkfHVuc3VwcG9ydGVkKSBjbGFpbXMgdnMgaHVtYW4gInByb2JsZW0iIGxhYmVscywKICAgICAgICAgICAgICAgYW5kIHRoZSBjb250aW5nZW5jeSB0YWJsZS4gRGVzY3JpcHRpdmUgb25seSDigJQgbm8gZml4ZWQKICAgICAgICAgICAgICAgcGFzcyB0aHJlc2hvbGRzIChCNS43KS4KCiAgLS1mcm9tLWZlZWRiYWNrICBleHBvcnQgZmVlZGJhY2stbG9nIHJvd3MgKGNsYWltX3RleHQgcHJlc2VudCkgYXMgYQogICAgICAgICAgICAgICBjbGFpbS1ldmFsIENTViBmb3IgaHVtYW4gbGFiZWxpbmcgLyB0aHJlc2hvbGQgdHVuaW5nLgoKICAtLXRpZXIzICAgICAgY2l0YXRpb24gcmVjYWxsQGsgb3ZlciB0aGUgZG9jdW1lbnQgaW5kZXg6IHJlYWRzIGEgQ1NWIHdpdGgKICAgICAgICAgICAgICAgW3F1ZXJ5LCBnb2xkX3NvdXJjZV0gcm93cyBhbmQgcmVwb3J0cyB3aGV0aGVyIHRoZSBnb2xkIHBhc3NhZ2UKICAgICAgICAgICAgICAgYXBwZWFycyBpbiB0aGUgdG9wLWsgcmV0cmlldmVkIHBhc3NhZ2VzIChrID0gMSwzLDUpLgoKUnVuIChyZXBvIHJvb3QsIC52ZW52OyAtLWJ1aWxkIGFuZCAtLXRpZXIzIG5lZWQgdGhlIGhlYXZ5IG1vZGVscyBsb2FkZWQpOgogIHB5dGhvbiBzcmMvbW9kZWxzL2V2YWxfY2xhaW1zLnB5IC0tYnVpbGQgLS1uIDQwCiAgcHl0aG9uIHNyYy9tb2RlbHMvZXZhbF9jbGFpbXMucHkgLS1ldmFsdWF0ZSBhcnRpZmFjdHMvcmVzdWx0cy9iNS9jbGFpbV9ldmFsX2Nhc2VzX3Jldmlld2VkLmNzdgogIHB5dGhvbiBzcmMvbW9kZWxzL2V2YWxfY2xhaW1zLnB5IC0tZnJvbS1mZWVkYmFjayBkYXRhL3Byb2Nlc3NlZC9mZWVkYmFja19ldmFsLmNzdgogIHB5dGhvbiBzcmMvbW9kZWxzL2V2YWxfY2xhaW1zLnB5IC0tdGllcjMgZGF0YS9wcm9jZXNzZWQvdGllcjNfcXVlcmllcy5jc3YKIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUk9PVCkpCgpCNV9SRVZJRVdfQ0FTRVMgPSBST09UIC8gImFydGlmYWN0cyIgLyAicmVzdWx0cyIgLyAiYjUiIC8gImI1X3Jldmlld19jYXNlcy5qc29uIgpFVkFMX0NTViA9IFJPT1QgLyAiYXJ0aWZhY3RzIiAvICJyZXN1bHRzIiAvICJiNSIgLyAiY2xhaW1fZXZhbF9jYXNlcy5jc3YiCgpIVU1BTl9MQUJFTFMgPSB7InN1cHBvcnRlZCIsICJjb250cmFkaWN0ZWQiLCAidW5zdXBwb3J0ZWQiLCAiIn0gICMgIiIgPSBub3QgeWV0IHJldmlld2VkCgpDT0xVTU5TID0gWyJzYW1wbGVfaWQiLCAiY2xhaW1faWQiLCAiY2xhaW1fdGV4dCIsICJjb250ZXh0IiwgIm1vZGVsX3ZlcmRpY3QiLAogICAgICAgICAgICJtb2RlbF9jb25maWRlbmNlIiwgImh1bWFuX3ZlcmRpY3QiLCAibm90ZXMiXQoKCmRlZiBfYnVpbGRfZXZhbF9zZXQobjogaW50ID0gNDAsIHNlZWQ6IGludCA9IDQyKSAtPiBsaXN0OgogICAgIiIiU2FtcGxlIGFuc3dlcnMgLT4gY2xhaW1zIC0+IHZlcmRpY3RzIChyZXF1aXJlcyBOTEkgbW9kZWxzICsgY29udGV4dHMpLiIiIgogICAgZnJvbSBzcmMuYXBpLm1haW4gaW1wb3J0IGxvYWRfZmVhdHVyZV9tb2RlbHMgICMgcmV1c2UgdGhlIEFQSSdzIG1vZGVsIGxvYWRpbmcKICAgIGZyb20gc3JjLmNsYWltcy5kZWNvbXBvc2UgaW1wb3J0IHNwbGl0X2NsYWltcwogICAgZnJvbSBzcmMuY2xhaW1zLnZlcmlmeSBpbXBvcnQgdmVyaWZ5X2NsYWltcwoKICAgIGNhc2VzID0ganNvbi5sb2FkcyhCNV9SRVZJRVdfQ0FTRVMucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBwaWNrcyA9IHJuZy5jaG9pY2UobGVuKGNhc2VzKSwgc2l6ZT1taW4obiwgbGVuKGNhc2VzKSksIHJlcGxhY2U9RmFsc2UpCgogICAgbmxpID0gbG9hZF9mZWF0dXJlX21vZGVscygpLmdldCgibmxpIikKICAgIGlmIG5saSBpcyBOb25lOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTkxJIG1vZGVsIG5vdCBsb2FkZWQgKHN0YXJ0IHZpYSB0aGUgQVBJIG9yIEhBTFVfQVBJX1BSRUxPQUQ9MSkiKQoKICAgIHJvd3MgPSBbXQogICAgZm9yIGkgaW4gcGlja3M6CiAgICAgICAgY2FzZSA9IGNhc2VzW2ludChpKV0KICAgICAgICBjbGFpbXMgPSBzcGxpdF9jbGFpbXMoY2FzZS5nZXQoImFuc3dlciIsICIiKSkKICAgICAgICBpZiBub3QgY2xhaW1zOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJlc3VsdCA9IHZlcmlmeV9jbGFpbXMoY2xhaW1zLCBjYXNlLmdldCgiY29udGV4dCIsICIiKSwgbmxpKQogICAgICAgIGZvciBjIGluIHJlc3VsdFsiY2xhaW1zIl06CiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsKICAgICAgICAgICAgICAgICJzYW1wbGVfaWQiOiBjYXNlWyJzYW1wbGVfaWQiXSwKICAgICAgICAgICAgICAgICJjbGFpbV9pZCI6IGNbImlkIl0sCiAgICAgICAgICAgICAgICAiY2xhaW1fdGV4dCI6IGNbInRleHQiXSwKICAgICAgICAgICAgICAgICJjb250ZXh0IjogKGNhc2UuZ2V0KCJjb250ZXh0Iikgb3IgIiIpWzoyMDAwXSwKICAgICAgICAgICAgICAgICJtb2RlbF92ZXJkaWN0IjogY1sidmVyZGljdCJdLAogICAgICAgICAgICAgICAgIm1vZGVsX2NvbmZpZGVuY2UiOiBjWyJjb25maWRlbmNlIl0sCiAgICAgICAgICAgICAgICAiaHVtYW5fdmVyZGljdCI6ICIiLAogICAgICAgICAgICAgICAgIm5vdGVzIjogIiIsCiAgICAgICAgICAgIH0pCiAgICByZXR1cm4gcm93cwoKCmRlZiB3cml0ZV9ldmFsX2Nzdihyb3dzOiBsaXN0LCBwYXRoOiBQYXRoID0gRVZBTF9DU1YpIC0+IE5vbmU6CiAgICBpbXBvcnQgY3N2CgogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgZjoKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihmLCBmaWVsZG5hbWVzPUNPTFVNTlMpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICB3cml0ZXIud3JpdGVyb3dzKHJvd3MpCiAgICBwcmludChmIldyb3RlIHtsZW4ocm93cyl9IGNsYWltIGNhc2VzIHRvIHtwYXRofSIpCiAgICBwcmludCgiRmlsbCB0aGUgaHVtYW5fdmVyZGljdCBjb2x1bW4gKHN1cHBvcnRlZHxjb250cmFkaWN0ZWR8dW5zdXBwb3J0ZWQpIGJ5IGhhbmQsIikKICAgIHByaW50KCJzYXZlIGFzIGNsYWltX2V2YWxfY2FzZXNfcmV2aWV3ZWQuY3N2LCB0aGVuIHJ1biAtLWV2YWx1YXRlIG9uIGl0LiIpCgoKZGVmIGZsYWdnZWQodmVyZGljdDogc3RyKSAtPiBib29sOgogICAgcmV0dXJuIHZlcmRpY3QgaW4gKCJjb250cmFkaWN0ZWQiLCAidW5zdXBwb3J0ZWQiKQoKCmRlZiBldmFsdWF0ZV9zaGVldChwYXRoOiBQYXRoKSAtPiBkaWN0OgogICAgaW1wb3J0IGNzdgoKICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBmOgogICAgICAgIHJvd3MgPSBbciBmb3IgciBpbiBjc3YuRGljdFJlYWRlcihmKSBpZiAoci5nZXQoImh1bWFuX3ZlcmRpY3QiKSBvciAiIikuc3RyaXAoKV0KCiAgICByZXZpZXdlZCA9IFtyIGZvciByIGluIHJvd3MgaWYgKHIuZ2V0KCJodW1hbl92ZXJkaWN0Iikgb3IgIiIpLnN0cmlwKCkgaW4gSFVNQU5fTEFCRUxTIC0geyIifV0KICAgIGlmIG5vdCByZXZpZXdlZDoKICAgICAgICBwcmludChmIk5vIHJldmlld2VkIHJvd3MgaW4ge3BhdGh9IChodW1hbl92ZXJkaWN0IGVtcHR5KS4iKQogICAgICAgIHJldHVybiB7fQoKICAgIGNvbnRpbmdlbmN5ID0geyJ0cCI6IDAsICJmcCI6IDAsICJ0biI6IDAsICJmbiI6IDB9CiAgICBwZXJfY2xhc3NfYWdyZWVtZW50ID0ge30KICAgIGZvciB2IGluICgic3VwcG9ydGVkIiwgImNvbnRyYWRpY3RlZCIsICJ1bnN1cHBvcnRlZCIpOgogICAgICAgIHN1YiA9IFtyIGZvciByIGluIHJldmlld2VkIGlmIHJbImh1bWFuX3ZlcmRpY3QiXSA9PSB2XQogICAgICAgIGFncmVlID0gc3VtKDEgZm9yIHIgaW4gc3ViIGlmIHJbIm1vZGVsX3ZlcmRpY3QiXSA9PSB2KQogICAgICAgIHBlcl9jbGFzc19hZ3JlZW1lbnRbdl0gPSB7Im4iOiBsZW4oc3ViKSwgIm1vZGVsX2FncmVlcyI6IGFncmVlfQoKICAgIGZvciByIGluIHJldmlld2VkOgogICAgICAgIGh1bWFuX2ZsYWcgPSBmbGFnZ2VkKHJbImh1bWFuX3ZlcmRpY3QiXSkKICAgICAgICBtb2RlbF9mbGFnID0gZmxhZ2dlZChyWyJtb2RlbF92ZXJkaWN0Il0pCiAgICAgICAgaWYgbW9kZWxfZmxhZyBhbmQgaHVtYW5fZmxhZzoKICAgICAgICAgICAgY29udGluZ2VuY3lbInRwIl0gKz0gMQogICAgICAgIGVsaWYgbW9kZWxfZmxhZyBhbmQgbm90IGh1bWFuX2ZsYWc6CiAgICAgICAgICAgIGNvbnRpbmdlbmN5WyJmcCJdICs9IDEKICAgICAgICBlbGlmIG5vdCBtb2RlbF9mbGFnIGFuZCBub3QgaHVtYW5fZmxhZzoKICAgICAgICAgICAgY29udGluZ2VuY3lbInRuIl0gKz0gMQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNvbnRpbmdlbmN5WyJmbiJdICs9IDEKCiAgICB0cCwgZnAsIHRuLCBmbiA9IChjb250aW5nZW5jeVtrXSBmb3IgayBpbiAoInRwIiwgImZwIiwgInRuIiwgImZuIikpCiAgICBwcmVjaXNpb24gPSB0cCAvICh0cCArIGZwKSBpZiB0cCArIGZwIGVsc2UgTm9uZQogICAgcmVjYWxsID0gdHAgLyAodHAgKyBmbikgaWYgdHAgKyBmbiBlbHNlIE5vbmUKICAgIGFjY3VyYWN5ID0gKHRwICsgdG4pIC8gKHRwICsgZnAgKyB0biArIGZuKSBpZiAodHAgKyBmcCArIHRuICsgZm4pIGVsc2UgTm9uZQogICAgb3ZlcmFsbF9hZ3JlZSA9IHN1bSgxIGZvciByIGluIHJldmlld2VkIGlmIHJbIm1vZGVsX3ZlcmRpY3QiXSA9PSByWyJodW1hbl92ZXJkaWN0Il0pIC8gbGVuKHJldmlld2VkKQoKICAgIHByaW50KGYiUmV2aWV3ZWQgY2xhaW1zOiB7bGVuKHJldmlld2VkKX0iKQogICAgZm9yIHYsIHN0YXRzIGluIHBlcl9jbGFzc19hZ3JlZW1lbnQuaXRlbXMoKToKICAgICAgICBwcmludChmIiAge3Y6PDE0fSBuPXtzdGF0c1snbiddOjw0fSBtb2RlbCBhZ3JlZXM9e3N0YXRzWydtb2RlbF9hZ3JlZXMnXX0iKQogICAgcHJpbnQoZiJCaW5hcnkgZmxhZ2dpbmcgKGNvbnRyYWRpY3RlZHx1bnN1cHBvcnRlZCBhcyAncHJvYmxlbScpOiIpCiAgICBwcmludChmIiAgVFA9e3RwfSBGUD17ZnB9IFROPXt0bn0gRk49e2ZufSIpCiAgICBwcmludChmIiAgcHJlY2lzaW9uPXtwcmVjaXNpb246LjNmfSIgaWYgcHJlY2lzaW9uIGlzIG5vdCBOb25lIGVsc2UgIiAgcHJlY2lzaW9uPW4vYSIpCiAgICBwcmludChmIiAgcmVjYWxsPXtyZWNhbGw6LjNmfSIgaWYgcmVjYWxsIGlzIG5vdCBOb25lIGVsc2UgIiAgcmVjYWxsPW4vYSIpCiAgICBwcmludChmIiAgYWNjdXJhY3k9e2FjY3VyYWN5Oi4zZn0iIGlmIGFjY3VyYWN5IGlzIG5vdCBOb25lIGVsc2UgIiAgYWNjdXJhY3k9bi9hIikKICAgIHByaW50KGYiICBvdmVyYWxsIHZlcmRpY3QgYWdyZWVtZW50PXtvdmVyYWxsX2FncmVlOi4zZn0iKQogICAgcmV0dXJuIHsiY29udGluZ2VuY3kiOiBjb250aW5nZW5jeSwgInBlcl9jbGFzc19hZ3JlZW1lbnQiOiBwZXJfY2xhc3NfYWdyZWVtZW50LAogICAgICAgICAgICAicHJlY2lzaW9uIjogcHJlY2lzaW9uLCAicmVjYWxsIjogcmVjYWxsLCAiYWNjdXJhY3kiOiBhY2N1cmFjeSwKICAgICAgICAgICAgIm92ZXJhbGxfYWdyZWVtZW50Ijogb3ZlcmFsbF9hZ3JlZX0KCgpkZWYgZXhwb3J0X2ZlZWRiYWNrX2Nzdihyb3dzOiBsaXN0LCBwYXRoOiBQYXRoKSAtPiBpbnQ6CiAgICAiIiJGZWVkYmFjayByb3dzIHdpdGggY2xhaW1fdGV4dCAtPiBjbGFpbS1ldmFsIHNoZWV0IHJvd3MgZm9yIGxhYmVsaW5nLiIiIgogICAgaW1wb3J0IGNzdgoKICAgIGV4cG9ydCA9IFtdCiAgICBmb3IgaSwgciBpbiBlbnVtZXJhdGUocm93cyk6CiAgICAgICAgY2xhaW0gPSAoci5nZXQoImNsYWltX3RleHQiKSBvciAiIikuc3RyaXAoKQogICAgICAgIGlmIG5vdCBjbGFpbToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBleHBvcnQuYXBwZW5kKHsKICAgICAgICAgICAgInNhbXBsZV9pZCI6IGYiZmVlZGJhY2s6e2l9IiwKICAgICAgICAgICAgImNsYWltX2lkIjogMCwKICAgICAgICAgICAgImNsYWltX3RleHQiOiBjbGFpbSwKICAgICAgICAgICAgImNvbnRleHQiOiAoci5nZXQoImNvbnRleHQiKSBvciAiIilbOjIwMDBdLAogICAgICAgICAgICAibW9kZWxfdmVyZGljdCI6IHIuZ2V0KCJ2ZXJkaWN0Iikgb3IgIiIsCiAgICAgICAgICAgICJtb2RlbF9jb25maWRlbmNlIjogIiIsCiAgICAgICAgICAgICJodW1hbl92ZXJkaWN0IjogIiIsCiAgICAgICAgICAgICJub3RlcyI6IGYiZmVlZGJhY2s9e3IuZ2V0KCdmZWVkYmFjaycpfSIsCiAgICAgICAgfSkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIGY6CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1DT0xVTU5TKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyhleHBvcnQpCiAgICByZXR1cm4gbGVuKGV4cG9ydCkKCgpkZWYgY2l0YXRpb25fcmVjYWxsKHF1ZXJpZXM6IGxpc3QsIGluZGV4LCBrX2xpc3Q9KDEsIDMsIDUpKSAtPiBkaWN0OgogICAgIiIicmVjYWxsQGs6IGZyYWN0aW9uIG9mIHF1ZXJpZXMgd2hvc2UgZ29sZCBwYXNzYWdlIGlzIGluIHRoZSB0b3Atay4iIiIKICAgIGltcG9ydCBjc3YKCiAgICBoaXRzID0ge2s6IDAgZm9yIGsgaW4ga19saXN0fQogICAgbiA9IDAKICAgIGZvciBxIGluIHF1ZXJpZXM6CiAgICAgICAgZ29sZCA9IChxLmdldCgiZ29sZF9zb3VyY2UiKSBvciAiIikuc3RyaXAoKQogICAgICAgIGlmIG5vdCBnb2xkOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG4gKz0gMQogICAgICAgIHRvcCA9IGluZGV4LnNlYXJjaChxLmdldCgicXVlcnkiLCAiIiksIHRvcF9rPW1heChrX2xpc3QpKQogICAgICAgIGZvciBrIGluIGtfbGlzdDoKICAgICAgICAgICAgdG9wayA9IHRvcFs6a10KICAgICAgICAgICAgaWRzID0ge3AuZ2V0KCJpZCIpIGZvciBwIGluIHRvcGt9CiAgICAgICAgICAgIHNyY3MgPSB7cC5nZXQoInNvdXJjZSIpIGZvciBwIGluIHRvcGt9CiAgICAgICAgICAgIGlmIGdvbGQgaW4gaWRzIG9yIGdvbGQgaW4gc3JjczoKICAgICAgICAgICAgICAgIGhpdHNba10gKz0gMQogICAgcmV0dXJuIHsibiI6IG4sICJyZWNhbGxfYXRfayI6IHtrOiAoaGl0c1trXSAvIG4gaWYgbiBlbHNlIE5vbmUpIGZvciBrIGluIGtfbGlzdH19CgoKZGVmIGxvYWRfY3N2X3Jvd3MocGF0aDogUGF0aCkgLT4gbGlzdDoKICAgIGltcG9ydCBjc3YKCiAgICB3aXRoIG9wZW4ocGF0aCwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgZjoKICAgICAgICByZXR1cm4gW2RpY3QocikgZm9yIHIgaW4gY3N2LkRpY3RSZWFkZXIoZildCgoKZGVmIF9sb2FkX2ZlZWRiYWNrX2xvZyhwYXRoOiBQYXRoID0gUk9PVCAvICJkYXRhIiAvICJwcm9jZXNzZWQiIC8gImZlZWRiYWNrX2xvZy5qc29ubCIpIC0+IGxpc3Q6CiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICByZXR1cm4gW10KICAgIHJvd3MgPSBbXQogICAgZm9yIGxpbmUgaW4gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcm93cy5hcHBlbmQoanNvbi5sb2FkcyhsaW5lKSkKICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgY29udGludWUKICAgIHJldHVybiByb3dzCgoKZGVmIG1haW4oYXJndjogbGlzdCB8IE5vbmUgPSBOb25lKSAtPiBpbnQ6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iVDItVDQgY2xhaW0tdmVyaWZpY2F0aW9uIGV2YWx1YXRpb24gc2V0ICsgbWV0cmljcyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJ1aWxkIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0iYnVpbGQgdGhlIGh1bWFuLWxhYmVsaW5nIENTViIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW4iLCB0eXBlPWludCwgZGVmYXVsdD00MCwgaGVscD0iYW5zd2VycyB0byBzYW1wbGUgZm9yIC0tYnVpbGQiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWV2YWx1YXRlIiwgbWV0YXZhcj0iQ1NWIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJldmFsdWF0ZSBhIHJldmlld2VkIHNoZWV0IikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZnJvbS1mZWVkYmFjayIsIG1ldGF2YXI9Ik9VVF9DU1YiLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9ImV4cG9ydCBmZWVkYmFjay1sb2cgY2xhaW0gcm93cyBhcyBhIGxhYmVsaW5nIENTViIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRpZXIzIiwgbWV0YXZhcj0iUVVFUklFU19DU1YiLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9ImNpdGF0aW9uIHJlY2FsbEBrIG92ZXIgdGhlIGRvY3VtZW50IGluZGV4IChuZWVkcyBpbmRleCArIGVtYmVkZGVyKSIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoYXJndikKCiAgICBpZiBhcmdzLmJ1aWxkOgogICAgICAgIHdyaXRlX2V2YWxfY3N2KF9idWlsZF9ldmFsX3NldChuPWFyZ3Mubiwgc2VlZD1hcmdzLnNlZWQpKQogICAgICAgIHJldHVybiAwCiAgICBpZiBhcmdzLmV2YWx1YXRlOgogICAgICAgIGV2YWx1YXRlX3NoZWV0KFBhdGgoYXJncy5ldmFsdWF0ZSkpCiAgICAgICAgcmV0dXJuIDAKICAgIGlmIGFyZ3MuZnJvbV9mZWVkYmFjazoKICAgICAgICByb3dzID0gX2xvYWRfZmVlZGJhY2tfbG9nKCkKICAgICAgICBuID0gZXhwb3J0X2ZlZWRiYWNrX2Nzdihyb3dzLCBQYXRoKGFyZ3MuZnJvbV9mZWVkYmFjaykpCiAgICAgICAgcHJpbnQoZiJFeHBvcnRlZCB7bn0gZmVlZGJhY2sgcm93cyB0byB7YXJncy5mcm9tX2ZlZWRiYWNrfSAoZmlsbCBodW1hbl92ZXJkaWN0LCB0aGVuIC0tZXZhbHVhdGUpLiIpCiAgICAgICAgcmV0dXJuIDAKICAgIGlmIGFyZ3MudGllcjM6CiAgICAgICAgZnJvbSBzcmMucmV0cmlldmFsLmluZGV4IGltcG9ydCBSZXRyaWV2YWxJbmRleAoKICAgICAgICBpbmRleCA9IFJldHJpZXZhbEluZGV4KGVtYmVkX2ZuPV9lbWJlZF9mcm9tX2FwaSkKICAgICAgICBpZiBpbmRleC5zdGF0dXMoKVsibl9wYXNzYWdlcyJdID09IDA6CiAgICAgICAgICAgIHByaW50KCJEb2N1bWVudCBpbmRleCBpcyBlbXB0eSDigJQgdXBsb2FkIGRvY3VtZW50cyBmaXJzdCAoUE9TVCAvYXBpL21sL2luZGV4KS4iKQogICAgICAgICAgICByZXR1cm4gMQogICAgICAgIHF1ZXJpZXMgPSBsb2FkX2Nzdl9yb3dzKFBhdGgoYXJncy50aWVyMykpCiAgICAgICAgcmVwb3J0ID0gY2l0YXRpb25fcmVjYWxsKHF1ZXJpZXMsIGluZGV4KQogICAgICAgIHByaW50KGYiQ2l0YXRpb24gcmVjYWxsQGsgb3ZlciB7cmVwb3J0WyduJ119IHF1ZXJpZXM6IikKICAgICAgICBmb3IgaywgdiBpbiByZXBvcnRbInJlY2FsbF9hdF9rIl0uaXRlbXMoKToKICAgICAgICAgICAgcHJpbnQoZiIgIHJlY2FsbEB7a306IHt2Oi4zZn0iIGlmIHYgaXMgbm90IE5vbmUgZWxzZSBmIiAgcmVjYWxsQHtrfTogbi9hIikKICAgICAgICByZXR1cm4gMAogICAgcGFyc2VyLnByaW50X2hlbHAoKQogICAgcmV0dXJuIDEKCgpkZWYgX2VtYmVkX2Zyb21fYXBpKHRleHRzOiBsaXN0KSAtPiBucC5uZGFycmF5OgogICAgIiIiUmV1c2UgdGhlIEFQSSdzIGVtYmVkZGVyIHdpdGhvdXQgc3RhcnRpbmcgdGhlIHNlcnZlci4iIiIKICAgIGZyb20gc3JjLmFwaS5tYWluIGltcG9ydCBfZW1iZWRfdGV4dHMKCiAgICByZXR1cm4gX2VtYmVkX3RleHRzKHRleHRzKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg==",
 "src/models/eval_efficiency.py": "IiIiCkhhbHVSSVNDIGVmZmljaWVuY3kgJiBjb3N0IGFuYWx5c2lzIChibHVlcHJpbnQgwqc4LjEgZWZmaWNpZW5jeSBibG9jaykuCgpPbiBhIHNhbXBsZSBvZiB0aGUgdGVzdCBzZXQsIHRpbWVzIGVhY2ggZmVhdHVyZS1leHRyYWN0aW9uIGdyb3VwLCBtb2RlbApwcmVkaWN0aW9uLCBhbmQgU0hBUCBleHBsYW5hdGlvbjsgcmVwb3J0cyBwNTAvcDk1LCBtb2RlbCBhcnRpZmFjdCBzaXplLCBhbmQKYW4gZXN0aW1hdGVkIGNvc3QgcGVyIDEsMDAwIHByZWRpY3Rpb25zIHZzIGFuIExMTSBqdWRnZS4KClJ1biAocmVwbyByb290LCAudmVudik6CiAgcHl0aG9uIHNyYy9tb2RlbHMvZXZhbF9lZmZpY2llbmN5LnB5CiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkKCmltcG9ydCBqb2JsaWIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoImV2YWxfZWZmaWNpZW5jeSIpCgpmcm9tIHNyYy5tb2RlbHMuY29uZmlnIGltcG9ydCBGRUFUVVJFU19GQUxMQkFDSywgRkVBVFVSRVNfRlVMTCwgTU9ERUxTX0RJUiwgUUFfQ0xFQU4sIFJFU1VMVFNfRElSLCBTQU1QTEVfU0VFRAoKTl9TQU1QTEVTID0gMjAwCgoKZGVmIHBlcmNlbnRpbGUodmFscywgcCk6CiAgICByZXR1cm4gZmxvYXQobnAucGVyY2VudGlsZSh2YWxzLCBwKSkKCgpkZWYgbWFpbigpOgogICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXSkpCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5leHRyYWN0X2ZlYXR1cmVzIGltcG9ydCAoCiAgICAgICAgZXh0cmFjdF9oZWRnaW5nX2ZlYXR1cmVzLAogICAgICAgIGV4dHJhY3RfbGVuZ3RoX2ZlYXR1cmVzLAogICAgICAgIGV4dHJhY3RfbGV4aWNhbF9mZWF0dXJlcywKICAgICAgICBleHRyYWN0X251bWVyaWNfZmVhdHVyZXMsCiAgICAgICAgbG9hZF9oZWF2eV9tb2RlbHMsCiAgICApCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5lbnRpdHlfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3RfZW50aXR5X2ZlYXR1cmVzCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5ubGlfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3RfbmxpX2ZlYXR1cmVzCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5zZW1hbnRpY19mZWF0dXJlcyBpbXBvcnQgZXh0cmFjdF9zZW1hbnRpY19mZWF0dXJlcwoKICAgIHBhdGggPSBGRUFUVVJFU19GVUxMIGlmIEZFQVRVUkVTX0ZVTEwuZXhpc3RzKCkgZWxzZSBGRUFUVVJFU19GQUxMQkFDSwogICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQocGF0aCkKICAgIGNsZWFuID0gcGQucmVhZF9wYXJxdWV0KFFBX0NMRUFOKQogICAgdGV4dF9jb2xzID0gW2MgZm9yIGMgaW4gWyJxdWVzdGlvbiIsICJjb250ZXh0IiwgImFuc3dlciJdIGlmIGMgaW4gY2xlYW4uY29sdW1uc10KICAgIGlmIHRleHRfY29sczoKICAgICAgICBkZiA9IHBkLmNvbmNhdChbZGYsIGNsZWFuW3RleHRfY29sc11dLCBheGlzPTEpCiAgICBmZWF0dXJlX2NvbHMgPSBqc29uLmxvYWRzKChNT0RFTFNfRElSIC8gImZlYXR1cmVfbmFtZXMuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgdGVzdCA9IGRmW2RmWyJzcGxpdCJdID09ICJ0ZXN0Il0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhTQU1QTEVfU0VFRCkKICAgIGlkeCA9IHJuZy5jaG9pY2UobGVuKHRlc3QpLCBzaXplPW1pbihOX1NBTVBMRVMsIGxlbih0ZXN0KSksIHJlcGxhY2U9RmFsc2UpCiAgICBzYW1wbGUgPSB0ZXN0Lmlsb2NbaWR4XQoKICAgIGJ1bmRsZSA9IGpvYmxpYi5sb2FkKE1PREVMU19ESVIgLyAibW9kZWxfeGdib29zdF9jYWxpYnJhdGVkLmpvYmxpYiIpCiAgICBpZiBpc2luc3RhbmNlKGJ1bmRsZSwgZGljdCkgYW5kIGJ1bmRsZS5nZXQoImtpbmQiKSA9PSAieGdiK3BsYXR0IjoKICAgICAgICByYXcsIHBsYXR0ID0gYnVuZGxlWyJtb2RlbCJdLCBidW5kbGVbImNhbGlicmF0b3IiXQoKICAgICAgICBkZWYgcHJlZGljdF9wcm9iYShYKToKICAgICAgICAgICAgcCA9IHJhdy5wcmVkaWN0X3Byb2JhKFgpWzosIDFdCiAgICAgICAgICAgIHJldHVybiBwbGF0dC5wcmVkaWN0X3Byb2JhKHAucmVzaGFwZSgtMSwgMSkpWzosIDFdCgogICAgZWxzZToKICAgICAgICByYXcsIHBsYXR0ID0gTm9uZSwgTm9uZQogICAgICAgIHByZWRpY3RfcHJvYmEgPSBidW5kbGUucHJlZGljdF9wcm9iYQoKICAgIGltcG9ydCBzaGFwCgogICAgZXhwbGFpbmVyID0gam9ibGliLmxvYWQoTU9ERUxTX0RJUiAvICJzaGFwX2V4cGxhaW5lci5qb2JsaWIiKQoKICAgIG1vZGVscyA9IGxvYWRfaGVhdnlfbW9kZWxzKCkKICAgIG5scCwgbmxpLCBlbWJlZGRlciA9IG1vZGVsc1sibmxwIl0sIG1vZGVsc1sibmxpIl0sIG1vZGVsc1siZW1iZWRkZXIiXQoKICAgIHRpbWluZ3MgPSB7ZzogW10gZm9yIGcgaW4gWyJjb3JlX2xleGljYWwiLCAiZW50aXR5IiwgIm5saSIsICJzZW1hbnRpYyIsICJtb2RlbCIsICJzaGFwIl19CiAgICBmb3IgXywgcm93IGluIHNhbXBsZS5pdGVycm93cygpOgogICAgICAgIHEsIGMsIGEgPSBzdHIocm93WyJxdWVzdGlvbiJdKSwgc3RyKHJvd1siY29udGV4dCJdKSwgc3RyKHJvd1siYW5zd2VyIl0pCgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGV4dHJhY3RfbGVuZ3RoX2ZlYXR1cmVzKHEsIGMsIGEpCiAgICAgICAgZXh0cmFjdF9sZXhpY2FsX2ZlYXR1cmVzKHEsIGMsIGEpCiAgICAgICAgZXh0cmFjdF9udW1lcmljX2ZlYXR1cmVzKHEsIGMsIGEpCiAgICAgICAgZXh0cmFjdF9oZWRnaW5nX2ZlYXR1cmVzKHEsIGMsIGEpCiAgICAgICAgdGltaW5nc1siY29yZV9sZXhpY2FsIl0uYXBwZW5kKCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMCkKCiAgICAgICAgdDAgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZXh0cmFjdF9lbnRpdHlfZmVhdHVyZXMocSwgYywgYSwgbmxwKQogICAgICAgIHRpbWluZ3NbImVudGl0eSJdLmFwcGVuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwKSAqIDEwMDApCgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGV4dHJhY3RfbmxpX2ZlYXR1cmVzKHEsIGMsIGEsIG5saSkKICAgICAgICB0aW1pbmdzWyJubGkiXS5hcHBlbmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgKiAxMDAwKQoKICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBleHRyYWN0X3NlbWFudGljX2ZlYXR1cmVzKHEsIGMsIGEsIGVtYmVkZGVyKQogICAgICAgIHRpbWluZ3NbInNlbWFudGljIl0uYXBwZW5kKCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMCkKCiAgICAgICAgZmVhdHMgPSB7CiAgICAgICAgICAgICJuX2NoYXJzIjogMCwgIm5fd29yZHMiOiAwLCAibl9zZW50ZW5jZXMiOiAxLCAiYXZnX3dvcmRfbGVuIjogMCwKICAgICAgICAgICAgIm92ZXJsYXBfYW5zd2VyX2NvbnRleHQiOiAwLjAsICJvdmVybGFwX2Fuc3dlcl9xdWVzdGlvbiI6IDAuMCwKICAgICAgICAgICAgImphY2NhcmRfYW5zX2N0eCI6IDAuMCwgImphY2NhcmRfYW5zX3EiOiAwLjAsCiAgICAgICAgICAgICJuX2VudGl0aWVzX2Fuc3dlciI6IDAsICJuX2VudGl0aWVzX2NvbnRleHQiOiAwLAogICAgICAgICAgICAiZW50aXR5X292ZXJsYXBfcmF0aW8iOiAxLjAsICJub3ZlbF9lbnRpdHlfcmF0aW8iOiAwLjAsCiAgICAgICAgICAgICJubGlfY3R4X2VudGFpbHNfYW5zIjogMSAvIDMsICJubGlfY3R4X2NvbnRyYWRpY3RzX2FucyI6IDEgLyAzLCAibmxpX2N0eF9uZXV0cmFsX2FucyI6IDEgLyAzLAogICAgICAgICAgICAibmxpX2Fuc19lbnRhaWxzX2N0eCI6IDEgLyAzLCAibmxpX2Fuc19jb250cmFkaWN0c19jdHgiOiAxIC8gMywgIm5saV9hbnNfbmV1dHJhbF9jdHgiOiAxIC8gMywKICAgICAgICAgICAgIm5fbnVtYmVyc19hbnN3ZXIiOiAwLCAibl9udW1iZXJzX2NvbnRleHQiOiAwLCAibnVtYmVyX292ZXJsYXBfcmF0aW8iOiAxLjAsICJub3ZlbF9udW1iZXJzIjogMCwKICAgICAgICAgICAgImhlZGdlX2NvdW50IjogMCwgImhlZGdlX2RlbnNpdHkiOiAwLjAsCiAgICAgICAgICAgICJjb3NpbmVfY3R4X2FucyI6IDAuMCwgImNvc2luZV9xX2FucyI6IDAuMCwKICAgICAgICB9CiAgICAgICAgZmVhdHMudXBkYXRlKHtrOiBmbG9hdChyb3dba10pIGZvciBrIGluIGZlYXR1cmVfY29scyBpZiBrIGluIHJvd30pCgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHByZWRpY3RfcHJvYmEobnAuYXJyYXkoW1tmZWF0c1tjXSBmb3IgYyBpbiBmZWF0dXJlX2NvbHNdXSwgZHR5cGU9bnAuZmxvYXQ2NCkpCiAgICAgICAgdGltaW5nc1sibW9kZWwiXS5hcHBlbmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCkgKiAxMDAwKQoKICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBleHBsYWluZXIuc2hhcF92YWx1ZXMobnAuYXJyYXkoW1tmZWF0c1tjXSBmb3IgYyBpbiBmZWF0dXJlX2NvbHNdXSwgZHR5cGU9bnAuZmxvYXQ2NCkpCiAgICAgICAgdGltaW5nc1sic2hhcCJdLmFwcGVuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwKSAqIDEwMDApCgogICAgc3VtbWFyeSA9IHsKICAgICAgICAibl9zYW1wbGVzIjogaW50KGxlbihzYW1wbGUpKSwKICAgICAgICAibGF0ZW5jeV9tcyI6IHsKICAgICAgICAgICAgZ3JvdXA6IHsicDUwIjogcm91bmQocGVyY2VudGlsZSh2LCA1MCksIDIpLCAicDk1Ijogcm91bmQocGVyY2VudGlsZSh2LCA5NSksIDIpLCAibWVhbiI6IHJvdW5kKGZsb2F0KG5wLm1lYW4odikpLCAyKX0KICAgICAgICAgICAgZm9yIGdyb3VwLCB2IGluIHRpbWluZ3MuaXRlbXMoKQogICAgICAgIH0sCiAgICAgICAgInRvdGFsX3Blcl9zYW1wbGVfbXMiOiB7CiAgICAgICAgICAgICJwNTAiOiByb3VuZChmbG9hdChucC5tZWRpYW4oW3N1bSh0aW1pbmdzW2ddW2ldIGZvciBnIGluIHRpbWluZ3MpIGZvciBpIGluIHJhbmdlKGxlbihzYW1wbGUpKV0pKSwgMikKICAgICAgICB9LAogICAgICAgICJtb2RlbF9hcnRpZmFjdF9tYiI6IHsKICAgICAgICAgICAgIm1vZGVsX3hnYm9vc3RfY2FsaWJyYXRlZC5qb2JsaWIiOiByb3VuZChvcy5wYXRoLmdldHNpemUoTU9ERUxTX0RJUiAvICJtb2RlbF94Z2Jvb3N0X2NhbGlicmF0ZWQuam9ibGliIikgLyAxZTYsIDIpLAogICAgICAgICAgICAibW9kZWxfeGdib29zdF9yYXcuam9ibGliIjogcm91bmQob3MucGF0aC5nZXRzaXplKE1PREVMU19ESVIgLyAibW9kZWxfeGdib29zdF9yYXcuam9ibGliIikgLyAxZTYsIDIpLAogICAgICAgIH0sCiAgICAgICAgImNvc3RfcGVyXzEwMDBfcHJlZGljdGlvbnNfdXNkIjogewogICAgICAgICAgICAiaGFsdXJpc2NfbG9jYWwiOiAwLjAwMSwgICMgZWxlY3RyaWNpdHkgb25seTsgbm8gQVBJIGNvc3QKICAgICAgICAgICAgImxsbV9qdWRnZV9lc3RpbWF0ZSI6IDAuMTEsICAjIDEwMDAgeCB+MTEwMCB0b2tlbnMgYXQgJDAuMjAvTSBpbiArICQxLjIwL00gb3V0CiAgICAgICAgfSwKICAgIH0KCiAgICB3aXRoIG9wZW4oUkVTVUxUU19ESVIgLyAibGF0ZW5jeV9hbmFseXNpcy5qc29uIiwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChzdW1tYXJ5LCBmLCBpbmRlbnQ9MikKCiAgICBwcmludCgiXG4iICsgIj0iICogODApCiAgICBwcmludChmIiBIYWx1UklTQyBMYXRlbmN5IEFuYWx5c2lzIChuPXtsZW4oc2FtcGxlKX0gdGVzdCBzYW1wbGVzKSIpCiAgICBwcmludCgiPSIgKiA4MCkKICAgIHByaW50KGYieydjb21wb25lbnQnOjwxNn17J3A1MCBtcyc6PjEwfXsncDk1IG1zJzo+MTB9eydtZWFuIG1zJzo+MTB9IikKICAgIGZvciBncm91cCwgdiBpbiB0aW1pbmdzLml0ZW1zKCk6CiAgICAgICAgcHJpbnQoZiJ7Z3JvdXA6PDE2fXtwZXJjZW50aWxlKHYsNTApOj4xMC4yZn17cGVyY2VudGlsZSh2LDk1KTo+MTAuMmZ9e25wLm1lYW4odik6PjEwLjJmfSIpCiAgICBwcmludCgiPSIgKiA4MCkKICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQgbGF0ZW5jeV9hbmFseXNpcy5qc29uIHRvIHtSRVNVTFRTX0RJUn0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
 "src/models/eval_llm_judge.py": "IiIiCkhhbHVSSVNDIExMTS1hcy1qdWRnZSBjb21wYXJpc29uIChibHVlcHJpbnQgwqc4LjQgZXh0ZXJuYWwgYmFzZWxpbmUgKyBjb3N0IHRhYmxlKS4KClJ1bnMgR1BUIDUuNiBMdW5hIGFzIGEgaGFsbHVjaW5hdGlvbiBqdWRnZSBvbiBhIGJhbGFuY2VkIHNhbXBsZSBvZiB0aGUgdGVzdCBzZXQKYW5kIGNvbXBhcmVzIGFnYWluc3QgdGhlIFhHQm9vc3QgbW9kZWw6IGFjY3VyYWN5L3ByZWNpc2lvbi9yZWNhbGwvRjEsIGFncmVlbWVudCwKbGF0ZW5jeSwgYW5kIGEgcmVhbCB0b2tlbi1jb3N0IGVzdGltYXRlLgoKQ29zdDogfjIwMCBzYW1wbGVzIHggfjEuMUsgdG9rZW5zIOKJiCAkMC4wNS0wLjE1IGRlcGVuZGluZyBvbiBtb2RlbCBwcmljaW5nLgpPdmVycmlkZSBzYW1wbGUgc2l6ZSB3aXRoIEhBTFVfSlVER0VfTi4KClJ1biAocmVwbyByb290LCAudmVudik6CiAgcHl0aG9uIHNyYy9tb2RlbHMvZXZhbF9sbG1fanVkZ2UucHkKIiIiCgppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0pKQoKaW1wb3J0IGpvYmxpYgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIGRvdGVudiBpbXBvcnQgbG9hZF9kb3RlbnYKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGFjY3VyYWN5X3Njb3JlLCBmMV9zY29yZSwgcHJlY2lzaW9uX3Njb3JlLCByZWNhbGxfc2NvcmUKCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoImV2YWxfbGxtX2p1ZGdlIikKCmxvYWRfZG90ZW52KCkgICMgT1BFTkFJX0FQSV9LRVksIE9QRU5BSV9NT0RFTAoKZnJvbSBzcmMubW9kZWxzLmNvbmZpZyBpbXBvcnQgRkVBVFVSRVNfRkFMTEJBQ0ssIEZFQVRVUkVTX0ZVTEwsIE1PREVMU19ESVIsIFFBX0NMRUFOLCBSRVNVTFRTX0RJUiwgU0FNUExFX1NFRUQKCk5fU0FNUExFUyA9IGludChvcy5lbnZpcm9uLmdldCgiSEFMVV9KVURHRV9OIiwgIjIwMCIpKQoKUFJJQ0lORyA9IHsiaW5wdXRfcGVyX210b2siOiAwLjIwLCAib3V0cHV0X3Blcl9tdG9rIjogMS4yMH0gICMgR1BUIDUuNiBMdW5hIChyb2FkbWFwIFBoYXNlIDUpCgoKZGVmIGxvYWRfdGVzdF9zZXQoKToKICAgIHBhdGggPSBGRUFUVVJFU19GVUxMIGlmIEZFQVRVUkVTX0ZVTEwuZXhpc3RzKCkgZWxzZSBGRUFUVVJFU19GQUxMQkFDSwogICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQocGF0aCkKICAgIGNsZWFuID0gcGQucmVhZF9wYXJxdWV0KFFBX0NMRUFOKQogICAgdGV4dF9jb2xzID0gW2MgZm9yIGMgaW4gWyJxdWVzdGlvbiIsICJjb250ZXh0IiwgImFuc3dlciJdIGlmIGMgaW4gY2xlYW4uY29sdW1uc10KICAgIGlmIHRleHRfY29sczoKICAgICAgICBkZiA9IHBkLmNvbmNhdChbZGYsIGNsZWFuW3RleHRfY29sc11dLCBheGlzPTEpCiAgICBmZWF0dXJlX2NvbHMgPSBqc29uLmxvYWRzKChNT0RFTFNfRElSIC8gImZlYXR1cmVfbmFtZXMuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgdGVzdCA9IGRmW2RmWyJzcGxpdCJdID09ICJ0ZXN0Il0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgcmV0dXJuIHRlc3QsIGZlYXR1cmVfY29scwoKCmRlZiB4Z2JfcHJvYmFiaWxpdGllcyh0ZXN0LCBmZWF0dXJlX2NvbHMpOgogICAgYnVuZGxlID0gam9ibGliLmxvYWQoTU9ERUxTX0RJUiAvICJtb2RlbF94Z2Jvb3N0X2NhbGlicmF0ZWQuam9ibGliIikKICAgIGlmIGlzaW5zdGFuY2UoYnVuZGxlLCBkaWN0KSBhbmQgYnVuZGxlLmdldCgia2luZCIpID09ICJ4Z2IrcGxhdHQiOgogICAgICAgIHJhdywgcGxhdHQgPSBidW5kbGVbIm1vZGVsIl0sIGJ1bmRsZVsiY2FsaWJyYXRvciJdCgogICAgICAgIGRlZiBwcmVkaWN0X3Byb2JhKFgpOgogICAgICAgICAgICBwID0gcmF3LnByZWRpY3RfcHJvYmEoWClbOiwgMV0KICAgICAgICAgICAgcmV0dXJuIHBsYXR0LnByZWRpY3RfcHJvYmEocC5yZXNoYXBlKC0xLCAxKSlbOiwgMV0KCiAgICBlbHNlOgogICAgICAgIHByZWRpY3RfcHJvYmEgPSBidW5kbGUucHJlZGljdF9wcm9iYQogICAgWCA9IHRlc3RbZmVhdHVyZV9jb2xzXS52YWx1ZXMKICAgIHJldHVybiBwcmVkaWN0X3Byb2JhKFgpCgoKSlVER0VfU1lTVEVNID0gKAogICAgIllvdSBhcmUgYW4gZXhwZXJ0IGhhbGx1Y2luYXRpb24tanVkZ2UuIEdpdmVuIGEgcXVlc3Rpb24sIGEgcmVmZXJlbmNlIGNvbnRleHQsIGFuZCBhbiBhbnN3ZXIsICIKICAgICJkZWNpZGUgd2hldGhlciB0aGUgYW5zd2VyIGNvbnRhaW5zIGhhbGx1Y2luYXRlZCBjb250ZW50ICh1bnN1cHBvcnRlZCwgY29udHJhZGljdG9yeSwgb3IgZmFicmljYXRlZCAiCiAgICAiaW5mb3JtYXRpb24gcmVsYXRpdmUgdG8gdGhlIGNvbnRleHQpLiBSZXNwb25kIHdpdGggSlNPTiBvbmx5OiAiCiAgICAneyJqdWRnbWVudCI6ICJoYWxsdWNpbmF0ZWQifCJncm91bmRlZCIsICJjb25maWRlbmNlIjogMC4wLTEuMH0nCikKCgpkZWYganVkZ2Vfb25lKGNsaWVudCwgbW9kZWxfbmFtZSwgcSwgYywgYSk6CiAgICBpbXBvcnQganNvbiBhcyBfanNvbgogICAgaW1wb3J0IHJlCgogICAgdXNlciA9IGYiUXVlc3Rpb246IHtxfVxuQ29udGV4dDoge2Mgb3IgJyhub25lKSd9XG5BbnN3ZXI6IHthfSIKICAgIHJlc3AgPSBjbGllbnQuY2hhdC5jb21wbGV0aW9ucy5jcmVhdGUoCiAgICAgICAgbW9kZWw9bW9kZWxfbmFtZSwKICAgICAgICBtZXNzYWdlcz1bCiAgICAgICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IEpVREdFX1NZU1RFTX0sCiAgICAgICAgICAgIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiB1c2VyfSwKICAgICAgICBdLAogICAgICAgIG1heF9jb21wbGV0aW9uX3Rva2Vucz0xMDAwLAogICAgICAgIHJlc3BvbnNlX2Zvcm1hdD17InR5cGUiOiAianNvbl9vYmplY3QifSwKICAgICkKICAgIGNvbnRlbnQgPSByZXNwLmNob2ljZXNbMF0ubWVzc2FnZS5jb250ZW50LnN0cmlwKCkKICAgIGNvbnRlbnQgPSByZS5zdWIociJeYGBgKD86anNvbik/fGBgYCQiLCAiIiwgY29udGVudCwgZmxhZ3M9cmUuTVVMVElMSU5FKS5zdHJpcCgpCiAgICB0cnk6CiAgICAgICAgZGF0YSA9IF9qc29uLmxvYWRzKGNvbnRlbnRbY29udGVudC5maW5kKCJ7IikgOiBjb250ZW50LnJmaW5kKCJ9IikgKyAxXSkKICAgIGV4Y2VwdCBfanNvbi5KU09ORGVjb2RlRXJyb3I6CiAgICAgICAganVkZ21lbnQgPSAiaGFsbHVjaW5hdGVkIiBpZiAiaGFsbHVjaW5hdGVkIiBpbiBjb250ZW50Lmxvd2VyKCkgZWxzZSAiZ3JvdW5kZWQiCiAgICAgICAgcmV0dXJuIGp1ZGdtZW50LCAwLjUsIHJlc3AudXNhZ2UKICAgIGp1ZGdtZW50ID0gZGF0YS5nZXQoImp1ZGdtZW50IiwgImdyb3VuZGVkIikKICAgIGNvbmZpZGVuY2UgPSBmbG9hdChkYXRhLmdldCgiY29uZmlkZW5jZSIsIDAuNSkpCiAgICByZXR1cm4ganVkZ21lbnQsIGNvbmZpZGVuY2UsIHJlc3AudXNhZ2UKCgpkZWYgbWFpbigpOgogICAgZnJvbSBvcGVuYWkgaW1wb3J0IE9wZW5BSQoKICAgIGFwaV9rZXkgPSBvcy5lbnZpcm9uLmdldCgiT1BFTkFJX0FQSV9LRVkiKQogICAgaWYgbm90IGFwaV9rZXk6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiT1BFTkFJX0FQSV9LRVkgbm90IHNldCBpbiAuZW52IOKAlCBjYW5ub3QgcnVuIExMTSBqdWRnZS4iKQoKICAgIG1vZGVsX25hbWUgPSBvcy5lbnZpcm9uLmdldCgiT1BFTkFJX01PREVMIiwgImdwdC01LjYtbHVuYSIpCiAgICBjbGllbnQgPSBPcGVuQUkoYXBpX2tleT1hcGlfa2V5KQoKICAgIHRlc3QsIGZlYXR1cmVfY29scyA9IGxvYWRfdGVzdF9zZXQoKQogICAgeV9wcm9iX3hnYiA9IHhnYl9wcm9iYWJpbGl0aWVzKHRlc3QsIGZlYXR1cmVfY29scykKICAgIHlfcHJlZF94Z2IgPSAoeV9wcm9iX3hnYiA+PSAwLjUpLmFzdHlwZShpbnQpCiAgICB5X3RydWUgPSB0ZXN0WyJsYWJlbCJdLnZhbHVlcwoKICAgICMgQmFsYW5jZWQgc2FtcGxlIChoYWxmIGhhbGx1Y2luYXRlZCwgaGFsZiBncm91bmRlZCkKICAgIHBvcyA9IG5wLndoZXJlKHlfdHJ1ZSA9PSAxKVswXQogICAgbmVnID0gbnAud2hlcmUoeV90cnVlID09IDApWzBdCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoU0FNUExFX1NFRUQpCiAgICBuX2hhbGYgPSBOX1NBTVBMRVMgLy8gMgogICAgc2FtcGxlX2lkeCA9IG5wLmNvbmNhdGVuYXRlKFtybmcuY2hvaWNlKHBvcywgbl9oYWxmLCByZXBsYWNlPUZhbHNlKSwgcm5nLmNob2ljZShuZWcsIG5faGFsZiwgcmVwbGFjZT1GYWxzZSldKQoKICAgIGp1ZGdtZW50cywgY29uZnMsIGxhdGVuY2llcyA9IFtdLCBbXSwgW10KICAgIGluX3Rva2VucyA9IG91dF90b2tlbnMgPSAwCiAgICBmb3IgaSwgaWR4IGluIGVudW1lcmF0ZShzYW1wbGVfaWR4KToKICAgICAgICByb3cgPSB0ZXN0Lmlsb2NbaWR4XQogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGp1ZGdtZW50LCBjb25mLCB1c2FnZSA9IGp1ZGdlX29uZShjbGllbnQsIG1vZGVsX25hbWUsIHJvd1sicXVlc3Rpb24iXSwgcm93WyJjb250ZXh0Il0sIHJvd1siYW5zd2VyIl0pCiAgICAgICAgbGF0ZW5jaWVzLmFwcGVuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwKSAqIDEwMDApCiAgICAgICAganVkZ21lbnRzLmFwcGVuZCgxIGlmIGp1ZGdtZW50ID09ICJoYWxsdWNpbmF0ZWQiIGVsc2UgMCkKICAgICAgICBjb25mcy5hcHBlbmQoY29uZikKICAgICAgICBpbl90b2tlbnMgKz0gdXNhZ2UucHJvbXB0X3Rva2VucwogICAgICAgIG91dF90b2tlbnMgKz0gdXNhZ2UuY29tcGxldGlvbl90b2tlbnMKICAgICAgICBpZiAoaSArIDEpICUgNTAgPT0gMDoKICAgICAgICAgICAgbG9nZ2VyLmluZm8oZiJqdWRnZWQge2kgKyAxfS97bGVuKHNhbXBsZV9pZHgpfSIpCgogICAgeV9wcmVkX2p1ZGdlID0gbnAuYXJyYXkoanVkZ21lbnRzKQogICAgeV90cnVlX3N1YiA9IHlfdHJ1ZVtzYW1wbGVfaWR4XQogICAgeV94Z2Jfc3ViID0geV9wcmVkX3hnYltzYW1wbGVfaWR4XQoKICAgICMgTWNOZW1hcjoganVkZ2UgdnMgWEdCb29zdCBvbiB0aGUgc2FtZSAyMDAgc2FtcGxlcyAob2ZmLWRpYWdvbmFsID0gZGlzY29yZGFudCkKICAgIGp1ZGdlX3dyb25nID0geV9wcmVkX2p1ZGdlICE9IHlfdHJ1ZV9zdWIKICAgIHhnYl93cm9uZyA9IHlfeGdiX3N1YiAhPSB5X3RydWVfc3ViCiAgICBib3RoX3dyb25nID0gaW50KChqdWRnZV93cm9uZyAmIHhnYl93cm9uZykuc3VtKCkpCiAgICBqdWRnZV93cm9uZ194Z2JfcmlnaHQgPSBpbnQoKGp1ZGdlX3dyb25nICYgfnhnYl93cm9uZykuc3VtKCkpCiAgICBqdWRnZV9yaWdodF94Z2Jfd3JvbmcgPSBpbnQoKH5qdWRnZV93cm9uZyAmIHhnYl93cm9uZykuc3VtKCkpCiAgICBib3RoX3JpZ2h0ID0gaW50KCh+anVkZ2Vfd3JvbmcgJiB+eGdiX3dyb25nKS5zdW0oKSkKICAgIGZyb20gc3RhdHNtb2RlbHMuc3RhdHMuY29udGluZ2VuY3lfdGFibGVzIGltcG9ydCBtY25lbWFyCgogICAgbWNuID0gbWNuZW1hcigKICAgICAgICBbW2JvdGhfd3JvbmcsIGp1ZGdlX3dyb25nX3hnYl9yaWdodF0sIFtqdWRnZV9yaWdodF94Z2Jfd3JvbmcsIGJvdGhfcmlnaHRdXSwKICAgICAgICBleGFjdD1GYWxzZSwKICAgICAgICBjb3JyZWN0aW9uPVRydWUsCiAgICApCiAgICBtY25lbWFyX3AgPSBmbG9hdChtY24ucHZhbHVlKQoKICAgIGNvc3QgPSAoaW5fdG9rZW5zIC8gMWU2KSAqIFBSSUNJTkdbImlucHV0X3Blcl9tdG9rIl0gKyAob3V0X3Rva2VucyAvIDFlNikgKiBQUklDSU5HWyJvdXRwdXRfcGVyX210b2siXQoKICAgIHJlc3VsdHMgPSB7CiAgICAgICAgIm5fc2FtcGxlcyI6IGludChsZW4oc2FtcGxlX2lkeCkpLAogICAgICAgICJtb2RlbCI6IG1vZGVsX25hbWUsCiAgICAgICAgImp1ZGdlIjogewogICAgICAgICAgICAiYWNjdXJhY3kiOiByb3VuZChmbG9hdChhY2N1cmFjeV9zY29yZSh5X3RydWVfc3ViLCB5X3ByZWRfanVkZ2UpKSwgNCksCiAgICAgICAgICAgICJwcmVjaXNpb24iOiByb3VuZChmbG9hdChwcmVjaXNpb25fc2NvcmUoeV90cnVlX3N1YiwgeV9wcmVkX2p1ZGdlLCB6ZXJvX2RpdmlzaW9uPTApKSwgNCksCiAgICAgICAgICAgICJyZWNhbGwiOiByb3VuZChmbG9hdChyZWNhbGxfc2NvcmUoeV90cnVlX3N1YiwgeV9wcmVkX2p1ZGdlLCB6ZXJvX2RpdmlzaW9uPTApKSwgNCksCiAgICAgICAgICAgICJmMSI6IHJvdW5kKGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZV9zdWIsIHlfcHJlZF9qdWRnZSwgemVyb19kaXZpc2lvbj0wKSksIDQpLAogICAgICAgICAgICAibGF0ZW5jeV9tc19wNTAiOiByb3VuZChmbG9hdChucC5tZWRpYW4obGF0ZW5jaWVzKSksIDEpLAogICAgICAgICAgICAibGF0ZW5jeV9tc19wOTUiOiByb3VuZChmbG9hdChucC5wZXJjZW50aWxlKGxhdGVuY2llcywgOTUpKSwgMSksCiAgICAgICAgfSwKICAgICAgICAieGdib29zdF9vbl9zYW1lX3N1YnNldCI6IHsKICAgICAgICAgICAgImFjY3VyYWN5Ijogcm91bmQoZmxvYXQoYWNjdXJhY3lfc2NvcmUoeV90cnVlX3N1YiwgeV94Z2Jfc3ViKSksIDQpLAogICAgICAgICAgICAicHJlY2lzaW9uIjogcm91bmQoZmxvYXQocHJlY2lzaW9uX3Njb3JlKHlfdHJ1ZV9zdWIsIHlfeGdiX3N1YiwgemVyb19kaXZpc2lvbj0wKSksIDQpLAogICAgICAgICAgICAicmVjYWxsIjogcm91bmQoZmxvYXQocmVjYWxsX3Njb3JlKHlfdHJ1ZV9zdWIsIHlfeGdiX3N1YiwgemVyb19kaXZpc2lvbj0wKSksIDQpLAogICAgICAgICAgICAiZjEiOiByb3VuZChmbG9hdChmMV9zY29yZSh5X3RydWVfc3ViLCB5X3hnYl9zdWIsIHplcm9fZGl2aXNpb249MCkpLCA0KSwKICAgICAgICB9LAogICAgICAgICJhZ3JlZW1lbnRfd2l0aF94Z2Jvb3N0Ijogcm91bmQoZmxvYXQoKHlfcHJlZF9qdWRnZSA9PSB5X3hnYl9zdWIpLm1lYW4oKSksIDQpLAogICAgICAgICJtY25lbWFyX2p1ZGdlX3ZzX3hnYm9vc3RfcCI6IG1jbmVtYXJfcCwKICAgICAgICAiZGlzY29yZGFudF9wYWlycyI6IHsianVkZ2Vfd3JvbmdfeGdiX3JpZ2h0IjoganVkZ2Vfd3JvbmdfeGdiX3JpZ2h0LCAianVkZ2VfcmlnaHRfeGdiX3dyb25nIjoganVkZ2VfcmlnaHRfeGdiX3dyb25nfSwKICAgICAgICAiY29zdF91c2QiOiByb3VuZChjb3N0LCA0KSwKICAgICAgICAiY29zdF9wZXJfMTAwMF91c2QiOiByb3VuZChjb3N0IC8gbGVuKHNhbXBsZV9pZHgpICogMTAwMCwgMyksCiAgICAgICAgInRva2VucyI6IHsiaW5wdXQiOiBpbnQoaW5fdG9rZW5zKSwgIm91dHB1dCI6IGludChvdXRfdG9rZW5zKX0sCiAgICAgICAgInByaWNpbmciOiBQUklDSU5HLAogICAgfQoKICAgIHdpdGggb3BlbihSRVNVTFRTX0RJUiAvICJsbG1fanVkZ2VfcmVzdWx0cy5qc29uIiwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChyZXN1bHRzLCBmLCBpbmRlbnQ9MikKCiAgICBwcmludCgiXG4iICsgIj0iICogODApCiAgICBwcmludChmIiBMTE0tYXMtSnVkZ2UgKEdQVCA1LjYgTHVuYSkgdnMgWEdCb29zdCBvbiB7bGVuKHNhbXBsZV9pZHgpfSB0ZXN0IHNhbXBsZXMiKQogICAgcHJpbnQoIj0iICogODApCiAgICBmb3IgbmFtZSwgbSBpbiBbKCJKdWRnZSIsIHJlc3VsdHNbImp1ZGdlIl0pLCAoIlhHQm9vc3QiLCByZXN1bHRzWyJ4Z2Jvb3N0X29uX3NhbWVfc3Vic2V0Il0pXToKICAgICAgICBwcmludChmIntuYW1lOjwxMH0gYWNjPXttWydhY2N1cmFjeSddOi40Zn0gUD17bVsncHJlY2lzaW9uJ106LjRmfSBSPXttWydyZWNhbGwnXTouNGZ9IEYxPXttWydmMSddOi40Zn0iKQogICAgcHJpbnQoZiJBZ3JlZW1lbnQ6IHtyZXN1bHRzWydhZ3JlZW1lbnRfd2l0aF94Z2Jvb3N0J106LjRmfSB8IE1jTmVtYXIgcD17bWNuZW1hcl9wOi4yZX0gfCAiCiAgICAgICAgICBmIkp1ZGdlIGNvc3Q6ICR7cmVzdWx0c1snY29zdF91c2QnXTouNGZ9ICh7cmVzdWx0c1snY29zdF9wZXJfMTAwMF91c2QnXTouM2Z9LzFLKSB8IHA1MCB7cmVzdWx0c1snanVkZ2UnXVsnbGF0ZW5jeV9tc19wNTAnXX1tcyIpCiAgICBwcmludCgiPSIgKiA4MCkKICAgIGxvZ2dlci5pbmZvKGYiU2F2ZWQgbGxtX2p1ZGdlX3Jlc3VsdHMuanNvbiB0byB7UkVTVUxUU19ESVJ9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
 "src/models/eval_ragtruth.py": "IiIiClplcm8tc2hvdCBleHRlcm5hbCB2YWxpZGF0aW9uIG9mIHRoZSBIYWx1UklTQyBtb2RlbCBvbiBSQUdUcnV0aCBRQSAoYmx1ZXByaW50IMKnOC40LApyb2FkbWFwIFBoYXNlIDUgIkV4dGVybmFsIGNvbXBhcmlzb24iKS4KClJ1bnMgdGhlIGZpbmFsIGNhbGlicmF0ZWQgWEdCb29zdCBtb2RlbCB3aXRoIE5PIHRyYWluaW5nIG9uIFJBR1RydXRoIGRhdGE6CiAgZmVhdHVyZXMgYXJlIGV4dHJhY3RlZCB3aXRoIHRoZSBzYW1lIHBpcGVsaW5lLCB0aGVuIHByZWRpY3QuCgpSdW4gKHJlcG8gcm9vdCwgLnZlbnYpOgogIHB5dGhvbiBzcmMvbW9kZWxzL2V2YWxfcmFndHJ1dGgucHkKIiIiCgppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBqb2JsaWIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlLCBicmllcl9zY29yZV9sb3NzLCBmMV9zY29yZSwgbWF0dGhld3NfY29ycmNvZWYsIHByZWNpc2lvbl9zY29yZSwgcmVjYWxsX3Njb3JlLCByb2NfYXVjX3Njb3JlCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJldmFsX3JhZ3RydXRoIikKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXQpSQUdUUlVUSF9QQVRIID0gUk9PVCAvICJkYXRhIiAvICJyYXciIC8gInJhZ3RydXRoIiAvICJyYWd0cnV0aF9xYS5wYXJxdWV0IgpVTklGSUVEX1BBVEggPSBST09UIC8gImRhdGEiIC8gInByb2Nlc3NlZCIgLyAidW5pZmllZF9yZWNvcmRzLnBhcnF1ZXQiCk1PREVMU19ESVIgPSBST09UIC8gImFydGlmYWN0cyIgLyAibW9kZWxzIgpSRVNVTFRTX0RJUiA9IFJPT1QgLyAiYXJ0aWZhY3RzIiAvICJyZXN1bHRzIgoKTl9TQU1QTEVTID0gMjAwMAoKCmRlZiBsb2FkX3JhZ3RydXRoX2ZyYW1lKG46IGludCA9IE5fU0FNUExFUykgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiUkFHVHJ1dGggUUEgcm93cyBmb3IgemVyby1zaG90IHZhbGlkYXRpb24uCgogICAgUHJpb3JpdHk6ICgxKSBsZWdhY3kgVmVyc2lvbiBBIHBhcnF1ZXQgaWYgcHJlc2VudCwgKDIpIHRoZSBCMSB1bmlmaWVkCiAgICBkYXRhc2V0IGJ1aWx0IGJ5IGNlbGwgN2QgKGRhdGEvcHJvY2Vzc2VkL3VuaWZpZWRfcmVjb3Jkcy5wYXJxdWV0KSwgd2hpY2gKICAgIGlzIHdyaXR0ZW4gdG8gdGhlIGxlZ2FjeSBwYXRoIG9uIGZpcnN0IHVzZSBzbyByZXBlYXQgcnVucyBza2lwIHRoZSBtZXJnZS4KICAgICIiIgogICAgaWYgUkFHVFJVVEhfUEFUSC5leGlzdHMoKToKICAgICAgICByZXR1cm4gcGQucmVhZF9wYXJxdWV0KFJBR1RSVVRIX1BBVEgpLmhlYWQobikKICAgIGlmIG5vdCBVTklGSUVEX1BBVEguZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiUkFHVHJ1dGggZGF0YSBtaXNzaW5nOiBydW4gY2VsbCA3ZCAoQjEgdW5pZmllZCBidWlsZCkgb3IgcGxhY2UgIgogICAgICAgICAgICBmIntSQUdUUlVUSF9QQVRIfSIKICAgICAgICApCiAgICB1bmlmaWVkID0gcGQucmVhZF9wYXJxdWV0KAogICAgICAgIFVOSUZJRURfUEFUSCwgY29sdW1ucz1bInNvdXJjZV9kYXRhc2V0IiwgInRhc2siLCAicXVlc3Rpb24iLCAiY29udGV4dCIsICJhbnN3ZXIiLCAibGFiZWwiXQogICAgKQogICAgcWEgPSB1bmlmaWVkWyh1bmlmaWVkWyJzb3VyY2VfZGF0YXNldCJdID09ICJyYWd0cnV0aCIpICYgKHVuaWZpZWRbInRhc2siXSA9PSAicWEiKV0KICAgIHFhID0gcWFbWyJxdWVzdGlvbiIsICJjb250ZXh0IiwgImFuc3dlciIsICJsYWJlbCJdXS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICBSQUdUUlVUSF9QQVRILnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBxYS50b19wYXJxdWV0KFJBR1RSVVRIX1BBVEgsIGluZGV4PUZhbHNlKQogICAgbG9nZ2VyLmluZm8oZiJCdWlsdCB7UkFHVFJVVEhfUEFUSH0gZnJvbSB0aGUgdW5pZmllZCBkYXRhc2V0ICh7bGVuKHFhKX0gUUEgcm93cykiKQogICAgcmV0dXJuIHFhLmhlYWQobikKCgpkZWYgZWNlKHlfdHJ1ZSwgeV9wcm9iLCBuX2JpbnM6IGludCA9IDEwKSAtPiBmbG9hdDoKICAgIGZyb20gc3JjLm1vZGVscy50cmFpbl9waXBlbGluZSBpbXBvcnQgZWNlIGFzIGVjZV9mbgoKICAgIHJldHVybiBlY2VfZm4oeV90cnVlLCB5X3Byb2IsIG5fYmlucykKCgpkZWYgbWFpbigpOgogICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihST09UKSkKICAgIGZyb20gc3JjLmZlYXR1cmVzLmV4dHJhY3RfZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3RfYWxsX2ZlYXR1cmVzX3NpbmdsZSwgbG9hZF9oZWF2eV9tb2RlbHMKCiAgICBkZiA9IGxvYWRfcmFndHJ1dGhfZnJhbWUoTl9TQU1QTEVTKQogICAgbG9nZ2VyLmluZm8oZiJSQUdUcnV0aCBRQSBob2xkb3V0OiB7bGVuKGRmKX0gc2FtcGxlcyAobGFiZWwgYmFsYW5jZToge2RmWydsYWJlbCddLnZhbHVlX2NvdW50cygpLnRvX2RpY3QoKX0pIikKCiAgICBidW5kbGUgPSBqb2JsaWIubG9hZChNT0RFTFNfRElSIC8gIm1vZGVsX3hnYm9vc3RfY2FsaWJyYXRlZC5qb2JsaWIiKQogICAgaWYgaXNpbnN0YW5jZShidW5kbGUsIGRpY3QpIGFuZCBidW5kbGUuZ2V0KCJraW5kIikgPT0gInhnYitwbGF0dCI6CiAgICAgICAgcmF3LCBwbGF0dCA9IGJ1bmRsZVsibW9kZWwiXSwgYnVuZGxlWyJjYWxpYnJhdG9yIl0KCiAgICAgICAgZGVmIHByZWRpY3RfcHJvYmEoWCk6CiAgICAgICAgICAgIHAgPSByYXcucHJlZGljdF9wcm9iYShYKVs6LCAxXQogICAgICAgICAgICByZXR1cm4gcGxhdHQucHJlZGljdF9wcm9iYShwLnJlc2hhcGUoLTEsIDEpKQoKICAgIGVsc2U6CiAgICAgICAgcHJlZGljdF9wcm9iYSA9IGJ1bmRsZS5wcmVkaWN0X3Byb2JhCiAgICBmZWF0dXJlX2NvbHMgPSBqc29uLmxvYWRzKChNT0RFTFNfRElSIC8gImZlYXR1cmVfbmFtZXMuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgbW9kZWxzID0gbG9hZF9oZWF2eV9tb2RlbHMoKQoKICAgIGxvZ2dlci5pbmZvKCJFeHRyYWN0aW5nIGZlYXR1cmVzIG9uIFJBR1RydXRoICh6ZXJvLXNob3QpLi4uIikKICAgIHJvd3MgPSBbXQogICAgZm9yIF8sIHIgaW4gZGYuaXRlcnJvd3MoKToKICAgICAgICByb3dzLmFwcGVuZChleHRyYWN0X2FsbF9mZWF0dXJlc19zaW5nbGUoclsicXVlc3Rpb24iXSwgclsiY29udGV4dCJdLCByWyJhbnN3ZXIiXSwgbW9kZWxzKSkKICAgIFggPSBwZC5EYXRhRnJhbWUocm93cylbZmVhdHVyZV9jb2xzXS52YWx1ZXMKICAgIHkgPSBkZlsibGFiZWwiXS52YWx1ZXMKCiAgICB5X3Byb2IgPSBwcmVkaWN0X3Byb2JhKFgpWzosIDFdCiAgICB5X3ByZWQgPSAoeV9wcm9iID49IDAuNSkuYXN0eXBlKGludCkKCiAgICByZXN1bHRzID0gewogICAgICAgICJuX3NhbXBsZXMiOiBpbnQobGVuKHkpKSwKICAgICAgICAicHJlY2lzaW9uIjogZmxvYXQocHJlY2lzaW9uX3Njb3JlKHksIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKSksCiAgICAgICAgInJlY2FsbCI6IGZsb2F0KHJlY2FsbF9zY29yZSh5LCB5X3ByZWQsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJmMSI6IGZsb2F0KGYxX3Njb3JlKHksIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKSksCiAgICAgICAgImF1cm9jIjogZmxvYXQocm9jX2F1Y19zY29yZSh5LCB5X3Byb2IpKSwKICAgICAgICAicHJfYXVjIjogZmxvYXQoYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUoeSwgeV9wcm9iKSksCiAgICAgICAgIm1jYyI6IGZsb2F0KG1hdHRoZXdzX2NvcnJjb2VmKHksIHlfcHJlZCkpLAogICAgICAgICJlY2UiOiBmbG9hdChlY2UoeSwgeV9wcm9iKSksCiAgICAgICAgImJyaWVyIjogZmxvYXQoYnJpZXJfc2NvcmVfbG9zcyh5LCB5X3Byb2IpKSwKICAgICAgICAibGFiZWxfZGlzdHJpYnV0aW9uIjogZGZbImxhYmVsIl0udmFsdWVfY291bnRzKCkudG9fZGljdCgpLAogICAgfQogICAgbG9nZ2VyLmluZm8oZiJSQUdUcnV0aCB6ZXJvLXNob3Q6IHtqc29uLmR1bXBzKHtrOiAocm91bmQodiwgNCkgaWYgaXNpbnN0YW5jZSh2LCBmbG9hdCkgZWxzZSB2KSBmb3IgaywgdiBpbiByZXN1bHRzLml0ZW1zKCl9LCBpbmRlbnQ9Mil9IikKCiAgICBvcy5tYWtlZGlycyhSRVNVTFRTX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3BlbihSRVNVTFRTX0RJUiAvICJyYWd0cnV0aF9yZXN1bHRzLmpzb24iLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHJlc3VsdHMsIGYsIGluZGVudD0yKQogICAgbG9nZ2VyLmluZm8oZiJTYXZlZCB7UkVTVUxUU19ESVIgLyAncmFndHJ1dGhfcmVzdWx0cy5qc29uJ30iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
 "src/models/fit_display_calibrator.py": "IiIiRml0IHRoZSBkaXNwbGF5IGNhbGlicmF0b3IgZm9yIHRoZSBBUEkgb24gbmF0dXJhbC1zdHlsZSBSQUdUcnV0aCBRQSBkYXRhLgoKVGhlIEIyIG1vZGVsIHNhdHVyYXRlcyBuZWFyIDEuMCBvbiBmdWxsLXNlbnRlbmNlIGlucHV0cyBiZWNhdXNlIGl0IHdhcyB0cmFpbmVkCm9uIEhhbHVFdmFsJ3MgdGVyc2Ugc3ludGhldGljIGFuc3dlcnMuIFRoZSBkZXBsb3llZCBBUEkgdGhlcmVmb3JlIHVzZXMgYQpzZXBhcmF0ZSBjYWxpYnJhdG9yIGZpdHRlZCBvbiBuYXR1cmFsIFJBR1RydXRoIFFBIHJlc3BvbnNlcyAodGhlIEI0IHRhcmdldApjYWxpYnJhdGlvbiBzZXQpIHNvIHRoZSBzaG93biBwZXJjZW50YWdlIHJlZmxlY3RzIHRoZSBldmlkZW5jZSBzdHlsZSBvZiByZWFsCmNoYXQgYW5zd2Vycy4KCkZpdHMgUGxhdHQsIGlzb3RvbmljLCBhbmQgdGVtcGVyYXR1cmUgc2NhbGluZyBvbiA1LDAzNCBSQUdUcnV0aCBRQSByb3dzIGFuZApldmFsdWF0ZXMgb24gdGhlIGRpc2pvaW50IDkwMC1yb3cgdGVzdCBzZXQuIFNhdmVzIHRoZSBiZXN0IG1ldGhvZCAobG93ZXN0CkVDRSwgdGllLWJyZWFrIE5MTCkgcGx1cyBhIG1ldHJpY3MgcmVwb3J0LiBSdW4gZnJvbSB0aGUgcmVwbyByb290OgoKICAudmVudi9TY3JpcHRzL3B5dGhvbi5leGUgc3JjL21vZGVscy9maXRfZGlzcGxheV9jYWxpYnJhdG9yLnB5CgpBcnRpZmFjdHMgd3JpdHRlbjoKICBhcnRpZmFjdHMvbW9kZWxzL2I0L2NhbGlicmF0b3JfZGlzcGxheS5qb2JsaWIKICBhcnRpZmFjdHMvbW9kZWxzL2I0L2Rpc3BsYXlfY2FsaWJyYXRpb24uanNvbgoiIiIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNrbGVhcm4uaXNvdG9uaWMgaW1wb3J0IElzb3RvbmljUmVncmVzc2lvbgpmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb24KZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGJyaWVyX3Njb3JlX2xvc3MKZnJvbSBzY2lweS5vcHRpbWl6ZSBpbXBvcnQgbWluaW1pemVfc2NhbGFyCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KQ0FMX0ZJTEUgPSBST09UIC8gImFydGlmYWN0cyIgLyAicmVzdWx0cyIgLyAiYjQiIC8gIl9zdGFnZXMiIC8gInFhX2NhbF9jbGVhbi5wYXJxdWV0IgpURVNUX0ZJTEUgPSBST09UIC8gImFydGlmYWN0cyIgLyAicmVzdWx0cyIgLyAiYjQiIC8gImI0X3ByZWRpY3Rpb25zLnBhcnF1ZXQiCk9VVF9NT0RFTCA9IFJPT1QgLyAiYXJ0aWZhY3RzIiAvICJtb2RlbHMiIC8gImI0IiAvICJjYWxpYnJhdG9yX2Rpc3BsYXkuam9ibGliIgpPVVRfTUVUQSA9IFJPT1QgLyAiYXJ0aWZhY3RzIiAvICJtb2RlbHMiIC8gImI0IiAvICJkaXNwbGF5X2NhbGlicmF0aW9uLmpzb24iCkJJTlMgPSAxMAoKCmRlZiBlY2UoeTogbnAubmRhcnJheSwgcDogbnAubmRhcnJheSwgYmluczogaW50ID0gQklOUykgLT4gZmxvYXQ6CiAgICBlZGdlcyA9IG5wLmxpbnNwYWNlKDAuMCwgMS4wLCBiaW5zICsgMSkKICAgIHRvdGFsID0gMC4wCiAgICBuID0gbGVuKHkpCiAgICBmb3IgbG8sIGhpIGluIHppcChlZGdlc1s6LTFdLCBlZGdlc1sxOl0pOgogICAgICAgIG0gPSAocCA+PSBsbykgJiAocCA8IGhpKQogICAgICAgIGlmIG0uc3VtKCkgPT0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBhY2MgPSB5W21dLm1lYW4oKQogICAgICAgIGNvbmYgPSBwW21dLm1lYW4oKQogICAgICAgIHRvdGFsICs9IG0uc3VtKCkgLyBuICogYWJzKGFjYyAtIGNvbmYpCiAgICByZXR1cm4gZmxvYXQodG90YWwpCgoKZGVmIG5sbCh5OiBucC5uZGFycmF5LCBwOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgIHAgPSBucC5jbGlwKHAsIDFlLTEyLCAxIC0gMWUtMTIpCiAgICByZXR1cm4gZmxvYXQoLSh5ICogbnAubG9nKHApICsgKDEgLSB5KSAqIG5wLmxvZygxIC0gcCkpLm1lYW4oKSkKCgpkZWYgbG9naXQocDogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgIHAgPSBucC5jbGlwKHAsIDFlLTEyLCAxIC0gMWUtMTIpCiAgICByZXR1cm4gbnAubG9nKHAgLyAoMSAtIHApKQoKCmRlZiBzaWdtb2lkKHo6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICByZXR1cm4gMS4wIC8gKDEuMCArIG5wLmV4cCgteikpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgY2FsID0gcGQucmVhZF9wYXJxdWV0KENBTF9GSUxFKQogICAgdGVzdCA9IHBkLnJlYWRfcGFycXVldChURVNUX0ZJTEUpCiAgICB0ZXN0ID0gdGVzdFsodGVzdC5zdWJzZXQgPT0gInJhZ3RydXRoX3FhX3Rlc3QiKSAmICh0ZXN0Lm1ldGhvZCA9PSAicmF3IildCgogICAgeV9jYWwgPSBjYWxbImxhYmVsIl0udG9fbnVtcHkoKQogICAgcF9jYWwgPSBjYWxbInNjb3JlXzQyIl0udG9fbnVtcHkoKQogICAgeV90ZXN0ID0gdGVzdFsibGFiZWwiXS50b19udW1weSgpCiAgICBwX3Rlc3QgPSB0ZXN0WyJzY29yZSJdLnRvX251bXB5KCkKCiAgICB6X2NhbCwgel90ZXN0ID0gbG9naXQocF9jYWwpLCBsb2dpdChwX3Rlc3QpCgogICAgcGxhdHQgPSBMb2dpc3RpY1JlZ3Jlc3Npb24oQz0xZTYpCiAgICBwbGF0dC5maXQoel9jYWwucmVzaGFwZSgtMSwgMSksIHlfY2FsKQogICAgcF9wbGF0dCA9IHBsYXR0LnByZWRpY3RfcHJvYmEoel90ZXN0LnJlc2hhcGUoLTEsIDEpKVs6LCAxXQoKICAgIGlzbyA9IElzb3RvbmljUmVncmVzc2lvbihvdXRfb2ZfYm91bmRzPSJjbGlwIiwgaW5jcmVhc2luZz1UcnVlKQogICAgaXNvLmZpdChwX2NhbCwgeV9jYWwpCiAgICBwX2lzbyA9IGlzby5wcmVkaWN0KHBfdGVzdCkKCiAgICBkZWYgdGVtcF9ubGwodDogZmxvYXQpIC0+IGZsb2F0OgogICAgICAgIHJldHVybiBubGwoeV9jYWwsIHNpZ21vaWQoel9jYWwgLyB0KSkKCiAgICByZXMgPSBtaW5pbWl6ZV9zY2FsYXIodGVtcF9ubGwsIGJvdW5kcz0oMC4wNSwgNS4wKSwgbWV0aG9kPSJib3VuZGVkIikKICAgIHRfYmVzdCA9IHJlcy54CiAgICBwX3RlbXAgPSBzaWdtb2lkKHpfdGVzdCAvIHRfYmVzdCkKCiAgICBtZXRob2RzID0gewogICAgICAgICJwbGF0dCI6IChwX3BsYXR0LCBwbGF0dCksCiAgICAgICAgImlzb3RvbmljIjogKHBfaXNvLCBpc28pLAogICAgICAgICJ0ZW1wZXJhdHVyZSI6IChwX3RlbXAsIHRfYmVzdCksCiAgICB9CgogICAgcmVwb3J0ID0gewogICAgICAgICJuX2NhbGlicmF0aW9uX3Jvd3MiOiBpbnQobGVuKHlfY2FsKSksCiAgICAgICAgIm5fdGVzdF9yb3dzIjogaW50KGxlbih5X3Rlc3QpKSwKICAgICAgICAibGFiZWxfcG9zaXRpdmVfcmF0ZV90ZXN0IjogZmxvYXQoeV90ZXN0Lm1lYW4oKSksCiAgICAgICAgIm1ldGhvZHMiOiB7fSwKICAgIH0KICAgIGZvciBuYW1lLCAocCwgXykgaW4gbWV0aG9kcy5pdGVtcygpOgogICAgICAgIHJlcG9ydFsibWV0aG9kcyJdW25hbWVdID0gewogICAgICAgICAgICAiZWNlIjogZWNlKHlfdGVzdCwgcCksCiAgICAgICAgICAgICJicmllciI6IGJyaWVyX3Njb3JlX2xvc3MoeV90ZXN0LCBwKSwKICAgICAgICAgICAgIm5sbCI6IG5sbCh5X3Rlc3QsIHApLAogICAgICAgICAgICAicHJlZF9wb3NpdGl2ZV9yYXRlIjogZmxvYXQoKHAgPj0gMC41KS5tZWFuKCkpLAogICAgICAgICAgICAic2NvcmVfcDUwIjogZmxvYXQobnAubWVkaWFuKHApKSwKICAgICAgICAgICAgInNjb3JlX3A5MCI6IGZsb2F0KG5wLnF1YW50aWxlKHAsIDAuOSkpLAogICAgICAgICAgICAic2NvcmVfbWVhbiI6IGZsb2F0KHAubWVhbigpKSwKICAgICAgICB9CiAgICByZXBvcnRbInJhd19yZWZlcmVuY2UiXSA9IHsKICAgICAgICAiZWNlIjogZWNlKHlfdGVzdCwgcF90ZXN0KSwKICAgICAgICAiYnJpZXIiOiBicmllcl9zY29yZV9sb3NzKHlfdGVzdCwgcF90ZXN0KSwKICAgICAgICAibmxsIjogbmxsKHlfdGVzdCwgcF90ZXN0KSwKICAgICAgICAicHJlZF9wb3NpdGl2ZV9yYXRlIjogZmxvYXQoKHBfdGVzdCA+PSAwLjUpLm1lYW4oKSksCiAgICAgICAgInNjb3JlX3A1MCI6IGZsb2F0KG5wLm1lZGlhbihwX3Rlc3QpKSwKICAgIH0KCiAgICBiZXN0ID0gbWluKHJlcG9ydFsibWV0aG9kcyJdLCBrZXk9bGFtYmRhIG06IChyZXBvcnRbIm1ldGhvZHMiXVttXVsiZWNlIl0sIHJlcG9ydFsibWV0aG9kcyJdW21dWyJubGwiXSkpCiAgICByZXBvcnRbImNob3NlbiJdID0gYmVzdAoKICAgIGltcG9ydCBqb2JsaWIKCiAgICBPVVRfTU9ERUwucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGpvYmxpYi5kdW1wKHsibWV0aG9kIjogYmVzdCwgImNhbGlicmF0b3IiOiBtZXRob2RzW2Jlc3RdWzFdfSwgT1VUX01PREVMKQogICAgT1VUX01FVEEud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgcHJpbnQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yKSkKICAgIHByaW50KGYic2F2ZWQge09VVF9NT0RFTH0gKGNob3Nlbjoge2Jlc3R9KSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
 "src/models/make_manifest.py": "IiIiCkhhbHVSSVNDIGFydGlmYWN0IG1hbmlmZXN0IGdlbmVyYXRvciAoYmx1ZXByaW50IMKnMTEgLyByb2FkbWFwIEI2KS4KCldyaXRlcyBhcnRpZmFjdHMvcmVzdWx0cy9tYW5pZmVzdC5qc29uIHdpdGg6CiAgLSBkYXRhc2V0IGhhc2hlcyAocHJvY2Vzc2VkICsgcmF3IHNvdXJjZXMgd2l0aCByZXZpc2lvbiBmaWxlcykKICAtIHNwbGl0IHJlcG9ydCAoc3BsaXQgaGFzaCB2aWEgc3BsaXRfaW5kaWNlcy5qc29uKQogIC0gc2VlZHMgWzQyLCAxMjMsIDQ1Nl0sIGZlYXR1cmUgZ3JvdXBzLCBtb2RlbC9mZWF0dXJlIHZlcnNpb25zCiAgLSBwYWNrYWdlIHZlcnNpb25zICsgaGFyZHdhcmUgKENQVS9SQU0vR1BVKQogIC0gZ2l0IGNvbW1pdCAod2hlbiBydW4gaW5zaWRlIGEgY2xvbmUpIE9SIHNvdXJjZV9maW5nZXJwcmludCAoc2hhMjU2IG92ZXIKICAgIHRoZSBub3RlYm9vayBjZWxsLTMgZW1iZWRkZWQgSEFTSEVTLCBwYXNzZWQgdmlhIEhBTFVfU09VUkNFX0ZJTkdFUlBSSU5UOwogICAgdGhpcyBmaW5nZXJwcmludHMgdGhlIGV4YWN0IHNoaXBwZWQgc291cmNlIGV2ZW4gb24gQ29sYWIgd2l0aG91dCBnaXQpCiAgLSBIQUxVXyogZW52aXJvbm1lbnQgY29uZmlndXJhdGlvbiAobm9uLXNlY3JldDsgQVBJIGtleXMgYXJlIG5ldmVyIGluY2x1ZGVkKQogIC0gdGhlIGxpc3Qgb2YgcHJvZHVjZWQgYXJ0aWZhY3RzIChpbnRlcm5hbCBjaGVja3BvaW50cyBleGNsdWRlZCkKCkNvbGFiLXNhZmU6IHJlcG8tcm9vdC1yZWxhdGl2ZSBwYXRocyBvbmx5LgoKUnVuIChyZXBvIHJvb3QsIC52ZW52IG9yIENvbGFiKToKICBweXRob24gc3JjL21vZGVscy9tYWtlX21hbmlmZXN0LnB5CiIiIgoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0pKQoKZnJvbSBzcmMubW9kZWxzLmNvbmZpZyBpbXBvcnQgKCAgIyBub3FhOiBFNDAyCiAgICBBUlRJRkFDVFNfRElSLAogICAgREFUQV9QUk9DRVNTRUQsCiAgICBNT0RFTFNfRElSLAogICAgUkVTVUxUU19ESVIsCiAgICBTRUVEUywKKQpmcm9tIHNyYy5tb2RlbHMudHJhaW5fcGlwZWxpbmUgaW1wb3J0IEZFQVRVUkVfR1JPVVBTICAjIG5vcWE6IEU0MDIKCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoIm1ha2VfbWFuaWZlc3QiKQoKUkFXX0RJUiA9IEFSVElGQUNUU19ESVIucGFyZW50IC8gImRhdGEiIC8gInJhdyIKCiMgSW50ZXJuYWwvbm9uLXB1YmxpY2F0aW9uIHBhdGhzIG5ldmVyIGxpc3RlZCBpbiB0aGUgbWFuaWZlc3QgKG9yIGZ1dHVyZSB6aXBzKS4KRVhDTFVERV9GUkFHTUVOVFMgPSAoIl9zdGFnZXMiLCAiYjJfc21va2VfdGVzdCIsICJiMl90ZXN0X3RtcCIsICJiNV9jcmFzaC5sb2ciLCAiX19weWNhY2hlX18iKQoKU0hBX0ZJTEVTID0gWwogICAgREFUQV9QUk9DRVNTRUQgLyAicWFfY2xlYW4ucGFycXVldCIsCiAgICBEQVRBX1BST0NFU1NFRCAvICJmZWF0dXJlc19mdWxsLnBhcnF1ZXQiLAogICAgTU9ERUxTX0RJUiAvICJtb2RlbF94Z2Jvb3N0X3Jhdy5qb2JsaWIiLAogICAgTU9ERUxTX0RJUiAvICJtb2RlbF94Z2Jvb3N0X2NhbGlicmF0ZWQuam9ibGliIiwKICAgIEFSVElGQUNUU19ESVIgLyAic3BsaXRfaW5kaWNlcy5qc29uIiwKICAgIEFSVElGQUNUU19ESVIgLyAic3BsaXRfaW50ZWdyaXR5X3JlcG9ydC5qc29uIiwKXQoKCmRlZiBzaGEyNTYocGF0aDogUGF0aCkgLT4gc3RyIHwgTm9uZToKICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOgogICAgICAgIHJldHVybiBOb25lCiAgICBoID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBvcGVuKHBhdGgsICJyYiIpIGFzIGY6CiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBmLnJlYWQoMSA8PCAyMCksIGIiIik6CiAgICAgICAgICAgIGgudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgZ2l0X2NvbW1pdCgpIC0+IHN0ciB8IE5vbmU6CiAgICB0cnk6CiAgICAgICAgb3V0ID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgIFsiZ2l0IiwgInJldi1wYXJzZSIsICJIRUFEIl0sIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwgdGltZW91dD0xMAogICAgICAgICkKICAgICAgICByZXR1cm4gb3V0LnN0ZG91dC5zdHJpcCgpIG9yIE5vbmUKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIE5vbmUKCgpkZWYgdmVyc2lvbnMoKSAtPiBkaWN0OgogICAgZGVmIHZlcihuYW1lOiBzdHIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgbW9kID0gX19pbXBvcnRfXyhuYW1lKQogICAgICAgICAgICByZXR1cm4gZ2V0YXR0cihtb2QsICJfX3ZlcnNpb25fXyIsICJ1bmtub3duIikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICByZXR1cm4gTm9uZQoKICAgIHJldHVybiB7CiAgICAgICAgInB5dGhvbiI6IHBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksCiAgICAgICAgInBsYXRmb3JtIjogcGxhdGZvcm0ucGxhdGZvcm0oKSwKICAgICAgICAidG9yY2giOiB2ZXIoInRvcmNoIiksCiAgICAgICAgIm51bXB5IjogdmVyKCJudW1weSIpLAogICAgICAgICJwYW5kYXMiOiB2ZXIoInBhbmRhcyIpLAogICAgICAgICJzY2lraXRfbGVhcm4iOiB2ZXIoInNrbGVhcm4iKSwKICAgICAgICAieGdib29zdCI6IHZlcigieGdib29zdCIpLAogICAgICAgICJzaGFwIjogdmVyKCJzaGFwIiksCiAgICAgICAgInNwYWN5IjogdmVyKCJzcGFjeSIpLAogICAgICAgICJzZW50ZW5jZV90cmFuc2Zvcm1lcnMiOiB2ZXIoInNlbnRlbmNlX3RyYW5zZm9ybWVycyIpLAogICAgICAgICJmYXN0YXBpIjogdmVyKCJmYXN0YXBpIiksCiAgICAgICAgInB5eWFtbCI6IHZlcigieWFtbCIpLAogICAgfQoKCmRlZiBoYXJkd2FyZSgpIC0+IGRpY3Q6CiAgICBodyA9IHsiY3B1IjogcGxhdGZvcm0ucHJvY2Vzc29yKCksICJjdWRhIjogRmFsc2UsICJncHVfbmFtZSI6IE5vbmUsICJncHVfdG90YWxfbWVtb3J5X2diIjogTm9uZX0KICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgaHdbImN1ZGEiXSA9IFRydWUKICAgICAgICAgICAgaHdbImdwdV9uYW1lIl0gPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKQogICAgICAgICAgICBod1siZ3B1X3RvdGFsX21lbW9yeV9nYiJdID0gcm91bmQodG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkudG90YWxfbWVtb3J5IC8gMioqMzAsIDIpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgcHN1dGlsCgogICAgICAgIGh3WyJyYW1fdG90YWxfZ2IiXSA9IHJvdW5kKHBzdXRpbC52aXJ0dWFsX21lbW9yeSgpLnRvdGFsIC8gMioqMzAsIDIpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGh3WyJyYW1fdG90YWxfZ2IiXSA9IE5vbmUKICAgIHJldHVybiBodwoKCmRlZiByYXdfaGFzaGVzKCkgLT4gZGljdDoKICAgICIiInNoYTI1NiBvZiBldmVyeSByYXcgaW5wdXQgZmlsZSAoaW5jbC4gZG93bmxvYWQgcmV2aXNpb24uanNvbiBmaWxlcykuIiIiCiAgICBpZiBub3QgUkFXX0RJUi5leGlzdHMoKToKICAgICAgICByZXR1cm4ge30KICAgIG91dCA9IHt9CiAgICBmb3IgcCBpbiBzb3J0ZWQoUkFXX0RJUi5yZ2xvYigiKiIpKToKICAgICAgICBpZiBwLmlzX2ZpbGUoKToKICAgICAgICAgICAgb3V0W3N0cihwLnJlbGF0aXZlX3RvKFJBV19ESVIucGFyZW50KSldID0gc2hhMjU2KHApCiAgICByZXR1cm4gb3V0CgoKZGVmIHJlYWRfanNvbihwYXRoOiBQYXRoKToKICAgIHRyeToKICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dCgpKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gTm9uZQoKCmRlZiBtYWluKCk6CiAgICBwYXJhbXMgPSByZWFkX2pzb24oTU9ERUxTX0RJUiAvICJwYXJhbXMuanNvbiIpIG9yIHt9CiAgICBzcGxpdF9yZXBvcnQgPSByZWFkX2pzb24oQVJUSUZBQ1RTX0RJUiAvICJzcGxpdF9pbnRlZ3JpdHlfcmVwb3J0Lmpzb24iKQogICAgbmxpX3VzZWQgPSByZWFkX2pzb24oREFUQV9QUk9DRVNTRUQgLyAibmxpX21vZGVsX3VzZWQuanNvbiIpCgogICAgZGVmIHNoYV9vcl9ub25lKHJlbF9wYXRoOiBQYXRoKToKICAgICAgICByZXR1cm4gc2hhMjU2KHJlbF9wYXRoKSBpZiByZWxfcGF0aC5leGlzdHMoKSBlbHNlIE5vbmUKCiAgICBiX3BhdGhzID0gewogICAgICAgICJiMl9ydW5fY29uZmlnLmpzb24iOiBSRVNVTFRTX0RJUiAvICJiMiIgLyAiYjJfcnVuX2NvbmZpZy5qc29uIiwKICAgICAgICAiYjJfbW9kZWxfY29tcGFyaXNvbi5qc29uIjogUkVTVUxUU19ESVIgLyAiYjIiIC8gImIyX21vZGVsX2NvbXBhcmlzb24uanNvbiIsCiAgICAgICAgImIyX3ByZWRpY3Rpb25zLnBhcnF1ZXQiOiBSRVNVTFRTX0RJUiAvICJiMiIgLyAiYjJfcHJlZGljdGlvbnMucGFycXVldCIsCiAgICAgICAgImIyX3hnYm9vc3Rfc2VlZF80Mi5qb2JsaWIiOiBNT0RFTFNfRElSIC8gImIyIiAvICJ4Z2Jvb3N0X3NlZWRfNDIuam9ibGliIiwKICAgICAgICAiYjNfcHJlZGljdGlvbnMucGFycXVldCI6IFJFU1VMVFNfRElSIC8gImIzIiAvICJiM19wcmVkaWN0aW9ucy5wYXJxdWV0IiwKICAgICAgICAiYjNfZGF0YXNldF9tZXRyaWNzLmpzb24iOiBSRVNVTFRTX0RJUiAvICJiMyIgLyAiYjNfZGF0YXNldF9tZXRyaWNzLmpzb24iLAogICAgICAgICJiM19ib290c3RyYXBfY2lzLmpzb24iOiBSRVNVTFRTX0RJUiAvICJiMyIgLyAiYjNfYm9vdHN0cmFwX2Npcy5qc29uIiwKICAgICAgICAiYjNfcnVuX2NvbmZpZy5qc29uIjogUkVTVUxUU19ESVIgLyAiYjMiIC8gImIzX3J1bl9jb25maWcuanNvbiIsCiAgICAgICAgImI0X3ByZWRpY3Rpb25zLnBhcnF1ZXQiOiBSRVNVTFRTX0RJUiAvICJiNCIgLyAiYjRfcHJlZGljdGlvbnMucGFycXVldCIsCiAgICAgICAgImI0X2NhbGlicmF0aW9uX21ldHJpY3MuanNvbiI6IFJFU1VMVFNfRElSIC8gImI0IiAvICJiNF9jYWxpYnJhdGlvbl9tZXRyaWNzLmpzb24iLAogICAgICAgICJiNF90YXJnZXRfY2FsaWJyYXRpb24uanNvbiI6IFJFU1VMVFNfRElSIC8gImI0IiAvICJiNF90YXJnZXRfY2FsaWJyYXRpb24uanNvbiIsCiAgICAgICAgImI0X2NhbGlicmF0b3JfcGxhdHRfc291cmNlX3NlZWRfNDIuam9ibGliIjogTU9ERUxTX0RJUiAvICJiNCIgLyAiY2FsaWJyYXRvcl9wbGF0dF9zb3VyY2Vfc2VlZF80Mi5qb2JsaWIiLAogICAgICAgICJiNF9jYWxpYnJhdG9yX2lzb3RvbmljX3NvdXJjZV9zZWVkXzQyLmpvYmxpYiI6IE1PREVMU19ESVIgLyAiYjQiIC8gImNhbGlicmF0b3JfaXNvdG9uaWNfc291cmNlX3NlZWRfNDIuam9ibGliIiwKICAgICAgICAidW5pZmllZF9yZWNvcmRzLnBhcnF1ZXQiOiBEQVRBX1BST0NFU1NFRCAvICJ1bmlmaWVkX3JlY29yZHMucGFycXVldCIsCiAgICAgICAgImIzX2V4dGVybmFsX2ZlYXR1cmVzLnBhcnF1ZXQiOiBEQVRBX1BST0NFU1NFRCAvICJiM19leHRlcm5hbF9mZWF0dXJlcy5wYXJxdWV0IiwKICAgICAgICAiYjVfcnVuX2NvbmZpZy5qc29uIjogUkVTVUxUU19ESVIgLyAiYjUiIC8gImI1X3J1bl9jb25maWcuanNvbiIsCiAgICAgICAgImI1X2ZlYXR1cmVfaW1wb3J0YW5jZS5qc29uIjogUkVTVUxUU19ESVIgLyAiYjUiIC8gImI1X2ZlYXR1cmVfaW1wb3J0YW5jZS5qc29uIiwKICAgICAgICAiYjVfbmV1dHJhbGl6YXRpb24uanNvbiI6IFJFU1VMVFNfRElSIC8gImI1IiAvICJiNV9uZXV0cmFsaXphdGlvbi5qc29uIiwKICAgICAgICAiYjVfc3RhYmlsaXR5X2Jvb3RzdHJhcC5qc29uIjogUkVTVUxUU19ESVIgLyAiYjUiIC8gImI1X3N0YWJpbGl0eV9ib290c3RyYXAuanNvbiIsCiAgICAgICAgImI1X3BlcnR1cmJhdGlvbl9hZ2dyZWdhdGVzLmNzdiI6IFJFU1VMVFNfRElSIC8gImI1IiAvICJiNV9wZXJ0dXJiYXRpb25fYWdncmVnYXRlcy5jc3YiLAogICAgICAgICJiNV9yZXZpZXdfY2FzZXMuY3N2IjogUkVTVUxUU19ESVIgLyAiYjUiIC8gImI1X3Jldmlld19jYXNlcy5jc3YiLAogICAgICAgICJiNV9mYWlsdXJlX2Nhc2VzLmpzb24iOiBSRVNVTFRTX0RJUiAvICJiNSIgLyAiYjVfZmFpbHVyZV9jYXNlcy5qc29uIiwKICAgIH0KCiAgICBlbnZfY29uZmlnID0gewogICAgICAgIG5hbWU6IG9zLmVudmlyb25bbmFtZV0KICAgICAgICBmb3IgbmFtZSBpbiAoIkhBTFVfQVBJX0RFVklDRSIsICJIQUxVX0FQSV9QUkVMT0FEIiwgIkhBTFVfWEdCX0RFVklDRSIsCiAgICAgICAgICAgICAgICAgICAgICJIQUxVX05MSV9NT0RFTCIsICJIQUxVX0pVREdFX04iKQogICAgICAgIGlmIG5hbWUgaW4gb3MuZW52aXJvbgogICAgfQogICAgc291cmNlX2ZpbmdlcnByaW50ID0gb3MuZW52aXJvbi5nZXQoIkhBTFVfU09VUkNFX0ZJTkdFUlBSSU5UIikKCiAgICAjIFRoZSBkZXBsb3llZCBtb2RlbCBpcyBFQy1YR0Igd2hlbiB0aGUgQjYgYXJ0aWZhY3RzIGV4aXN0OyB0aGUgZnJvemVuCiAgICAjIHBhcmFtcy5qc29uIHN0aWxsIGRlc2NyaWJlcyB0aGUgQjIgYmFzZSBtb2RlbC4KICAgIGVjX21vZGVsID0gUkVTVUxUU19ESVIucGFyZW50IC8gIm1vZGVscyIgLyAiYjYiIC8gInhnYm9vc3RfbTNfc2VlZF80Mi5qb2JsaWIiCgogICAgbWFuaWZlc3QgPSB7CiAgICAgICAgImdlbmVyYXRlZF9hdCI6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLAogICAgICAgICJnaXRfY29tbWl0IjogZ2l0X2NvbW1pdCgpLAogICAgICAgICJzb3VyY2VfZmluZ2VycHJpbnQiOiBzb3VyY2VfZmluZ2VycHJpbnQsCiAgICAgICAgIm1vZGVsX3ZlcnNpb24iOiAiYjYtZWMteGdiLXYxLjAiIGlmIGVjX21vZGVsLmV4aXN0cygpIGVsc2UgcGFyYW1zLmdldCgibW9kZWxfdmVyc2lvbiIpLAogICAgICAgICJkZXBsb3llZF9tb2RlbCI6ICJFQy1YR0IgKEV2aWRlbmNlLUNvbnNpc3RlbnQgWEdCb29zdCkiIGlmIGVjX21vZGVsLmV4aXN0cygpIGVsc2UgIkIyIFhHQm9vc3QiLAogICAgICAgICJmZWF0dXJlX3ZlcnNpb24iOiAiY291cnNlLXYxLjAiLAogICAgICAgICJuX2ZlYXR1cmVzIjogcGFyYW1zLmdldCgibl9mZWF0dXJlcyIpLAogICAgICAgICJubGlfbW9kZWwiOiBwYXJhbXMuZ2V0KCJubGlfbW9kZWwiKSwKICAgICAgICAibmxpX3Byb3ZlbmFuY2UiOiBubGlfdXNlZCwKICAgICAgICAic2VlZHMiOiBsaXN0KFNFRURTKSwKICAgICAgICAiZmVhdHVyZV9ncm91cHMiOiBGRUFUVVJFX0dST1VQUywKICAgICAgICAic3BsaXRfcmVwb3J0Ijogc3BsaXRfcmVwb3J0LAogICAgICAgICJkYXRhc2V0X3NoYTI1NiI6IHsKICAgICAgICAgICAgInFhX2NsZWFuLnBhcnF1ZXQiOiBzaGEyNTYoU0hBX0ZJTEVTWzBdKSwKICAgICAgICAgICAgImZlYXR1cmVzX2Z1bGwucGFycXVldCI6IHNoYTI1NihTSEFfRklMRVNbMV0pLAogICAgICAgICAgICAibW9kZWxfeGdib29zdF9yYXcuam9ibGliIjogc2hhMjU2KFNIQV9GSUxFU1syXSksCiAgICAgICAgICAgICJtb2RlbF94Z2Jvb3N0X2NhbGlicmF0ZWQuam9ibGliIjogc2hhMjU2KFNIQV9GSUxFU1szXSksCiAgICAgICAgICAgICJzcGxpdF9pbmRpY2VzLmpzb24iOiBzaGEyNTYoU0hBX0ZJTEVTWzRdKSwKICAgICAgICAgICAgInNwbGl0X2ludGVncml0eV9yZXBvcnQuanNvbiI6IHNoYTI1NihTSEFfRklMRVNbNV0pLAogICAgICAgIH0sCiAgICAgICAgInJhd19zaGEyNTYiOiByYXdfaGFzaGVzKCksCiAgICAgICAgImJfYXJ0aWZhY3RzX3NoYTI1NiI6IHtuYW1lOiBzaGFfb3Jfbm9uZShwKSBmb3IgbmFtZSwgcCBpbiBiX3BhdGhzLml0ZW1zKCl9LAogICAgICAgICJ2ZXJzaW9ucyI6IHZlcnNpb25zKCksCiAgICAgICAgImhhcmR3YXJlIjogaGFyZHdhcmUoKSwKICAgICAgICAiZW52IjogZW52X2NvbmZpZywKICAgICAgICAiYXJ0aWZhY3RzIjogc29ydGVkKAogICAgICAgICAgICBzdHIocC5yZWxhdGl2ZV90byhBUlRJRkFDVFNfRElSLnBhcmVudCkpCiAgICAgICAgICAgIGZvciBwIGluIFsKICAgICAgICAgICAgICAgICpBUlRJRkFDVFNfRElSLnJnbG9iKCIqIiksCiAgICAgICAgICAgICAgICAqREFUQV9QUk9DRVNTRUQuZ2xvYigiKi5wYXJxdWV0IiksCiAgICAgICAgICAgICAgICBEQVRBX1BST0NFU1NFRCAvICJubGlfbW9kZWxfdXNlZC5qc29uIiwKICAgICAgICAgICAgXQogICAgICAgICAgICBpZiBwLmlzX2ZpbGUoKSBhbmQgbm90IGFueShmcmFnIGluIHN0cihwKSBmb3IgZnJhZyBpbiBFWENMVURFX0ZSQUdNRU5UUykKICAgICAgICApLAogICAgfQoKICAgIFJFU1VMVFNfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG91dCA9IFJFU1VMVFNfRElSIC8gIm1hbmlmZXN0Lmpzb24iCiAgICBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKG1hbmlmZXN0LCBpbmRlbnQ9MikpCiAgICBsb2dnZXIuaW5mbyhmIlNhdmVkIG1hbmlmZXN0IHRvIHtvdXR9IChnaXRfY29tbWl0PXttYW5pZmVzdFsnZ2l0X2NvbW1pdCddfSwgIgogICAgICAgICAgICAgICAgZiJzb3VyY2VfZmluZ2VycHJpbnQ9eydzZXQnIGlmIHNvdXJjZV9maW5nZXJwcmludCBlbHNlIE5vbmV9LCAiCiAgICAgICAgICAgICAgICBmInJhdyBmaWxlcz17bGVuKG1hbmlmZXN0WydyYXdfc2hhMjU2J10pfSkiKQogICAgcmV0dXJuIG1hbmlmZXN0CgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
 "src/models/review_tally.py": "IiIiCkI1LjUgcmV2aWV3IHRhbGx5IOKAlCByZWFkcyB0aGUgbWFudWFsbHkgZmlsbGVkIHJldmlld2VyIHNoZWV0IGFuZCBwcmludHMKZGVzY3JpcHRpdmUgYWdyZWVtZW50IHN0YXRpc3RpY3MuIE5vIE1MLCBubyB0aHJlc2hvbGRzIChCNS43OiBkaXN0cmlidXRpb25zCm9ubHkpLiBSZXZpZXdzIGFyZSBhIGh1bWFuIHN0ZXA7IHRoaXMgc2NyaXB0IG9ubHkgdGFsbGllcyB3aGF0IGh1bWFucyB3cm90ZS4KCklucHV0IChkZWZhdWx0KTogYXJ0aWZhY3RzL3Jlc3VsdHMvYjUvYjVfcmV2aWV3X2Nhc2VzX3Jldmlld2VkLmNzdgpDb2x1bW5zIGV4cGVjdGVkOiAuLi4gcmV2aWV3ZXJfMSwgcmV2aWV3ZXJfMiwgYWdyZWVtZW50IChhcyBwcm9kdWNlZCBieSB0aGUKQjUgZXhwb3J0OyByZXZpZXdlcnMgZmlsbCByZXZpZXdlcl8xL3Jldmlld2VyXzIvYWdyZWVtZW50IGJ5IGhhbmQpLgoKUnVuIChyZXBvIHJvb3QsIC52ZW52KToKICBweXRob24gc3JjL21vZGVscy9yZXZpZXdfdGFsbHkucHkgW3BhdGgvdG8vcmV2aWV3ZWQuY3N2XQoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgc3lzCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KREVGQVVMVF9TSEVFVCA9IFJPT1QgLyAiYXJ0aWZhY3RzIiAvICJyZXN1bHRzIiAvICJiNSIgLyAiYjVfcmV2aWV3X2Nhc2VzX3Jldmlld2VkLmNzdiIKClZBTElEX01BUktTID0geyJwbGF1c2libGUiLCAiaW1wbGF1c2libGUiLCAidW5zdXJlIn0KCgpkZWYgbWFpbihhcmd2OiBsaXN0IHwgTm9uZSA9IE5vbmUpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJUYWxseSBCNS41IHJldmlld2VyIGFncmVlbWVudCAoZGVzY3JpcHRpdmUgb25seSkiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgic2hlZXQiLCBuYXJncz0iPyIsIGRlZmF1bHQ9c3RyKERFRkFVTFRfU0hFRVQpLCBoZWxwPSJwYXRoIHRvIHRoZSByZXZpZXdlZCBDU1YiKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKGFyZ3YpCgogICAgc2hlZXQgPSBQYXRoKGFyZ3Muc2hlZXQpCiAgICBpZiBub3Qgc2hlZXQuZXhpc3RzKCk6CiAgICAgICAgcHJpbnQoZiJTaGVldCBub3QgZm91bmQ6IHtzaGVldH0iKQogICAgICAgIHByaW50KCJSdW4gdGhlIG1hbnVhbCByZXZpZXcgZmlyc3QgKGRvY3MvYjUtZXhwbGFuYXRpb24tcmVsaWFiaWxpdHkubWQpIGFuZCBzYXZlIGFzIikKICAgICAgICBwcmludCgiYXJ0aWZhY3RzL3Jlc3VsdHMvYjUvYjVfcmV2aWV3X2Nhc2VzX3Jldmlld2VkLmNzdiIpCiAgICAgICAgcmV0dXJuIDEKCiAgICBpbXBvcnQgY3N2CgogICAgd2l0aCBvcGVuKHNoZWV0LCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBmOgogICAgICAgIHJvd3MgPSBsaXN0KGNzdi5EaWN0UmVhZGVyKGYpKQoKICAgIGlmIG5vdCByb3dzOgogICAgICAgIHByaW50KCJTaGVldCBpcyBlbXB0eS4iKQogICAgICAgIHJldHVybiAxCgogICAgZGVmIG1hcmsocm93LCBjb2w6IHN0cik6CiAgICAgICAgdiA9IChyb3cuZ2V0KGNvbCkgb3IgIiIpLnN0cmlwKCkubG93ZXIoKQogICAgICAgIHJldHVybiB2IGlmIHYgaW4gVkFMSURfTUFSS1MgZWxzZSBOb25lCgogICAgcjEgPSBDb3VudGVyKG1hcmsociwgInJldmlld2VyXzEiKSBmb3IgciBpbiByb3dzKQogICAgcjIgPSBDb3VudGVyKG1hcmsociwgInJldmlld2VyXzIiKSBmb3IgciBpbiByb3dzKQogICAgcmV2aWV3ZWQgPSBbciBmb3IgciBpbiByb3dzIGlmIG1hcmsociwgInJldmlld2VyXzEiKSBvciBtYXJrKHIsICJyZXZpZXdlcl8yIildCiAgICBhZ3JlZWRfeWVzID0gc3VtKDEgZm9yIHIgaW4gcm93cyBpZiAoci5nZXQoImFncmVlbWVudCIpIG9yICIiKS5zdHJpcCgpLmxvd2VyKCkgPT0gInllcyIpCiAgICBhZ3JlZWRfbm8gPSBzdW0oMSBmb3IgciBpbiByb3dzIGlmIChyLmdldCgiYWdyZWVtZW50Iikgb3IgIiIpLnN0cmlwKCkubG93ZXIoKSA9PSAibm8iKQoKICAgIHByaW50KGYiQ2FzZXMgaW4gc2hlZXQ6ICAgICAgICB7bGVuKHJvd3MpfSIpCiAgICBwcmludChmIkNhc2VzIHdpdGgg4omlMSBtYXJrOiAgICB7bGVuKHJldmlld2VkKX0iKQogICAgcHJpbnQoZiJBZ3JlZW1lbnQ9eWVzOiAgICAgICAgIHthZ3JlZWRfeWVzfSAgKHthZ3JlZWRfeWVzIC8gbGVuKHJvd3MpOi4wJX0gb2YgYWxsIGNhc2VzKSIgaWYgcm93cyBlbHNlICIiKQogICAgcHJpbnQoZiJBZ3JlZW1lbnQ9bm86ICAgICAgICAgIHthZ3JlZWRfbm99IikKICAgIHByaW50KGYiUmV2aWV3ZXIgMSBtYXJrczogICAgICB7ZGljdChyMSl9IikKICAgIHByaW50KGYiUmV2aWV3ZXIgMiBtYXJrczogICAgICB7ZGljdChyMil9IikKCiAgICAjIENyb3NzLWNoZWNrOiBhZ3JlZW1lbnQgY29sdW1uIHZzIHJldmlld2VyIG1hcmtzIChkaXNjcmVwYW5jaWVzIGFyZSB3b3J0aCBhIGxvb2spCiAgICBmbGFnZ2VkID0gW10KICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgYSA9IChyLmdldCgiYWdyZWVtZW50Iikgb3IgIiIpLnN0cmlwKCkubG93ZXIoKQogICAgICAgIG0xLCBtMiA9IG1hcmsociwgInJldmlld2VyXzEiKSwgbWFyayhyLCAicmV2aWV3ZXJfMiIpCiAgICAgICAgaWYgbTEgYW5kIG0yOgogICAgICAgICAgICBjb21wdXRlZCA9ICJ5ZXMiIGlmIG0xID09IG0yIGVsc2UgIm5vIgogICAgICAgICAgICBpZiBhIGluICgieWVzIiwgIm5vIikgYW5kIGEgIT0gY29tcHV0ZWQ6CiAgICAgICAgICAgICAgICBmbGFnZ2VkLmFwcGVuZChyLmdldCgic2FtcGxlX2lkIikpCiAgICBpZiBmbGFnZ2VkOgogICAgICAgIHByaW50KGYiV0FSTjogYWdyZWVtZW50IGNvbHVtbiBjb250cmFkaWN0cyByZXZpZXdlciBtYXJrcyBmb3Ige2xlbihmbGFnZ2VkKX0gY2FzZXM6IHtmbGFnZ2VkWzoxMF19eycgLi4uJyBpZiBsZW4oZmxhZ2dlZCkgPiAxMCBlbHNlICcnfSIpCiAgICAgICAgcHJpbnQoIihEaXNhZ3JlZW1lbnRzIGFyZSBhIGRlbGl2ZXJhYmxlIOKAlCByZXBvcnQgdGhlbSwgZG9uJ3QgaGlkZSB0aGVtLikiKQoKICAgIGRpc2FncmVlbWVudHMgPSBbci5nZXQoInNhbXBsZV9pZCIpIGZvciByIGluIHJvd3MgaWYgKHIuZ2V0KCJhZ3JlZW1lbnQiKSBvciAiIikuc3RyaXAoKS5sb3dlcigpID09ICJubyJdCiAgICBpZiBkaXNhZ3JlZW1lbnRzOgogICAgICAgIHByaW50KGYiUmVjb3JkZWQgZGlzYWdyZWVtZW50cyAoe2xlbihkaXNhZ3JlZW1lbnRzKX0pOiB7ZGlzYWdyZWVtZW50c1s6MTVdfXsnIC4uLicgaWYgbGVuKGRpc2FncmVlbWVudHMpID4gMTUgZWxzZSAnJ30iKQogICAgcmV0dXJuIDAKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgc3lzLmV4aXQobWFpbigpKQo=",
 "src/models/run_all_experiments.py": "IiIiCkI2IOKAlCBSZXByb2R1Y2libGUgcHVibGljYXRpb24gcHJvdG9jb2w6IGNvbmZpZy1kcml2ZW4gb3JjaGVzdHJhdG9yIChibHVlcHJpbnQgwqcxMSkuCgpSdW5zIHRoZSBmdWxsIFZlcnNpb24gQiBwaXBlbGluZSAoQjEgZGF0YSAtPiBCMiBiYXNlbGluZXMgLT4gQjMgY3Jvc3MtZG9tYWluIC0+CkI0IGNhbGlicmF0aW9uIHNoaWZ0IC0+IEI1IGV4cGxhbmF0aW9uIHJlbGlhYmlsaXR5IC0+IG1hbmlmZXN0IC0+IHZlcmlmeSkgYnkKc3VicHJvY2Vzcy1pbnZva2luZyB0aGUgZXhpc3RpbmcgcGVyLXBoYXNlIHJ1bm5lcnMuIFRoZSBwZXItcGhhc2UgcnVubmVycyBzdGF5CnRoZSBzaW5nbGUgc291cmNlIG9mIHRydXRoOyB0aGlzIHNjcmlwdCBvbmx5IG9yZGVycyB0aGVtIGFuZCByZWNvcmRzIHdoYXQgcmFuLgoKQ29uZmlnOiBjb25maWdzL3ZlcnNpb25fYi55YW1sIChzY2hlbWEgYjYtcnVuLWFsbC12MSkuIFBsYWNlaG9sZGVycyB7ZGV2aWNlfQphbmQge2JhdGNoX3NpemV9IGFyZSByZXNvbHZlZCBmcm9tIGNvbmZpZyBkZWZhdWx0cyBvciBDTEkgb3ZlcnJpZGVzLgoKVXNhZ2UgKHJlcG8gcm9vdCk6CiAgcHl0aG9uIHNyYy9tb2RlbHMvcnVuX2FsbF9leHBlcmltZW50cy5weSAtLWRyeS1ydW4KICBweXRob24gc3JjL21vZGVscy9ydW5fYWxsX2V4cGVyaW1lbnRzLnB5IC0tZGV2aWNlIGN1ZGEgLS1mcm9tIGIyIC0tdG8gYjUKICBweXRob24gc3JjL21vZGVscy9ydW5fYWxsX2V4cGVyaW1lbnRzLnB5IC0ta2VlcC1nb2luZwoKRXZlcnkgcGhhc2Ugd3JpdGVzIGEgc3RhdHVzICsgZHVyYXRpb24gaW50byBhcnRpZmFjdHMvcmVzdWx0cy9ydW5fYWxsX3JlcG9ydC5qc29uCihnaXQgY29tbWl0IGluY2x1ZGVkIHdoZW4gcnVuIGluc2lkZSBhIGdpdCBjbG9uZTsgbnVsbCBvdGhlcndpc2UsIG1hdGNoaW5nIHRoZQpwZXItcGhhc2UgcnVuIGNvbmZpZ3MpLgoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB5YW1sCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoInJ1bl9hbGxfZXhwZXJpbWVudHMiKQoKQ09ORklHX1NDSEVNQSA9ICJiNi1ydW4tYWxsLXYxIgpERUZBVUxUX0NPTkZJRyA9IFJPT1QgLyAiY29uZmlncyIgLyAidmVyc2lvbl9iLnlhbWwiCkRFRkFVTFRfUkVQT1JUID0gUk9PVCAvICJhcnRpZmFjdHMiIC8gInJlc3VsdHMiIC8gInJ1bl9hbGxfcmVwb3J0Lmpzb24iCgoKZGVmIGdpdF9jb21taXQoKSAtPiBzdHIgfCBOb25lOgogICAgdHJ5OgogICAgICAgIG91dCA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgICAgICBbImdpdCIsICJyZXYtcGFyc2UiLCAiSEVBRCJdLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsIHRpbWVvdXQ9MTAsIGN3ZD1ST09UCiAgICAgICAgKQogICAgICAgIHJldHVybiBvdXQuc3Rkb3V0LnN0cmlwKCkgb3IgTm9uZQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gTm9uZQoKCmRlZiBsb2FkX2NvbmZpZyhwYXRoOiBQYXRoKSAtPiBkaWN0OgogICAgY2ZnID0geWFtbC5zYWZlX2xvYWQocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBpZiBub3QgaXNpbnN0YW5jZShjZmcsIGRpY3QpIG9yIGNmZy5nZXQoInNjaGVtYSIpICE9IENPTkZJR19TQ0hFTUE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntwYXRofSBpcyBub3QgYSB7Q09ORklHX1NDSEVNQX0gY29uZmlnIikKICAgIHBoYXNlcyA9IGNmZy5nZXQoInBoYXNlcyIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShwaGFzZXMsIGxpc3QpIG9yIG5vdCBwaGFzZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY29uZmlnIG11c3QgZGVjbGFyZSBhIG5vbi1lbXB0eSBwaGFzZXMgbGlzdCIpCiAgICBpZHMgPSBbcC5nZXQoImlkIikgZm9yIHAgaW4gcGhhc2VzXQogICAgaWYgYW55KG5vdCBpIG9yIGlkcy5jb3VudChpKSAhPSAxIGZvciBpIGluIGlkcyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInBoYXNlIGlkcyBtdXN0IGJlIHVuaXF1ZSBhbmQgbm9uLWVtcHR5OiB7aWRzfSIpCiAgICBmb3IgcCBpbiBwaGFzZXM6CiAgICAgICAgc2NyaXB0ID0gcC5nZXQoInNjcmlwdCIpCiAgICAgICAgaWYgbm90IHNjcmlwdCBvciBub3QgKFJPT1QgLyBzY3JpcHQpLmV4aXN0cygpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYicGhhc2UgJ3twLmdldCgnaWQnKX0nOiBzY3JpcHQgbm90IGZvdW5kOiB7c2NyaXB0fSIpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UocC5nZXQoImFyZ3MiLCBbXSksIGxpc3QpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYicGhhc2UgJ3twLmdldCgnaWQnKX0nOiBhcmdzIG11c3QgYmUgYSBsaXN0IikKICAgIHJldHVybiBjZmcKCgpkZWYgcmVzb2x2ZV9hcmdzKGFyZ3M6IGxpc3QsIGRlZmF1bHRzOiBkaWN0LCBvdmVycmlkZXM6IGRpY3QpIC0+IGxpc3Q6CiAgICB0b2tlbnMgPSB7azogb3ZlcnJpZGVzLmdldChrLCBkZWZhdWx0cy5nZXQoaykpIGZvciBrIGluICgiZGV2aWNlIiwgImJhdGNoX3NpemUiKX0KICAgIHJldHVybiBbc3RyKGEpLmZvcm1hdCgqKnRva2VucykgaWYgaXNpbnN0YW5jZShhLCBzdHIpIGVsc2UgYSBmb3IgYSBpbiBhcmdzXQoKCmRlZiBydW5fcGhhc2UocGhhc2U6IGRpY3QsIGFyZ3M6IGxpc3QsIGtlZXBfZ29pbmc6IGJvb2wpIC0+IGRpY3Q6CiAgICBzY3JpcHQgPSBST09UIC8gcGhhc2VbInNjcmlwdCJdCiAgICBjbWQgPSBbc3lzLmV4ZWN1dGFibGUsIHN0cihzY3JpcHQpLCAqYXJnc10KICAgIGxvZ2dlci5pbmZvKCI9PT4gJXM6ICVzIiwgcGhhc2VbImlkIl0sICIgIi5qb2luKHN0cihjKSBmb3IgYyBpbiBjbWQpKQogICAgc3RhcnRlZCA9IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCh0aW1lc3BlYz0ic2Vjb25kcyIpCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICB0cnk6CiAgICAgICAgcmVzdWx0ID0gc3VicHJvY2Vzcy5ydW4oY21kLCBjd2Q9Uk9PVCkKICAgICAgICBleGl0X2NvZGUsIHN0YXR1cyA9IHJlc3VsdC5yZXR1cm5jb2RlLCAib2siIGlmIHJlc3VsdC5yZXR1cm5jb2RlID09IDAgZWxzZSAiZmFpbGVkIgogICAgZXhjZXB0IE9TRXJyb3IgYXMgZToKICAgICAgICBleGl0X2NvZGUsIHN0YXR1cyA9IC0xLCBmInNwYXduIGZhaWxlZDoge2V9IgogICAgcmV0dXJuIHsKICAgICAgICAiaWQiOiBwaGFzZVsiaWQiXSwKICAgICAgICAic2NyaXB0IjogcGhhc2VbInNjcmlwdCJdLAogICAgICAgICJhcmdzIjogYXJncywKICAgICAgICAiZXhpdF9jb2RlIjogZXhpdF9jb2RlLAogICAgICAgICJzdGF0dXMiOiBzdGF0dXMsCiAgICAgICAgImR1cmF0aW9uX3MiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHQwLCAyKSwKICAgICAgICAic3RhcnRlZF9hdF91dGMiOiBzdGFydGVkLAogICAgfQoKCmRlZiBtYWluKGFyZ3Y6IGxpc3QgfCBOb25lID0gTm9uZSkgLT4gZGljdDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJCNiBjb25maWctZHJpdmVuIFZlcnNpb24gQiBwaXBlbGluZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9c3RyKERFRkFVTFRfQ09ORklHKSwgaGVscD0icGF0aCB0byBZQU1MIGNvbmZpZyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZyb20iLCBkZXN0PSJmcm9tX3BoYXNlIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJzdGFydCBwaGFzZSBpZCAoaW5jbHVzaXZlKSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRvIiwgZGVzdD0idG9fcGhhc2UiLCBkZWZhdWx0PU5vbmUsIGhlbHA9ImVuZCBwaGFzZSBpZCAoaW5jbHVzaXZlKSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0ib3ZlcnJpZGUgZGV2aWNlIChjcHV8Y3VkYXxhdXRvKSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLCBoZWxwPSJvdmVycmlkZSBiYXRjaCBzaXplIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZHJ5LXJ1biIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9InByaW50IHRoZSByZXNvbHZlZCBwbGFuIG9ubHkiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1rZWVwLWdvaW5nIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0iY29udGludWUgYWZ0ZXIgYSBmYWlsZWQgcGhhc2UiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yZXBvcnQiLCBkZWZhdWx0PXN0cihERUZBVUxUX1JFUE9SVCksIGhlbHA9Im91dHB1dCByZXBvcnQgcGF0aCIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoYXJndikKCiAgICBjZmcgPSBsb2FkX2NvbmZpZyhQYXRoKGFyZ3MuY29uZmlnKSkKICAgIHBoYXNlcyA9IGNmZ1sicGhhc2VzIl0KICAgIGlkcyA9IFtwWyJpZCJdIGZvciBwIGluIHBoYXNlc10KICAgIHN0YXJ0ID0gaWRzLmluZGV4KGFyZ3MuZnJvbV9waGFzZSkgaWYgYXJncy5mcm9tX3BoYXNlIGVsc2UgMAogICAgZW5kID0gaWRzLmluZGV4KGFyZ3MudG9fcGhhc2UpICsgMSBpZiBhcmdzLnRvX3BoYXNlIGVsc2UgbGVuKGlkcykKICAgIGlmIHN0YXJ0ID4gZW5kOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiItLWZyb20ge2FyZ3MuZnJvbV9waGFzZX0gY29tZXMgYWZ0ZXIgLS10byB7YXJncy50b19waGFzZX0iKQogICAgcGhhc2VzID0gcGhhc2VzW3N0YXJ0OmVuZF0KCiAgICBvdmVycmlkZXMgPSB7azogdiBmb3IgaywgdiBpbiB7ImRldmljZSI6IGFyZ3MuZGV2aWNlLCAiYmF0Y2hfc2l6ZSI6IGFyZ3MuYmF0Y2hfc2l6ZX0uaXRlbXMoKSBpZiB2IGlzIG5vdCBOb25lfQogICAgZGVmYXVsdHMgPSBjZmcuZ2V0KCJkZWZhdWx0cyIsIHt9KQogICAgcGxhbiA9IFsKICAgICAgICB7ImlkIjogcFsiaWQiXSwgInNjcmlwdCI6IHBbInNjcmlwdCJdLCAiYXJncyI6IHJlc29sdmVfYXJncyhwLmdldCgiYXJncyIsIFtdKSwgZGVmYXVsdHMsIG92ZXJyaWRlcyksCiAgICAgICAgICJkZXNjcmlwdGlvbiI6IHAuZ2V0KCJkZXNjcmlwdGlvbiIsICIiKX0KICAgICAgICBmb3IgcCBpbiBwaGFzZXMKICAgIF0KCiAgICBpZiBhcmdzLmRyeV9ydW46CiAgICAgICAgcHJpbnQoZiJEUlkgUlVOICh7bGVuKHBsYW4pfSBwaGFzZXMsIGRldmljZT17b3ZlcnJpZGVzLmdldCgnZGV2aWNlJywgZGVmYXVsdHMuZ2V0KCdkZXZpY2UnKSl9LCAiCiAgICAgICAgICAgICAgZiJiYXRjaF9zaXplPXtvdmVycmlkZXMuZ2V0KCdiYXRjaF9zaXplJywgZGVmYXVsdHMuZ2V0KCdiYXRjaF9zaXplJykpfSkiKQogICAgICAgIGZvciBwIGluIHBsYW46CiAgICAgICAgICAgIHByaW50KGYiICBbe3BbJ2lkJ106PjIwfV0ge3BbJ3NjcmlwdCddfSB7JyAnLmpvaW4ocFsnYXJncyddKX0iKQogICAgICAgIHJldHVybiB7ImRyeV9ydW4iOiBUcnVlLCAicGhhc2VzIjogcGxhbn0KCiAgICByZXN1bHRzID0gW10KICAgIG92ZXJhbGwgPSAib2siCiAgICBmb3IgcGhhc2UgaW4gcGxhbjoKICAgICAgICByZXN1bHQgPSBydW5fcGhhc2UocGhhc2UsIHBoYXNlWyJhcmdzIl0sIGFyZ3Mua2VlcF9nb2luZykKICAgICAgICByZXN1bHRzLmFwcGVuZChyZXN1bHQpCiAgICAgICAgaWYgcmVzdWx0WyJzdGF0dXMiXSAhPSAib2siOgogICAgICAgICAgICBvdmVyYWxsID0gImZhaWxlZCIKICAgICAgICAgICAgaWYgbm90IGFyZ3Mua2VlcF9nb2luZzoKICAgICAgICAgICAgICAgIGxvZ2dlci5lcnJvcigicGhhc2UgJXMgZmFpbGVkIChleGl0ICVzKTsgYWJvcnRpbmcgKHVzZSAtLWtlZXAtZ29pbmcgdG8gY29udGludWUpIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwaGFzZVsiaWQiXSwgcmVzdWx0WyJleGl0X2NvZGUiXSkKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgcmVwb3J0ID0gewogICAgICAgICJzY2hlbWEiOiAiYjYtcnVuLWFsbC1yZXBvcnQtdjEiLAogICAgICAgICJnZW5lcmF0ZWRfYXRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KHRpbWVzcGVjPSJzZWNvbmRzIiksCiAgICAgICAgImdpdF9jb21taXQiOiBnaXRfY29tbWl0KCksCiAgICAgICAgImNvbmZpZyI6IHN0cihQYXRoKGFyZ3MuY29uZmlnKSksCiAgICAgICAgInJlc29sdmVkX2RlZmF1bHRzIjogeyoqZGVmYXVsdHMsICoqb3ZlcnJpZGVzfSwKICAgICAgICAib3ZlcmFsbF9zdGF0dXMiOiBvdmVyYWxsLAogICAgICAgICJwaGFzZXMiOiByZXN1bHRzLAogICAgfQogICAgcmVwb3J0X3BhdGggPSBQYXRoKGFyZ3MucmVwb3J0KQogICAgcmVwb3J0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHJlcG9ydF9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIGxvZ2dlci5pbmZvKCJSZXBvcnQgd3JpdHRlbiB0byAlcyAob3ZlcmFsbDogJXMpIiwgcmVwb3J0X3BhdGgsIG92ZXJhbGwpCiAgICByZXR1cm4gcmVwb3J0CgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJlcG9ydCA9IG1haW4oKQogICAgb2sgPSByZXBvcnQuZ2V0KCJkcnlfcnVuIikgb3IgcmVwb3J0LmdldCgib3ZlcmFsbF9zdGF0dXMiKSA9PSAib2siCiAgICBzeXMuZXhpdCgwIGlmIG9rIGVsc2UgMSkK",
 "src/models/run_b2_baselines.py": "...",
 "src/models/run_b3_cross_domain.py": "...",
 "src/models/run_b4_calibration_shift.py": "...",
 "src/models/run_b5_explanation_reliability.py": "...",
 "src/models/train_baselines.py": "IiIiCkJhc2VsaW5lIG1vZGVsaW5nIHNjcmlwdCBmb3IgSGFsdVJJU0MuClRyYWlucyBhbmQgZXZhbHVhdGVzIEhldXJpc3RpYyBSdWxlLCBMb2dpc3RpYyBSZWdyZXNzaW9uLCBSYW5kb20gRm9yZXN0LCBhbmQgWEdCb29zdCBtb2RlbHMKb24gZXh0cmFjdGVkIGZlYXR1cmVzLCBwcmludGluZyBhIHBlcmZvcm1hbmNlIGNvbXBhcmlzb24gdGFibGUuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2tsZWFybi5wcmVwcm9jZXNzaW5nIGltcG9ydCBTdGFuZGFyZFNjYWxlcgpmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb24KZnJvbSBza2xlYXJuLmVuc2VtYmxlIGltcG9ydCBSYW5kb21Gb3Jlc3RDbGFzc2lmaWVyCmZyb20geGdib29zdCBpbXBvcnQgWEdCQ2xhc3NpZmllcgpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgKAogICAgcHJlY2lzaW9uX3Njb3JlLCByZWNhbGxfc2NvcmUsIGYxX3Njb3JlLCAKICAgIHJvY19hdWNfc2NvcmUsIGF2ZXJhZ2VfcHJlY2lzaW9uX3Njb3JlLCBtYXR0aGV3c19jb3JyY29lZgopCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAtICUobGV2ZWxuYW1lKXMgLSAlKG1lc3NhZ2UpcyIpCgpGRUFUVVJFU19QQVRIID0gb3MucGF0aC5qb2luKCJkYXRhIiwgInByb2Nlc3NlZCIsICJmZWF0dXJlc19jb3JlLnBhcnF1ZXQiKQpSRVNVTFRTX0RJUiA9IG9zLnBhdGguam9pbigiYXJ0aWZhY3RzIiwgInJlc3VsdHMiKQpNT0RFTFNfRElSID0gb3MucGF0aC5qb2luKCJhcnRpZmFjdHMiLCAibW9kZWxzIikKCmRlZiBldmFsdWF0ZV9wcmVkaWN0aW9ucyh5X3RydWUsIHlfcHJlZCwgeV9wcm9iKSAtPiBkaWN0OgogICAgIiIiQ2FsY3VsYXRlcyBldmFsdWF0aW9uIG1ldHJpY3MgZm9yIGJpbmFyeSBjbGFzc2lmaWNhdGlvbi4iIiIKICAgIHJldHVybiB7CiAgICAgICAgInByZWNpc2lvbiI6IGZsb2F0KHByZWNpc2lvbl9zY29yZSh5X3RydWUsIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKSksCiAgICAgICAgInJlY2FsbCI6IGZsb2F0KHJlY2FsbF9zY29yZSh5X3RydWUsIHlfcHJlZCwgemVyb19kaXZpc2lvbj0wKSksCiAgICAgICAgImYxIjogZmxvYXQoZjFfc2NvcmUoeV90cnVlLCB5X3ByZWQsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJhdXJvYyI6IGZsb2F0KHJvY19hdWNfc2NvcmUoeV90cnVlLCB5X3Byb2IpKSwKICAgICAgICAicHJfYXVjIjogZmxvYXQoYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUoeV90cnVlLCB5X3Byb2IpKSwKICAgICAgICAibWNjIjogZmxvYXQobWF0dGhld3NfY29ycmNvZWYoeV90cnVlLCB5X3ByZWQpKQogICAgfQoKZGVmIHJ1bl9oZXVyaXN0aWNfYmFzZWxpbmUodmFsX2RmOiBwZC5EYXRhRnJhbWUsIHRlc3RfZGY6IHBkLkRhdGFGcmFtZSk6CiAgICAiIiJSdWxlLWJhc2VkIGhldXJpc3RpYzogaGlnaCBvdmVybGFwX2Fuc3dlcl9jb250ZXh0IC0+IGxvdyByaXNrICgwKSwgbG93IG92ZXJsYXAgLT4gaGlnaCByaXNrICgxKS4iIiIKICAgICMgVGhyZXNob2xkIHR1bmVkIG9uIHZhbCBzZXQKICAgIHRocmVzaG9sZHMgPSBucC5saW5zcGFjZSgwLCAxLCAxMDEpCiAgICBiZXN0X3RocmVzaCA9IDAuNQogICAgYmVzdF92YWxfZjEgPSAwLjAKCiAgICBmb3IgdCBpbiB0aHJlc2hvbGRzOgogICAgICAgIHZhbF9wcmVkcyA9ICh2YWxfZGZbIm92ZXJsYXBfYW5zd2VyX2NvbnRleHQiXSA8IHQpLmFzdHlwZShpbnQpCiAgICAgICAgZjEgPSBmMV9zY29yZSh2YWxfZGZbImxhYmVsIl0sIHZhbF9wcmVkcywgemVyb19kaXZpc2lvbj0wKQogICAgICAgIGlmIGYxID4gYmVzdF92YWxfZjE6CiAgICAgICAgICAgIGJlc3RfdmFsX2YxID0gZjEKICAgICAgICAgICAgYmVzdF90aHJlc2ggPSB0CgogICAgbG9nZ2luZy5pbmZvKGYiSGV1cmlzdGljIGJlc3Qgb3ZlcmxhcCB0aHJlc2hvbGQgb24gVmFsOiB7YmVzdF90aHJlc2g6LjJmfSAoRjE6IHtiZXN0X3ZhbF9mMTouNGZ9KSIpCgogICAgIyBQcmVkaWN0IG9uIHRlc3QKICAgIHRlc3RfcHJvYnMgPSAxLjAgLSB0ZXN0X2RmWyJvdmVybGFwX2Fuc3dlcl9jb250ZXh0Il0KICAgIHRlc3RfcHJlZHMgPSAodGVzdF9kZlsib3ZlcmxhcF9hbnN3ZXJfY29udGV4dCJdIDwgYmVzdF90aHJlc2gpLmFzdHlwZShpbnQpCgogICAgbWV0cmljcyA9IGV2YWx1YXRlX3ByZWRpY3Rpb25zKHRlc3RfZGZbImxhYmVsIl0sIHRlc3RfcHJlZHMsIHRlc3RfcHJvYnMpCiAgICByZXR1cm4gbWV0cmljcywgYmVzdF90aHJlc2gKCmRlZiB0cmFpbl9hbmRfZXZhbF9hbGwoKToKICAgIG9zLm1ha2VkaXJzKFJFU1VMVFNfRElSLCBleGlzdF9vaz1UcnVlKQogICAgb3MubWFrZWRpcnMoTU9ERUxTX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoRkVBVFVSRVNfUEFUSCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7RkVBVFVSRVNfUEFUSH0gbm90IGZvdW5kLiBSdW4gc3JjL2ZlYXR1cmVzL2V4dHJhY3RfZmVhdHVyZXMucHkgZmlyc3QuIikKCiAgICBkZiA9IHBkLnJlYWRfcGFycXVldChGRUFUVVJFU19QQVRIKQogICAgZmVhdHVyZV9jb2xzID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiBjIG5vdCBpbiBbInNhbXBsZV9pZCIsICJpdGVtX2lkeCIsICJsYWJlbCIsICJzcGxpdCJdXQoKICAgIHRyYWluX2RmID0gZGZbZGZbInNwbGl0Il0gPT0gInRyYWluIl0KICAgIHZhbF9kZiA9IGRmW2RmWyJzcGxpdCJdID09ICJ2YWwiXQogICAgdGVzdF9kZiA9IGRmW2RmWyJzcGxpdCJdID09ICJ0ZXN0Il0KCiAgICBsb2dnaW5nLmluZm8oZiJUcmFpbiBzYW1wbGVzOiB7bGVuKHRyYWluX2RmKX0sIFZhbDoge2xlbih2YWxfZGYpfSwgVGVzdDoge2xlbih0ZXN0X2RmKX0iKQogICAgbG9nZ2luZy5pbmZvKGYiRmVhdHVyZSBsaXN0ICh7bGVuKGZlYXR1cmVfY29scyl9KToge2ZlYXR1cmVfY29sc30iKQoKICAgIFhfdHJhaW4sIHlfdHJhaW4gPSB0cmFpbl9kZltmZWF0dXJlX2NvbHNdLCB0cmFpbl9kZlsibGFiZWwiXQogICAgWF92YWwsIHlfdmFsID0gdmFsX2RmW2ZlYXR1cmVfY29sc10sIHZhbF9kZlsibGFiZWwiXQogICAgWF90ZXN0LCB5X3Rlc3QgPSB0ZXN0X2RmW2ZlYXR1cmVfY29sc10sIHRlc3RfZGZbImxhYmVsIl0KCiAgICByZXN1bHRzID0ge30KCiAgICAjIDEuIEhldXJpc3RpYyBCYXNlbGluZQogICAgaGV1cl9tZXRyaWNzLCBoZXVyX3RocmVzaCA9IHJ1bl9oZXVyaXN0aWNfYmFzZWxpbmUodmFsX2RmLCB0ZXN0X2RmKQogICAgcmVzdWx0c1siSGV1cmlzdGljIChPdmVybGFwKSJdID0gaGV1cl9tZXRyaWNzCgogICAgIyAyLiBMb2dpc3RpYyBSZWdyZXNzaW9uIChTY2FsZWQpCiAgICBzY2FsZXIgPSBTdGFuZGFyZFNjYWxlcigpCiAgICBYX3RyYWluX3NjYWxlZCA9IHNjYWxlci5maXRfdHJhbnNmb3JtKFhfdHJhaW4pCiAgICBYX3Rlc3Rfc2NhbGVkID0gc2NhbGVyLnRyYW5zZm9ybShYX3Rlc3QpCgogICAgbHIgPSBMb2dpc3RpY1JlZ3Jlc3Npb24obWF4X2l0ZXI9MjAwMCwgcmFuZG9tX3N0YXRlPTQyKQogICAgbHIuZml0KFhfdHJhaW5fc2NhbGVkLCB5X3RyYWluKQogICAgbHJfcHJvYnMgPSBsci5wcmVkaWN0X3Byb2JhKFhfdGVzdF9zY2FsZWQpWzosIDFdCiAgICBscl9wcmVkcyA9IChscl9wcm9icyA+PSAwLjUpLmFzdHlwZShpbnQpCiAgICByZXN1bHRzWyJMb2dpc3RpYyBSZWdyZXNzaW9uIl0gPSBldmFsdWF0ZV9wcmVkaWN0aW9ucyh5X3Rlc3QsIGxyX3ByZWRzLCBscl9wcm9icykKCiAgICAjIDMuIFJhbmRvbSBGb3Jlc3QKICAgIHJmID0gUmFuZG9tRm9yZXN0Q2xhc3NpZmllcihuX2VzdGltYXRvcnM9MzAwLCBtaW5fc2FtcGxlc19sZWFmPTUsIHJhbmRvbV9zdGF0ZT00Miwgbl9qb2JzPS0xKQogICAgcmYuZml0KFhfdHJhaW4sIHlfdHJhaW4pCiAgICByZl9wcm9icyA9IHJmLnByZWRpY3RfcHJvYmEoWF90ZXN0KVs6LCAxXQogICAgcmZfcHJlZHMgPSAocmZfcHJvYnMgPj0gMC41KS5hc3R5cGUoaW50KQogICAgcmVzdWx0c1siUmFuZG9tIEZvcmVzdCJdID0gZXZhbHVhdGVfcHJlZGljdGlvbnMoeV90ZXN0LCByZl9wcmVkcywgcmZfcHJvYnMpCgogICAgIyA0LiBYR0Jvb3N0CiAgICB4Z2IgPSBYR0JDbGFzc2lmaWVyKAogICAgICAgIG5fZXN0aW1hdG9ycz0zMDAsIG1heF9kZXB0aD01LCBsZWFybmluZ19yYXRlPTAuMDUsIAogICAgICAgIGV2YWxfbWV0cmljPSJsb2dsb3NzIiwgcmFuZG9tX3N0YXRlPTQyLCBuX2pvYnM9LTEKICAgICkKICAgIHhnYi5maXQoWF90cmFpbiwgeV90cmFpbikKICAgIHhnYl9wcm9icyA9IHhnYi5wcmVkaWN0X3Byb2JhKFhfdGVzdClbOiwgMV0KICAgIHhnYl9wcmVkcyA9ICh4Z2JfcHJvYnMgPj0gMC41KS5hc3R5cGUoaW50KQogICAgcmVzdWx0c1siWEdCb29zdCAoRGVmYXVsdCkiXSA9IGV2YWx1YXRlX3ByZWRpY3Rpb25zKHlfdGVzdCwgeGdiX3ByZWRzLCB4Z2JfcHJvYnMpCgogICAgIyBDb252ZXJ0IHJlc3VsdHMgdG8gRGF0YUZyYW1lIGFuZCBkaXNwbGF5CiAgICByZXN1bHRzX2RmID0gcGQuRGF0YUZyYW1lKHJlc3VsdHMpLlQKICAgIHJlc3VsdHNfZGYgPSByZXN1bHRzX2RmW1sicHJlY2lzaW9uIiwgInJlY2FsbCIsICJmMSIsICJhdXJvYyIsICJwcl9hdWMiLCAibWNjIl1dCiAgICAKICAgIHByaW50KCJcbiIgKyAiPSIqODApCiAgICBwcmludCgiIEhhbHVSSVNDIEJhc2VsaW5lIE1vZGVsIENvbXBhcmlzb24gb24gVGVzdCBTZXQgKENvcmUgRmVhdHVyZXMpIikKICAgIHByaW50KCI9Iio4MCkKICAgIHByaW50KHJlc3VsdHNfZGYudG9fc3RyaW5nKCkpCiAgICBwcmludCgiPSIqODAgKyAiXG4iKQoKICAgICMgU2F2ZSByZXN1bHRzIHRvIEpTT04gYW5kIENTVgogICAgcmVzdWx0c19kZi50b19jc3Yob3MucGF0aC5qb2luKFJFU1VMVFNfRElSLCAiYmFzZWxpbmVfcmVzdWx0cy5jc3YiKSkKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oUkVTVUxUU19ESVIsICJiYXNlbGluZV9yZXN1bHRzLmpzb24iKSwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChyZXN1bHRzLCBmLCBpbmRlbnQ9MikKCiAgICBsb2dnaW5nLmluZm8oZiJTYXZlZCBiYXNlbGluZSBldmFsdWF0aW9uIHJlc3VsdHMgdG8ge1JFU1VMVFNfRElSfSIpCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgdHJhaW5fYW5kX2V2YWxfYWxsKCkK",
 "src/models/train_pipeline.py": "...",
 "src/models/tune_thresholds.py": "IiIiVDQ6IGZlZWRiYWNrLWRyaXZlbiBOTEkgdGhyZXNob2xkIHR1bmluZyAocmVwb3J0LW9ubHkgYnkgZGVmYXVsdCkuCgpTZWFyY2hlcyB0aGUgZW50YWlsbWVudC9jb250cmFkaWN0aW9uIHRocmVzaG9sZCBncmlkIGZvciB0aGUgY29tYmluYXRpb24gdGhhdAptYXhpbWl6ZXMgdmVyZGljdCBhZ3JlZW1lbnQgb24gTEFCRUxFRCBmZWVkYmFjayByb3dzIChyb3dzIHdpdGggYQpgY29ycmVjdF92ZXJkaWN0YCBmaWVsZCDigJQgZmlsbGVkIG1hbnVhbGx5IG9yIGV4cG9ydGVkIGZyb20gdGhlIGNsYWltLWV2YWwKc2hlZXQpLiBSb3dzIHdpdGggb25seSBhZ3JlZS9kaXNhZ3JlZSBmZWVkYmFjayBhcmUgcmVwb3J0ZWQgYXMgdW5yZXNvbHZlZC4KCiAgcHl0aG9uIHNyYy9tb2RlbHMvdHVuZV90aHJlc2hvbGRzLnB5ICAgICAgICAgICAgICAgICAgICAjIHJlcG9ydCBvbmx5CiAgcHl0aG9uIHNyYy9tb2RlbHMvdHVuZV90aHJlc2hvbGRzLnB5IC0tYXBwbHkgICAgICAgICAgICAjIHdyaXRlIHRocmVzaG9sZHMKICBweXRob24gc3JjL21vZGVscy90dW5lX3RocmVzaG9sZHMucHkgLS1mcm9tIGV2YWwuY3N2ICAgICMgdXNlIGEgcmV2aWV3ZWQgc2hlZXQKCi0tYXBwbHkgd3JpdGVzIGRhdGEvcHJvY2Vzc2VkL3ZlcmRpY3RfdGhyZXNob2xkcy5qc29uIHdoaWNoIHRoZSB2ZXJpZmllcgpyZWFkcyBhdCBydW50aW1lIChhIGRlbGliZXJhdGUsIGV4cGxpY2l0IGh1bWFuIHN0ZXA7IGRlZmF1bHQgdGhyZXNob2xkcyBzdGF5CmVudGFpbD0wLjUgLyBjb250cmE9MC41KS4KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQganNvbgppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0Kc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihST09UKSkKCkZFRURCQUNLX0xPRyA9IFJPT1QgLyAiZGF0YSIgLyAicHJvY2Vzc2VkIiAvICJmZWVkYmFja19sb2cuanNvbmwiClRIUkVTSE9MRFNfRklMRSA9IFJPT1QgLyAiZGF0YSIgLyAicHJvY2Vzc2VkIiAvICJ2ZXJkaWN0X3RocmVzaG9sZHMuanNvbiIKREVGQVVMVF9USFJFU0hPTERTID0geyJlbnRhaWwiOiAwLjUsICJjb250cmEiOiAwLjV9CgpHUklEID0gbnAuYXJhbmdlKDAuMzUsIDAuODEsIDAuMDUpCgoKZGVmIGxvYWRfcm93cyhzb3VyY2U6IFBhdGggfCBOb25lID0gTm9uZSkgLT4gbGlzdDoKICAgICIiIkZlZWRiYWNrIHJvd3Mgd2l0aCBjbGFpbV90ZXh0ICsgY29ycmVjdF92ZXJkaWN0ICgrIGV2aWRlbmNlX3NlbnRlbmNlKS4iIiIKICAgIHJvd3MgPSBbXQogICAgaWYgc291cmNlIGlzIE5vbmU6CiAgICAgICAgcGF0aCA9IEZFRURCQUNLX0xPRwogICAgICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOgogICAgICAgICAgICByZXR1cm4gW10KICAgICAgICBmb3IgbGluZSBpbiBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zcGxpdGxpbmVzKCk6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKGpzb24ubG9hZHMobGluZSkpCiAgICAgICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgICAgICAgICAgY29udGludWUKICAgIGVsc2U6CiAgICAgICAgd2l0aCBvcGVuKHNvdXJjZSwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgZjoKICAgICAgICAgICAgcm93cyA9IFtkaWN0KHIpIGZvciByIGluIGNzdi5EaWN0UmVhZGVyKGYpXQogICAgcmV0dXJuIHJvd3MKCgpkZWYgbGFiZWxlZF9yb3dzKHJvd3M6IGxpc3QpIC0+IGxpc3Q6CiAgICBvdXQgPSBbXQogICAgZm9yIHIgaW4gcm93czoKICAgICAgICBjdiA9IChyLmdldCgiY29ycmVjdF92ZXJkaWN0Iikgb3IgIiIpLnN0cmlwKCkKICAgICAgICBpZiBjdiBpbiAoInN1cHBvcnRlZCIsICJjb250cmFkaWN0ZWQiLCAidW5zdXBwb3J0ZWQiKSBhbmQgKHIuZ2V0KCJjbGFpbV90ZXh0Iikgb3IgIiIpLnN0cmlwKCk6CiAgICAgICAgICAgIG91dC5hcHBlbmQoeyoqciwgImNvcnJlY3RfdmVyZGljdCI6IGN2fSkKICAgIHJldHVybiBvdXQKCgpkZWYgX3ZlcmRpY3QocHJvYnM6IG5wLm5kYXJyYXksIGVudF90aHI6IGZsb2F0LCBjb25fdGhyOiBmbG9hdCkgLT4gc3RyOgogICAgY29udHJhLCBlbnRhaWwgPSBwcm9ic1swXSwgcHJvYnNbMV0KICAgIGlmIGVudGFpbCA+PSBlbnRfdGhyIGFuZCBlbnRhaWwgPj0gY29udHJhOgogICAgICAgIHJldHVybiAic3VwcG9ydGVkIgogICAgaWYgY29udHJhID49IGNvbl90aHI6CiAgICAgICAgcmV0dXJuICJjb250cmFkaWN0ZWQiCiAgICByZXR1cm4gInVuc3VwcG9ydGVkIgoKCmRlZiB0dW5lKHJvd3M6IGxpc3QsIG5saV9tb2RlbCkgLT4gZGljdDoKICAgICIiIkdyaWQgc2VhcmNoIG92ZXIgdGhyZXNob2xkczsgcmV0dXJucyB7YmVzdCwgZ3JpZCwgdW5yZXNvbHZlZH0uIiIiCiAgICBsYWJlbGVkID0gbGFiZWxlZF9yb3dzKHJvd3MpCiAgICB1bnJlc29sdmVkID0gW3IgZm9yIHIgaW4gcm93cyBpZiByIG5vdCBpbiBsYWJlbGVkXQogICAgcmVzdWx0cyA9IFtdCiAgICBpZiBsYWJlbGVkOgogICAgICAgIGZvciBlbnQgaW4gR1JJRDoKICAgICAgICAgICAgZm9yIGNvbiBpbiBHUklEOgogICAgICAgICAgICAgICAgYWdyZWUgPSAwCiAgICAgICAgICAgICAgICBmb3IgciBpbiBsYWJlbGVkOgogICAgICAgICAgICAgICAgICAgIHByb2JzID0gbnAuYXNhcnJheShubGlfbW9kZWwucHJlZGljdCgKICAgICAgICAgICAgICAgICAgICAgICAgW1tyLmdldCgiZXZpZGVuY2Vfc2VudGVuY2UiLCAiIiksIHJbImNsYWltX3RleHQiXV1dLAogICAgICAgICAgICAgICAgICAgICAgICBiYXRjaF9zaXplPTEsIGFwcGx5X3NvZnRtYXg9VHJ1ZSkpWzBdCiAgICAgICAgICAgICAgICAgICAgYWdyZWUgKz0gaW50KF92ZXJkaWN0KHByb2JzLCBmbG9hdChlbnQpLCBmbG9hdChjb24pKSA9PSByWyJjb3JyZWN0X3ZlcmRpY3QiXSkKICAgICAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHsiZW50YWlsIjogcm91bmQoZmxvYXQoZW50KSwgMiksICJjb250cmEiOiByb3VuZChmbG9hdChjb24pLCAyKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiYWdyZWVtZW50IjogYWdyZWUgLyBsZW4obGFiZWxlZCl9KQogICAgICAgIGJlc3QgPSBtYXgocmVzdWx0cywga2V5PWxhbWJkYSB4OiB4WyJhZ3JlZW1lbnQiXSkKICAgIGVsc2U6CiAgICAgICAgYmVzdCA9IE5vbmUKICAgIHJldHVybiB7ImJlc3QiOiBiZXN0LCAiZ3JpZCI6IHJlc3VsdHMsICJuX2xhYmVsZWQiOiBsZW4obGFiZWxlZCksCiAgICAgICAgICAgICJuX3VucmVzb2x2ZWQiOiBsZW4odW5yZXNvbHZlZCksICJkZWZhdWx0cyI6IERFRkFVTFRfVEhSRVNIT0xEU30KCgpkZWYgbWFpbihhcmd2OiBsaXN0IHwgTm9uZSA9IE5vbmUpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJUNCBmZWVkYmFjay1kcml2ZW4gTkxJIHRocmVzaG9sZCB0dW5pbmcgKHJlcG9ydC1vbmx5IGJ5IGRlZmF1bHQpIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYXBwbHkiLCBhY3Rpb249InN0b3JlX3RydWUiLCBoZWxwPSJ3cml0ZSB0aGUgYmVzdCB0aHJlc2hvbGRzIChleHBsaWNpdCBodW1hbiBzdGVwKSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZyb20iLCBkZXN0PSJzb3VyY2UiLCBkZWZhdWx0PU5vbmUsIGhlbHA9IkNTViBzb3VyY2UgaW5zdGVhZCBvZiB0aGUgZmVlZGJhY2sgbG9nIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncyhhcmd2KQoKICAgIHJvd3MgPSBsb2FkX3Jvd3MoUGF0aChhcmdzLnNvdXJjZSkgaWYgYXJncy5zb3VyY2UgZWxzZSBOb25lKQogICAgaWYgbm90IHJvd3M6CiAgICAgICAgcHJpbnQoZiJObyBmZWVkYmFjayByb3dzIGZvdW5kICh7YXJncy5zb3VyY2Ugb3IgRkVFREJBQ0tfTE9HfSkuIENvbGxlY3QgZmVlZGJhY2sgZmlyc3QsIG9yIHBhc3MgLS1mcm9tLiIpCiAgICAgICAgcmV0dXJuIDEKCiAgICBmcm9tIHNyYy5mZWF0dXJlcy5leHRyYWN0X2ZlYXR1cmVzIGltcG9ydCBsb2FkX2hlYXZ5X21vZGVscwoKICAgIHByaW50KCJMb2FkaW5nIE5MSSBtb2RlbCAoZmlyc3QgcnVuIGRvd25sb2FkcyBpdCkuLi4iKQogICAgbmxpID0gbG9hZF9oZWF2eV9tb2RlbHMoZGV2aWNlPSJjcHUiKS5nZXQoIm5saSIpCiAgICByZXN1bHQgPSB0dW5lKHJvd3MsIG5saSkKCiAgICBwcmludChmIkxhYmVsZWQgcm93czoge3Jlc3VsdFsnbl9sYWJlbGVkJ119ICB8IHVucmVzb2x2ZWQgKGFncmVlL2Rpc2FncmVlIG9ubHkpOiB7cmVzdWx0WyduX3VucmVzb2x2ZWQnXX0iKQogICAgaWYgcmVzdWx0WyJiZXN0Il06CiAgICAgICAgYiA9IHJlc3VsdFsiYmVzdCJdCiAgICAgICAgcHJpbnQoZiJCZXN0IHRocmVzaG9sZHM6IGVudGFpbD17YlsnZW50YWlsJ119IGNvbnRyYT17YlsnY29udHJhJ119IGFncmVlbWVudD17YlsnYWdyZWVtZW50J106LjNmfSIpCiAgICAgICAgaWYgYXJncy5hcHBseToKICAgICAgICAgICAgVEhSRVNIT0xEU19GSUxFLndyaXRlX3RleHQoanNvbi5kdW1wcygKICAgICAgICAgICAgICAgIHsiZW50YWlsIjogYlsiZW50YWlsIl0sICJjb250cmEiOiBiWyJjb250cmEiXSwKICAgICAgICAgICAgICAgICAiYWdyZWVtZW50Ijogcm91bmQoYlsiYWdyZWVtZW50Il0sIDQpLAogICAgICAgICAgICAgICAgICJuX2xhYmVsZWQiOiByZXN1bHRbIm5fbGFiZWxlZCJdfSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICBwcmludChmIldyb3RlIHtUSFJFU0hPTERTX0ZJTEV9ICh2ZXJpZmllciByZWFkcyBpdCBhdCBydW50aW1lKS4iKQogICAgZWxzZToKICAgICAgICBwcmludCgiTm8gbGFiZWxlZCByb3dzIHdpdGggY29ycmVjdF92ZXJkaWN0IOKAlCBhZGQgbGFiZWxzIChjbGFpbS1ldmFsIHNoZWV0KSBiZWZvcmUgdHVuaW5nLiIpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg==",
 "src/models/verify_artifacts.py": "IiIiClBvc3QtcnVuIGFydGlmYWN0IHZlcmlmaWNhdGlvbiAoQ29sYWIgY2VsbCA3aSArIGxvY2FsIHBvc3QtZG93bmxvYWQgY2hlY2spLgoKTG9hZHMgZXZlcnkgYXJ0aWZhY3QgdGhlIHBpcGVsaW5lIHByb2R1Y2VzIGFuZCBwcm92ZXMgaXQgY2FuIGJlIHVzZWQgb24gVEhJUwptYWNoaW5lLCBwcmV2ZW50aW5nIHRoZSBoaXN0b3JpY2FsICJkb3dubG9hZGVkIG1vZGVsIGZhaWxzIHRvIGxvYWQiIGNsYXNzIG9mCmVycm9ycyAoQ1VEQS10cmFpbmVkIGJvb3N0ZXJzIG5vdCBwb3J0aW5nIGFjcm9zcyBwbGF0Zm9ybXMpOgoKICAtIEIyIFhHQm9vc3QgYm9vc3RlcnMgKDMgc2VlZHMpIGxvYWQgYW5kIHByZWRpY3Qgd2l0aGluIFswLCAxXQogIC0gQjQgc291cmNlICsgdGFyZ2V0IGNhbGlicmF0b3JzIChwdXJlIHNrbGVhcm4pIGxvYWQgYW5kIGFwcGx5CiAgLSBCMi9CMy9CNCBwcmVkaWN0aW9uIHBhcnF1ZXQgZmlsZXMgZXhpc3Qgd2l0aCB0aGUgZXhwZWN0ZWQgc2NoZW1hCiAgLSBGZWF0dXJlLWNvbHVtbiBvcmRlciBtYXRjaGVzIHRoZSBCMiBjb25maWcKCkV4aXQgY29kZSAwID0gZXZlcnl0aGluZyB1c2FibGU7IDEgPSBmaXJzdCBmYWlsaW5nIGNoZWNrIChjbGVhciBtZXNzYWdlKS4KClJ1biAocmVwbyByb290LCAudmVudiBvciBDb2xhYiBhZnRlciBCMy9CNCk6CiAgcHl0aG9uIHNyYy9tb2RlbHMvdmVyaWZ5X2FydGlmYWN0cy5weQoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzJdKSkKCmltcG9ydCBqb2JsaWIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmxvZ2dpbmcuYmFzaWNDb25maWcobGV2ZWw9bG9nZ2luZy5JTkZPLCBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoInZlcmlmeV9hcnRpZmFjdHMiKQoKZnJvbSBzcmMubW9kZWxzLmNvbmZpZyBpbXBvcnQgREFUQV9QUk9DRVNTRUQsIE1PREVMU19ESVIsIFJFU1VMVFNfRElSICAjIG5vcWE6IEU0MDIKCkIyX01PREVMU19ESVIgPSBNT0RFTFNfRElSIC8gImIyIgpCMl9SRVNVTFRTID0gUkVTVUxUU19ESVIgLyAiYjIiCkIzX1JFU1VMVFMgPSBSRVNVTFRTX0RJUiAvICJiMyIKQjRfUkVTVUxUUyA9IFJFU1VMVFNfRElSIC8gImI0IgpCNF9NT0RFTFMgPSBNT0RFTFNfRElSIC8gImI0IgpGRUFUVVJFU19GVUxMID0gREFUQV9QUk9DRVNTRUQgLyAiZmVhdHVyZXNfZnVsbC5wYXJxdWV0IgpTRUVEUyA9IFs0MiwgMTIzLCA0NTZdCgpGQUlMVVJFUyA9IFtdCgoKZGVmIGNoZWNrKG5hbWUsIGZuKToKICAgIHRyeToKICAgICAgICBmbigpCiAgICAgICAgbG9nZ2VyLmluZm8oZiJPSyAge25hbWV9IikKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBGQUlMVVJFUy5hcHBlbmQobmFtZSkKICAgICAgICBsb2dnZXIuZXJyb3IoZiJGQUlMIHtuYW1lfToge2V9IikKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIGIyX2NmZyA9IGpzb24ubG9hZHMoKEIyX1JFU1VMVFMgLyAiYjJfcnVuX2NvbmZpZy5qc29uIikucmVhZF90ZXh0KCkpCiAgICBmZWF0dXJlX2NvbHMgPSBsaXN0KGIyX2NmZ1siZmVhdHVyZV9jb2xzIl0pCiAgICBsb2dnZXIuaW5mbyhmIkIyIGNvbmZpZyBsb2FkZWQ6IHtsZW4oZmVhdHVyZV9jb2xzKX0gZmVhdHVyZXMiKQoKICAgIGlmIG5vdCBGRUFUVVJFU19GVUxMLmV4aXN0cygpOgogICAgICAgIGxvZ2dlci5lcnJvcigiZmVhdHVyZXNfZnVsbC5wYXJxdWV0IG1pc3NpbmcgKHJ1biBjZWxsIDYpIikKICAgICAgICByZXR1cm4gMQogICAgZmVhdHVyZXMgPSBwZC5yZWFkX3BhcnF1ZXQoRkVBVFVSRVNfRlVMTCkKICAgIHNhbXBsZSA9IGZlYXR1cmVzW2ZlYXR1cmVzWyJzcGxpdCJdID09ICJ2YWwiXS5oZWFkKDMyKQoKICAgIGZvciBzZWVkIGluIFNFRURTOgogICAgICAgIGRlZiBfbG9hZF9wcmVkaWN0KHNlZWQ9c2VlZCk6CiAgICAgICAgICAgIG1vZGVsID0gam9ibGliLmxvYWQoQjJfTU9ERUxTX0RJUiAvIGYieGdib29zdF9zZWVkX3tzZWVkfS5qb2JsaWIiKQogICAgICAgICAgICBwID0gbW9kZWwucHJlZGljdF9wcm9iYShzYW1wbGVbZmVhdHVyZV9jb2xzXS52YWx1ZXMpWzosIDFdCiAgICAgICAgICAgIGFzc2VydCAocCA+PSAwLjApLmFsbCgpIGFuZCAocCA8PSAxLjApLmFsbCgpLCAicHJvYmFiaWxpdGllcyBvdXQgb2YgWzAsMV0iCiAgICAgICAgY2hlY2soZiJCMiB4Z2Jvb3N0X3NlZWRfe3NlZWR9IGxvYWRzIGFuZCBwcmVkaWN0cyIsIF9sb2FkX3ByZWRpY3QpCgogICAgZm9yIG5hbWUgaW4gKCJtb2RlbF94Z2Jvb3N0X3Jhdy5qb2JsaWIiLCAibW9kZWxfeGdib29zdF9jYWxpYnJhdGVkLmpvYmxpYiIpOgogICAgICAgIHBhdGggPSBNT0RFTFNfRElSIC8gbmFtZQogICAgICAgIGlmIHBhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIGRlZiBfbG9hZF92ZXJzaW9uX2EocGF0aD1wYXRoKToKICAgICAgICAgICAgICAgIGJ1bmRsZSA9IGpvYmxpYi5sb2FkKHBhdGgpCiAgICAgICAgICAgICAgICBtb2RlbCA9IGJ1bmRsZVsibW9kZWwiXSBpZiBpc2luc3RhbmNlKGJ1bmRsZSwgZGljdCkgYW5kICJtb2RlbCIgaW4gYnVuZGxlIGVsc2UgYnVuZGxlCiAgICAgICAgICAgICAgICBwID0gbW9kZWwucHJlZGljdF9wcm9iYShzYW1wbGVbZmVhdHVyZV9jb2xzXS52YWx1ZXMpWzosIDFdCiAgICAgICAgICAgICAgICBhc3NlcnQgKHAgPj0gMC4wKS5hbGwoKSBhbmQgKHAgPD0gMS4wKS5hbGwoKSwgIlZlcnNpb24gQSBwcm9iYWJpbGl0aWVzIG91dCBvZiBbMCwxXSIKICAgICAgICAgICAgY2hlY2soZiJWZXJzaW9uIEEge25hbWV9IGxvYWRzIGFuZCBwcmVkaWN0cyIsIF9sb2FkX3ZlcnNpb25fYSkKCiAgICBkZWYgX2NoZWNrX2IyX3BhcnF1ZXQoKToKICAgICAgICBwcmVkcyA9IHBkLnJlYWRfcGFycXVldChCMl9SRVNVTFRTIC8gImIyX3ByZWRpY3Rpb25zLnBhcnF1ZXQiKQogICAgICAgIGFzc2VydCB7InNhbXBsZV9pZCIsICJtb2RlbCIsICJzY29yZSIsICJwcmVkIiwgImxhYmVsIn0gPD0gc2V0KHByZWRzLmNvbHVtbnMpCiAgICAgICAgYXNzZXJ0IHByZWRzWyJtb2RlbCJdLm51bmlxdWUoKSA+PSAxCiAgICBjaGVjaygiQjIgYjJfcHJlZGljdGlvbnMucGFycXVldCBzY2hlbWEiLCBfY2hlY2tfYjJfcGFycXVldCkKCiAgICBkZWYgX2NoZWNrX2IzKCk6CiAgICAgICAgcHJlZHMgPSBwZC5yZWFkX3BhcnF1ZXQoQjNfUkVTVUxUUyAvICJiM19wcmVkaWN0aW9ucy5wYXJxdWV0IikKICAgICAgICBhc3NlcnQgeyJzYW1wbGVfaWQiLCAic291cmNlX2RhdGFzZXQiLCAic291cmNlX2dyb3VwX2lkIiwgInRhc2siLCAibGFiZWwiLCAibW9kZWwiLCAic2NvcmUiLCAicHJlZCJ9IDw9IHNldChwcmVkcy5jb2x1bW5zKQogICAgICAgIGFzc2VydCB7InhnYm9vc3Rfc2VlZF80MiIsICJ4Z2Jvb3N0X3NlZWRfMTIzIiwgInhnYm9vc3Rfc2VlZF80NTYifSA8PSBzZXQocHJlZHNbIm1vZGVsIl0pCiAgICAgICAganNvbi5sb2FkcygoQjNfUkVTVUxUUyAvICJiM19kYXRhc2V0X21ldHJpY3MuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgICAgIGpzb24ubG9hZHMoKEIzX1JFU1VMVFMgLyAiYjNfYm9vdHN0cmFwX2Npcy5qc29uIikucmVhZF90ZXh0KCkpCiAgICAgICAgZXJyb3JfcGF0aCA9IEIzX1JFU1VMVFMgLyAiYjNfZXJyb3JfY2FzZXMuanNvbiIKICAgICAgICBpZiBlcnJvcl9wYXRoLmV4aXN0cygpOgogICAgICAgICAgICBmb3IgY2FzZSBpbiBqc29uLmxvYWRzKGVycm9yX3BhdGgucmVhZF90ZXh0KCkpOgogICAgICAgICAgICAgICAgaWYgY2FzZS5nZXQoInNvdXJjZV9kYXRhc2V0IikgPT0gImZhaXRoYmVuY2giOgogICAgICAgICAgICAgICAgICAgIGFzc2VydCBub3QgYW55KGNhc2UuZ2V0KGssICIiKSBmb3IgayBpbiAoInF1ZXN0aW9uIiwgImNvbnRleHQiLCAiYW5zd2VyIiwgInNwYW5fYW5ub3RhdGlvbnMiKSksIFwKICAgICAgICAgICAgICAgICAgICAgICAgInVucmVkYWN0ZWQgRmFpdGhCZW5jaCB0ZXh0IGluIEIzIGVycm9yIGNhc2VzIgogICAgY2hlY2soIkIzIHByZWRpY3Rpb25zICsgcmVwb3J0cyIsIF9jaGVja19iMykKCiAgICBmb3IgbmFtZSBpbiAoImNhbGlicmF0b3JfcGxhdHRfc291cmNlX3NlZWRfNDIuam9ibGliIiwKICAgICAgICAgICAgICAgICAiY2FsaWJyYXRvcl9pc290b25pY19zb3VyY2Vfc2VlZF80Mi5qb2JsaWIiLAogICAgICAgICAgICAgICAgICJjYWxpYnJhdG9yX3BsYXR0X3RhcmdldF9yYWd0cnV0aF9xYV9zZWVkXzQyLmpvYmxpYiIsCiAgICAgICAgICAgICAgICAgImNhbGlicmF0b3JfaXNvdG9uaWNfdGFyZ2V0X3JhZ3RydXRoX3FhX3NlZWRfNDIuam9ibGliIik6CiAgICAgICAgZGVmIF9sb2FkX2NhbChuYW1lPW5hbWUpOgogICAgICAgICAgICBjYWwgPSBqb2JsaWIubG9hZChCNF9NT0RFTFMgLyBuYW1lKQogICAgICAgICAgICBwID0gY2FsLnByZWRpY3RfcHJvYmEobnAuYXJyYXkoW1swLjFdLCBbMC41XSwgWzAuOV1dKSlbOiwgMV0gaWYgInBsYXR0IiBpbiBuYW1lIFwKICAgICAgICAgICAgICAgIGVsc2UgY2FsLnByZWRpY3QobnAuYXJyYXkoWzAuMSwgMC41LCAwLjldKSkKICAgICAgICAgICAgYXNzZXJ0IChwID49IDAuMCkuYWxsKCkgYW5kIChwIDw9IDEuMCkuYWxsKCksICJjYWxpYnJhdGVkIHNjb3JlcyBvdXQgb2YgWzAsMV0iCiAgICAgICAgY2hlY2soZiJCNCB7bmFtZX0gbG9hZHMgYW5kIGFwcGxpZXMiLCBfbG9hZF9jYWwpCgogICAgZGVmIF9jaGVja19iNCgpOgogICAgICAgIHByZWRzID0gcGQucmVhZF9wYXJxdWV0KEI0X1JFU1VMVFMgLyAiYjRfcHJlZGljdGlvbnMucGFycXVldCIpCiAgICAgICAgYXNzZXJ0IHsic2FtcGxlX2lkIiwgIm1ldGhvZCIsICJzY29yZSIsICJwcmVkIiwgImxhYmVsIn0gPD0gc2V0KHByZWRzLmNvbHVtbnMpCiAgICAgICAgYXNzZXJ0IHsicmF3IiwgInBsYXR0IiwgImlzb3RvbmljIn0gPD0gc2V0KHByZWRzWyJtZXRob2QiXSkKICAgICAgICBqc29uLmxvYWRzKChCNF9SRVNVTFRTIC8gImI0X2NhbGlicmF0aW9uX21ldHJpY3MuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgICAgIGpzb24ubG9hZHMoKEI0X1JFU1VMVFMgLyAiYjRfdGFyZ2V0X2NhbGlicmF0aW9uLmpzb24iKS5yZWFkX3RleHQoKSkKICAgIGNoZWNrKCJCNCBwcmVkaWN0aW9ucyArIHJlcG9ydHMiLCBfY2hlY2tfYjQpCgogICAgaWYgRkFJTFVSRVM6CiAgICAgICAgbG9nZ2VyLmVycm9yKGYiVkVSSUZJQ0FUSU9OIEZBSUxFRCAoe2xlbihGQUlMVVJFUyl9KToge0ZBSUxVUkVTfSIpCiAgICAgICAgcmV0dXJuIDEKICAgIGxvZ2dlci5pbmZvKCJBTEwgQVJUSUZBQ1RTIFZFUklGSUVEOiBwb3J0YWJsZSBtb2RlbHMvY2FsaWJyYXRvcnMgbG9hZCBhbmQgcHJlZGljdCBvbiB0aGlzIG1hY2hpbmUuIikKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK",
 "src/retrieval/__init__.py": "IiIiVDM6IHJldHJpZXZhbCBwYWNrYWdlIOKAlCBldmlkZW5jZSBhY3F1aXNpdGlvbiBmb3IgY2xhaW0gdmVyaWZpY2F0aW9uLgoKTW9kdWxlczoKICBjaHVuay5weSAgICAgZG9jdW1lbnQgY2h1bmtpbmcgKHBsYWluIHRleHQ7IHB5cGRmL3B5dGhvbi1kb2N4IHBhcnNpbmcgaGVscGVycykKICBpbmRleC5weSAgICAgZGlzay1wZXJzaXN0ZWQgaHlicmlkIGluZGV4IChCTTI1ICsgRkFJU1MgZGVuc2UgKyBSUkYgZnVzaW9uKQogIHdlYl9zZWFyY2gucHkgQnJhdmUtYmFja2VkIHdlYiBzZWFyY2ggKExMTSBDb250ZXh0ICsgV2ViIHNuaXBwZXRzLCBUYXZpbHkgZmFsbGJhY2spCiAgcmVyYW5rLnB5ICAgIGxhenkgY3Jvc3MtZW5jb2RlciByZXJhbmtlciAobXMtbWFyY28tTWluaUxNLUwtNi12MikKIiIiCgpmcm9tIC5icmF2ZV9hbnN3ZXJzIGltcG9ydCBCcmF2ZUFuc3dlcnMgICMgbm9xYTogRjQwMQpmcm9tIC5jaHVuayBpbXBvcnQgY2h1bmtfdGV4dCwgZXh0cmFjdF90ZXh0X2Zyb21fYnl0ZXMgICMgbm9xYTogRjQwMQpmcm9tIC5pbmRleCBpbXBvcnQgUmV0cmlldmFsSW5kZXggICMgbm9xYTogRjQwMQpmcm9tIC53ZWJfc2VhcmNoIGltcG9ydCBXZWJTZWFyY2ggICMgbm9xYTogRjQwMQo=",
 "src/retrieval/chunk.py": "IiIiVDM6IGRvY3VtZW50IGNodW5raW5nICsgZmlsZSBwYXJzaW5nLgoKQ2h1bmtzIGFyZSB+Q0hVTktfU0laRSBjaGFycyB3aXRoIE9WRVJMQVAgY2hhcnMgb2YgY29udGV4dDsgY2h1bmsgYm91bmRhcmllcwpwcmVmZXIgc2VudGVuY2UgYm91bmRhcmllcyB3aGVuIGF2YWlsYWJsZS4gUERGL0RPQ1gvVFhUIHBhcnNpbmcgaXMgbGF6eSBzbwp0ZXN0cyBuZXZlciBuZWVkIHRoZSBvcHRpb25hbCBwYWNrYWdlcy4KIiIiCgppbXBvcnQgcmUKCkNIVU5LX1NJWkUgPSAxNTAwCk9WRVJMQVAgPSAyMDAKTUFYX0ZJTEVfQllURVMgPSA1ICogMTAyNCAqIDEwMjQgICMgNSBNQiBwZXIgdXBsb2FkCgpfU0VOVEVOQ0VfQk9VTkRBUlkgPSByZS5jb21waWxlKHIiKD88PVsuIT9dKVxzKyIpCgoKZGVmIGV4dHJhY3RfdGV4dF9mcm9tX2J5dGVzKGZpbGVuYW1lOiBzdHIsIGRhdGE6IGJ5dGVzKSAtPiBzdHI6CiAgICAiIiJQYXJzZSBhIFBERi9ET0NYL1RYVC9NRCB1cGxvYWQgaW50byBwbGFpbiB0ZXh0LiIiIgogICAgbG93ZXIgPSBmaWxlbmFtZS5sb3dlcigpCiAgICBpZiBsb3dlci5lbmRzd2l0aCgiLnBkZiIpOgogICAgICAgIGltcG9ydCBpbwoKICAgICAgICBmcm9tIHB5cGRmIGltcG9ydCBQZGZSZWFkZXIKCiAgICAgICAgcmVhZGVyID0gUGRmUmVhZGVyKGlvLkJ5dGVzSU8oZGF0YSkpCiAgICAgICAgcmV0dXJuICJcbiIuam9pbihwYWdlLmV4dHJhY3RfdGV4dCgpIG9yICIiIGZvciBwYWdlIGluIHJlYWRlci5wYWdlcykKICAgIGlmIGxvd2VyLmVuZHN3aXRoKCIuZG9jeCIpOgogICAgICAgIGltcG9ydCBpbwoKICAgICAgICBmcm9tIGRvY3ggaW1wb3J0IERvY3VtZW50CgogICAgICAgIGRvYyA9IERvY3VtZW50KGlvLkJ5dGVzSU8oZGF0YSkpCiAgICAgICAgcmV0dXJuICJcbiIuam9pbihwLnRleHQgZm9yIHAgaW4gZG9jLnBhcmFncmFwaHMgaWYgcC50ZXh0KQogICAgIyB0eHQgLyBtZCAvIGFueXRoaW5nIGVsc2U6IGRlY29kZSB0ZXh0CiAgICByZXR1cm4gZGF0YS5kZWNvZGUoInV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIikKCgpkZWYgY2h1bmtfdGV4dCh0ZXh0OiBzdHIsIGNodW5rX3NpemU6IGludCA9IENIVU5LX1NJWkUsIG92ZXJsYXA6IGludCA9IE9WRVJMQVApIC0+IGxpc3Q6CiAgICAiIiJGaXhlZC1zaXplIGNodW5rcyBwcmVmZXJyaW5nIHNlbnRlbmNlIGJvdW5kYXJpZXM7IG92ZXJsYXAgZm9yIGNvbnRpbnVpdHkuIiIiCiAgICB0ZXh0ID0gdGV4dC5zdHJpcCgpCiAgICBpZiBub3QgdGV4dDoKICAgICAgICByZXR1cm4gW10KICAgIGlmIGxlbih0ZXh0KSA8PSBjaHVua19zaXplOgogICAgICAgIHJldHVybiBbdGV4dF0KCiAgICAjIHByZS1zcGxpdCBpbnRvIHNlbnRlbmNlIHRva2VucyBzbyBib3VuZGFyaWVzIGFyZSBwcmVmZXJyZWQKICAgIHNlbnRlbmNlcyA9IFtzIGZvciBzIGluIF9TRU5URU5DRV9CT1VOREFSWS5zcGxpdCh0ZXh0KSBpZiBzLnN0cmlwKCldCiAgICBjaHVua3M6IGxpc3QgPSBbXQogICAgY3VycmVudCA9ICIiCiAgICBmb3Igc2VudGVuY2UgaW4gc2VudGVuY2VzOgogICAgICAgIGlmIGxlbihjdXJyZW50KSArIGxlbihzZW50ZW5jZSkgPD0gY2h1bmtfc2l6ZToKICAgICAgICAgICAgY3VycmVudCA9IGYie2N1cnJlbnR9IHtzZW50ZW5jZX0iLnN0cmlwKCkKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBjdXJyZW50OgogICAgICAgICAgICBjaHVua3MuYXBwZW5kKGN1cnJlbnQpCiAgICAgICAgICAgIHRhaWwgPSBjdXJyZW50Wy1vdmVybGFwOl0gaWYgb3ZlcmxhcCBlbHNlICIiCiAgICAgICAgICAgIGN1cnJlbnQgPSBmInt0YWlsfSB7c2VudGVuY2V9Ii5zdHJpcCgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgIyBzaW5nbGUgc2VudGVuY2UgbG9uZ2VyIHRoYW4gY2h1bmtfc2l6ZTogaGFyZCBjdXQKICAgICAgICAgICAgY2h1bmtzLmFwcGVuZChzZW50ZW5jZVs6Y2h1bmtfc2l6ZV0pCiAgICAgICAgICAgIGN1cnJlbnQgPSBzZW50ZW5jZVtjaHVua19zaXplIC0gb3ZlcmxhcCA6XSBpZiBvdmVybGFwIGVsc2UgIiIKICAgIGlmIGN1cnJlbnQ6CiAgICAgICAgY2h1bmtzLmFwcGVuZChjdXJyZW50KQogICAgcmV0dXJuIFtjIGZvciBjIGluIGNodW5rcyBpZiBjLnN0cmlwKCldCg==",
 "src/retrieval/index.py": "IiIiVDM6IGRpc2stcGVyc2lzdGVkIGh5YnJpZCByZXRyaWV2YWwgaW5kZXggKEJNMjUgKyBGQUlTUyBkZW5zZSArIFJSRiBmdXNpb24pLgoKLSBEb2N1bWVudHMgYXJlIGNodW5rZWQsIGVtYmVkZGVkIHdpdGggdGhlIHNoYXJlZCBTQkVSVCBlbWJlZGRlciAodGhlIHNhbWUKICBzZW50ZW5jZS10cmFuc2Zvcm1lcnMgbW9kZWwgdGhlIEFQSSBhbHJlYWR5IGxvYWRzKSwgYW5kIHN0b3JlZCBib3RoIGluIGEKICByYW5rX2JtMjUgY29ycHVzIGFuZCBhIEZBSVNTIGluZGV4LgotIFJldHJpZXZhbCBmdXNlcyBCTTI1IGFuZCBkZW5zZSByYW5raW5ncyB3aXRoIHJlY2lwcm9jYWwgcmFuayBmdXNpb24gKFJSRik7CiAgYW4gb3B0aW9uYWwgY3Jvc3MtZW5jb2RlciByZXJhbmtlciByZW9yZGVycyB0aGUgZnVzZWQgdG9wLWsuCi0gVGhlIGluZGV4IHBlcnNpc3RzIHVuZGVyIGRhdGEvcHJvY2Vzc2VkL3JldHJpZXZhbF9pbmRleC8gKGdpdGlnbm9yZWQpIGFuZAogIHJlbG9hZHMgb24gQVBJIHN0YXJ0dXAuCiIiIgoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHRocmVhZGluZwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKSU5ERVhfRElSX0RFRkFVTFQgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXSAvICJkYXRhIiAvICJwcm9jZXNzZWQiIC8gInJldHJpZXZhbF9pbmRleCIKClJSRl9LID0gNjAKREVGQVVMVF9UT1BfSyA9IDUKUkVMRVZBTkNFX0ZMT09SID0gMWUtNiAgIyBiZWxvdyB0aGlzIGZ1c2VkIHNjb3JlIC0+IGFic3RhaW4gKG5vIGV2aWRlbmNlKQoKCmNsYXNzIFJldHJpZXZhbEluZGV4OgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkX2ZuLCBpbmRleF9kaXI6IFBhdGggPSBJTkRFWF9ESVJfREVGQVVMVCwgdG9wX2s6IGludCA9IERFRkFVTFRfVE9QX0spOgogICAgICAgICIiImVtYmVkX2ZuKHRleHRzOiBsaXN0W3N0cl0pIC0+IG5wLm5kYXJyYXkgKG4sIGRpbSkuIiIiCiAgICAgICAgc2VsZi5fZW1iZWQgPSBlbWJlZF9mbgogICAgICAgIHNlbGYuX2RpciA9IGluZGV4X2RpcgogICAgICAgIHNlbGYuX3RvcF9rID0gdG9wX2sKICAgICAgICBzZWxmLl9sb2NrID0gdGhyZWFkaW5nLkxvY2soKQogICAgICAgIHNlbGYuX3Bhc3NhZ2VzOiBsaXN0W2RpY3RdID0gW10gICAgICAgICAgIyB7aWQsIHNvdXJjZSwgdXJsLCB0ZXh0LCBzaGEyNTZ9CiAgICAgICAgc2VsZi5fbWV0YTogZGljdCA9IHsiZG9jdW1lbnRzIjogW10sICJkaW0iOiBOb25lfQogICAgICAgIHNlbGYuX2ZhaXNzID0gTm9uZQogICAgICAgIHNlbGYuX2JtMjUgPSBOb25lCiAgICAgICAgc2VsZi5fY29ycHVzX3Rva2VuczogbGlzdFtsaXN0XSA9IFtdCiAgICAgICAgc2VsZi5sb2FkKCkKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBwYXRocwogICAgQHByb3BlcnR5CiAgICBkZWYgcGFzc2FnZXNfcGF0aChzZWxmKSAtPiBQYXRoOgogICAgICAgIHJldHVybiBzZWxmLl9kaXIgLyAicGFzc2FnZXMuanNvbiIKCiAgICBAcHJvcGVydHkKICAgIGRlZiBtZXRhX3BhdGgoc2VsZikgLT4gUGF0aDoKICAgICAgICByZXR1cm4gc2VsZi5fZGlyIC8gIm1ldGEuanNvbiIKCiAgICBAcHJvcGVydHkKICAgIGRlZiBpbmRleF9wYXRoKHNlbGYpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYuX2RpciAvICJ2ZWN0b3JzLmZhaXNzIgoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHN0YXRlCiAgICBkZWYgX3Rva2VuaXplKHNlbGYsIHRleHQ6IHN0cikgLT4gbGlzdDoKICAgICAgICByZXR1cm4gdGV4dC5sb3dlcigpLnNwbGl0KCkKCiAgICBkZWYgbG9hZChzZWxmKSAtPiBOb25lOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgaWYgbm90IHNlbGYucGFzc2FnZXNfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBzZWxmLl9wYXNzYWdlcyA9IGpzb24ubG9hZHMoc2VsZi5wYXNzYWdlc19wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICAgICAgICAgIHNlbGYuX21ldGEgPSBqc29uLmxvYWRzKHNlbGYubWV0YV9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICAgICAgICAgIGlmIHNlbGYuX3Bhc3NhZ2VzOgogICAgICAgICAgICAgICAgICAgIHNlbGYuX2NvcnB1c190b2tlbnMgPSBbc2VsZi5fdG9rZW5pemUocFsidGV4dCJdKSBmb3IgcCBpbiBzZWxmLl9wYXNzYWdlc10KICAgICAgICAgICAgICAgICAgICBzZWxmLl9ibTI1ID0gc2VsZi5fbWFrZV9ibTI1KHNlbGYuX2NvcnB1c190b2tlbnMpCiAgICAgICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBzZWxmLl9wYXNzYWdlcyA9IFtdCiAgICAgICAgICAgICAgICBzZWxmLl9tZXRhID0geyJkb2N1bWVudHMiOiBbXSwgImRpbSI6IE5vbmV9CgogICAgZGVmIF9tYWtlX2JtMjUoc2VsZiwgY29ycHVzOiBsaXN0KToKICAgICAgICBmcm9tIHJhbmtfYm0yNSBpbXBvcnQgQk0yNU9rYXBpCgogICAgICAgIHJldHVybiBCTTI1T2thcGkoY29ycHVzKQoKICAgIGRlZiBfYnVpbGRfZmFpc3Moc2VsZiwgZGltOiBpbnQpIC0+IE5vbmU6CiAgICAgICAgaW1wb3J0IGZhaXNzCgogICAgICAgIHNlbGYuX2ZhaXNzID0gZmFpc3MuSW5kZXhGbGF0SVAoZGltKQoKICAgIGRlZiBhZGRfZG9jdW1lbnRzKHNlbGYsIGRvY3VtZW50czogbGlzdFtkaWN0XSwgZW1iZWRfZm49Tm9uZSkgLT4gZGljdDoKICAgICAgICAiIiJkb2N1bWVudHM6IFt7c291cmNlLCB1cmw/LCB0ZXh0fV0uIFJldHVybnMgcGVyLWRvYyBjaHVuayBjb3VudHMuIiIiCiAgICAgICAgZW1iZWQgPSBlbWJlZF9mbiBvciBzZWxmLl9lbWJlZAogICAgICAgIG5ld19wYXNzYWdlczogbGlzdFtkaWN0XSA9IFtdCiAgICAgICAgbmV3X3ZlY3M6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgICAgIHJlc3VsdCA9IFtdCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBleGlzdGluZyA9IHtwWyJzaGEyNTYiXSBmb3IgcCBpbiBzZWxmLl9wYXNzYWdlc30KICAgICAgICAgICAgZm9yIGRvYyBpbiBkb2N1bWVudHM6CiAgICAgICAgICAgICAgICBzb3VyY2UgPSBkb2NbInNvdXJjZSJdCiAgICAgICAgICAgICAgICB1cmwgPSBkb2MuZ2V0KCJ1cmwiKSBvciAiIgogICAgICAgICAgICAgICAgY2h1bmtzID0gW2MgZm9yIGMgaW4gc2VsZi5fY2h1bmtzX29mKGRvY1sidGV4dCJdKSBpZiBjLnN0cmlwKCldCiAgICAgICAgICAgICAgICBjaHVua192ZWNzID0gZW1iZWQoY2h1bmtzKQogICAgICAgICAgICAgICAgbl9hZGRlZCA9IDAKICAgICAgICAgICAgICAgIGZvciBpLCAoY2h1bmssIHZlYykgaW4gZW51bWVyYXRlKHppcChjaHVua3MsIGNodW5rX3ZlY3MpKToKICAgICAgICAgICAgICAgICAgICBzaGEgPSBoYXNobGliLnNoYTI1NihjaHVuay5lbmNvZGUoInV0Zi04IikpLmhleGRpZ2VzdCgpICAjIGNvbnRlbnQtYmFzZWQgZGVkdXAKICAgICAgICAgICAgICAgICAgICBpZiBzaGEgaW4gZXhpc3Rpbmc6CiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgZXhpc3RpbmcuYWRkKHNoYSkKICAgICAgICAgICAgICAgICAgICBuZXdfcGFzc2FnZXMuYXBwZW5kKHsiaWQiOiBzaGEsICJzb3VyY2UiOiBzb3VyY2UsICJ1cmwiOiB1cmwsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRleHQiOiBjaHVuaywgInNoYTI1NiI6IHNoYX0pCiAgICAgICAgICAgICAgICAgICAgbmV3X3ZlY3MuYXBwZW5kKG5wLmFzYXJyYXkodmVjLCBkdHlwZT0iZmxvYXQzMiIpKQogICAgICAgICAgICAgICAgICAgIG5fYWRkZWQgKz0gMQogICAgICAgICAgICAgICAgcmVzdWx0LmFwcGVuZCh7InNvdXJjZSI6IHNvdXJjZSwgImNodW5rcyI6IGxlbihjaHVua3MpLCAiYWRkZWQiOiBuX2FkZGVkfSkKICAgICAgICAgICAgaWYgbmV3X3Bhc3NhZ2VzOgogICAgICAgICAgICAgICAgc2VsZi5fcGFzc2FnZXMuZXh0ZW5kKG5ld19wYXNzYWdlcykKICAgICAgICAgICAgICAgIHZlY3MgPSBucC5zdGFjayhuZXdfdmVjcykuYXN0eXBlKCJmbG9hdDMyIikKICAgICAgICAgICAgICAgIGRpbSA9IHZlY3Muc2hhcGVbMV0KICAgICAgICAgICAgICAgIHNlbGYuX21ldGFbImRpbSJdID0gZGltCiAgICAgICAgICAgICAgICBpZiBzZWxmLl9mYWlzcyBpcyBOb25lIG9yIHNlbGYuX2ZhaXNzLmQgIT0gZGltOgogICAgICAgICAgICAgICAgICAgIHNlbGYuX2J1aWxkX2ZhaXNzKGRpbSkKICAgICAgICAgICAgICAgIHNlbGYuX2ZhaXNzLmFkZCh2ZWNzKQogICAgICAgICAgICAgICAgc2VsZi5fY29ycHVzX3Rva2Vucy5leHRlbmQoc2VsZi5fdG9rZW5pemUocFsidGV4dCJdKSBmb3IgcCBpbiBuZXdfcGFzc2FnZXMpCiAgICAgICAgICAgICAgICBzZWxmLl9ibTI1ID0gc2VsZi5fbWFrZV9ibTI1KHNlbGYuX2NvcnB1c190b2tlbnMpCiAgICAgICAgICAgICAgICBzZWxmLl9zYXZlKCkKICAgICAgICAgICAgcmV0dXJuIHJlc3VsdAoKICAgIGRlZiBfY2h1bmtzX29mKHNlbGYsIHRleHQ6IHN0cikgLT4gbGlzdDoKICAgICAgICBmcm9tIC5jaHVuayBpbXBvcnQgY2h1bmtfdGV4dAoKICAgICAgICByZXR1cm4gY2h1bmtfdGV4dCh0ZXh0KQoKICAgIGRlZiBfc2F2ZShzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgc2VsZi5wYXNzYWdlc19wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhzZWxmLl9wYXNzYWdlcywgZW5zdXJlX2FzY2lpPUZhbHNlKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICBzZWxmLm1ldGFfcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMoc2VsZi5fbWV0YSwgZW5zdXJlX2FzY2lpPUZhbHNlKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICBpZiBzZWxmLl9mYWlzcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgaW1wb3J0IGZhaXNzCgogICAgICAgICAgICBmYWlzcy53cml0ZV9pbmRleChzZWxmLl9mYWlzcywgc3RyKHNlbGYuaW5kZXhfcGF0aCkpCgogICAgZGVmIGNsZWFyKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBzZWxmLl9wYXNzYWdlcyA9IFtdCiAgICAgICAgICAgIHNlbGYuX21ldGEgPSB7ImRvY3VtZW50cyI6IFtdLCAiZGltIjogTm9uZX0KICAgICAgICAgICAgc2VsZi5fZmFpc3MgPSBOb25lCiAgICAgICAgICAgIHNlbGYuX2JtMjUgPSBOb25lCiAgICAgICAgICAgIHNlbGYuX2NvcnB1c190b2tlbnMgPSBbXQogICAgICAgICAgICBzZWxmLl9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICBmb3IgcCBpbiAoc2VsZi5wYXNzYWdlc19wYXRoLCBzZWxmLm1ldGFfcGF0aCwgc2VsZi5pbmRleF9wYXRoKToKICAgICAgICAgICAgICAgIGlmIHAuZXhpc3RzKCk6CiAgICAgICAgICAgICAgICAgICAgcC51bmxpbmsoKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHN0YXRzCiAgICBkZWYgc3RhdHVzKHNlbGYpIC0+IGRpY3Q6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICByZXR1cm4gewogICAgICAgICAgICAgICAgIm5fcGFzc2FnZXMiOiBsZW4oc2VsZi5fcGFzc2FnZXMpLAogICAgICAgICAgICAgICAgIm5fZG9jdW1lbnRzIjogbGVuKHNlbGYuX21ldGEuZ2V0KCJkb2N1bWVudHMiLCBbXSkpLAogICAgICAgICAgICAgICAgImRpbSI6IHNlbGYuX21ldGEuZ2V0KCJkaW0iKSwKICAgICAgICAgICAgICAgICJpbmRleF9kaXIiOiBzdHIoc2VsZi5fZGlyKSwKICAgICAgICAgICAgfQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHNlYXJjaAogICAgZGVmIHNlYXJjaChzZWxmLCBxdWVyeTogc3RyLCB0b3BfazogaW50IHwgTm9uZSA9IE5vbmUsIHJlcmFua19mbj1Ob25lKSAtPiBsaXN0OgogICAgICAgICIiIkh5YnJpZCBCTTI1ICsgZGVuc2UgcmV0cmlldmFsIHdpdGggUlJGIGZ1c2lvbiwgb3B0aW9uYWwgcmVyYW5rLiIiIgogICAgICAgIGsgPSB0b3BfayBvciBzZWxmLl90b3BfawogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAgICAgICAgICAgaWYgbm90IHNlbGYuX3Bhc3NhZ2VzOgogICAgICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgICAgIHFfdG9rZW5zID0gc2VsZi5fdG9rZW5pemUocXVlcnkpCgogICAgICAgICAgICBkZW5zZV9zY29yZXM6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZQogICAgICAgICAgICBpZiBzZWxmLl9mYWlzcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF2ID0gbnAuYXNhcnJheShzZWxmLl9lbWJlZChbcXVlcnldKVswXSwgZHR5cGU9ImZsb2F0MzIiKS5yZXNoYXBlKDEsIC0xKQogICAgICAgICAgICAgICAgc2NvcmVzLCBpZHhzID0gc2VsZi5fZmFpc3Muc2VhcmNoKHF2LCBtaW4oayAqIDMsIGxlbihzZWxmLl9wYXNzYWdlcykpKQogICAgICAgICAgICAgICAgZGVuc2Vfc2NvcmVzID0gbnAuemVyb3MobGVuKHNlbGYuX3Bhc3NhZ2VzKSkKICAgICAgICAgICAgICAgIGZvciBzLCBpIGluIHppcChzY29yZXNbMF0sIGlkeHNbMF0pOgogICAgICAgICAgICAgICAgICAgIGlmIGkgPj0gMDoKICAgICAgICAgICAgICAgICAgICAgICAgZGVuc2Vfc2NvcmVzW2ldID0gcwoKICAgICAgICAgICAgYm0yNV9zY29yZXMgPSBucC5hc2FycmF5KHNlbGYuX2JtMjUuZ2V0X3Njb3JlcyhxX3Rva2VucykpIGlmIHNlbGYuX2JtMjUgZWxzZSBOb25lCgogICAgICAgICAgICBkZWYgcnJmKHNjb3JlcywgYXNjZW5kaW5nPUZhbHNlKToKICAgICAgICAgICAgICAgIG9yZGVyID0gbnAuYXJnc29ydChzY29yZXMpIGlmIGFzY2VuZGluZyBlbHNlIG5wLmFyZ3NvcnQoc2NvcmVzKVs6Oi0xXQogICAgICAgICAgICAgICAgb3V0ID0gbnAuemVyb3MobGVuKHNjb3JlcykpCiAgICAgICAgICAgICAgICBmb3IgcmFuaywgaSBpbiBlbnVtZXJhdGUob3JkZXIpOgogICAgICAgICAgICAgICAgICAgIG91dFtpXSA9IDEuMCAvIChSUkZfSyArIHJhbmsgKyAxKQogICAgICAgICAgICAgICAgcmV0dXJuIG91dAoKICAgICAgICAgICAgZnVzZWQgPSBucC56ZXJvcyhsZW4oc2VsZi5fcGFzc2FnZXMpKQogICAgICAgICAgICBpZiBkZW5zZV9zY29yZXMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBmdXNlZCArPSBycmYoZGVuc2Vfc2NvcmVzKQogICAgICAgICAgICBpZiBibTI1X3Njb3JlcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGZ1c2VkICs9IHJyZihibTI1X3Njb3JlcykKICAgICAgICAgICAgaWYgZGVuc2Vfc2NvcmVzIGlzIE5vbmUgYW5kIGJtMjVfc2NvcmVzIGlzIE5vbmU6CiAgICAgICAgICAgICAgICByZXR1cm4gW10KCiAgICAgICAgICAgIG9yZGVyID0gbnAuYXJnc29ydChmdXNlZClbOjotMV0KICAgICAgICAgICAgY2FuZGlkYXRlcyA9IFsKICAgICAgICAgICAgICAgIHsqKnNlbGYuX3Bhc3NhZ2VzW2ldLCAic2NvcmUiOiBmbG9hdChmdXNlZFtpXSl9CiAgICAgICAgICAgICAgICBmb3IgaSBpbiBvcmRlcls6IGsgKiAyXQogICAgICAgICAgICAgICAgaWYgZnVzZWRbaV0gPiBSRUxFVkFOQ0VfRkxPT1IKICAgICAgICAgICAgXQoKICAgICAgICBpZiBjYW5kaWRhdGVzIGFuZCByZXJhbmtfZm4gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGNhbmRpZGF0ZXMgPSByZXJhbmtfZm4ocXVlcnksIGNhbmRpZGF0ZXMsIGspCgogICAgICAgIHJldHVybiBjYW5kaWRhdGVzWzprXQo=",
 "src/retrieval/rerank.py": "IiIiVDM6IGxhenkgY3Jvc3MtZW5jb2RlciByZXJhbmtlci4KCkxvYWRlZCBvbmNlIChIRiBjYWNoZSksIH45MCBNQi4gRmFsbHMgYmFjayB0byB0aGUgZnVzZWQgb3JkZXIgd2hlbiB0aGUgbW9kZWwKaXMgdW5hdmFpbGFibGUgKGUuZy4sIG9mZmxpbmUpLiBNb2RlbCBjb25maWd1cmFibGUgdmlhIEhBTFVfUkVSQU5LRVJfTU9ERUwuCiIiIgoKaW1wb3J0IG9zCmltcG9ydCB0aHJlYWRpbmcKClJFUkFOS0VSX01PREVMID0gb3MuZW52aXJvbi5nZXQoIkhBTFVfUkVSQU5LRVJfTU9ERUwiLCAiY3Jvc3MtZW5jb2Rlci9tcy1tYXJjby1NaW5pTE0tTC02LXYyIikKCl9sb2NrID0gdGhyZWFkaW5nLkxvY2soKQpfbW9kZWwgPSBOb25lCgoKZGVmIF9sb2FkKCk6CiAgICBnbG9iYWwgX21vZGVsCiAgICBpZiBfbW9kZWwgaXMgbm90IE5vbmU6CiAgICAgICAgcmV0dXJuIF9tb2RlbAogICAgd2l0aCBfbG9jazoKICAgICAgICBpZiBfbW9kZWwgaXMgTm9uZToKICAgICAgICAgICAgZnJvbSBzZW50ZW5jZV90cmFuc2Zvcm1lcnMgaW1wb3J0IENyb3NzRW5jb2RlcgoKICAgICAgICAgICAgX21vZGVsID0gQ3Jvc3NFbmNvZGVyKFJFUkFOS0VSX01PREVMKQogICAgcmV0dXJuIF9tb2RlbAoKCmRlZiByZXJhbmsocXVlcnk6IHN0ciwgY2FuZGlkYXRlczogbGlzdCwgdG9wX2s6IGludCkgLT4gbGlzdDoKICAgICIiIlJlb3JkZXIgY2FuZGlkYXRlcyBieSBjcm9zcy1lbmNvZGVyIHJlbGV2YW5jZTsgcmV0dXJucyB0b3Bfay4iIiIKICAgIHRyeToKICAgICAgICBtb2RlbCA9IF9sb2FkKCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIGNhbmRpZGF0ZXNbOnRvcF9rXQogICAgdHJ5OgogICAgICAgIHBhaXJzID0gW1txdWVyeSwgY1sidGV4dCJdXSBmb3IgYyBpbiBjYW5kaWRhdGVzXQogICAgICAgIHNjb3JlcyA9IG1vZGVsLnByZWRpY3QocGFpcnMpCiAgICAgICAgcmFua2VkID0gc29ydGVkKHppcChjYW5kaWRhdGVzLCBzY29yZXMpLCBrZXk9bGFtYmRhIHQ6IGZsb2F0KHRbMV0pLCByZXZlcnNlPVRydWUpCiAgICAgICAgcmV0dXJuIFt7KipjLCAic2NvcmUiOiBmbG9hdChzKX0gZm9yIGMsIHMgaW4gcmFua2VkWzp0b3Bfa11dCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBjYW5kaWRhdGVzWzp0b3Bfa10K",
 "src/retrieval/web_search.py": "IiIiVDMgd2ViIHNlYXJjaDogQnJhdmUgKGRlZmF1bHQpIG9yIFRhdmlseSwgcGVyLWNsYWltIHF1ZXJpZXMsIExSVS1jYWNoZWQuCgpQcm92aWRlciBzZWxlY3Rpb24gKEhBTFVfV0VCX1NFQVJDSF9QUk9WSURFUj1hdXRvfGJyYXZlfHRhdmlseSk6CgogIGF1dG8gICAgQnJhdmUgd2hlbiBCUkFWRV9TRUFSQ0hfQVBJX0tFWSBpcyBzZXQsIGVsc2UgVGF2aWx5LgogIGJyYXZlICAgQnJhdmUgb25seSAoZmFsbHMgYmFjayB0byBUYXZpbHkgb25seSB3aGVuIGl0cyBrZXkgaXMgYWxzbyBwcmVzZW50KS4KICB0YXZpbHkgIFRhdmlseSBvbmx5IChsZWdhY3kpLgoKQnJhdmUgZXZpZGVuY2UgcGF0aHMsIHRyaWVkIGluIG9yZGVyOgoKICAxLiBMTE0gQ29udGV4dCAoYGAvcmVzL3YxL2xsbS9jb250ZXh0YGApIHJldHVybnMgZXh0cmFjdGVkIHBhZ2UgY2h1bmtzIHRoYXQKICAgICBhcmUgcmFua2VkIGFuZCB0b2tlbi1idWRnZXRlZCBmb3IgbWFjaGluZSBjb25zdW1wdGlvbiwgc28gZWFjaCBwYXNzYWdlCiAgICAgYWxyZWFkeSByZWFkcyBsaWtlIHZlcmlmaWVyIGV2aWRlbmNlLgogIDIuIFdlYiBTZWFyY2ggKGBgL3Jlcy92MS93ZWIvc2VhcmNoYGAgd2l0aCBgYGV4dHJhX3NuaXBwZXRzPXRydWVgYCkgaXMgdGhlCiAgICAgZmFsbGJhY2sgd2hlbiB0aGUgY29udGV4dCBlbmRwb2ludCByZXR1cm5zIG5vdGhpbmcuCgpUaGUgQnJhdmUgZnJlZSB0aWVyIGFsbG93cyAyIHJlcXVlc3RzL3NlY29uZCAoa2V5ZWQgcmVxdWVzdHM6IDEvc2Vjb25kKSwgc28KZXZlcnkgQnJhdmUgY2FsbCBwYXNzZXMgdGhyb3VnaCBhIG1pbi1pbnRlcnZhbCB0aHJvdHRsZQooSEFMVV9CUkFWRV9NSU5fSU5URVJWQUwsIGRlZmF1bHQgMC41NSBzKS4gUGFzc2FnZXMgYXJlIHNoYXBlZCBsaWtlIHRoZQpkb2N1bWVudCBpbmRleCBlbnRyaWVzIHNvIHRoZSB2ZXJpZmllciB0cmVhdHMgd2ViIGFuZCBkb2N1bWVudCBldmlkZW5jZQppZGVudGljYWxseS4gVGhlIGNsYXNzIGRpc2FibGVzIGl0c2VsZiBncmFjZWZ1bGx5IHdoZW4gdGhlIGFjdGl2ZSBwcm92aWRlcgpoYXMgbm8ga2V5IG9yIHdoZW4gZXZlcnkgY2FsbCBmYWlscy4KIiIiCgppbXBvcnQgb3MKaW1wb3J0IHRocmVhZGluZwppbXBvcnQgdGltZQoKU0VBUkNIX0xSVV9NQVggPSAxMjgKTUFYX1JFU1VMVFMgPSA1CkJSQVZFX0JBU0UgPSAiaHR0cHM6Ly9hcGkuc2VhcmNoLmJyYXZlLmNvbS9yZXMvdjEiCkRFRkFVTFRfTUlOX0lOVEVSVkFMID0gMC41NQoKCmNsYXNzIFdlYlNlYXJjaDoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGFwaV9rZXk6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgICAgIG1heF9yZXN1bHRzOiBpbnQgPSBNQVhfUkVTVUxUUywKICAgICAgICBwcm92aWRlcjogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgYnJhdmVfYXBpX2tleTogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICApOgogICAgICAgICMgYXBpX2tleSBrZWVwcyB0aGUgbGVnYWN5IG1lYW5pbmcgKFRhdmlseSBrZXkgb3ZlcnJpZGUpLgogICAgICAgIHNlbGYuX3RhdmlseV9rZXkgPSBhcGlfa2V5IGlmIGFwaV9rZXkgaXMgbm90IE5vbmUgZWxzZSBvcy5lbnZpcm9uLmdldCgiVEFWSUxZX0FQSV9LRVkiLCAiIikKICAgICAgICBzZWxmLl9icmF2ZV9rZXkgPSBicmF2ZV9hcGlfa2V5IGlmIGJyYXZlX2FwaV9rZXkgaXMgbm90IE5vbmUgZWxzZSBvcy5lbnZpcm9uLmdldCgiQlJBVkVfU0VBUkNIX0FQSV9LRVkiLCAiIikKICAgICAgICBzZWxmLl9tYXhfcmVzdWx0cyA9IG1heF9yZXN1bHRzCiAgICAgICAgc2VsZi5fcHJvdmlkZXIgPSBzZWxmLl9yZXNvbHZlX3Byb3ZpZGVyKHByb3ZpZGVyKQogICAgICAgIHNlbGYuX2xvY2sgPSB0aHJlYWRpbmcuTG9jaygpCiAgICAgICAgc2VsZi5fcmF0ZV9sb2NrID0gdGhyZWFkaW5nLkxvY2soKQogICAgICAgIHNlbGYuX2xhc3RfY2FsbCA9IDAuMAogICAgICAgIHNlbGYuX21pbl9pbnRlcnZhbCA9IGZsb2F0KG9zLmVudmlyb24uZ2V0KCJIQUxVX0JSQVZFX01JTl9JTlRFUlZBTCIsIERFRkFVTFRfTUlOX0lOVEVSVkFMKSkKICAgICAgICBzZWxmLl90aW1lb3V0ID0gZmxvYXQob3MuZW52aXJvbi5nZXQoIkhBTFVfQlJBVkVfVElNRU9VVCIsICIxMCIpKQogICAgICAgIHNlbGYuX2NhY2hlOiBkaWN0W3N0ciwgbGlzdF0gPSB7fQoKICAgIGRlZiBfcmVzb2x2ZV9wcm92aWRlcihzZWxmLCBwcm92aWRlcjogc3RyIHwgTm9uZSkgLT4gc3RyOgogICAgICAgIG1vZGUgPSAocHJvdmlkZXIgb3Igb3MuZW52aXJvbi5nZXQoIkhBTFVfV0VCX1NFQVJDSF9QUk9WSURFUiIsICJhdXRvIikpLmxvd2VyKCkKICAgICAgICBpZiBtb2RlIGluICgiYnJhdmUiLCAidGF2aWx5Iik6CiAgICAgICAgICAgIHJldHVybiBtb2RlCiAgICAgICAgcmV0dXJuICJicmF2ZSIgaWYgc2VsZi5fYnJhdmVfa2V5IGVsc2UgInRhdmlseSIKCiAgICBAcHJvcGVydHkKICAgIGRlZiBwcm92aWRlcihzZWxmKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIHNlbGYuX3Byb3ZpZGVyCgogICAgQHByb3BlcnR5CiAgICBkZWYgZW5hYmxlZChzZWxmKSAtPiBib29sOgogICAgICAgIGlmIHNlbGYuX3Byb3ZpZGVyID09ICJicmF2ZSI6CiAgICAgICAgICAgIHJldHVybiBib29sKHNlbGYuX2JyYXZlX2tleSkKICAgICAgICByZXR1cm4gYm9vbChzZWxmLl90YXZpbHlfa2V5KQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBicmF2ZQogICAgZGVmIF9icmF2ZV9oZWFkZXJzKHNlbGYpIC0+IGRpY3Q6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIkFjY2VwdCI6ICJhcHBsaWNhdGlvbi9qc29uIiwKICAgICAgICAgICAgIkFjY2VwdC1FbmNvZGluZyI6ICJnemlwIiwKICAgICAgICAgICAgIlgtU3Vic2NyaXB0aW9uLVRva2VuIjogc2VsZi5fYnJhdmVfa2V5LAogICAgICAgIH0KCiAgICBkZWYgX2JyYXZlX2NhbGwoc2VsZiwgcGF0aDogc3RyLCBwYXJhbXM6IGRpY3QpIC0+IGRpY3Q6CiAgICAgICAgIiIiVGhyb3R0bGVkIEJyYXZlIEdFVCAoZnJlZSB0aWVyOiAyIHJlcXVlc3RzL3NlY29uZCkuIiIiCiAgICAgICAgaW1wb3J0IGh0dHB4CgogICAgICAgIHdpdGggc2VsZi5fcmF0ZV9sb2NrOgogICAgICAgICAgICB3YWl0ID0gc2VsZi5fbWluX2ludGVydmFsIC0gKHRpbWUubW9ub3RvbmljKCkgLSBzZWxmLl9sYXN0X2NhbGwpCiAgICAgICAgICAgIGlmIHdhaXQgPiAwOgogICAgICAgICAgICAgICAgdGltZS5zbGVlcCh3YWl0KQogICAgICAgICAgICBzZWxmLl9sYXN0X2NhbGwgPSB0aW1lLm1vbm90b25pYygpCiAgICAgICAgICAgIHJlc3AgPSBodHRweC5nZXQoCiAgICAgICAgICAgICAgICBmIntCUkFWRV9CQVNFfS97cGF0aH0iLAogICAgICAgICAgICAgICAgcGFyYW1zPXBhcmFtcywKICAgICAgICAgICAgICAgIGhlYWRlcnM9c2VsZi5fYnJhdmVfaGVhZGVycygpLAogICAgICAgICAgICAgICAgdGltZW91dD1zZWxmLl90aW1lb3V0LAogICAgICAgICAgICApCiAgICAgICAgICAgIHJlc3AucmFpc2VfZm9yX3N0YXR1cygpCiAgICAgICAgICAgIHJldHVybiByZXNwLmpzb24oKQoKICAgIGRlZiBfYnJhdmVfY29udGV4dChzZWxmLCBxdWVyeTogc3RyKSAtPiBsaXN0OgogICAgICAgIGRhdGEgPSBzZWxmLl9icmF2ZV9jYWxsKAogICAgICAgICAgICAibGxtL2NvbnRleHQiLAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAicSI6IHF1ZXJ5LAogICAgICAgICAgICAgICAgImNvdW50Ijogc2VsZi5fbWF4X3Jlc3VsdHMsCiAgICAgICAgICAgICAgICAibWF4aW11bV9udW1iZXJfb2ZfdXJscyI6IHNlbGYuX21heF9yZXN1bHRzLAogICAgICAgICAgICAgICAgIm1heGltdW1fbnVtYmVyX29mX3Rva2VucyI6IGludChvcy5lbnZpcm9uLmdldCgiSEFMVV9CUkFWRV9DT05URVhUX1RPS0VOUyIsICI0MDk2IikpLAogICAgICAgICAgICAgICAgImNvbnRleHRfdGhyZXNob2xkX21vZGUiOiBvcy5lbnZpcm9uLmdldCgiSEFMVV9CUkFWRV9DT05URVhUX1RIUkVTSE9MRCIsICJiYWxhbmNlZCIpLAogICAgICAgICAgICB9LAogICAgICAgICkKICAgICAgICBnZW5lcmljID0gKGRhdGEuZ2V0KCJncm91bmRpbmciKSBvciB7fSkuZ2V0KCJnZW5lcmljIikgb3IgW10KICAgICAgICBwYXNzYWdlcyA9IFtdCiAgICAgICAgZm9yIHJhbmssIGl0ZW0gaW4gZW51bWVyYXRlKGdlbmVyaWMpOgogICAgICAgICAgICB1cmwgPSBpdGVtLmdldCgidXJsIikgb3IgIiIKICAgICAgICAgICAgdGV4dCA9ICJcbiIuam9pbihzIGZvciBzIGluIChpdGVtLmdldCgic25pcHBldHMiKSBvciBbXSkgaWYgcykuc3RyaXAoKQogICAgICAgICAgICBpZiBub3QgdXJsIG9yIG5vdCB0ZXh0OgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcGFzc2FnZXMuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJzb3VyY2UiOiBmIndlYjp7dXJsfSIsCiAgICAgICAgICAgICAgICAgICAgInVybCI6IHVybCwKICAgICAgICAgICAgICAgICAgICAidGV4dCI6IHRleHRbOjQwMDBdLAogICAgICAgICAgICAgICAgICAgICJzY29yZSI6IG1heCgwLjAsIDEuMCAtIDAuMDUgKiByYW5rKSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQogICAgICAgIHJldHVybiBwYXNzYWdlcwoKICAgIGRlZiBfYnJhdmVfd2ViKHNlbGYsIHF1ZXJ5OiBzdHIpIC0+IGxpc3Q6CiAgICAgICAgZGF0YSA9IHNlbGYuX2JyYXZlX2NhbGwoCiAgICAgICAgICAgICJ3ZWIvc2VhcmNoIiwKICAgICAgICAgICAgeyJxIjogcXVlcnksICJjb3VudCI6IHNlbGYuX21heF9yZXN1bHRzLCAiZXh0cmFfc25pcHBldHMiOiAidHJ1ZSJ9LAogICAgICAgICkKICAgICAgICByZXN1bHRzID0gKGRhdGEuZ2V0KCJ3ZWIiKSBvciB7fSkuZ2V0KCJyZXN1bHRzIikgb3IgW10KICAgICAgICBwYXNzYWdlcyA9IFtdCiAgICAgICAgZm9yIHJhbmssIHIgaW4gZW51bWVyYXRlKHJlc3VsdHMpOgogICAgICAgICAgICB1cmwgPSByLmdldCgidXJsIikgb3IgIiIKICAgICAgICAgICAgcGFydHMgPSBbci5nZXQoImRlc2NyaXB0aW9uIikgb3IgIiIsICooci5nZXQoImV4dHJhX3NuaXBwZXRzIikgb3IgW10pXQogICAgICAgICAgICB0ZXh0ID0gIlxuIi5qb2luKHAgZm9yIHAgaW4gcGFydHMgaWYgcCkuc3RyaXAoKQogICAgICAgICAgICBpZiBub3QgdXJsIG9yIG5vdCB0ZXh0OgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcGFzc2FnZXMuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJzb3VyY2UiOiBmIndlYjp7dXJsfSIsCiAgICAgICAgICAgICAgICAgICAgInVybCI6IHVybCwKICAgICAgICAgICAgICAgICAgICAidGV4dCI6IHRleHRbOjIwMDBdLAogICAgICAgICAgICAgICAgICAgICJzY29yZSI6IG1heCgwLjAsIDEuMCAtIDAuMDUgKiByYW5rKSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQogICAgICAgIHJldHVybiBwYXNzYWdlcwoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSB0YXZpbHkKICAgIGRlZiBfdGF2aWx5KHNlbGYpOgogICAgICAgIGZyb20gdGF2aWx5IGltcG9ydCBUYXZpbHlDbGllbnQKCiAgICAgICAgcmV0dXJuIFRhdmlseUNsaWVudChhcGlfa2V5PXNlbGYuX3RhdmlseV9rZXkpCgogICAgZGVmIF90YXZpbHlfc2VhcmNoKHNlbGYsIHF1ZXJ5OiBzdHIpIC0+IGxpc3Q6CiAgICAgICAgcmVzcCA9IHNlbGYuX3RhdmlseSgpLnNlYXJjaChxdWVyeT1xdWVyeSwgbWF4X3Jlc3VsdHM9c2VsZi5fbWF4X3Jlc3VsdHMsIHNlYXJjaF9kZXB0aD0iYmFzaWMiKQogICAgICAgIHJlc3VsdHMgPSByZXNwLmdldCgicmVzdWx0cyIsIFtdKSBpZiBpc2luc3RhbmNlKHJlc3AsIGRpY3QpIGVsc2UgW10KICAgICAgICByZXR1cm4gWwogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAic291cmNlIjogZiJ3ZWI6e3IuZ2V0KCd1cmwnLCAnJyl9IiwKICAgICAgICAgICAgICAgICJ1cmwiOiByLmdldCgidXJsIiwgIiIpLAogICAgICAgICAgICAgICAgInRleHQiOiAoci5nZXQoImNvbnRlbnQiKSBvciAiIilbOjIwMDBdLAogICAgICAgICAgICAgICAgInNjb3JlIjogZmxvYXQoci5nZXQoInNjb3JlIikgb3IgMC4wKSwKICAgICAgICAgICAgfQogICAgICAgICAgICBmb3IgciBpbiByZXN1bHRzCiAgICAgICAgICAgIGlmIHIuZ2V0KCJjb250ZW50IikKICAgICAgICBdCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHB1YmxpYwogICAgZGVmIF9zZWFyY2hfcHJvdmlkZXIoc2VsZiwgcXVlcnk6IHN0cikgLT4gbGlzdDoKICAgICAgICBpZiBzZWxmLl9wcm92aWRlciA9PSAiYnJhdmUiOgogICAgICAgICAgICBwYXNzYWdlczogbGlzdCA9IFtdCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHBhc3NhZ2VzID0gc2VsZi5fYnJhdmVfY29udGV4dChxdWVyeSkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHBhc3NhZ2VzID0gW10KICAgICAgICAgICAgaWYgbm90IHBhc3NhZ2VzOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHBhc3NhZ2VzID0gc2VsZi5fYnJhdmVfd2ViKHF1ZXJ5KQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICBwYXNzYWdlcyA9IFtdCiAgICAgICAgICAgIGlmIG5vdCBwYXNzYWdlcyBhbmQgc2VsZi5fdGF2aWx5X2tleToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwYXNzYWdlcyA9IHNlbGYuX3RhdmlseV9zZWFyY2gocXVlcnkpICAjIGdyYWNlZnVsIGNyb3NzLXByb3ZpZGVyIGZhbGxiYWNrCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgIHBhc3NhZ2VzID0gW10KICAgICAgICAgICAgcmV0dXJuIHBhc3NhZ2VzCiAgICAgICAgcmV0dXJuIHNlbGYuX3RhdmlseV9zZWFyY2gocXVlcnkpCgogICAgZGVmIHNlYXJjaChzZWxmLCBxdWVyeTogc3RyKSAtPiBsaXN0OgogICAgICAgICIiIlJldHVybnMgW3tzb3VyY2U6ICd3ZWI6PHVybD4nLCB1cmwsIHRleHQsIHNjb3JlfV0gb3IgW10gb24gYW55IGZhaWx1cmUuIiIiCiAgICAgICAgaWYgbm90IHNlbGYuZW5hYmxlZDoKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBjYWNoZWQgPSBzZWxmLl9jYWNoZS5nZXQocXVlcnkpCiAgICAgICAgICAgIGlmIGNhY2hlZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHJldHVybiBjYWNoZWQKICAgICAgICB0cnk6CiAgICAgICAgICAgIHBhc3NhZ2VzID0gc2VsZi5fc2VhcmNoX3Byb3ZpZGVyKHF1ZXJ5KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3NhZ2VzID0gW10KICAgICAgICBpZiBwYXNzYWdlczoKICAgICAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICAgICAgc2VsZi5fY2FjaGVbcXVlcnldID0gcGFzc2FnZXMKICAgICAgICAgICAgICAgIGlmIGxlbihzZWxmLl9jYWNoZSkgPiBTRUFSQ0hfTFJVX01BWDoKICAgICAgICAgICAgICAgICAgICBzZWxmLl9jYWNoZS5wb3AobmV4dChpdGVyKHNlbGYuX2NhY2hlKSkpCiAgICAgICAgcmV0dXJuIHBhc3NhZ2VzCg==",
 "colab/drive_cache.py": "IiIiCkRyaXZlLWNhY2hlIGhlbHBlcnMgZm9yIHRoZSBDb2xhYiBub3RlYm9vayAoY2VsbHMgNWIgLyA2YiAvIDdkLjUgLyA3ZSkuCgpEZXRlcm1pbmlzdGljIGhlYXZ5IGFydGlmYWN0cyAoSGFsdUV2YWwgZmVhdHVyZXMsIEIzIGV4dGVybmFsIGZlYXR1cmVzKSBhcmUKY2FjaGVkIG9uIEdvb2dsZSBEcml2ZSBiZXR3ZWVuIHNlc3Npb25zIHNvIGV4dHJhY3Rpb24gZG9lcyBub3QgcmVwZWF0IG9uIGV2ZXJ5CnJ1bi4gRXZlcnkgcmVzdG9yZSBpcyBWRVJJRklFRCBhZ2FpbnN0IGZyZXNobHkgcHJlcGFyZWQgZGF0YTsgYSBtaXNtYXRjaCBmYWxscwpiYWNrIHRvIHJlLWV4dHJhY3Rpb24gYXV0b21hdGljYWxseS4KCk5vIGdvb2dsZS5jb2xhYiBpbXBvcnRzIGhlcmUsIHNvIHRoZXNlIGZ1bmN0aW9ucyBhcmUgdW5pdC10ZXN0YWJsZSBsb2NhbGx5LgoiIiIKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKRVhQRUNURURfUk9XUyA9IDIwMDAwCkVYUEVDVEVEX0xBQkVMUyA9IHswOiAxMDAwMCwgMTogMTAwMDB9CkVYUEVDVEVEX1NQTElUUyA9IHsidHJhaW4iOiAxNDAwMCwgInZhbCI6IDMwMDAsICJ0ZXN0IjogMzAwMH0KIyBTYW5pdHkgZmVhdHVyZSBuYW1lcyB0aGF0IG11c3QgYmUgcHJlc2VudCAoZnVsbCAyNiBhcmUgY2hlY2tlZCBieSB0aGUgcnVubmVycykKU0FOSVRZX0ZFQVRVUkVTID0gWyJuX2NoYXJzIiwgIm5fd29yZHMiLCAib3ZlcmxhcF9hbnN3ZXJfY29udGV4dCIsCiAgICAgICAgICAgICAgICAgICAibmxpX2N0eF9lbnRhaWxzX2FucyIsICJjb3NpbmVfY3R4X2FucyJdCgoKZGVmIHNoYTI1Nl9maWxlKHBhdGgpIC0+IHN0cjoKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwgInJiIikgYXMgZjoKICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IGYucmVhZCgxIDw8IDIwKSwgYiIiKToKICAgICAgICAgICAgaC51cGRhdGUoY2h1bmspCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiB2ZXJpZnlfaGFsdWV2YWxfZmVhdHVyZXMoZmVhdHVyZXNfcGF0aCwgcWFfcGF0aCkgLT4gZGljdDoKICAgICIiIlZlcmlmeSBhIGNhY2hlZCBmZWF0dXJlc19mdWxsLnBhcnF1ZXQgYWdhaW5zdCB0aGUgZnJlc2hseSBidWlsdCBxYV9jbGVhbi5wYXJxdWV0LgoKICAgIFJldHVybnMgeyJvayI6IGJvb2wsICJjaGVja3MiOiBbcmVhc29uc119LgogICAgIiIiCiAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCgogICAgdHJ5OgogICAgICAgIGZlYXRzID0gcGQucmVhZF9wYXJxdWV0KGZlYXR1cmVzX3BhdGgpCiAgICAgICAgcWEgPSBwZC5yZWFkX3BhcnF1ZXQocWFfcGF0aCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICMgY29ycnVwdCBvciB1bnJlYWRhYmxlIGNhY2hlCiAgICAgICAgcmV0dXJuIHsib2siOiBGYWxzZSwgImNoZWNrcyI6IFtmInJlYWQgZmFpbGVkOiB7ZX0iXX0KCiAgICBjaGVja3MgPSBbXQogICAgaWYgbGVuKGZlYXRzKSAhPSBFWFBFQ1RFRF9ST1dTOgogICAgICAgIGNoZWNrcy5hcHBlbmQoZiJyb3dzIHtsZW4oZmVhdHMpfSAhPSB7RVhQRUNURURfUk9XU30iKQogICAgaWYgc2V0KGZlYXRzWyJzYW1wbGVfaWQiXSkgIT0gc2V0KHFhWyJzYW1wbGVfaWQiXSk6CiAgICAgICAgY2hlY2tzLmFwcGVuZCgic2FtcGxlX2lkIHNldHMgZGlmZmVyIGZyb20gcWFfY2xlYW4iKQogICAgaWYgZmVhdHNbImxhYmVsIl0udmFsdWVfY291bnRzKCkudG9fZGljdCgpICE9IEVYUEVDVEVEX0xBQkVMUzoKICAgICAgICBjaGVja3MuYXBwZW5kKGYibGFiZWwgYmFsYW5jZSB7ZmVhdHNbJ2xhYmVsJ10udmFsdWVfY291bnRzKCkudG9fZGljdCgpfSAhPSB7RVhQRUNURURfTEFCRUxTfSIpCiAgICBjb3VudHMgPSBmZWF0cy5ncm91cGJ5KCJzcGxpdCIpLnNpemUoKS50b19kaWN0KCkKICAgIGlmIGNvdW50cyAhPSBFWFBFQ1RFRF9TUExJVFM6CiAgICAgICAgY2hlY2tzLmFwcGVuZChmInNwbGl0IGNvdW50cyB7Y291bnRzfSAhPSB7RVhQRUNURURfU1BMSVRTfSIpCiAgICBpZiAiaXRlbV9pZHgiIGluIGZlYXRzLmNvbHVtbnMgYW5kIGZlYXRzLmdyb3VwYnkoIml0ZW1faWR4IilbInNwbGl0Il0ubnVuaXF1ZSgpLm1heCgpICE9IDE6CiAgICAgICAgY2hlY2tzLmFwcGVuZCgiaXRlbV9pZHggZ3JvdXBzIHNwYW4gbXVsdGlwbGUgc3BsaXRzIChsZWFrYWdlKSIpCiAgICBtaXNzaW5nX2ZlYXRzID0gW2MgZm9yIGMgaW4gU0FOSVRZX0ZFQVRVUkVTIGlmIGMgbm90IGluIGZlYXRzLmNvbHVtbnNdCiAgICBpZiBtaXNzaW5nX2ZlYXRzOgogICAgICAgIGNoZWNrcy5hcHBlbmQoZiJtaXNzaW5nIGZlYXR1cmUgY29sdW1ucyB7bWlzc2luZ19mZWF0c30iKQogICAgcmV0dXJuIHsib2siOiBub3QgY2hlY2tzLCAiY2hlY2tzIjogY2hlY2tzfQoKCmRlZiByZWFkX2NhY2hlX21ldGEobWV0YV9wYXRoKSAtPiBkaWN0OgogICAgdHJ5OgogICAgICAgIHJldHVybiBqc29uLmxvYWRzKFBhdGgobWV0YV9wYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgIHJldHVybiB7fQoKCmRlZiBjYWNoZV9rZXlfbWF0Y2hlcyhtZXRhOiBkaWN0LCB1bmlmaWVkX3BhcnF1ZXRfcGF0aCkgLT4gYm9vbDoKICAgICIiIkIzIGV4dGVybmFsLWZlYXR1cmUgY2FjaGUgaXMgcmV1c2FibGUgb25seSB3aGVuIHRoZSB1bmlmaWVkIHBhcnF1ZXQgaGFzaCBtYXRjaGVzLiIiIgogICAgaWYgbm90IG1ldGEuZ2V0KCJpbnB1dF9zaGEyNTYiKToKICAgICAgICByZXR1cm4gRmFsc2UKICAgIHRyeToKICAgICAgICByZXR1cm4gbWV0YVsiaW5wdXRfc2hhMjU2Il0gPT0gc2hhMjU2X2ZpbGUodW5pZmllZF9wYXJxdWV0X3BhdGgpCiAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgcmVzdG9yZV9jb25maWdfaGFzaChjb25maWdfanNvbl9wYXRoLCBrZXk6IHN0ciwgZmlsZV9wYXRoKSAtPiBib29sOgogICAgIiIiR2VuZXJpYzogY2FjaGVkIHJ1bi1jb25maWcgcmVjb3JkcyBpbnB1dCBoYXNoZXM7IHJlc3RvcmUgb25seSB3aGVuIHRoZXkgbWF0Y2guCgogICAga2V5IGlzIHRoZSBkb3R0ZWQgcGF0aCBpbnRvIHRoZSBjb25maWcsIGUuZy4gImlucHV0cy5mZWF0dXJlc19mdWxsLnBhcnF1ZXQiCiAgICBvciAidW5pZmllZF9wYXJxdWV0X3NoYTI1NiIuIENvbmZpZyBrZXlzIHRoZW1zZWx2ZXMgbWF5IGNvbnRhaW4gZG90cwogICAgKCJmZWF0dXJlc19mdWxsLnBhcnF1ZXQiKSwgc28gYXQgZXZlcnkgbGV2ZWwgdGhlIGxvbmdlc3QgbGl0ZXJhbCByZW1haW5kZXIKICAgIGlzIHRyaWVkIGZpcnN0LiBSZXR1cm5zIEZhbHNlIG9uIGFueSBtaXNtYXRjaC9taXNzaW5nIGZpbGUuCiAgICAiIiIKICAgIGltcG9ydCBqc29uCgogICAgdHJ5OgogICAgICAgIGNmZyA9IGpzb24ubG9hZHMoUGF0aChjb25maWdfanNvbl9wYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgcGFydHMgPSBrZXkuc3BsaXQoIi4iKQogICAgICAgIG5vZGUgPSBjZmcKICAgICAgICBmb3IgaSwgcGFydCBpbiBlbnVtZXJhdGUocGFydHMpOgogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShub2RlLCBkaWN0KToKICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgICAgICByZXN0ID0gIi4iLmpvaW4ocGFydHNbaTpdKQogICAgICAgICAgICBpZiByZXN0IGluIG5vZGU6ICAjIGxpdGVyYWwga2V5IGNvbnRhaW5zIGRvdHM6IHRha2UgdGhlIHdob2xlIHJlbWFpbmRlcgogICAgICAgICAgICAgICAgbm9kZSA9IG5vZGVbcmVzdF0KICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGlmIHBhcnQgbm90IGluIG5vZGU6CiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICAgICAgbm9kZSA9IG5vZGVbcGFydF0KICAgICAgICByZXR1cm4gbm9kZSA9PSBzaGEyNTZfZmlsZShmaWxlX3BhdGgpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IsIFR5cGVFcnJvcik6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIGIyX3Jlc3RvcmVfdmFsaWQoYjJfY29uZmlnX3BhdGgsIGZlYXR1cmVzX3BhdGgpIC0+IGJvb2w6CiAgICAiIiJCMiBhcnRpZmFjdHMgYXJlIHJldXNhYmxlIHdoZW4gdGhlIGNhY2hlZCBydW4gY29uc3VtZWQgVEhJUyBmZWF0dXJlc19mdWxsLnBhcnF1ZXQuIiIiCiAgICByZXR1cm4gcmVzdG9yZV9jb25maWdfaGFzaChiMl9jb25maWdfcGF0aCwgImlucHV0cy5mZWF0dXJlc19mdWxsLnBhcnF1ZXQiLCBmZWF0dXJlc19wYXRoKQoKCmRlZiBiM19yZXN0b3JlX3ZhbGlkKGIzX2NvbmZpZ19wYXRoLCB1bmlmaWVkX3BhdGgsIGIyX21vZGVsc19kaXIpIC0+IGJvb2w6CiAgICAiIiJCMyByZXN1bHRzIGFyZSByZXVzYWJsZSB3aGVuIHVuaWZpZWQgcGFycXVldCBBTkQgQjIgbW9kZWxzIG1hdGNoIHRoZSBjYWNoZWQgcnVuLiIiIgogICAgaW1wb3J0IGpzb24KCiAgICB0cnk6CiAgICAgICAgY2ZnID0ganNvbi5sb2FkcyhQYXRoKGIzX2NvbmZpZ19wYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgIHJldHVybiBGYWxzZQogICAgdHJ5OgogICAgICAgIGlmIGNmZy5nZXQoInVuaWZpZWRfcGFycXVldF9zaGEyNTYiKSAhPSBzaGEyNTZfZmlsZSh1bmlmaWVkX3BhdGgpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBoYXNoZXMgPSBjZmcuZ2V0KCJiMl9tb2RlbF9oYXNoZXMiKSBvciB7fQogICAgICAgIHJldHVybiBhbGwoCiAgICAgICAgICAgIGhhc2hlcy5nZXQoZiJzZWVkX3tzfSIpID09IHNoYTI1Nl9maWxlKFBhdGgoYjJfbW9kZWxzX2RpcikgLyBmInhnYm9vc3Rfc2VlZF97c30uam9ibGliIikKICAgICAgICAgICAgZm9yIHMgaW4gKDQyLCAxMjMsIDQ1NikKICAgICAgICApCiAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgYjRfcmVzdG9yZV92YWxpZChiNF9jb25maWdfcGF0aCwgYjNfcHJlZGljdGlvbnNfcGF0aCkgLT4gYm9vbDoKICAgICIiIkI0IGFydGlmYWN0cyBhcmUgcmV1c2FibGUgd2hlbiB0aGV5IHdlcmUgcHJvZHVjZWQgZnJvbSBUSElTIGIzIHByZWRpY3Rpb25zIGZpbGUuIiIiCiAgICByZXR1cm4gcmVzdG9yZV9jb25maWdfaGFzaChiNF9jb25maWdfcGF0aCwgImlucHV0cy5iM19wcmVkaWN0aW9ucy5wYXJxdWV0IiwgYjNfcHJlZGljdGlvbnNfcGF0aCkKCgpkZWYgdmVyc2lvbl9hX3Jlc3RvcmVfdmFsaWQobWFya2VyX3BhdGgsIGZlYXR1cmVzX3BhdGgsIHFhX3BhdGgsIHNwbGl0X3JlcG9ydF9wYXRoLCBjYWNoZV9kaXIpIC0+IGJvb2w6CiAgICAiIiJWYWxpZGF0ZSB0aGUgY2FjaGVkIHJvb3QgVmVyc2lvbiBBIGFydGlmYWN0cyBhZ2FpbnN0IGN1cnJlbnQgaW5wdXRzLiIiIgogICAgdHJ5OgogICAgICAgIG1hcmtlciA9IGpzb24ubG9hZHMoUGF0aChtYXJrZXJfcGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGV4cGVjdGVkID0gewogICAgICAgICAgICAiZmVhdHVyZXNfZnVsbC5wYXJxdWV0Ijogc2hhMjU2X2ZpbGUoZmVhdHVyZXNfcGF0aCksCiAgICAgICAgICAgICJxYV9jbGVhbi5wYXJxdWV0Ijogc2hhMjU2X2ZpbGUocWFfcGF0aCksCiAgICAgICAgICAgICJzcGxpdF9pbnRlZ3JpdHlfcmVwb3J0Lmpzb24iOiBzaGEyNTZfZmlsZShzcGxpdF9yZXBvcnRfcGF0aCksCiAgICAgICAgfQogICAgICAgIGlmIG1hcmtlci5nZXQoImlucHV0cyIpICE9IGV4cGVjdGVkOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICByZXR1cm4gYWxsKChQYXRoKGNhY2hlX2RpcikgLyByZWwpLmV4aXN0cygpIGZvciByZWwgaW4gbWFya2VyLmdldCgiYXJ0aWZhY3RzIiwgW10pKQogICAgZXhjZXB0IChPU0Vycm9yLCBWYWx1ZUVycm9yLCBUeXBlRXJyb3IpOgogICAgICAgIHJldHVybiBGYWxzZQoKCmRlZiBiM19mZWF0dXJlX2NhY2hlX3NhZmUoY2FjaGVfcGF0aCkgLT4gYm9vbDoKICAgICIiIlJlamVjdCBvbGQgQjMgY2FjaGVzIHRoYXQgYWNjaWRlbnRhbGx5IGNvbnRhaW4gcmF3IGV4dGVybmFsIHRleHQuIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHBhbmRhcyBhcyBwZAoKICAgICAgICBjb2x1bW5zID0gc2V0KHBkLnJlYWRfcGFycXVldChjYWNoZV9wYXRoLCBjb2x1bW5zPU5vbmUpLmNvbHVtbnMpCiAgICAgICAgZm9yYmlkZGVuID0geyJxdWVzdGlvbiIsICJjb250ZXh0IiwgImFuc3dlciIsICJzcGFuX2Fubm90YXRpb25zIn0KICAgICAgICByZXR1cm4gbm90IChjb2x1bW5zICYgZm9yYmlkZGVuKSBhbmQgInNhbXBsZV9pZCIgaW4gY29sdW1ucwogICAgZXhjZXB0IChPU0Vycm9yLCBWYWx1ZUVycm9yLCBJbXBvcnRFcnJvcik6CiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIGIzX3Jlc3VsdHNfc2FmZShyZXN1bHRzX2RpcikgLT4gYm9vbDoKICAgICIiIlJlamVjdCBjYWNoZWQgQjMgZXJyb3IgY2FzZXMgY29udGFpbmluZyB1bnJlZGFjdGVkIEZhaXRoQmVuY2ggdGV4dC4iIiIKICAgIGltcG9ydCBqc29uCgogICAgZXJyb3JfcGF0aCA9IFBhdGgocmVzdWx0c19kaXIpIC8gImIzX2Vycm9yX2Nhc2VzLmpzb24iCiAgICBpZiBub3QgZXJyb3JfcGF0aC5leGlzdHMoKToKICAgICAgICByZXR1cm4gVHJ1ZQogICAgdHJ5OgogICAgICAgIGNhc2VzID0ganNvbi5sb2FkcyhlcnJvcl9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBmb3IgY2FzZSBpbiBjYXNlczoKICAgICAgICAgICAgaWYgY2FzZS5nZXQoInNvdXJjZV9kYXRhc2V0IikgPT0gImZhaXRoYmVuY2giOgogICAgICAgICAgICAgICAgaWYgYW55KGNhc2UuZ2V0KGZpZWxkLCAiIikgZm9yIGZpZWxkIGluICgicXVlc3Rpb24iLCAiY29udGV4dCIsICJhbnN3ZXIiLCAic3Bhbl9hbm5vdGF0aW9ucyIpKToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICByZXR1cm4gVHJ1ZQogICAgZXhjZXB0IChPU0Vycm9yLCBWYWx1ZUVycm9yLCBUeXBlRXJyb3IpOgogICAgICAgIHJldHVybiBGYWxzZQoKCmRlZiBiM19wcmVkaWN0aW9uc19zYWZlKHByZWRpY3Rpb25zX3BhdGgpIC0+IGJvb2w6CiAgICAiIiJSZWplY3QgQjMgcHJlZGljdGlvbiBjYWNoZXMgcG9sbHV0ZWQgYnkgb3ZlcmxhcHBpbmcgc3Vic2V0IGFwcGVuZHMuIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHBhbmRhcyBhcyBwZAoKICAgICAgICBwcmVkcyA9IHBkLnJlYWRfcGFycXVldChwcmVkaWN0aW9uc19wYXRoLCBjb2x1bW5zPVsic2FtcGxlX2lkIiwgIm1vZGVsIl0pCiAgICAgICAgZXhwZWN0ZWQgPSB7InhnYm9vc3Rfc2VlZF80MiIsICJ4Z2Jvb3N0X3NlZWRfMTIzIiwgInhnYm9vc3Rfc2VlZF80NTYifQogICAgICAgIHJldHVybiAoCiAgICAgICAgICAgIGxlbihwcmVkcykgPiAwCiAgICAgICAgICAgIGFuZCBzZXQocHJlZHNbIm1vZGVsIl0uZHJvcG5hKCkudW5pcXVlKCkpID09IGV4cGVjdGVkCiAgICAgICAgICAgIGFuZCBub3QgcHJlZHMuZHVwbGljYXRlZChbInNhbXBsZV9pZCIsICJtb2RlbCJdKS5hbnkoKQogICAgICAgICkKICAgIGV4Y2VwdCAoT1NFcnJvciwgVmFsdWVFcnJvciwgSW1wb3J0RXJyb3IsIEtleUVycm9yKToKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgYjRfcHJlZGljdGlvbnNfc2FmZShwcmVkaWN0aW9uc19wYXRoKSAtPiBib29sOgogICAgIiIiUmVqZWN0IEI0IHByZWRpY3Rpb24gY2FjaGVzIHdpdGggZHVwbGljYXRlIHNhbXBsZS9tZXRob2Qgcm93cy4iIiIKICAgIHRyeToKICAgICAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCgogICAgICAgIHByZWRzID0gcGQucmVhZF9wYXJxdWV0KAogICAgICAgICAgICBwcmVkaWN0aW9uc19wYXRoLAogICAgICAgICAgICBjb2x1bW5zPVsic2FtcGxlX2lkIiwgInNvdXJjZV9kYXRhc2V0IiwgInN1YnNldCIsICJtZXRob2QiXSwKICAgICAgICApCiAgICAgICAga2V5ID0gWyJzYW1wbGVfaWQiLCAic291cmNlX2RhdGFzZXQiLCAic3Vic2V0IiwgIm1ldGhvZCJdCiAgICAgICAgcmV0dXJuIGxlbihwcmVkcykgPiAwIGFuZCBub3QgcHJlZHMuZHVwbGljYXRlZChrZXkpLmFueSgpCiAgICBleGNlcHQgKE9TRXJyb3IsIFZhbHVlRXJyb3IsIEltcG9ydEVycm9yLCBLZXlFcnJvcik6CiAgICAgICAgcmV0dXJuIEZhbHNlCg==",
 "colab/requirements-colab.txt": "IyBIYWx1UklTQyDigJQgQ29sYWIgZGVwZW5kZW5jaWVzIChubyB0b3JjaDogQ29sYWIgc2hpcHMgaXRzIG93biBHUFUgdG9yY2gpCiMgUGluIGV2ZXJ5dGhpbmcgZWxzZSBpZGVudGljYWxseSB0byByZXF1aXJlbWVudHMudHh0ICgyMDI2LTA4LTA0KQoKbnVtcHk9PTIuNC42CnBhbmRhcz09My4wLjUKcHlhcnJvdz09MjUuMC4wCmpvYmxpYj09MS41LjMKc2Npa2l0LWxlYXJuPT0xLjkuMAp4Z2Jvb3N0PT0zLjMuMApzY2lweT09MS4xNy4xCnN0YXRzbW9kZWxzPT0wLjE0LjYKc2hhcD09MC41Mi4wCnRyYW5zZm9ybWVycz09NS4xNC4xCnNlbnRlbmNlLXRyYW5zZm9ybWVycz09NS42LjEKc3BhY3k9PTMuOC4xNApkYXRhc2V0cz09NS4wLjEKaHVnZ2luZ2ZhY2UtaHViPT0xLjI2LjAKbWF0cGxvdGxpYj09My4xMS4xCm9wZW5haT09Mi41My4wCnB5dGhvbi1kb3RlbnY9PTEuMi4yCnB5eWFtbD09Ni4wLjMKdGF2aWx5LXB5dGhvbj09MC43LjI3CmZhaXNzLWNwdT09MS4xNS4wCnB5cGRmPT02LjE1LjAKcHl0aG9uLWRvY3g9PTEuMi4wCnJhbmtfYm0yNT09MC4yLjIKcHl0aG9uLW11bHRpcGFydD09MC4wLjMyCnNsb3dhcGk9PTAuMS4xMAo="
}
HASHES = {
 "src/__init__.py": "2e015d29f635fda3f7a3758a283191cc0203f3c913bdebe133d4b04f8b9604b9",
 "src/api/main.py": "536e9e50e4262eb6df3a4cb733317dcf3f5505d2e58e86a0c1ae604e83c06bd4",
 "src/claims/decompose.py": "7c525fc6656e03dd3ba0fc39736d1dfe88c1ab9f85c3632e8a62ec5989a42cce",
 "src/claims/judge.py": "24b1de9b914029a9cec9a9f0ddf5cc4f94070623a3d9b21d32cf874f999ff18c",
 "src/claims/verify.py": "2216abf9e1f4f07de87c5952d17481dface97564866de0c045abc8b5b8e05013",
 "src/data/download.py": "b4276c40e5981e144d60dcbaf6350d0736fec3a591a3a6f109674a7f06087f0d",
 "src/data/download_faithbench.py": "a71b43e84e0785639046bfb4224e429d448bfaeb713d3919a1945b4ecf0a4118",
 "src/data/download_ragtruth.py": "fde317a9a3d339162f772ad01c1fcc0da91f286d83c4afacf970331aa7177806",
 "src/data/mappings.py": "49b4813f288e74f792e853baf635bdf2b02b48f2018a6a722b178bee10a7d00c",
 "src/data/prepare.py": "69d6973af1d819346dc92787c0da2e841c8294ed34063a347cc667958d95300b",
 "src/data/prepare_unified.py": "a4b2478a20e5e05c31ccb7fcd745e07d1b4dd14312e3e14598649b2aaa6ce02a",
 "src/data/registry.py": "ead69514382cdc4d90d499689dab5f9b54250eb7a4f87271a4bda6a7ba44a25a",
 "src/data/schema.py": "931a2d0d8ed6108b3c5af1db0d7e81ce6fe9bfa5b39dbc842aa71d691c12aac3",
 "src/explain/fig_b5_importance.py": "17ea3b936b5932804805d39a1fb50f4a343e927e204ad48498a654c909fb211b",
 "src/explain/shap_analysis.py": "4199090cb0652614d847be0857fe54efc35f0adebaee2e8b89fe56de8607dbac",
 "src/features/entity_features.py": "07e982cc6983df5798bc80af2018a963a8429763f7604149b938ae80ca5c129f",
 "src/features/extract_features.py": "083b04da1c16a005cbc945df106a845a44f6647047c50e12e18b7dc97cedbfc3",
 "src/features/nli_features.py": "b6bc54fada99e6a4d770f2b8f924f26240c955007c3deac8d542dd9c0f475f45",
 "src/features/semantic_features.py": "f899c9f77894d568a492e996c9e79412d44a8200854017cb6853e3ae43ca4394",
 "src/models/config.py": "a94bc38d5004743cafa9c771f9418416d9afad7653b50ab8b75b051cf32d12f9",
 "src/models/error_analysis.py": "d59fec9f2199432b3d3a2be3e4ac72df179ad972f54fd6b1e69d7aac9a828511",
 "src/models/eval_claims.py": "c0f2a942c1e21380e28f88b7326a828fe54f02b79d9a611dff6b7cefe33f6c8b",
 "src/models/eval_efficiency.py": "5cce019c81e6264d0726ed6950732f33088d8351738b1c54147bb3e75665b7b0",
 "src/models/eval_llm_judge.py": "b00d34f9989a043176b118a1988bac51296e1b9a028430650502437f3e785366",
 "src/models/eval_ragtruth.py": "176cb33fadfa88a46da31390389b5d743382ebeaaec8aeb105d287f92ae8ec90",
 "src/models/fit_display_calibrator.py": "258bbca1221c0ec00b77f19c57400aa89487f1570311448fb8d88b0701aa8d9c",
 "src/models/make_manifest.py": "0a547bfd57cfcecd0a5d31b035533118fb8c2c905453c7b0de4a87c5d23cdd18",
 "src/models/review_tally.py": "2f03ad5d13805e7e22578cb7ba8a2183520bc7d60f1aa03ac8378dd96996be8d",
 "src/models/run_all_experiments.py": "1c95f8cca4043d8dc8d0901737a54918de0d4571fa8815db033b3f8d1f3e238c",
 "src/models/run_b2_baselines.py": "ba3c21ee649648c0811ea65464a9d78d7ff58d0a5037f7af8fb84bde3507e746",
 "src/models/run_b3_cross_domain.py": "5a65e4f4497f921cb3970b600293c13545e89b45b2a96eb18e4c03b08bf239d6",
 "src/models/run_b4_calibration_shift.py": "21b21bacdb09b0a03ce83f4edecc139fddd2d5f5ab0b6e4f44d9462bbdea5dfc",
 "src/models/run_b5_explanation_reliability.py": "b6964690ec8d93399c21bd7fea07f84dffeedb9e2df645e46d6afeeda86bff83",
 "src/models/train_baselines.py": "81099aee9f3d676e576572047a940ee6531ff696a4117e80081c3a57cae3ec89",
 "src/models/train_pipeline.py": "a7b9d3feea11ac08c9fdce27dc211263a25b6b3c33e0c1645e0d874b7190fa77",
 "src/models/tune_thresholds.py": "f5b537b4eb12e6b26a13ee791410b35f10475d85ca0e0fdf6ef114184feb15d5",
 "src/models/verify_artifacts.py": "dd552460e5b9dbd8d55cee4e12b223dd4a04836825e0d42b33a23d22a98cb231",
 "src/retrieval/__init__.py": "8ce98300a979a1bb20fba6c455673fef1b650a669d31b58abc1fe13ed0fe63d4",
 "src/retrieval/chunk.py": "958cd942348ac69d17c5863dcbd1d113246e8db4104d6de57c10ede313014349",
 "src/retrieval/index.py": "6f9c4d59a4f3f87d07e75d45d34b3fd86fc68fb54831ffa6a298fe401db83378",
 "src/retrieval/rerank.py": "52ee24de71b72cd61aebff86d4ce8ba5653d2dbcbb4465b861fd093ad22a3367",
 "src/retrieval/web_search.py": "a1908b46a9d8a3b20bd887a9ea2fc7d45db44bf7a626e94375a1d358e5e4cab9",
 "colab/drive_cache.py": "d0b743c0e437ac0f3633342422713d5bb2acb507b778dfba938c5a42a97578be",
 "colab/requirements-colab.txt": "7640cb603a9851751ab5506791e1de9ffee4254459b9fde110f8685d84b95a97"
}

for rel, b64 in EMBEDDED.items():
    dest = os.path.join(ROOT, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, 'wb') as f:
        f.write(base64.b64decode(b64))

bad = [rel for rel in EMBEDDED
       if hashlib.sha256(open(os.path.join(ROOT, rel), 'rb').read()).hexdigest() != HASHES[rel]]
assert not bad, f'embedded source mismatch: {bad}'
os.chdir(ROOT)
print(f'Self-contained source ready: {len(EMBEDDED)} files in {ROOT}')
print('src present:', os.path.exists(os.path.join(ROOT, 'src')))


In [ ]:
# 4) Install pinned dependencies (Colab keeps its own torch)
!pip install -q -r colab/requirements-colab.txt
# Colab preinstalls a newer numpy that removed numpy._core.umath._center,
# which the xgboost 3.3.0 / scipy 1.17.1 wheels need at import; force the
# pinned numpy if the preinstalled one lacks it (jax etc. are never used).
import numpy
try:
    from numpy._core.umath import _center
except ImportError:
    !pip install -q --force-reinstall --no-deps numpy==2.4.6
    from numpy._core.umath import _center  # needs a fresh kernel if this fails
!python -m spacy download en_core_web_sm -q
print('deps OK')


In [ ]:
# 5) HaluEval download + prepare with GROUP-AWARE split (item_idx) + integrity check
!python src/data/download.py
!python src/data/prepare.py
import json
rep = json.load(open('artifacts/split_integrity_report.json'))
print(json.dumps(rep, indent=2))
assert rep['leakage_free'] and rep['groups_spanning_multiple_splits'] == 0, 'Split leakage detected!'


In [ ]:
# 5b) Restore cached HaluEval features from Drive when valid (cell 6 shortcut)
# Saves 5-10 min on repeat runs. The cache is VERIFIED against the freshly built
# qa_clean.parquet (rows / sample_ids / labels / splits / leakage) - a mismatch
# falls back to full extraction in cell 6. Set CACHE_OK = True on success.
import os, sys, json, shutil
sys.path.insert(0, '.')
from colab.drive_cache import verify_halueval_features

DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
cache_feat = os.path.join(DRIVE_CACHE, 'features_full.parquet')
CACHE_OK = False
if os.path.exists(cache_feat):
    tmp = '/content/_cache_features.parquet'
    shutil.copy(cache_feat, tmp)
    v = verify_halueval_features(tmp, 'data/processed/qa_clean.parquet')
    if v['ok']:
        shutil.move(tmp, 'data/processed/features_full.parquet')
        cache_nli = os.path.join(DRIVE_CACHE, 'nli_model_used.json')
        if os.path.exists(cache_nli):
            shutil.copy(cache_nli, 'data/processed/nli_model_used.json')
        CACHE_OK = True
        print('FEATURE CACHE RESTORED from Drive and verified against qa_clean.')
    else:
        print('Cached features INVALID - cell 6 will re-extract:', v['checks'])
        os.remove(tmp)
else:
    print('No HaluEval feature cache on Drive yet - cell 6 will extract.')


In [ ]:
# 6) Full feature extraction (7 groups, ~40K NLI pairs on GPU; ~5-10 min)
# Skipped automatically when cell 5b restored a verified Drive cache.
# --device cuda -> models load in fp16 + CUDA; batch size from cell 2.
# Default NLI model: cross-encoder/nli-deberta-v3-base. Fallback: --nli-model cross-encoder/nli-MiniLM2-L6-H768
import json, os
if CACHE_OK:
    print('Using Drive-cached features (verified against qa_clean.parquet).')
    print('NLI provenance:', json.load(open('data/processed/nli_model_used.json')))
else:
    os.system(f'python src/features/extract_features.py --device cuda --batch-size {BATCH_SIZE}')
    print('NLI provenance:', json.load(open('data/processed/nli_model_used.json')))


In [ ]:
# 6b) Upload the HaluEval feature cache to Drive (next session skips extraction)
# Safe: features_full.parquet contains no FaithBench text (HaluEval is MIT).
import os, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
shutil.copy('data/processed/features_full.parquet', os.path.join(DRIVE_CACHE, 'features_full.parquet'))
shutil.copy('data/processed/nli_model_used.json', os.path.join(DRIVE_CACHE, 'nli_model_used.json'))
print('Uploaded HaluEval feature cache to Drive (halurisc_cache/).')


In [ ]:
# 7.0) Restore Version A cell-7 artifacts (LOCAL first, then Drive)
import os, json, shutil
from colab.drive_cache import version_a_restore_valid, sha256_file

DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
VA_DIR = os.path.join(DRIVE_CACHE, 'version_a')
VA_MARKER = os.path.join(VA_DIR, 'marker.json')
VA_FILES = [
    ('models/model_xgboost_raw.joblib', 'artifacts/models/model_xgboost_raw.joblib'),
    ('models/model_xgboost_calibrated.joblib', 'artifacts/models/model_xgboost_calibrated.joblib'),
    ('models/calibrator_platt.joblib', 'artifacts/models/calibrator_platt.joblib'),
    ('models/scaler.joblib', 'artifacts/models/scaler.joblib'),
    ('models/params.json', 'artifacts/models/params.json'),
    ('models/feature_names.json', 'artifacts/models/feature_names.json'),
    ('results/final_results.json', 'artifacts/results/final_results.json'),
    ('results/seed_metrics.json', 'artifacts/results/seed_metrics.json'),
    ('results/model_comparison.csv', 'artifacts/results/model_comparison.csv'),
    ('results/ablation_results.csv', 'artifacts/results/ablation_results.csv'),
]
VA_OK = False
if os.path.exists('artifacts/results/final_results.json'):
    VA_OK = True
    print('Version A cell-7 artifacts already present locally - skipping.')
elif os.path.exists(VA_MARKER) and version_a_restore_valid(
    VA_MARKER, 'data/processed/features_full.parquet', 'data/processed/qa_clean.parquet',
    'artifacts/split_integrity_report.json', VA_DIR
):
    for rel, local in VA_FILES:
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(os.path.join(VA_DIR, rel), local)
    VA_OK = True
    print('Version A cell-7 artifacts RESTORED from Drive (input hashes verified).')
else:
    print('No Version A cache found - cell 7 will run if selected.')


In [ ]:
# 7) Full Version A experiment (optional; CPU, ~10-20 min)
# HALU_XGB_DEVICE=cpu keeps boosters portable after download.
# RESUMABLE: restores from 7.0 or checkpoints immediately after success.
import datetime, json, os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
if VA_OK:
    print('Version A cell 7 already complete - skipping training.')
else:
    rc = os.system('python src/models/train_pipeline.py')
    if rc != 0:
        raise RuntimeError(f'Version A cell 7 failed with exit code {rc}')
    os.makedirs(VA_DIR, exist_ok=True)
    for rel, local in VA_FILES:
        dest = os.path.join(VA_DIR, rel)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy(local, dest)
    marker = {
        'schema': 'version-a-checkpoint-v1',
        'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'inputs': {
            'features_full.parquet': sha256_file('data/processed/features_full.parquet'),
            'qa_clean.parquet': sha256_file('data/processed/qa_clean.parquet'),
            'split_integrity_report.json': sha256_file('artifacts/split_integrity_report.json'),
        },
        'artifacts': [rel for rel, _ in VA_FILES],
    }
    with open(VA_MARKER, 'w', encoding='utf-8') as f:
        json.dump(marker, f, indent=2)
    print('Checkpointed Version A cell-7 artifacts to Drive (halurisc_cache/version_a).')


In [ ]:
# 7b.0) Restore B2 artifacts (LOCAL first, then Drive; 7b shortcut)
import os, json, shutil, hashlib
from colab.drive_cache import b2_restore_valid
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
B2_OK = False
B2_FORCE_RESTORE = False  # set True to copy the Drive cache even if the hash differs (prints a warning)

def b2_why_not(cfg_path, parquet_path):
    if not os.path.exists(cfg_path):
        return 'config missing: ' + cfg_path
    if not os.path.isdir(os.path.join(DRIVE_CACHE, 'b2_models')):
        return 'models dir missing: ' + os.path.join(DRIVE_CACHE, 'b2_models')
    if not os.path.exists(parquet_path):
        return 'local features missing: ' + parquet_path
    rec = json.load(open(cfg_path)).get('inputs', {}).get('features_full.parquet', '')
    cur = hashlib.sha256(open(parquet_path, 'rb').read()).hexdigest()
    return 'hash mismatch: config %s.. vs local %s..' % (rec[:16], cur[:16])

if (os.path.exists('artifacts/results/b2/b2_run_config.json')
        and os.path.isdir('artifacts/models/b2')
        and b2_restore_valid('artifacts/results/b2/b2_run_config.json', 'data/processed/features_full.parquet')):
    B2_OK = True
    print('B2 artifacts already present locally (hash verified) - skipping training.')
elif B2_FORCE_RESTORE:
    shutil.copytree(os.path.join(DRIVE_CACHE, 'b2'), 'artifacts/results/b2', dirs_exist_ok=True)
    shutil.copytree(os.path.join(DRIVE_CACHE, 'b2_models'), 'artifacts/models/b2', dirs_exist_ok=True)
    B2_OK = True
    print('WARNING: B2 artifacts FORCE-restored from Drive WITHOUT hash verification.')
else:
    b2_dir = os.path.join(DRIVE_CACHE, 'b2')
    cfg = os.path.join(b2_dir, 'b2_run_config.json')
    if (os.path.exists(cfg) and os.path.isdir(os.path.join(DRIVE_CACHE, 'b2_models'))
            and b2_restore_valid(cfg, 'data/processed/features_full.parquet')):
        shutil.copytree(b2_dir, 'artifacts/results/b2', dirs_exist_ok=True)
        shutil.copytree(os.path.join(DRIVE_CACHE, 'b2_models'), 'artifacts/models/b2', dirs_exist_ok=True)
        B2_OK = True
        print('B2 artifacts RESTORED from Drive (feature hash verified).')
    else:
        print('No valid B2 cache found:', b2_why_not(cfg, 'data/processed/features_full.parquet'))
        print('Cell 7b will train (8-15 min).')


In [ ]:
# 7b) B2: corrected baselines + artifact controls (grouped CV, TF-IDF shortcut checks)
# Runs: majority, overlap heuristic, TF-IDF (all/answer/context), NLI-only, LR, RF, tuned XGBoost
# seeds 42/123/456. Grouped 5-fold CV keyed by item_idx; thresholds: 0.5 / overlap tuned on val.
# ~5-10 min on L4. All outputs under artifacts/{results,models}/b2 (Version A untouched).
# RESUMABLE: skipped automatically when cell 7b.0 restored valid artifacts (B2_OK).
import os
os.environ['HALU_XGB_DEVICE'] = 'cpu'  # portable boosters, see cell 7 note
if B2_OK:
    print('B2 already complete (restored from Drive) - skipping training.')
else:
    rc = os.system('python src/models/run_b2_baselines.py')
    if rc != 0:
        raise RuntimeError(f'B2 run failed with exit code {rc}')


In [ ]:
# 7b.5) Upload B2 artifacts to Drive (crash-safe checkpoint)
import os, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(os.path.join(DRIVE_CACHE, 'b2'), exist_ok=True)
shutil.copytree('artifacts/results/b2', os.path.join(DRIVE_CACHE, 'b2'), dirs_exist_ok=True)
shutil.copytree('artifacts/models/b2', os.path.join(DRIVE_CACHE, 'b2_models'), dirs_exist_ok=True)
print('Checkpointed B2 artifacts to Drive (halurisc_cache/b2).')


In [ ]:
# 7c) Show B2 headline results (comparison table + leakage-removal impact)
import json, pandas as pd
comp = pd.read_csv('artifacts/results/b2/b2_model_comparison.csv', index_col=0)
cols = [c for c in ['precision_mean','recall_mean','f1_mean','auroc_mean','pr_auc_mean','mcc_mean','ece_mean','threshold'] if c in comp.columns]
print('B2 MODEL COMPARISON (test set, seeds 42/123/456)')
print(comp[cols].round(4).to_string())
print('\nLEAKAGE-REMOVAL IMPACT:')
print(json.dumps(json.load(open('artifacts/results/b2/b2_leakage_comparison.json')), indent=2))


In [ ]:
# 7d) B1: build the unified external dataset (official RAGTruth + FaithBench)
# Required by B3. Downloads RAGTruth response/source files (~35 MB) and FaithBench
# annotation batches (~3 MB). Raw files stay in the Colab VM (never committed;
# FaithBench is CC BY-NC-SA). Deterministic: rerunning gives byte-identical output.
!python src/data/download_ragtruth.py
!python src/data/download_faithbench.py
!python src/data/prepare_unified.py


In [ ]:
# 7d.5) B3 external-feature cache (LOCAL first, then Drive; 7e shortcut)
import os, json, shutil, hashlib
from colab.drive_cache import b3_feature_cache_safe
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')

def unified_sha():
    return hashlib.sha256(open('data/processed/unified_records.parquet', 'rb').read()).hexdigest()

B3_CACHE_OK = False
local_meta = 'data/processed/b3_external_features.meta.json'
local_cache = 'data/processed/b3_external_features.parquet'
if (os.path.exists(local_meta) and os.path.exists(local_cache)
        and json.load(open(local_meta)).get('input_sha256') == unified_sha()
        and json.load(open(local_meta)).get('complete', True)
        and b3_feature_cache_safe(local_cache)):
    B3_CACHE_OK = True
    print('B3 external feature cache already present locally (complete, hash verified).')
else:
    cache_meta = os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json')
    cache_features = os.path.join(DRIVE_CACHE, 'b3_external_features.parquet')
    if (os.path.exists(cache_meta) and os.path.exists(cache_features)
            and b3_feature_cache_safe(cache_features)):
        meta = json.load(open(cache_meta))
        if meta.get('input_sha256') == unified_sha():
            shutil.copy(cache_features, local_cache)
            shutil.copy(cache_meta, local_meta)
            B3_CACHE_OK = bool(meta.get('complete', True)) and b3_feature_cache_safe(cache_features)
            if B3_CACHE_OK:
                print('B3 external feature cache restored from Drive (complete, hash verified).')
            else:
                print('Partial B3 feature cache restored from Drive - cell 7e RESUMES extraction.')
        else:
            print('B3 cache stale (unified parquet changed) - cell 7e will re-extract.')
    else:
        print('No B3 external feature cache found - cell 7e will extract.')


In [ ]:
# 7d.6) B3 RESULTS (LOCAL first, then Drive; 7e shortcut)
import os, json, shutil
from colab.drive_cache import b3_restore_valid, b3_results_safe, b3_predictions_safe
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
B3_OK = False
if (os.path.exists('artifacts/results/b3/b3_run_config.json')
        and os.path.exists('artifacts/results/b3/b3_predictions.parquet')
        and b3_predictions_safe('artifacts/results/b3/b3_predictions.parquet')
        and b3_restore_valid('artifacts/results/b3/b3_run_config.json',
                             'data/processed/unified_records.parquet', 'artifacts/models/b2')):
    B3_OK = True
    print('B3 results already present locally (unified + B2 model hashes verified).')
else:
    b3_dir = os.path.join(DRIVE_CACHE, 'b3')
    figures_dir = os.path.join(DRIVE_CACHE, 'b3_figures')
    cfg = os.path.join(b3_dir, 'b3_run_config.json')
    if (os.path.exists(cfg) and os.path.exists(os.path.join(b3_dir, 'b3_predictions.parquet'))
            and os.path.isdir(figures_dir) and b3_results_safe(b3_dir)
            and b3_predictions_safe(os.path.join(b3_dir, 'b3_predictions.parquet'))
            and b3_restore_valid(cfg, 'data/processed/unified_records.parquet', 'artifacts/models/b2')):
        shutil.copytree(b3_dir, 'artifacts/results/b3', dirs_exist_ok=True)
        shutil.copytree(figures_dir, 'artifacts/figures/b3', dirs_exist_ok=True)
        B3_OK = True
        print('B3 results RESTORED from Drive (unified + B2 model hashes verified).')
    else:
        print('No valid B3 results cache found - cell 7e will run.')


In [ ]:
# 7e) B3: cross-domain zero-shot evaluation (HEAVY: ~18.5K external rows)
# Extracts the 26 features on RAGTruth + FaithBench, then evaluates the B2
# XGBoost models zero-shot (fixed threshold 0.5, source-group bootstrap CIs,
# subgroup + transfer-failure analysis, FaithBench label sensitivity).
# RESUMABLE: skipped when cell 7d.6 restored results (B3_OK); the runner
# reuses a complete local feature cache automatically (self-healing) or
# resumes a partial one. Feature cache uploads to Drive FIRST (most valuable).
import os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
if B3_OK:
    print('B3 already complete (results present) - skipping run.')
else:
    flag = '--skip-features' if B3_CACHE_OK else ''
    cmd = f'python src/models/run_b3_cross_domain.py --device cuda --batch-size {BATCH_SIZE} {flag}'.strip()
    print('Running:', cmd)
    rc = os.system(cmd)
    if rc != 0:
        cache_src = 'data/processed/b3_external_features.parquet'
        if os.path.exists(cache_src):
            shutil.copy(cache_src, os.path.join(DRIVE_CACHE, 'b3_external_features.parquet'))
            shutil.copy('data/processed/b3_external_features.meta.json', os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json'))
            print('B3 failed after feature cache creation; feature cache checkpointed.')
        raise RuntimeError(f'B3 run failed with exit code {rc}')
    # feature cache FIRST (most valuable), then results, then figures
    if os.path.exists('data/processed/b3_external_features.parquet'):
        shutil.copy('data/processed/b3_external_features.parquet', os.path.join(DRIVE_CACHE, 'b3_external_features.parquet'))
        shutil.copy('data/processed/b3_external_features.meta.json', os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json'))
        print('Checkpointed B3 external feature cache to Drive.')
    shutil.copytree('artifacts/results/b3', os.path.join(DRIVE_CACHE, 'b3'), dirs_exist_ok=True)
    print('Checkpointed B3 results to Drive (halurisc_cache/b3).')
    shutil.copytree('artifacts/figures/b3', os.path.join(DRIVE_CACHE, 'b3_figures'), dirs_exist_ok=True)
    print('Checkpointed B3 figures to Drive (halurisc_cache/b3_figures).')


In [ ]:
# 7f) Show B3 headline results (dataset metrics + transfer comparison)
import json, pandas as pd
m = json.load(open('artifacts/results/b3/b3_dataset_metrics.json'))
keep = ('n_rows','n_groups','f1_mean','auroc_mean','ece_mean','predicted_positive_rate','label_positive_rate')
rows = {k: {kk: (round(vv,4) if isinstance(vv,float) else vv) for kk,vv in v.items() if kk in keep} for k,v in m.items() if v}
print('B3 DATASET METRICS (zero-shot, B2 XGBoost, seeds 42/123/456)')
print(pd.DataFrame(rows).T.to_string())
print('\nTRANSFER COMPARISON:')
print(pd.read_csv('artifacts/results/b3/b3_transfer_comparison.csv').to_string(index=False))
print('\nFAITHBENCH LABEL SENSITIVITY:')
print(json.dumps(json.load(open('artifacts/results/b3/b3_label_sensitivity.json')), indent=2))


In [ ]:
# 7g.0) B4 artifacts (LOCAL first, then Drive; 7g shortcut)
import os, json, shutil
from colab.drive_cache import b4_restore_valid, b4_predictions_safe
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
B4_OK = False
if (os.path.exists('artifacts/results/b4/b4_run_config.json')
        and os.path.exists('artifacts/results/b4/b4_predictions.parquet')
        and b4_predictions_safe('artifacts/results/b4/b4_predictions.parquet')
        and b4_restore_valid('artifacts/results/b4/b4_run_config.json',
                             'artifacts/results/b3/b3_predictions.parquet')):
    B4_OK = True
    print('B4 artifacts already present locally (b3 predictions hash verified).')
else:
    b4_dir = os.path.join(DRIVE_CACHE, 'b4')
    cfg = os.path.join(b4_dir, 'b4_run_config.json')
    models_dir = os.path.join(DRIVE_CACHE, 'b4_models')
    figures_dir = os.path.join(DRIVE_CACHE, 'b4_figures')
    if (os.path.exists(cfg) and os.path.exists(os.path.join(b4_dir, 'b4_predictions.parquet'))
            and os.path.isdir(models_dir) and os.path.isdir(figures_dir)
            and b4_predictions_safe(os.path.join(b4_dir, 'b4_predictions.parquet'))
            and b4_restore_valid(cfg, 'artifacts/results/b3/b3_predictions.parquet')):
        shutil.copytree(b4_dir, 'artifacts/results/b4', dirs_exist_ok=True)
        shutil.copytree(models_dir, 'artifacts/models/b4', dirs_exist_ok=True)
        shutil.copytree(figures_dir, 'artifacts/figures/b4', dirs_exist_ok=True)
        B4_OK = True
        print('B4 artifacts RESTORED from Drive (b3 predictions hash verified).')
    else:
        print('No valid B4 cache found - cell 7g will run.')


In [ ]:
# 7g) B4: calibration under distribution shift
# Reuses B3 predictions, cached external features, and B2 models (NO new
# feature extraction, NO retraining). Fits calibrators ONLY on HaluEval
# validation (source) and RAGTruth QA train (target, disjoint source
# groups), then evaluates on HaluEval test / RAGTruth QA test / FaithBench.
# Fast on CPU (2-5 min). All B4 calibrators are pure sklearn -> portable.
# CRASH-SAFE: heavy stages checkpoint to artifacts/results/b4/_stages/ and
# the runner resumes automatically after a kernel kill (OOM/quota) - just
# re-run this cell, it continues instead of restarting. On a Python crash a
# b4_crash.log is written with the full traceback.
import os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
if B4_OK:
    print('B4 already complete (artifacts restored from Drive) - skipping run.')
else:
    rc = os.system('python src/models/run_b4_calibration_shift.py')
    if rc != 0:
        crash = 'artifacts/results/b4/b4_crash.log'
        if os.path.exists(crash):
            shutil.copy(crash, os.path.join(DRIVE_CACHE, 'b4_crash.log'))
            print('B4 crash log checkpointed to Drive.')
        raise RuntimeError(f'B4 run failed with exit code {rc}')
    shutil.copytree('artifacts/results/b4', os.path.join(DRIVE_CACHE, 'b4'), dirs_exist_ok=True)
    shutil.copytree('artifacts/models/b4', os.path.join(DRIVE_CACHE, 'b4_models'), dirs_exist_ok=True)
    shutil.copytree('artifacts/figures/b4', os.path.join(DRIVE_CACHE, 'b4_figures'), dirs_exist_ok=True)
    print('Checkpointed B4 artifacts to Drive (halurisc_cache/b4).')


In [ ]:
# 7h) Show B4 headline results (calibration metrics + target calibration)
import json, pandas as pd
m = json.load(open('artifacts/results/b4/b4_calibration_metrics.json'))
rows = []
for subset, methods in m.items():
    for method, mm in methods.items():
        rows.append({'subset': subset, 'method': method,
                     'ece': mm.get('ece_mean'), 'ace': mm.get('ace_mean'),
                     'brier': mm.get('brier_mean'), 'nll': mm.get('nll_mean'),
                     'slope': mm.get('slope_mean'), 'f1': mm.get('f1_mean')})
df = pd.DataFrame(rows).dropna(subset=['ece']).round(4)
print('B4 CALIBRATION METRICS (seeds 42/123/456; calibrators on HaluEval val only)')
print(df.to_string(index=False))
print('\nTARGET CALIBRATION (RAGTruth QA train -> test):')
t = json.load(open('artifacts/results/b4/b4_target_calibration.json'))
print('n_cal=%d n_test=%d overlap_groups_removed=%d' % (t['n_calibration_rows'], t['n_test_rows'], t['overlapping_groups_removed']))
for k, v in t['methods'].items():
    if 'ece_mean' in v:
        print('  %s: ece=%.4f brier=%.4f' % (k, v['ece_mean'], v['brier_mean']))


In [ ]:
# 7i) Verify every artifact loads on THIS machine BEFORE packaging
# Loads the B2 boosters, B4 calibrators, and all prediction parquet files;
# proves they predict correctly (prevents the old post-download loading error).
# Exit code 0 = safe to package and use locally after download.
import sys
sys.path.insert(0, os.getcwd())
from src.models.verify_artifacts import main
status = main()
if status != 0:
    raise RuntimeError(f'artifact verification failed with exit code {status}')


In [ ]:
# 7j.0) B5 artifacts (LOCAL first, then Drive; 7j shortcut)
import os, json, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
B5_OK = False
if (os.path.exists('artifacts/results/b5/b5_run_config.json')
        and os.path.exists('artifacts/results/b5/b5_feature_importance.json')):
    B5_OK = True
    print('B5 artifacts already present locally - skipping run.')
else:
    b5_dir = os.path.join(DRIVE_CACHE, 'b5')
    cfg = os.path.join(b5_dir, 'b5_run_config.json')
    if os.path.exists(cfg) and os.path.exists(os.path.join(b5_dir, 'b5_feature_importance.json')):
        shutil.copytree(b5_dir, 'artifacts/results/b5', dirs_exist_ok=True)
        shutil.copytree(os.path.join(DRIVE_CACHE, 'b5_figures'), 'artifacts/figures/b5', dirs_exist_ok=True)
        B5_OK = True
        print('B5 artifacts RESTORED from Drive.')
    else:
        print('No valid B5 cache found - cell 7j will run.')


In [ ]:
# 7j) B5: explanation reliability + error analysis
# SHAP vs permutation vs group ablation, top-k neutralization, text
# perturbations with FULL feature re-extraction, bootstrap CIs, and a
# 40-case reviewer export (10 FP / 10 FN / 20 borderline). CPU-ok;
# pass --device cuda for the perturbation re-extraction speedup.
# CRASH-SAFE: per-sample checkpoint (b5_perturbations.csv), rerun resumes.
import os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
if B5_OK:
    print('B5 already complete - skipping run.')
else:
    rc = os.system('python src/models/run_b5_explanation_reliability.py --device cuda')
    if rc != 0:
        raise RuntimeError(f'B5 run failed with exit code {rc}')
    shutil.copytree('artifacts/results/b5', os.path.join(DRIVE_CACHE, 'b5'), dirs_exist_ok=True)
    shutil.copytree('artifacts/figures/b5', os.path.join(DRIVE_CACHE, 'b5_figures'), dirs_exist_ok=True)
    print('Checkpointed B5 artifacts to Drive (halurisc_cache/b5).')


### CRASH RECOVERY (resume from Drive)

If the runtime dies mid-run, restart it and run cells **1-5**, **5b**, **6/6b**, then **7.0** and **7** if you want Version A. Completed phases restore automatically:

- **7.0** restores Version A cell-7 artifacts; cell **7** skips training.
- **7b.0** restores B2; cell **7b** skips training.
- **7d.5** restores B3 features; **7d.6** restores B3 results; cell **7e** skips extraction/evaluation.
- **7g.0** restores B4; cell **7g** skips calibration.

Continue from the first cell whose checkpoint is missing. Every completed heavy phase is written to `Drive/halurisc_cache/` immediately.


In [ ]:
# 8.0) Restore legacy Version A analyses (cells 8-12) from Drive
# Cell 7 cached the core Version A artifacts; this restores the auxiliary
# analyses (SHAP, RAGTruth, error, latency, optional LLM judge).
import os, json, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
LEGACY_DIR = os.path.join(DRIVE_CACHE, 'version_a', 'legacy')
LEGACY_FILES = [
    ('models/shap_explainer.joblib', 'artifacts/models/shap_explainer.joblib'),
    ('results/shap_summary.json', 'artifacts/results/shap_summary.json'),
    ('results/ragtruth_results.json', 'artifacts/results/ragtruth_results.json'),
    ('results/error_analysis.json', 'artifacts/results/error_analysis.json'),
    ('results/error_analysis_cases.json', 'artifacts/results/error_analysis_cases.json'),
    ('results/latency_analysis.json', 'artifacts/results/latency_analysis.json'),
    ('results/llm_judge_results.json', 'artifacts/results/llm_judge_results.json'),
]
LEGACY_OK = os.path.exists(os.path.join(LEGACY_DIR, 'marker.json'))
if LEGACY_OK:
    restored = 0
    for rel, local in LEGACY_FILES:
        src = os.path.join(LEGACY_DIR, rel)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(local), exist_ok=True)
            shutil.copy(src, local)
            restored += 1
    fig_dir = os.path.join(LEGACY_DIR, 'figures')
    if os.path.isdir(fig_dir):
        os.makedirs('artifacts/figures', exist_ok=True)
        for fn in os.listdir(fig_dir):
            shutil.copy(os.path.join(fig_dir, fn), os.path.join('artifacts/figures', fn))
    print(f'Legacy Version A analyses restored from Drive ({restored} files).')
else:
    print('No legacy cache yet - cells 8-12 will run once.')


In [ ]:
# 8) SHAP explanations + calibration/ROC/PR figures (skips when already present)
import os
if os.path.exists('artifacts/results/shap_summary.json'):
    print('SHAP analysis already present - skipping.')
else:
    rc = os.system('python src/explain/shap_analysis.py')
    if rc != 0:
        raise RuntimeError(f'SHAP analysis failed with exit code {rc}')


In [ ]:
# 9) RAGTruth zero-shot external validation (skips when already present)
# eval_ragtruth.py self-heals its input from the B1 unified dataset
# (cell 7d) - no separate download step needed.
import os
if os.path.exists('artifacts/results/ragtruth_results.json'):
    print('RAGTruth validation already present - skipping.')
else:
    rc = os.system('python src/models/eval_ragtruth.py')
    if rc != 0:
        raise RuntimeError(f'RAGTruth validation failed with exit code {rc}')


In [ ]:
# 10) Error analysis (10 FP + 10 FN, auto-tagged; skips when already present)
# NOTE: categories are heuristic and must be manually reviewed in
# artifacts/results/error_analysis_cases.json before the paper uses them.
import os
if os.path.exists('artifacts/results/error_analysis_cases.json'):
    print('Error analysis already present - skipping.')
else:
    rc = os.system('python src/models/error_analysis.py')
    if rc != 0:
        raise RuntimeError(f'Error analysis failed with exit code {rc}')


In [ ]:
# 11) Latency / efficiency analysis (skips when already present)
import os
if os.path.exists('artifacts/results/latency_analysis.json'):
    print('Latency analysis already present - skipping.')
else:
    rc = os.system('python src/models/eval_efficiency.py')
    if rc != 0:
        raise RuntimeError(f'Latency analysis failed with exit code {rc}')


In [ ]:
# 12) OPTIONAL: LLM-as-judge comparison (skips when already present or no key)
import os
if os.path.exists('artifacts/results/llm_judge_results.json'):
    print('LLM judge results already present - skipping.')
elif os.environ.get('OPENAI_API_KEY'):
    os.system('python src/models/eval_llm_judge.py')
else:
    print('OPENAI_API_KEY not set - skipping optional LLM-as-judge. You can run it later locally.')


In [ ]:
# 12.5) Checkpoint legacy Version A analyses (cells 8-12) to Drive
import os, json, shutil, datetime
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
LEGACY_DIR = os.path.join(DRIVE_CACHE, 'version_a', 'legacy')
LEGACY_FILES = [
    ('models/shap_explainer.joblib', 'artifacts/models/shap_explainer.joblib'),
    ('results/shap_summary.json', 'artifacts/results/shap_summary.json'),
    ('results/ragtruth_results.json', 'artifacts/results/ragtruth_results.json'),
    ('results/error_analysis.json', 'artifacts/results/error_analysis.json'),
    ('results/error_analysis_cases.json', 'artifacts/results/error_analysis_cases.json'),
    ('results/latency_analysis.json', 'artifacts/results/latency_analysis.json'),
    ('results/llm_judge_results.json', 'artifacts/results/llm_judge_results.json'),
]
os.makedirs(LEGACY_DIR, exist_ok=True)
saved = []
for rel, local in LEGACY_FILES:
    if os.path.exists(local):
        dest = os.path.join(LEGACY_DIR, rel)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy(local, dest)
        saved.append(rel)
fig_dir = os.path.join(LEGACY_DIR, 'figures')
os.makedirs(fig_dir, exist_ok=True)
if os.path.isdir('artifacts/figures'):
    for fn in sorted(os.listdir('artifacts/figures')):
        if fn.startswith('fig_') and os.path.isfile(os.path.join('artifacts/figures', fn)):
            shutil.copy(os.path.join('artifacts/figures', fn), os.path.join(fig_dir, fn))
with open(os.path.join(LEGACY_DIR, 'marker.json'), 'w', encoding='utf-8') as f:
    json.dump({'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
               'files': saved}, f, indent=2)
print(f'Checkpointed legacy Version A analyses to Drive ({len(saved)} files + figures).')


In [ ]:
# 13) Generate the artifact manifest (hashes, versions, hardware, split report)
# source_fingerprint = sha256 over the cell-3 embedded HASHES; fingerprints
# the exact shipped source even on Colab (no git repo there).
import os, hashlib, json
os.environ['HALU_SOURCE_FINGERPRINT'] = hashlib.sha256(
    json.dumps(HASHES, sort_keys=True).encode()).hexdigest()
!python src/models/make_manifest.py
import json
m = json.load(open('artifacts/results/manifest.json'))
print(json.dumps({k: m[k] for k in ['generated_at', 'git_commit', 'source_fingerprint', 'model_version', 'nli_model', 'split_report']}, indent=2))


In [ ]:
# 14) Show the headline results (model comparison, calibration, RAGTruth, per-seed rows)
import json, os, pandas as pd
res = json.load(open('artifacts/results/final_results.json'))
rows = {k: v for k, v in res.items() if isinstance(v, dict) and 'f1' in v and 'auroc' in v}
df = pd.DataFrame(rows).T[['precision','recall','f1','auroc','pr_auc','mcc']].round(4)
print('MODEL COMPARISON (test set, mean over seeds 42/123/456)')
print(df.to_string())
print('\nCALIBRATION:')
print(json.dumps(res['calibration'], indent=2))
print('\nABLATION:')
print(pd.DataFrame(res['ablation']).to_string(index=False))
print('\nPER-SEED XGBOOST:')
print(pd.DataFrame(json.load(open('artifacts/results/seed_metrics.json'))['xgboost']).round(4).to_string(index=False))
if os.path.exists('artifacts/results/ragtruth_results.json'):
    rt = json.load(open('artifacts/results/ragtruth_results.json'))
    print('\nRAGTRUTH ZERO-SHOT: f1=%.4f auroc=%.4f ece=%.4f (n=%d)' % (rt['f1'], rt['auroc'], rt['ece'], rt['n_samples']))
else:
    print('\nRAGTRUTH ZERO-SHOT: skipped (cell 9 not run) - not required for the B-run.')


In [ ]:
# 15) Package artifacts to Drive (persists across sessions) and offer a download link
import os, zipfile
from datetime import date
from google.colab import files

stamp = date.today().isoformat()
zip_path = f'{DRIVE_DIR}/halurisc_artifacts_{stamp}.zip'
EXCLUDE_DIRS = {'_stages', '__pycache__'}  # internal checkpoints never ship
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, fnames in os.walk('artifacts'):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
        for fn in fnames:
            p = os.path.join(root, fn)
            z.write(p, os.path.relpath(p, '.'))
    for fn in ['data/processed/features_full.parquet', 'data/processed/qa_clean.parquet',
               'data/processed/nli_model_used.json', 'data/processed/audit_50_samples.json']:
        if os.path.exists(fn):
            z.write(fn)
print('Saved:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
print()
print('NEXT: download the zip from your Drive, unzip at the repo root of your laptop.')
print('The API (uvicorn) and web dashboard will then load the corrected artifacts.')
files.download(zip_path)
